# 1) Install dependencies

In [0]:
%pip install seaborn -q

# 2) Importing packages

In [0]:
# 2) Importing the needed packages
from pyspark.sql.functions import (
    col, count as _count, countDistinct, min as spark_min, max as spark_max,
    when, isnan, isnull, desc, lit, length, trim, lower, upper, current_date,
    sum as _sum, year, month, date_format, explode, split, regexp_extract,
    mean, stddev, abs as spark_abs, skewness, kurtosis, size, array,
    percentile_approx, struct, coalesce
)
from pyspark.sql.types import StringType, DoubleType, LongType, StructType
import re
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 3) Reading raw data

In [0]:
# 3.1) Reading raw data from Unity Catalog
TABLE_NAME = "yelp_poc_uc.bronze_final.yelp_business"

In [0]:

# 3.2) Verify Delta format
table_info = spark.sql(f"DESCRIBE DETAIL {TABLE_NAME}").collect()[0]
assert table_info["format"].lower() == "delta", f"Expected Delta but found '{table_info['format']}'"
print(f"✅ Confirmed: {TABLE_NAME} is a Delta table")


In [0]:
# 3.3) Assign the table to a variable
df = spark.table(TABLE_NAME)

print(f"✅ Data loaded successfully from: {TABLE_NAME}")

# 4) General Exploration

In [0]:
# 4.1) Head - first 5 rows
print("\n📋 First 5 rows:")
display(df.limit(5))

In [0]:
# 4.2) Tail - last 6 rows (by business_id desc)
print("\n📋 Last 6 rows:")
display(df.orderBy(col("business_id").desc()).limit(6))


In [0]:
# 4.3) Dataset Info
total_rows = df.count()
print(f"\n📊 Dataset Info:")
print(f"   Total rows: {total_rows:,}")
print(f"   Total columns: {len(df.columns)}")
print(f"   Top-level columns: {df.columns}")

In [0]:
# 4.4) Schema info
print("\n📋 Schema Info:")
df.printSchema()

In [0]:
# 4.5) Recursive nested-data detection (schema-native + embedded inside strings)
#
# Bronze types can't be trusted: nested data may live in native STRUCT/ARRAY/MAP columns
# OR be serialized inside STRING leaves (JSON, Python-dict repr, quoted / double-encoded).
# Method:
#   1. Walk the schema recursively through structs -> every leaf becomes a (path, value-as-string) pair.
#      Native ARRAY/MAP leaves are serialized with to_json so they go through the same parser.
#   2. On the FULL table, pre-filter values that start like a container ({, [, '{, u'[ ...).
#   3. Decode candidates (json -> ast.literal_eval) and recursively walk the decoded object,
#      re-decoding any string that itself looks nested, collecting every key path at any depth.
#   4. Remaining flat strings are tested as DELIMITED LISTS (e.g. "Pizza, Italian, Restaurants"):
#      a delimiter appears in many rows AND the split items are heavily reused across rows
#      (few distinct items vs distinct values) -> list-like, not free text.
# Reusable: profile_nested(<any DataFrame>) -> (summary_df, detail_df)
import ast
import json
import re
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window

MAX_PARSE_DEPTH = 10
NESTED_START_RE = r"""^\s*[uUbB]?['"]?\s*[\[{]"""   # {..}  [..]  '{..}'  u'[..]'  "{..}"
_nested_start = re.compile(NESTED_START_RE)

# Delimited-list detection thresholds
LIST_DELIMITERS = [",", "|", ";"]
LIST_MIN_ROW_PCT = 30.0      # >= this % of non-null values contain the delimiter
LIST_MAX_ITEM_RATIO = 0.5    # distinct items / distinct delimited values (low = items reused = list)
LIST_MIN_AVG_ITEMS = 2.0     # avg items per delimited value
TOP_ITEMS = 5


# ---------- 1. Schema walk ----------
def walk_schema(dt, path="", expr=None, depth=0, leaves=None, containers=None):
    """Recurse through structs; return leaves as string expressions + struct containers."""
    leaves = [] if leaves is None else leaves
    containers = [] if containers is None else containers
    if isinstance(dt, T.StructType):
        if path:
            containers.append((path, "struct", depth, len(dt.fields)))
        for f in dt.fields:
            child_path = f"{path}.{f.name}" if path else f.name
            child_expr = F.col(f"`{f.name}`") if expr is None else expr.getField(f.name)
            walk_schema(f.dataType, child_path, child_expr, depth + 1 if path else 0, leaves, containers)
    elif isinstance(dt, (T.ArrayType, T.MapType)):
        leaves.append({"path": path, "schema_type": dt.simpleString(), "schema_depth": depth,
                       "native": True, "value": F.to_json(expr)})
    else:
        leaves.append({"path": path, "schema_type": dt.simpleString(), "schema_depth": depth,
                       "native": False, "value": expr.cast("string")})
    return leaves, containers


# ---------- 2. Recursive value parser (runs in a Python UDF) ----------
def _type_name(v):
    if v is None:
        return "null"
    if isinstance(v, bool):
        return "boolean"
    if isinstance(v, int):
        return "int"
    if isinstance(v, float):
        return "float"
    if isinstance(v, str):
        return "string"
    return type(v).__name__


def _decode(s):
    t = s.strip()
    for loader in (json.loads, ast.literal_eval):
        try:
            return True, loader(t)
        except Exception:
            pass
    return False, None


def _walk(obj, path, depth, out):
    """Collect key_path -> {value types}; return the max container depth reached."""
    key = path or "$"
    if depth > MAX_PARSE_DEPTH:
        out.setdefault(key, set()).add("depth_limit_exceeded")
        return depth
    if isinstance(obj, str) and _nested_start.match(obj):
        ok, inner = _decode(obj)
        if ok and not (isinstance(inner, str) and inner == obj):
            return _walk(inner, path, depth, out)          # embedded / double-encoded -> same path
        out.setdefault(key, set()).add("malformed_nested_string")
        return depth
    if isinstance(obj, dict):
        if not obj:
            out.setdefault(key, set()).add("empty_object")
            return depth + 1
        return max(_walk(v, f"{path}.{k}" if path else str(k), depth + 1, out) for k, v in obj.items())
    if isinstance(obj, (list, tuple, set)):
        p = f"{path}[]"
        if not obj:
            out.setdefault(p, set()).add("empty_array")
            return depth + 1
        return max(_walk(v, p, depth + 1, out) for v in obj)
    out.setdefault(key, set()).add(_type_name(obj))
    return depth


def _profile_value(s):
    if s is None or not _nested_start.match(s):
        return ("FLAT", 0, [])
    out = {}
    d = _walk(s, "", 0, out)
    if d == 0:
        malformed = any("malformed" in t for ts in out.values() for t in ts)
        return ("MALFORMED" if malformed else "FLAT", 0, [])
    return ("NESTED", d, sorted((k, sorted(v)) for k, v in out.items()))


_PROFILE_SCHEMA = T.StructType([
    T.StructField("status", T.StringType()),
    T.StructField("max_depth", T.IntegerType()),
    T.StructField("leaves", T.ArrayType(T.StructType([
        T.StructField("key_path", T.StringType()),
        T.StructField("value_types", T.ArrayType(T.StringType())),
    ]))),
])
profile_udf = F.udf(_profile_value, _PROFILE_SCHEMA)


# ---------- 3. Full-table profiling ----------
def profile_nested(sdf):
    leaves, containers = walk_schema(sdf.schema)

    # one scan: unpivot every leaf into (path, value)
    kv = (sdf.select(F.explode(F.array(*[
              F.struct(F.lit(l["path"]).alias("path"), l["value"].alias("value")) for l in leaves
          ])).alias("kv"))
          .select("kv.path", "kv.value"))
    is_candidate = F.col("value").rlike(NESTED_START_RE)

    base = kv.groupBy("path").agg(F.count("value").alias("non_null_rows"))

    parsed = (kv.filter(is_candidate)
                .withColumn("p", profile_udf("value"))
                .select("path", "p.status", "p.max_depth", "p.leaves"))

    per_path = parsed.groupBy("path").agg(
        F.count(F.when(F.col("status") == "NESTED", 1)).alias("nested_rows"),
        F.count(F.when(F.col("status") == "MALFORMED", 1)).alias("malformed_rows"),
        F.max("max_depth").alias("max_embedded_depth"))

    detail = (parsed.filter(F.col("status") == "NESTED")
                .select("path", F.explode("leaves").alias("leaf"))
                .groupBy("path", F.col("leaf.key_path").alias("sub_key_path"))
                .agg(F.count("*").alias("rows_present"),
                     F.array_sort(F.array_distinct(F.flatten(F.collect_list("leaf.value_types")))).alias("value_types"))
                .join(per_path.select("path", "nested_rows"), "path")
                .withColumn("presence_pct", F.round(100 * F.col("rows_present") / F.col("nested_rows"), 2))
                .withColumn("presence", F.when(F.col("rows_present") == F.col("nested_rows"), "ALWAYS")
                                         .otherwise("PARTIAL"))
                .select("path", "sub_key_path", "value_types", "rows_present", "nested_rows",
                        "presence_pct", "presence"))

    key_stats = detail.groupBy("path").agg(
        F.count("*").alias("distinct_sub_keys"),
        F.count(F.when(F.col("presence") == "PARTIAL", 1)).alias("partial_sub_keys"))

    # ----- delimited lists inside flat strings -----
    str_paths = [l["path"] for l in leaves if l["schema_type"] == "string"]
    flat_str = kv.filter(F.col("path").isin(str_paths) & F.col("value").isNotNull() & ~is_candidate)
    list_frames = []
    for d in LIST_DELIMITERS:
        with_d = flat_str.filter(F.col("value").contains(d))
        items = (with_d.select("path", "value", F.explode(F.split("value", re.escape(d))).alias("item"))
                       .withColumn("item", F.trim("item"))
                       .filter(F.col("item") != ""))
        stats = items.groupBy("path").agg(
            F.countDistinct("value").alias("list_distinct_values"),
            F.count("*").alias("list_items_total"),
            F.countDistinct("item").alias("distinct_items"))
        top = (items.groupBy("path", "item").agg(F.count("*").alias("item_rows"))
                    .withColumn("rk", F.row_number().over(
                        Window.partitionBy("path").orderBy(F.desc("item_rows"), "item")))
                    .filter(F.col("rk") <= TOP_ITEMS)
                    .groupBy("path")
                    .agg(F.sort_array(F.collect_list(F.struct("rk", "item"))).getField("item").alias("top_items")))
        list_frames.append(with_d.groupBy("path").agg(F.count("*").alias("list_rows"))
                                 .join(stats, "path").join(top, "path")
                                 .withColumn("list_delimiter", F.lit(d)))
    lists_all = list_frames[0]
    for lf in list_frames[1:]:
        lists_all = lists_all.unionByName(lf)
    lists = (lists_all.withColumn("rk", F.row_number().over(
                 Window.partitionBy("path").orderBy(F.desc("list_rows"))))
                      .filter(F.col("rk") == 1).drop("rk"))

    meta = spark.createDataFrame(
        [(l["path"], l["schema_type"], l["schema_depth"], l["native"]) for l in leaves],
        "path string, schema_type string, schema_depth int, native boolean")

    n = F.col("nested_rows")
    leaf_summary = (meta.join(base, "path", "left")
                        .join(per_path, "path", "left")
                        .join(key_stats, "path", "left")
                        .join(lists, "path", "left")
                        .na.fill(0, ["non_null_rows", "nested_rows", "malformed_rows", "max_embedded_depth",
                                     "distinct_sub_keys", "partial_sub_keys"])
                        .withColumn("nested_pct", F.when(F.col("non_null_rows") > 0,
                                    F.round(100 * n / F.col("non_null_rows"), 2)))
                        .withColumn("list_rows_pct", F.round(100 * F.col("list_rows") / F.col("non_null_rows"), 2))
                        .withColumn("avg_items", F.round(F.col("list_items_total") / F.col("list_rows"), 2))
                        .withColumn("is_list", F.coalesce(
                            (n == 0)
                            & (F.col("list_rows_pct") >= LIST_MIN_ROW_PCT)
                            & (F.col("distinct_items") / F.col("list_distinct_values") <= LIST_MAX_ITEM_RATIO)
                            & (F.col("avg_items") >= LIST_MIN_AVG_ITEMS), F.lit(False)))
                        .withColumn("max_embedded_depth",
                                    F.when(F.col("is_list"), F.lit(1)).otherwise(F.col("max_embedded_depth")))
                        .withColumn("total_depth", F.col("schema_depth") + F.col("max_embedded_depth"))
                        .withColumn("verdict",
                            F.when(F.col("native") & (n > 0), "NATIVE_NESTED")
                             .when((n > 0) & (n == F.col("non_null_rows")), "EMBEDDED_NESTED")
                             .when(n > 0, "MIXED_EMBEDDED")
                             .when(F.col("is_list"), "DELIMITED_LIST")
                             .when(F.col("malformed_rows") > 0, "SUSPECT_UNPARSEABLE")
                             .otherwise("FLAT")))
    # list stats are only meaningful where the list verdict fired
    leaf_summary = leaf_summary.select(
        *[c for c in leaf_summary.columns if c not in
          ("list_delimiter", "list_rows_pct", "avg_items", "distinct_items", "top_items")],
        *[F.when(F.col("is_list"), F.col(c)).alias(c) for c in
          ("list_delimiter", "list_rows_pct", "avg_items", "distinct_items", "top_items")])

    struct_summary = spark.createDataFrame(
        containers, "path string, schema_type string, schema_depth int, distinct_sub_keys int"
    ).withColumn("verdict", F.lit("NATIVE_STRUCT"))

    rank = F.create_map(*[F.lit(x) for kv_ in enumerate(
        ["NATIVE_STRUCT", "NATIVE_NESTED", "EMBEDDED_NESTED", "MIXED_EMBEDDED", "DELIMITED_LIST",
         "SUSPECT_UNPARSEABLE", "FLAT"])
        for x in kv_[::-1]])
    summary = (leaf_summary.unionByName(struct_summary, allowMissingColumns=True)
                 .orderBy(rank[F.col("verdict")], F.col("total_depth").desc_nulls_last(), "path")
                 .select("path", "verdict", "schema_type", "schema_depth", "max_embedded_depth", "total_depth",
                         "non_null_rows", "nested_rows", "nested_pct", "malformed_rows",
                         "distinct_sub_keys", "partial_sub_keys",
                         "list_delimiter", "list_rows_pct", "avg_items", "distinct_items", "top_items"))
    return summary, detail


# ---------- 4. Run on the business table ----------
nested_summary_df, nested_detail_df = profile_nested(df)
nested_summary_pd = nested_summary_df.toPandas()
_int_cols = ["schema_depth", "max_embedded_depth", "total_depth", "non_null_rows", "nested_rows",
             "malformed_rows", "distinct_sub_keys", "partial_sub_keys", "distinct_items"]
nested_summary_pd[_int_cols] = nested_summary_pd[_int_cols].astype("Int64")   # nullable ints, no 44279.0

counts = nested_summary_pd["verdict"].value_counts()
deepest = nested_summary_pd.dropna(subset=["total_depth"]).sort_values("total_depth", ascending=False).iloc[0]
print("Nested-data scan (full table):  " + "  |  ".join(f"{k}: {v}" for k, v in counts.items()))
print(f"Deepest path: {deepest['path']}  (total depth {int(deepest['total_depth'])})")
display(nested_summary_pd)

In [0]:
# 4.5.1) Nested sub-key inventory: every key path found inside nested values, at any depth
#   sub_key_path : dotted path inside the decoded value ('[]' = list element)
#   value_types  : Python/JSON types observed for that key across the full table
#   presence     : ALWAYS = key exists in every nested row of that field, PARTIAL = schema drift
nested_detail_pd = nested_detail_df.orderBy("path", "sub_key_path").toPandas()

drift = (nested_detail_pd[nested_detail_pd["presence"] == "PARTIAL"]
         .groupby("path")["sub_key_path"].count())
print(f"Sub-key paths found: {len(nested_detail_pd)} across {nested_detail_pd['path'].nunique()} nested fields")
print("Fields with inconsistent sub-keys (PARTIAL): " +
      (", ".join(f"{p} ({n})" for p, n in drift.items()) if len(drift) else "none"))
display(nested_detail_pd)

In [0]:
# 4.6) Missing values summary
print(f"\n📊 Missing Values Summary:")
for c in df.columns:
    null_count = df.filter(col(c).isNull()).count()
    if null_count > 0:
        print(f"   {c}: {null_count:,} nulls ({null_count/total_rows*100:.1f}%)")
    else:
        print(f"   {c}: ✅ No nulls")

# 5) Structural & Meta-Data Exploration

In [0]:
# 5.1) Dimensionality/Shape
total_rows = df.count()
total_cols = len(df.columns)
print(f"Shape: {total_rows:,} rows × {total_cols:,} columns")   

In [0]:
# 5.2) Data Types and Data Classification

print("="*80)
print("RAW DATA TYPES")
print("="*80)
df.printSchema()

# Collect dtype info and schema-level nullable metadata
dtypes = dict(df.dtypes)
schema_nullable = {field.name: field.nullable for field in df.schema.fields}
total_rows = df.count()

for column in df.columns:
    dtype = dtypes[column]
    n = total_rows
    unique = df.select(countDistinct(col(column))).collect()[0][0]
    null_count = df.filter(col(column).isNull()).count()
    null_pct = null_count / n * 100 if n > 0 else 0
    unique_pct = unique / n * 100 if n > 0 else 0
    col_lower = column.lower()
    
    # Sample values (up to 3)
    samples = df.select(col(column)).filter(col(column).isNotNull()).distinct().limit(3)
    sample_values = [row[0] for row in samples.collect()]
    
    # ---- CLASSIFICATION ----
    if unique == 1:
        classification = "CONSTANT"
    elif col_lower in ['id', 'host_id']:
        classification = "IDENTIFIER"
    elif dtype in ('timestamp', 'date'):
        classification = "DATETIME"
    elif col_lower in ['latitude', 'longitude']:
        classification = "COORDINATE"
    elif dtype in ('int', 'bigint'):
        min_val = df.select(spark_min(col(column))).collect()[0][0]
        if min_val is not None and min_val >= 0:
            if unique == 2:
                classification = "BINARY_FLAG"
            elif unique <= 3:
                classification = "ORDINAL"
            elif unique <= 10:
                classification = "ORDINAL"
            else:
                classification = "COUNT"
        else:
            classification = "NUMERIC_INTEGER"
    elif dtype in ('double', 'float'):
        classification = "CONTINUOUS"
    elif dtype == 'string':
        if unique <= 5:
            classification = "CATEGORICAL"
        elif unique > total_rows * 0.5:
            classification = "FREE_TEXT"
        else:
            classification = "CATEGORICAL"
    else:
        classification = "UNCLASSIFIED"
    
    print(f"\n{'='*60}")
    print(f"Column: {column}")
    print(f"{'='*60}")
    nullable = schema_nullable.get(column, None)
    print(f"  Classification:      {classification}")
    print(f"  Data Type:           {dtype}")
    print(f"  Schema Nullable:     {nullable}")
    print(f"  Unique Values:       {unique:,} ({unique_pct:.1f}%)")
    print(f"  Missing Values:      {null_count} ({null_pct:.1f}%)")
    if nullable and null_count == 0:
        print(f"  ⚠️  Schema allows NULLs but none found — consider adding a NOT NULL constraint")
    elif not nullable and null_count > 0:
        print(f"  🚨  Schema disallows NULLs but {null_count} found — data integrity issue!")
    print(f"  Sample Values:       {', '.join(str(v) for v in sample_values)}")
    
    # ---- NUMERIC PROFILING ----
    if dtype in ('int', 'bigint', 'double', 'float'):
        min_max = df.select(spark_min(col(column)), spark_max(col(column))).collect()[0]
        min_val = min_max[0]
        max_val = min_max[1]
        
        mean_val = df.select(mean(col(column))).collect()[0][0]
        median_val = df.select(percentile_approx(col(column), 0.5)).collect()[0][0]
        stddev_val = df.select(stddev(col(column))).collect()[0][0]
        skew_val = df.select(skewness(col(column))).collect()[0][0]
        
        zero_count = df.filter(col(column) == 0).count()
        neg_count = df.filter(col(column) < 0).count()
        
        q1 = df.select(percentile_approx(col(column), 0.25)).collect()[0][0]
        q3 = df.select(percentile_approx(col(column), 0.75)).collect()[0][0]
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outlier_count = df.filter((col(column) < lower_bound) | (col(column) > upper_bound)).count()
        
        if skew_val is not None:
            if abs(skew_val) < 0.5:
                skew_text = "Symmetric"
            elif abs(skew_val) < 1:
                skew_text = f"{'Right' if skew_val > 0 else 'Left'}-skewed (Moderate)"
            else:
                skew_text = f"{'Right' if skew_val > 0 else 'Left'}-skewed (High)"
        else:
            skew_text = "N/A"
        
        print(f"  Min:                 {min_val}")
        print(f"  Max:                 {max_val}")
        print(f"  Mean:                {mean_val:.2f}")
        print(f"  Median:              {median_val}")
        print(f"  Std Dev:             {stddev_val:.2f}")
        print(f"  Zero Count:          {zero_count}")
        print(f"  Negative Count:      {neg_count}")
        print(f"  Skewness:            {skew_val:.2f} ({skew_text})")
        print(f"  IQR Outliers:        {outlier_count} ({'Has outliers' if outlier_count > 0 else 'No outliers'})")
        
        if classification == "IDENTIFIER":
            action = "Use as key only - exclude from modeling"
        elif classification == "COORDINATE":
            action = "Use for geospatial analysis"
        elif classification == "ORDINAL":
            action = "Preserve order (ordinal encoding)"
        elif classification == "BINARY_FLAG":
            action = "Use as boolean (0/1)"
        elif zero_count > total_rows * 0.9:
            action = "High zero percentage - consider log transform"
        elif skew_val and abs(skew_val) > 1:
            action = "Scale & check outliers (high skew)"
        else:
            action = "Scale & check outliers"
        print(f"  Action:              {action}")
    
    # ---- STRING PROFILING ----
    elif dtype == 'string':
        empty_count = df.filter((col(column) == "") | (col(column).isNull())).count()
        whitespace_count = df.filter(length(col(column)) != length(trim(col(column)))).count()
        
        total_non_null = n - null_count
        if total_non_null > 0:
            all_upper = df.filter(col(column) == upper(col(column))).count() - null_count
            all_lower = df.filter(col(column) == lower(col(column))).count() - null_count
            upper_pct = all_upper / total_non_null * 100
            lower_pct = all_lower / total_non_null * 100
            
            if upper_pct > 80:
                casing = f"{upper_pct:.0f}% Uppercase"
            elif lower_pct > 80:
                casing = f"{lower_pct:.0f}% Lowercase"
            else:
                casing = "Mixed Casing"
        else:
            casing = "N/A (all null)"
        
        len_stats = df.select(
            spark_min(length(col(column))).alias("min_len"),
            spark_max(length(col(column))).alias("max_len"),
            mean(length(col(column))).alias("avg_len")
        ).collect()[0]
        
        print(f"  Empty Count:         {empty_count}")
        print(f"  Whitespace Issues:   {whitespace_count}")
        print(f"  Casing:              {casing}")
        print(f"  Length:              Min: {len_stats['min_len']}, Max: {len_stats['max_len']}, Avg: {len_stats['avg_len']:.2f}")
        
        if classification == "IDENTIFIER":
            action = "Use as key only - exclude from modeling"
        elif classification == "FREE_TEXT":
            action = "Use for text mining / NLP only"
        elif unique <= 5:
            action = "One-hot or frequency encode"
        elif unique <= 50:
            action = "Label encode or one-hot (careful with cardinality)"
        else:
            action = "Frequency encode or target encode"
        print(f"  Action:              {action}")
    
    # ---- DATETIME PROFILING ----
    elif dtype in ('timestamp', 'date'):
        min_max = df.select(spark_min(col(column)), spark_max(col(column))).collect()[0]
        min_val = min_max[0]
        max_val = min_max[1]
        future_count = df.filter(col(column) > current_date()).count()
        old_count = df.filter(col(column) < "2008-01-01").count()
        
        print(f"  Date Range:          {min_val} to {max_val}")
        print(f"  Future Dates:        {future_count}")
        print(f"  Before 2008:         {old_count}")
        print(f"  Action:              Convert to datetime, extract features")

In [0]:
# 5.3) Column Header Audit
import re

print("="*80)
print("COLUMN HEADER AUDIT — Databricks Naming Convention Check")
print("="*80)

# Databricks / Delta Lake column naming rules:
# - Must start with a letter or underscore
# - Can contain letters, digits, underscores, hyphens, spaces (with column mapping)
# - Should avoid special characters
# - Best practice: lowercase snake_case, no leading/trailing spaces

columns = df.columns
total_cols = len(columns)
issues_found = 0

for c in columns:
    issues = []

    # Check: starts with letter or underscore
    if not re.match(r'^[a-zA-Z_]', c):
        issues.append("❌ Does not start with a letter or underscore")

    # Check: contains only valid characters (letters, digits, underscores)
    if not re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*$', c):
        issues.append("⚠️  Contains characters beyond [a-zA-Z0-9_]")

    # Check: leading or trailing whitespace
    if c != c.strip():
        issues.append("❌ Has leading/trailing whitespace")

    # Check: contains spaces
    if ' ' in c:
        issues.append("⚠️  Contains spaces (use underscores instead)")

    # Check: contains hyphens
    if '-' in c:
        issues.append("⚠️  Contains hyphens (use underscores instead)")

    # Check: uppercase letters (best practice is lowercase snake_case)
    if c != c.lower():
        issues.append("⚠️  Contains uppercase letters (prefer lowercase snake_case)")

    # Check: consecutive underscores
    if '__' in c:
        issues.append("⚠️  Contains consecutive underscores")

    # Check: excessively long name
    if len(c) > 128:
        issues.append("⚠️  Column name exceeds 128 characters")

    # Check: reserved SQL keywords (common ones)
    reserved_keywords = {
        'select', 'from', 'where', 'table', 'column', 'index', 'group',
        'order', 'by', 'as', 'join', 'on', 'insert', 'update', 'delete',
        'create', 'drop', 'alter', 'into', 'values', 'set', 'null',
        'not', 'and', 'or', 'in', 'is', 'between', 'like', 'exists',
        'having', 'limit', 'offset', 'union', 'all', 'distinct', 'case',
        'when', 'then', 'else', 'end', 'cast', 'primary', 'key',
        'foreign', 'references', 'default', 'constraint', 'check',
        'unique', 'database', 'schema', 'view', 'replace', 'if',
        'true', 'false', 'date', 'time', 'timestamp', 'interval',
        'row', 'rows', 'window', 'partition', 'range', 'current',
        'user', 'role', 'grant', 'revoke'
    }
    if c.lower() in reserved_keywords:
        issues.append("⚠️  Column name is a SQL reserved keyword")

    # Report
    if issues:
        issues_found += 1
        print(f"\n  Column: '{c}'")
        for issue in issues:
            print(f"    {issue}")
        # Suggest corrected name
        suggested = re.sub(r'[^a-zA-Z0-9_]', '_', c)  # replace invalid chars
        suggested = re.sub(r'_+', '_', suggested)       # collapse multiple underscores
        suggested = suggested.strip('_').lower()         # lowercase, strip edge underscores
        if suggested != c:
            print(f"    💡 Suggested rename: '{c}' → '{suggested}'")
    else:
        print(f"\n  Column: '{c}' — ✅ Compliant")

print(f"\n{'='*80}")
print(f"SUMMARY: {total_cols} columns checked, {issues_found} with issues, {total_cols - issues_found} compliant")
if issues_found == 0:
    print("✅ All column names follow Databricks naming conventions!")
else:
    print(f"⚠️  {issues_found} column(s) have naming convention issues — review suggestions above.")
print("="*80)

In [0]:
# 5.4) Memory Footprint Estimation
print("="*80)
print("MEMORY FOOTPRINT ESTIMATION")
print("="*80)

total_rows = df.count()

# Estimate bytes per column based on data type
type_sizes = {
    'string': 50,      # average estimate
    'int': 4,
    'bigint': 8,
    'double': 8,
    'float': 4,
    'boolean': 1,
    'timestamp': 8,
    'date': 4,
}

total_estimated_bytes = 0
results = []

for field in df.schema.fields:
    col_name = field.name
    dtype = field.dataType.simpleString()
    
    if dtype == 'string':
        # Use actual average length for strings
        avg_len_row = df.select(mean(length(col(col_name)))).collect()[0][0]
        bytes_per_row = int(avg_len_row) if avg_len_row else 0
    else:
        bytes_per_row = type_sizes.get(dtype, 8)
    
    col_total = bytes_per_row * total_rows
    total_estimated_bytes += col_total
    results.append({
        "Column": col_name,
        "Type": dtype,
        "Est. Bytes/Row": bytes_per_row,
        "Est. Total (MB)": round(col_total / (1024**2), 2)
    })

# Add totals row
results.append({
    "Column": "TOTAL",
    "Type": "",
    "Est. Bytes/Row": "",
    "Est. Total (MB)": round(total_estimated_bytes / (1024**2), 2)
})

memory_df = pd.DataFrame(results)
display(memory_df)

print(f"\n  Total Estimated Size: {total_estimated_bytes / (1024**2):.2f} MB ({total_estimated_bytes / (1024**3):.2f} GB)")

# Warn if large
if total_estimated_bytes > 1 * 1024**3:
    print("  ⚠️  Estimated size > 1 GB — consider partitioning or filtering for local operations.")
else:
    print("  ✅ Dataset fits comfortably in memory.")
print("="*80)

In [0]:
# 5.5) Column Constraints & Key Candidates
print("="*80)
print("COLUMN CONSTRAINTS & KEY CANDIDATES")
print("="*80)

# 1) Table properties (Delta table properties, owner, etc.)
print("\n📋 Table Properties:")
try:
    props_df = spark.sql(f"SHOW TBLPROPERTIES {TABLE_NAME}")
    display(props_df)
except Exception as e:
    print(f"   Could not retrieve table properties: {e}")

# 2) NOT NULL constraints per column
print("\n📋 NOT NULL Constraints per Column:")
print(f"{'Column':<30} {'Data Type':<20} {'Nullable':<10}")
print("-" * 60)
for field in df.schema.fields:
    nullable_str = "YES" if field.nullable else "NO (🔒 enforced)"
    print(f"{field.name:<30} {field.dataType.simpleString():<20} {nullable_str:<10}")

# 3) Inferred uniqueness — identify candidate keys
print("\n📋 Candidate Key Analysis:")
total_rows = df.count()
for c in df.columns:
    distinct_count = df.select(countDistinct(col(c))).collect()[0][0]
    null_count = df.filter(col(c).isNull()).count()
    uniqueness_ratio = distinct_count / total_rows * 100 if total_rows > 0 else 0

    if distinct_count == total_rows and null_count == 0:
        status = "🔑 Candidate Primary Key (unique, no nulls)"
    elif distinct_count == total_rows:
        status = "🔑 Unique but has NULLs"
    elif uniqueness_ratio > 95:
        status = f"🔶 Near-unique ({uniqueness_ratio:.1f}%)"
    elif distinct_count <= 2:
        status = f"🔘 Low cardinality ({distinct_count} values)"
    else:
        status = f"📊 {distinct_count:,} distinct ({uniqueness_ratio:.1f}%)"

    print(f"   {c:<30} {status}")

print("\n" + "="*80)
print("META-DATA EXPLORATION COMPLETE")
print("="*80)

# 6) Data Quality Checks/Assessment


### 6.1) STRUCTURAL / SCHEMA CHECKS
*Verify the dataset's shape, schema, and declared types match expectations — and that declared types actually reflect the semantic content of the data.*

- **Column Presence & Order** — Expected columns exist; no missing or unexpected columns; order matches schema contract(if it exists).
  - *Example:* Expected `[id, fname, lname, hire_date]`, found `[id, name, hire_date]` → missing `fname`, `lname`; unexpected `name`.
  - *Columns present but reordered → flag if downstream consumers depend on positional access.*

- **Data Type Conformance** — Checks the column's declared type matches the type specified in the schema contract(if it exists).
  - *Examples:*
    - Contract says age INT, dataset declares age VARCHAR → schema mismatch.
    - Contract says hire_date DATE, dataset declares hire_date TIMESTAMP → widening mismatch (may be acceptable; configurable).
    - Contract says salary DECIMAL(10,2), dataset declares salary FLOAT → precision mismatch.

- **Semantic Data Type Violations** - The column's actual data values match the type the schema claims they are. The semantic type is inferred from the values themselves — not read from the schema — so the check is non-circular.
    - *Example:* A column can declare INT in the schema but contain "twenty-five", "unknown", "$1,200", or empty strings. Schema-vs-schema checks never catch this. Semantic inference does.

- **Schema Drift Detection** — Compare current schema against baseline/columns in the schema contract(if it exists); flag added/removed/retyped columns.
  - *Example:* `salary` changed from `INT` to `VARCHAR` between loads.

---

### 6.2) DOMAIN & RANGE CHECKS
*Verify values belong to allowed sets and fall within permitted bounds. All scope variants live here.*

- **Allowed Value Conformance (single column)** — Values belong to the permitted domain/category list.
  - *Example:* `sex` allows only `M`, `F`, `Other`; found `Male`, `1`, `Unknown`.
  - *Sub:* Geographic codes — `country_code` must be ISO-3166 alpha-2.
  - *Sub:* Department names — `department` must match organizational list.
  - *Sub:* Status codes — `employee_status` must be in `{Active, Terminated, Leave}`.

- **Range Conformance (single column)** — Values within min/max bounds.
  - *Example:* `age` must be 0–120; found `-3` or `250`.
  - *Sub:* Salary bands — `salary` within role-specific band.
  - *Sub:* Date bounds — `hire_date` after 1980-01-01.

- **Pattern Conformance** — Values match required format (email, phone, postal code, ID patterns).
  - *Example:* `email` must match RFC-5322; `phone` must match E.164.

- **Impossible Value Detection** — Values that cannot exist regardless of bounds.
  - *Example:* `birth_date` in the future; `age` negative; `death_date` before `birth_date` (single-column view).

---

### 6.3) FORMAT & ENCODING STANDARDIZATION
*Detect and normalize inconsistent representations. Detection only — normalization is a separate remediation step.*

- **Categorical Representation Consistency** — Same concept, different encodings.
  - *Example:* `Male`, `male`, `M`, `1` all representing the same category.
  - *Sub:* Category mapping consistency across related columns — `country_code = 'US'` but `country_name = 'Canada'`.

- **Date Format Consistency** — Dates stored in non-uniform formats.
  - *Example:* `2024-01-15`, `01/15/2024`, `15-Jan-2024` co-existing in one column.

- **Numeric Format Consistency** — Numbers stored with embedded symbols, separators, or units.
  - *Example:* `"$1,200.50"`, `"1.200,50 EUR"`, `"5 kg"` in a numeric column.

- **Boolean Representation Consistency** — Mixed boolean encodings co-existing.
  - *Example:* `Yes/No`, `True/False`, `1/0`, `Y/N` all present in one column.

- **Text Normalization Issues**
  - *Sub:* Leading/trailing whitespace — `" John "` vs `"John"`.
  - *Sub:* Multiple consecutive whitespaces — `"John    Doe"`.
  - *Sub:* Inconsistent capitalization — `"New York"`, `"new york"`, `"NEW YORK"`.
  - *Sub:* Escape/control characters — embedded `\n`, `\t`, `\r`, `\0`, non-printable characters that corrupt parsing.

- **Null Representation Consistency** — Non-standard nulls masquerading as data.
  - *Example:* `"N/A"`, `"NULL"`, `"None"`, `""`, `" "`, `"—"`, `"unknown"`, `"TBD"`, `"."`, `"NIL"`.
  - *Note:* This check only **detects and normalizes**. Counting missingness happens in 6.4.

---

### 6.4) COMPLETENESS CHECKS
*Measure missingness after null normalization. Understand patterns and impact.*

- **Missingness Measurement**
  - *Sub:* Per column — null percentage per column.
  - *Sub:* Per row — rows with excessive missing values (e.g., >50% null).
  - *Sub:* Per cell — total null rate after normalizing variants from 6.3.

- **Missingness Pattern Analysis**
  - *Sub:* Randomness classification — MCAR / MAR / MNAR.
  - *Sub:* Missing value correlation — missingness in `income` correlated with missingness in `employment_status`.
  - *Sub:* Temporal gaps — missing time periods in time-series data (e.g., no records for Q3 2023).

- **Critical Field Coverage** — Required fields (per business rules) must be non-null.
  - *Example:* `customer_id` must never be null.

---

### 6.5) UNIQUENESS CHECKS
*Ensure no duplicate data and each record is uniquely identifiable.*

- **Row-Level Uniqueness** — Entire rows duplicated.
  - *Example:* Two identical rows for `customer_id = 101` with all columns matching.

- **Key-Level Uniqueness** — Primary/unique key columns contain no duplicates.
  - *Example:* `order_id` appears twice.

- **Composite Key Uniqueness** — Multi-column keys are unique together.
  - *Example:* `(store_id, product_id, date)` should be unique; found duplicate.

- **Distribution Uniqueness** — Distinct value counts in categorical columns (too few/too many).
  - *Example:* `state` has 200 distinct values in a US-only dataset (expected ≤ 50) → likely encoding inconsistency (cross-reference 6.3).
  - *Example:* `country` has 1 distinct value in a global dataset → suspiciously low.

---

### 6.6) CROSS-COLUMN CONSISTENCY CHECKS
*Validate logical relationships and contradictions across columns within a row.*

- **Logical Relationship Validation** — Derived columns match their source columns.
  - *Example:* `full_name = fname + ' ' + lname` must hold.
  - *Example:* `total = subtotal + tax`.

- **Logical Contradiction Detection** — Values that conflict logically.
  - *Example:* `age < 18` AND `job_title = 'CEO'`.
  - *Example:* `marital_status = 'Single'` AND `spouse_name` is populated.

- **Cross-Column Range Logic** — Ranges that don't make sense together.
  - *Example:* `birth_date > hire_date`.
  - *Example:* `end_date < start_date`.
  - *Example:* `death_date < birth_date`.

- **Dependency Rule Validation** — Conditional required/forbidden fields.
  - *Example:* `employee_status = 'Terminated'` but `end_date` is null.
  - *Example:* `payment_method = 'Credit Card'` but `card_number` is null.

- **Category Mapping Consistency** — Codes and descriptions align across related columns.
  - *Example:* `country_code = 'US'` but `country_name = 'Canada'`.
  - *Example:* `dept_code = 'ENG'` but `dept_name = 'Sales'`.

---

### 6.7) REFERENTIAL INTEGRITY CHECKS
*Ensure relationships between tables are valid. One check, three views.*

- **Foreign Key Validity** — Every FK value exists in the parent table's PK.
  - *View 1 — Orphan records:* Child row's FK has no matching parent.
    - *Example:* `orders.customer_id = 999` but no `customers.id = 999`.
  - *View 2 — Missing parent records:* Parent row absent for an existing child reference.
    - *Example:* All orders for `customer_id = 500` exist, but customer 500 was deleted.
  - *View 3 — Invalid FK format:* FK value itself is malformed.
    - *Example:* `customer_id = "ABC"` in a numeric FK column.

- **Lookup Table Conformance** — Referenced values exist in lookup/reference tables.
  - *Example:* `orders.status_code = 'X9'` not present in `status_lookup`.

- **Cross-Table Consistency** — Aggregates reconcile across tables.
  - *Example:* `SUM(orders.amount)` per customer ≠ `customers.total_spend`.

---

### 6.8) BUSINESS RULE & ACCURACY CHECKS
*Domain-specific rules and statistical anomalies. All business rules live here — nowhere else.*

- **Business Rule Violations** — Values that break defined business constraints.
  - *Example:* `salary > 1,000,000` for a non-executive role.
  - *Example:* `discount_percent > 100`.
  - *Example:* `order_quantity > max_order_limit`.

- **Statistical Outlier Detection** — Values deviating significantly from mean/median.
  - *Example:* `age = 200` in a customer dataset (Z-score > 5).
  - *Example:* `transaction_amount` in the 99.99th percentile.

- **Distribution Anomalies** — Imbalances, sudden spikes/drops.
  - *Example:* Daily order count drops 90% on a single day with no holiday.
  - *Example:* `gender` distribution 99% `M` in a general population dataset.

---

### 6.9) TIMELINESS CHECKS
*Monitor data freshness, latency, and temporal logic. Only time-related checks live here.*

- **Data Freshness** — Age of data and last update timestamp.
  - *Example:* `last_updated` is 45 days old for a daily-refresh table.

- **Load Timestamp Analysis** — Consistency of ETL/load timestamps.
  - *Example:* `load_timestamp` < `source_timestamp` (loaded before generated).
  - *Example:* Load timestamps out of order across partitions.

- **Source-to-Target Latency** — Delay between source generation and target availability.
  - *Example:* Source event at 10:00, loaded at 14:00 → 4-hour latency exceeds 1-hour SLA.

- **Temporal Consistency** — Timestamps are logical.
  - *Example:* Future-dated `created_at`.
  - *Example:* `updated_at` before `created_at`.
  - *Example:* Data older than retention policy.

- **Temporal Gaps** — Missing time periods in time-series data.
  - *Note:* If treated as a **completeness** issue, keep in 6.4 only. Recommended home: **6.4** (missingness) with a cross-reference here for monitoring freshness cadence. Choose one and stick to it.
  - Currently I placed this check at **6.4** (missingness), so no need to implement it again here.


### 6.1) STRUCTURAL / SCHEMA CHECKS
*Verify the dataset's shape, schema, and declared types match expectations — and that declared types actually reflect the semantic content of the data.*

- **Column Presence & Order** — Expected columns exist; no missing or unexpected columns; order matches schema contract(if it exists).
  - *Example:* Expected `[id, fname, lname, hire_date]`, found `[id, name, hire_date]` → missing `fname`, `lname`; unexpected `name`.
  - *Columns present but reordered → flag if downstream consumers depend on positional access.*

- **Data Type Conformance** — Checks the column's declared type matches the type specified in the schema contract(if it exists).
  - *Examples:*
    - Contract says age INT, dataset declares age VARCHAR → schema mismatch.
    - Contract says hire_date DATE, dataset declares hire_date TIMESTAMP → widening mismatch (may be acceptable; configurable).
    - Contract says salary DECIMAL(10,2), dataset declares salary FLOAT → precision mismatch.

- **Semantic Data Type Violations** - The column's actual data values match the type the schema claims they are. The semantic type is inferred from the values themselves — not read from the schema — so the check is non-circular.
    - *Example:* A column can declare INT in the schema but contain "twenty-five", "unknown", "$1,200", or empty strings. Schema-vs-schema checks never catch this. Semantic inference does.

- **Schema Drift Detection** — Compare current schema against baseline/columns in the schema contract(if it exists); flag added/removed/retyped columns.
  - *Example:* `salary` changed from `INT` to `VARCHAR` between loads.


In [0]:
# 6.1.1) Column Presence & Order
# ─────────────────────────────────────────────────────────────────────
# DATA CONTRACT = 01_Business_Silver_Cleaned DDL (yelp_poc_uc.silver_cleaned, 11 tables).
# Every silver column in the DDL declares the bronze path it is derived from.
# This cell (1) materialises that DDL as Python metadata (SILVER_CONTRACT) that is
# reused by 6.1.2 – 6.1.4 and TE.1 – TE.2, and (2) verifies that every bronze path
# the contract consumes exists in bronze_final.yelp_business (top level AND nested),
# that no bronze field is silently left unmapped, and that top-level order is stable.
import pandas as pd
from pyspark.sql.functions import col
from pyspark.sql.types import StructType

SILVER_SCHEMA = "yelp_poc_uc.silver_cleaned"
df_bronze = spark.table(TABLE_NAME)          # always the untouched bronze source

# ══ DDL §2 — attributes STRUCT → yelp_business_attributes (24 BOOLEAN + 1 INT + 7 ENUM) ══
ATTR_BOOLEAN_MAP = {
    "AcceptsInsurance": "attr_accepts_insurance",
    "BikeParking": "attr_bike_parking",
    "BusinessAcceptsBitcoin": "attr_business_accepts_bitcoin",
    "BusinessAcceptsCreditCards": "attr_business_accepts_credit_cards",
    "ByAppointmentOnly": "attr_by_appointment_only",
    "BYOB": "attr_byob",
    "Caters": "attr_caters",
    "CoatCheck": "attr_coat_check",
    "Corkage": "attr_corkage",
    "DogsAllowed": "attr_dogs_allowed",
    "DriveThru": "attr_drive_thru",
    "GoodForDancing": "attr_good_for_dancing",
    "GoodForKids": "attr_good_for_kids",
    "HappyHour": "attr_happy_hour",
    "HasTV": "attr_has_tv",
    "Open24Hours": "attr_open_24_hours",
    "OutdoorSeating": "attr_outdoor_seating",
    "RestaurantsCounterService": "attr_restaurants_counter_service",
    "RestaurantsDelivery": "attr_restaurants_delivery",
    "RestaurantsGoodForGroups": "attr_restaurants_good_for_groups",
    "RestaurantsReservations": "attr_restaurants_reservations",
    "RestaurantsTableService": "attr_restaurants_table_service",
    "RestaurantsTakeOut": "attr_restaurants_take_out",
    "WheelchairAccessible": "attr_wheelchair_accessible",
}
ATTR_INT_MAP = {"RestaurantsPriceRange2": "attr_restaurants_price_range_2"}
ATTR_ENUM_MAP = {
    "AgesAllowed": "attr_ages_allowed",
    "Alcohol": "attr_alcohol",
    "BYOBCorkage": "attr_byob_corkage",
    "NoiseLevel": "attr_noise_level",
    "RestaurantsAttire": "attr_restaurants_attire",
    "Smoking": "attr_smoking",
    "WiFi": "attr_wifi",
}
# Allowed enum domains documented in the DDL column comments
ENUM_DOMAINS = {
    "attr_ages_allowed":       {"allages", "21plus", "18plus"},
    "attr_alcohol":            {"none", "full_bar", "beer_and_wine"},
    "attr_byob_corkage":       {"no", "yes_free", "yes_corkage"},
    "attr_noise_level":        {"quiet", "average", "loud", "very_loud"},
    "attr_restaurants_attire": {"casual", "dressy", "formal"},
    "attr_smoking":            {"no", "outdoor", "yes"},
    "attr_wifi":               {"free", "no", "paid"},
}

# ══ DDL §3 — hours STRUCT → yelp_business_hours ══
HOURS_MAP = {d: f"hours_{d.lower()}" for d in
             ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]}

# ══ DDL §5–11 — dict-strings nested inside attributes → parsed BOOLEAN tables ══
#   bronze attribute : (silver table, {bronze dict key: silver column})  — DDL column order
DICT_TABLE_MAP = {
    "Ambience":            ("yelp_business_ambience",            {k: k for k in ["romantic", "intimate", "touristy", "hipster", "divey", "classy", "trendy", "upscale", "casual"]}),
    "BestNights":          ("yelp_business_best_nights",         {k: k for k in ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]}),
    "BusinessParking":     ("yelp_business_parking",             {k: k for k in ["garage", "street", "validated", "lot", "valet"]}),
    "DietaryRestrictions": ("yelp_business_dietary_restrictions",{"dairy-free": "dairy_free", "gluten-free": "gluten_free", "vegan": "vegan", "kosher": "kosher",
                                                                  "halal": "halal", "soy-free": "soy_free", "vegetarian": "vegetarian"}),
    "GoodForMeal":         ("yelp_business_good_for_meal",       {k: k for k in ["breakfast", "brunch", "lunch", "dinner", "latenight", "dessert"]}),
    "HairSpecializesIn":   ("yelp_business_hair_specializes_in", {k: k for k in ["straightperms", "coloring", "extensions", "africanamerican", "curly", "kids", "perms", "asian"]}),
    "Music":               ("yelp_business_music",               {k: k for k in ["dj", "background_music", "no_music", "jukebox", "live", "video", "karaoke"]}),
}

# ══ SILVER_CONTRACT: table → ordered list of column specs (exactly as in the DDL) ══
def _c(column, dtype, source, transform="direct", nullable=True):
    return {"column": column, "type": dtype, "nullable": nullable, "source": source, "transform": transform}

_BID = _c("business_id", "string", "business_id", nullable=False)
SILVER_CONTRACT = {
    "yelp_business": [
        _BID,
        _c("name", "string", "name"), _c("address", "string", "address"), _c("city", "string", "city"),
        _c("state", "string", "state"), _c("postal_code", "string", "postal_code"),
        _c("latitude", "double", "latitude"), _c("longitude", "double", "longitude"),
        _c("stars", "decimal(2,1)", "stars", "cast_decimal"),
        _c("review_count", "bigint", "review_count"),
        _c("is_open", "boolean", "is_open", "cast_boolean_01"),
    ],
    "yelp_business_attributes": [_BID]
        + [_c(s, "boolean", f"attributes.{b}", "parse_bool_string") for b, s in ATTR_BOOLEAN_MAP.items()]
        + [_c(s, "int",     f"attributes.{b}", "parse_int_string")  for b, s in ATTR_INT_MAP.items()]
        + [_c(s, "string",  f"attributes.{b}", "clean_enum")        for b, s in ATTR_ENUM_MAP.items()],
    "yelp_business_hours": [_BID] + [_c(s, "string", f"hours.{d}", "time_range_string") for d, s in HOURS_MAP.items()],
    "yelp_business_categories": [
        _BID,
        _c("category_name", "string", "categories", "explode_csv", nullable=False),
        _c("category_position", "int", "categories", "explode_position"),
    ],
}
for attr, (tbl, keymap) in DICT_TABLE_MAP.items():
    SILVER_CONTRACT[tbl] = [_BID] + [_c(s, "boolean", f"attributes.{attr}['{k}']", "parse_dict_bool") for k, s in keymap.items()]

SILVER_PK = {t: ["business_id"] for t in SILVER_CONTRACT}
SILVER_PK["yelp_business_categories"] = ["business_id", "category_name"]
SILVER_CARDINALITY = {t: "0:1" for t in SILVER_CONTRACT}
SILVER_CARDINALITY.update({"yelp_business": "parent", "yelp_business_attributes": "1:1",
                           "yelp_business_hours": "1:1", "yelp_business_categories": "0:N"})

def bronze_leaves(schema, prefix=""):
    """Flatten a (possibly nested) schema into [(dotted_path, DataType)] leaves."""
    out = []
    for f in schema.fields:
        p = f"{prefix}{f.name}"
        out += bronze_leaves(f.dataType, p + ".") if isinstance(f.dataType, StructType) else [(p, f.dataType)]
    return out

def _base_path(src):            # attributes.Ambience['romantic'] → attributes.Ambience
    return src.split("[")[0]

print("=" * 80)
print("6.1.1) COLUMN PRESENCE & ORDER  —  bronze_final.yelp_business vs Silver DDL contract")
print("=" * 80)
print("\n📜 Contract loaded from 01_Business_Silver_Cleaned DDL:")
for t, cols in SILVER_CONTRACT.items():
    print(f"   {t:<38} {len(cols):>3} columns  | PK {SILVER_PK[t]}  | cardinality {SILVER_CARDINALITY[t]}")

findings = []

# ── 1) Top-level presence & order ──
expected_top = ["business_id", "name", "address", "city", "state", "postal_code", "latitude", "longitude",
                "stars", "review_count", "is_open", "attributes", "categories", "hours"]
actual_top = df_bronze.columns
missing_top    = [c for c in expected_top if c not in actual_top]
unexpected_top = [c for c in actual_top if c not in expected_top]
order_ok       = actual_top == expected_top
print(f"\n📌 Top level — expected {len(expected_top)}, actual {len(actual_top)}")
print(f"   {'✅' if not missing_top else '⚠️ '} Missing:    {missing_top or 'none'}")
print(f"   {'✅' if not unexpected_top else '⚠️ '} Unexpected: {unexpected_top or 'none'}")
print(f"   {'✅' if order_ok else '⚠️ '} Order {'matches' if order_ok else 'differs from'} the expected bronze layout")
if not order_ok:
    for i in range(max(len(expected_top), len(actual_top))):
        e = expected_top[i] if i < len(expected_top) else "<N/A>"
        a = actual_top[i] if i < len(actual_top) else "<N/A>"
        if e != a:
            print(f"      pos {i}: expected '{e}', got '{a}'")

findings += [
    {"Scope": "top-level", "Check": "Missing columns",    "Detail": str(missing_top) if missing_top else "None",       "Status": "PASS" if not missing_top else "FINDING"},
    {"Scope": "top-level", "Check": "Unexpected columns", "Detail": str(unexpected_top) if unexpected_top else "None", "Status": "PASS" if not unexpected_top else "FINDING"},
    {"Scope": "top-level", "Check": "Column order",       "Detail": "Matches" if order_ok else "Differs",               "Status": "PASS" if order_ok else "FINDING"},
]

# ── 2) Nested presence: every bronze path consumed by the contract must exist ──
leaf_paths = {p for p, _ in bronze_leaves(df_bronze.schema)}
required_paths = sorted({_base_path(c["source"]) for cols in SILVER_CONTRACT.values() for c in cols})
missing_paths = [p for p in required_paths if p not in leaf_paths]
print(f"\n📌 Nested — {len(required_paths)} bronze leaf paths required by the contract")
print(f"   {'✅' if not missing_paths else '⚠️ '} Missing source paths: {missing_paths or 'none'}")
findings.append({"Scope": "nested", "Check": "Contract source paths present",
                 "Detail": str(missing_paths) if missing_paths else f"All {len(required_paths)} present",
                 "Status": "PASS" if not missing_paths else "FINDING"})

# ── 3) Unmapped bronze fields: data that would be silently dropped by the silver model ──
unmapped = sorted(p for p in leaf_paths if p not in required_paths)
print(f"   {'✅' if not unmapped else '⚠️ '} Bronze leaves NOT consumed by any silver column: {unmapped or 'none'}")
findings.append({"Scope": "nested", "Check": "Unmapped bronze leaves (silent data loss)",
                 "Detail": str(unmapped) if unmapped else "None — every bronze leaf lands in silver",
                 "Status": "PASS" if not unmapped else "FINDING"})

# ── 4) Per-struct coverage (attributes 32 typed + 7 dicts = 39; hours 7) ──
for struct_name in ["attributes", "hours"]:
    actual_n = len(df_bronze.schema[struct_name].dataType.fields)
    expected_n = len([p for p in required_paths if p.startswith(struct_name + ".")])
    ok = actual_n == expected_n
    print(f"   {'✅' if ok else '⚠️ '} {struct_name}: {actual_n} bronze sub-fields vs {expected_n} mapped in contract")
    findings.append({"Scope": struct_name, "Check": "Sub-field coverage",
                     "Detail": f"{actual_n} bronze / {expected_n} contract", "Status": "PASS" if ok else "FINDING"})

print(f"\n{'='*80}\nSUMMARY\n{'='*80}")
findings_df = pd.DataFrame(findings)
display(findings_df)
violation_count = sum(f["Status"] == "FINDING" for f in findings)
print("\n✅ Bronze structure fully satisfies the silver contract." if violation_count == 0
      else f"\n⚠️  {violation_count} issue(s) detected — review above.")
print("COLUMN PRESENCE & ORDER CHECK COMPLETE")
print("=" * 80)

In [0]:
# 6.1.2) Data Type Conformance
# ─────────────────────────────────────────────────────────────────────
# Compares the DECLARED bronze type of every source path against the semantic type
# the Silver DDL (R5) requires. Each gap is classified by the remediation it needs,
# so the output doubles as the transformation spec for TE.1 / TE.2.
from pyspark.sql.types import StructType

print("=" * 80)
print("6.1.2) DATA TYPE CONFORMANCE  —  bronze declared type vs Silver DDL type")
print("=" * 80)

bronze_types = {p: dt.simpleString() for p, dt in bronze_leaves(df_bronze.schema)}
WIDENING = {("int", "bigint"), ("float", "double"), ("date", "timestamp"), ("int", "double")}

def classify(bronze_t, target_t, transform):
    if bronze_t is None:                       return "SOURCE MISSING", "FINDING"
    if transform == "explode_csv":             return "RESHAPE (CSV string → 1:N rows)", "FINDING"
    if transform == "explode_position":        return "DERIVED (ordinal from CSV split)", "INFO"
    if transform == "parse_dict_bool":         return "PARSE (dict-string key → BOOLEAN)", "FINDING"
    if bronze_t == target_t:                   return "MATCH", "PASS"
    if (target_t, bronze_t) in WIDENING:       return "NARROWING (bronze wider than contract)", "FINDING"
    if (bronze_t, target_t) in WIDENING:       return "WIDENING (bronze narrower than contract)", "FINDING"
    if bronze_t == "bigint" and target_t == "boolean":            return "CAST (0/1 BIGINT → BOOLEAN)", "FINDING"
    if bronze_t == "double" and target_t.startswith("decimal"):   return "PRECISION (DOUBLE → DECIMAL)", "FINDING"
    if bronze_t == "string" and target_t in ("boolean", "int"):   return f"PARSE (STRING → {target_t.upper()})", "FINDING"
    return "INCOMPATIBLE", "FINDING"

rows = []
for tbl, cols in SILVER_CONTRACT.items():
    for c in cols:
        if tbl != "yelp_business" and c["column"] == "business_id":
            continue                                   # FK copy of parent key — checked once in parent
        b_t = bronze_types.get(_base_path(c["source"]))
        category, status = classify(b_t, c["type"], c["transform"])
        rows.append({"Silver Table": tbl, "Silver Column": c["column"], "Bronze Source": c["source"],
                     "Bronze Type": b_t, "Contract Type": c["type"], "Conformance": category, "Status": status})

conf_df = pd.DataFrame(rows)

# Also: enum columns keep STRING in silver but bronze values carry Python repr quoting (u'x' / 'x')
enum_cols = set(ATTR_ENUM_MAP.values())
conf_df.loc[conf_df["Silver Column"].isin(enum_cols), "Conformance"] = "MATCH (type) — value cleaning needed (u'x' → x)"
conf_df.loc[conf_df["Silver Column"].isin(enum_cols), "Status"] = "FINDING"

print("\n📊 Conformance by category:")
display(conf_df.groupby(["Conformance", "Status"]).size().reset_index(name="Columns").sort_values("Columns", ascending=False))

print("\n📊 Conformance by silver table:")
by_tbl = conf_df.pivot_table(index="Silver Table", columns="Status", values="Silver Column", aggfunc="count", fill_value=0)
display(by_tbl.reset_index())

print("\n📋 Non-conforming columns (remediation required):")
display(conf_df[conf_df["Status"] == "FINDING"].reset_index(drop=True))

findings_df = conf_df
violation_count = int((conf_df["Status"] == "FINDING").sum())
print(f"\n{len(conf_df)} contract columns checked — {int((conf_df['Status']=='PASS').sum())} match as-is, "
      f"{violation_count} need a cast/parse/reshape, {int((conf_df['Status']=='INFO').sum())} derived.")
print("DATA TYPE CONFORMANCE CHECK COMPLETE")
print("=" * 80)

---
#### 🚨 DQ-BUSINESS-001 — Storage Type Non-Conformance vs Silver Contract

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-001` |
| **Severity** | FINDING (schema / data type conformance violation) |
| **Detected In** | Phase 6.1.2 — Data Type Conformance (Cell 27) |
| **Also Confirmed In** | Phase 6.1.3 — Semantic Data Type Violations (Cell 29): the values support every target type (58/58 leaves contract-aligned) |
| **Affected Columns** | 84 of 101 contract columns: `stars` (DOUBLE → DECIMAL(2,1)), `is_open` (BIGINT → BOOLEAN), 24 `attributes.*` STRING → BOOLEAN, `attributes.RestaurantsPriceRange2` STRING → INT, 7 enum attributes (Python-repr quoting), 49 flags packed in 7 dict-strings, `categories` (CSV string → 1:N rows) |
| **Total Affected Rows** | **150,346** out of 150,346 (100%). The issue is in the schema, so every row carries it |
| **Invalid Value** | Wrong storage types, e.g. `stars = 3.5` stored as DOUBLE, `is_open = 1` as BIGINT, `"True"` as STRING, `"{'lot': True, ...}"` as STRING |
| **Domain Rule** | Silver DDL rule R5 (Semantic Data Types): column types must reflect real-world meaning, not raw ingestion storage |

**Description:** Bronze keeps the Yelp JSON almost exactly as received. Ratings are floating-point, the open/closed flag is a 0/1 integer, and nearly all business attributes are free-form strings. Some of those strings are Python-repr booleans (`"True"`), some are quoted enums (`u'free'`) and some are whole Python dictionaries (`"{'romantic': False, ...}"`). Only 16 contract columns (identity, location text, coordinates, `review_count`, hours) can be used as-is. Left this way, analysts would have to compare `attr_wifi = "u'free'"` instead of `'free'`, filter on `"True"` strings instead of booleans, and they would hit floating-point drift on star averages. The 6.1.3 value-based inference confirms the problem is the storage types, not the contract: every target type in the DDL matches the values.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Typed cast + parse in the Silver loading script** (the TE.1/TE.2 logic: `CAST(stars AS DECIMAL(2,1))`, 0/1 → BOOLEAN, `'True'/'False'` → BOOLEAN, unquote enums, decode dict literals into MAP<STRING,BOOLEAN>, explode CSV) | Fixes every column once, at the bronze → silver boundary. Bronze stays a faithful raw copy (R1), and the TE.1 validation already showed 0 lossy casts and 0 out-of-domain enums. |
| 2 | **Fix types in bronze** (`ALTER` / rewrite `bronze_final.yelp_business`) | Downstream code gets simpler, but bronze would no longer be a raw replica of the source. That breaks the medallion contract and makes replays and audits harder. |
| 3 | **Expose typed views on top of bronze** | Nothing extra to store, but the parse and cast run on every query (dict regex parsing is costly), and you can't declare PK/FK constraints on views. |
| 4 | **Leave as STRING and cast at query time** | No engineering effort, but every consumer repeats the parsing logic, results drift between teams, and the DDL's R5 rule is ignored. |

**Best Logical Solution:** **Option 1: typed cast and parse in the Silver loading script.** The DDL already defines the target types, the 6.1.3 inference confirms the data supports them, and TE.1/TE.2 show the transformations work with no data loss. Doing this once at the silver boundary keeps bronze immutable and gives every consumer correctly typed, constraint-documented tables.

---

In [0]:
# 6.1.3) Semantic Data Type Violations
# ─────────────────────────────────────────────────────────────────────
# Non-circular check: infer the semantic type from the VALUES of every bronze leaf
# (top level + every STRUCT sub-field), then compare it with (a) the declared bronze
# storage type and (b) the semantic type the Silver DDL expects. This validates that
# the DDL's R5 type decisions are actually supported by the data.
# Single aggregation pass over the full table for all leaves.
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, FloatType, LongType, IntegerType

print("=" * 80)
print("6.1.3) SEMANTIC DATA TYPE VIOLATIONS  —  value-based inference vs storage & contract")
print("=" * 80)

TRANSFORM_SEMANTIC = {"parse_bool_string": "boolean", "parse_int_string": "int", "clean_enum": "enum",
                      "parse_dict_bool": "dict<string,boolean>", "time_range_string": "time_range",
                      "explode_csv": "csv_list", "cast_decimal": "decimal(2,1)", "cast_boolean_01": "boolean"}
expected_sem = {}
for cols in SILVER_CONTRACT.values():
    for c in cols:
        expected_sem.setdefault(_base_path(c["source"]), TRANSFORM_SEMANTIC.get(c["transform"], c["type"]))

leaves = bronze_leaves(df_bronze.schema)
total_rows = df_bronze.count()

# ── Build one wide aggregation ──
aggs = []
for i, (p, dt) in enumerate(leaves):
    c = F.col(p)
    aggs.append(F.count(c).alias(f"c{i}_nn"))
    if isinstance(dt, StringType):
        v = F.trim(c)
        flag = lambda cond, n: F.sum(cond.cast("int")).alias(f"c{i}_{n}")
        aggs += [
            flag(v == "None", "none"),
            flag(v.isin("True", "False"), "bool"),
            flag(v.rlike(r"^-?\d+$"), "int"),
            flag(v.rlike(r"^0\d"), "lead0"),
            flag(v.rlike(r"^u?'[^']*'$"), "enum"),
            flag(v.rlike(r"^u'"), "uprefix"),
            flag(v.rlike(r"^\{.*\}$"), "dict"),
            flag(v.rlike(r"^\d{1,2}:\d{1,2}-\d{1,2}:\d{1,2}$"), "trange"),
            flag(c.contains(", "), "csv"),
            flag(c != v, "ws"),
            F.avg(F.length(c)).alias(f"c{i}_avglen"),
            F.approx_count_distinct(c).alias(f"c{i}_ndv"),
        ]
    elif isinstance(dt, (DoubleType, FloatType, LongType, IntegerType)):
        aggs += [F.min(c).alias(f"c{i}_min"), F.max(c).alias(f"c{i}_max"),
                 F.approx_count_distinct(c).alias(f"c{i}_ndv"),
                 F.sum(((c * 2) != F.floor(c * 2)).cast("int")).alias(f"c{i}_offhalf")]
S = {k: (v if v is not None else 0) for k, v in df_bronze.agg(*aggs).collect()[0].asDict().items()}

STRING_LIKE = {"string", "identifier", "code (string)", "free_text"}
rows = []
for i, (p, dt) in enumerate(leaves):
    g = lambda n: S.get(f"c{i}_{n}", 0)
    nn, stype, notes, info = g("nn"), dt.simpleString(), [], []
    if isinstance(dt, StringType):
        none_tok = g("none"); eff = nn - none_tok
        r = lambda n: (g(n) / eff) if eff > 0 else 0
        if eff == 0:                 inferred = "all-null"
        elif r("dict") > 0.95:       inferred = "dict<string,boolean>"
        elif r("trange") > 0.95:     inferred = "time_range"
        elif r("bool") > 0.95:       inferred = "boolean"
        elif r("int") > 0.95:        inferred = "code (string)" if g("lead0") > 0 else "int"
        elif r("enum") > 0.95:       inferred = "enum"
        elif g("csv") / max(nn, 1) > 0.30 and g("avglen") <= 300: inferred = "csv_list"
        elif g("avglen") and abs(g("avglen") - 22) < 0.01 and g("ndv") > 0.95 * total_rows: inferred = "identifier"
        else:                        inferred = "string"
        if none_tok:  notes.append(f"{none_tok:,} literal 'None' strings masquerading as NULL")
        if g("ws"):   notes.append(f"{g('ws'):,} values with leading/trailing whitespace")
        if inferred == "enum" and 0 < g("uprefix") < g("enum"):
            notes.append(f"inconsistent Python repr: {g('uprefix'):,} u'x' vs {g('enum') - g('uprefix'):,} 'x'")
        if inferred == "code (string)":
            info.append(f"ℹ️ {g('lead0'):,} values with leading zeros — STRING is correct, numeric cast would corrupt them")
    else:
        mn, mx, ndv = g("min"), g("max"), g("ndv")
        if isinstance(dt, (LongType, IntegerType)) and ndv == 2 and mn == 0 and mx == 1:
            inferred = "boolean"; notes.append("binary 0/1 flag stored as integer")
        elif isinstance(dt, (DoubleType, FloatType)) and g("offhalf") == 0 and 0 <= mn and mx <= 5:
            inferred = "decimal(2,1)"; notes.append(f"bounded [{mn}, {mx}] in exact 0.5 steps — DOUBLE adds float imprecision")
        else:
            inferred = stype
    exp = expected_sem.get(p, "—")
    aligned = (inferred == exp) or (exp == "string" and inferred in STRING_LIKE) or (inferred == "all-null")
    storage_ok = inferred == stype or (stype == "string" and inferred in STRING_LIKE | {"enum"})
    status = "PASS" if (storage_ok and not notes) else "FINDING"
    rows.append({"Bronze Path": p, "Storage Type": stype, "Inferred Semantic": inferred, "Contract Semantic": exp,
                 "Contract Aligned": "Yes" if aligned else "NO", "Non-Null": nn, "Status": status,
                 "Detail": "; ".join(notes + info) if (notes or info) else ("Types aligned" if storage_ok else f"stored as {stype}, values are {inferred}")})

sem_df = pd.DataFrame(rows)
print("\n📊 Inferred semantic types (all bronze leaves):")
display(sem_df)

misaligned = sem_df[sem_df["Contract Aligned"] == "NO"]
print(f"\n{'✅' if misaligned.empty else '🚨'} Contract alignment: "
      f"{len(sem_df) - len(misaligned)}/{len(sem_df)} leaves support the DDL's semantic type")
if not misaligned.empty:
    display(misaligned)

# ── Level-2 drill-down: dict-string key inventory vs DDL columns ──
print(f"\n{'─'*60}\n📌 Dict-string key inventory (attributes.<Dict>) vs DDL boolean columns")
key_rows = []
for attr, (tbl, keymap) in DICT_TABLE_MAP.items():
    src = F.col(f"attributes.{attr}")
    kv = (df_bronze.where(src.rlike(r"^\s*\{"))
          .select(F.explode(F.regexp_extract_all(src, F.lit(r"'([^']+)'\s*:\s*([A-Za-z0-9_']+)"), F.lit(0))).alias("pair"))
          .select(F.regexp_extract("pair", r"'([^']+)'", 1).alias("k"),
                  F.regexp_extract("pair", r":\s*(.+)$", 1).alias("v"))
          .groupBy("k").agg(F.count("*").alias("n"), F.collect_set("v").alias("vals")).collect())
    found = {r["k"]: (r["n"], set(r["vals"])) for r in kv}
    unexpected = sorted(set(found) - set(keymap)); missing = sorted(set(keymap) - set(found))
    bad_vals = sorted({v for _, vs in found.values() for v in vs} - {"True", "False", "None"})
    ok = not (unexpected or missing or bad_vals)
    key_rows.append({"Dict Attribute": attr, "Silver Table": tbl, "DDL Keys": len(keymap), "Keys Found": len(found),
                     "Unexpected Keys": unexpected or "—", "Missing Keys": missing or "—",
                     "Values Outside {True,False,None}": bad_vals or "—",
                     "Inner 'None' values": sum(1 for _, vs in found.values() if "None" in vs),
                     "Status": "PASS" if ok else "FINDING"})
display(pd.DataFrame(key_rows))

findings_df = sem_df
violation_count = int((sem_df["Status"] == "FINDING").sum())
print(f"\n{len(sem_df)} bronze leaves checked, {violation_count} storage/value finding(s), "
      f"{len(misaligned)} contract misalignment(s).")
print("SEMANTIC DATA TYPE VIOLATIONS CHECK COMPLETE")
print("=" * 80)

---
#### 🚨 DQ-BUSINESS-002 — Literal `'None'` Strings Masquerading as NULL

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-002` |
| **Severity** | FINDING (completeness masking: false non-null values) |
| **Detected In** | Phase 6.1.3 — Semantic Data Type Violations (Cell 29) |
| **Also Confirmed In** | Phase TE.1 — Data Contract & Fix Data Types (Cell 34): all 12,827 tokens decoded as Python `None` → NULL (9,625 in single-value attributes + 3,202 whole-dict values), 0 values lost in decode; Phase TE.2 (Cell 35): inner `None` dict values kept as rows with `value = NULL` (e.g. 33,655 in `df_ambience`, 18,141 in `df_good_for_meal`, 9,654 in `df_business_parking`, 3,202 in `df_music`); Phase 6.1.3 dict-key inventory: inner `None` values inside 4 dict attributes |
| **Affected Columns** | **Level 1** (whole value = `'None'`): 37 of 39 `attributes.*` fields (all except `AgesAllowed` and `GoodForDancing`). Largest: `RestaurantsDelivery` (3,948), `BusinessParking` (2,271), `RestaurantsTakeOut` (2,105), `OutdoorSeating` (1,882), `DriveThru` (755), `Ambience` (551). **Level 2** (key value = `None` inside a dict): `Ambience` (9,754 dicts), `GoodForMeal` (7,337), `BusinessParking` (5,153), `Music` (1,632) |
| **Total Affected Rows** | **9,653** out of 150,346 (6.42%) with at least one level-1 `'None'` string, totalling 12,827 tokens (9,625 in typed columns + 3,202 in dict-strings) |
| **Invalid Value** | The 4-character string `'None'` (Python's `None` serialized by `str()`), stored where a SQL `NULL` belongs |
| **Domain Rule** | Missing information must be a SQL `NULL`. A non-null string must hold a real member of the attribute's domain (`True`/`False`, an enum value, or a dict) |

**Description:** The Yelp export was built by calling Python `str()` on attribute values, so Python's `None` became the literal text `'None'` instead of a JSON null. To Spark these values look present: `IS NOT NULL` is true, `COUNT(col)` includes them, and completeness metrics are overstated. A naive `CAST(... AS BOOLEAN)` sends them to NULL with no trace, and a naive enum cleaner could turn them into a fake `'none'` category. That collision matters here because `Alcohol` has a valid lowercase `'none'` value (no alcohol served). The same pattern repeats inside dict-strings, where single keys carry `None` (e.g. `{'lot': None, 'street': True}`), meaning "unknown", not `False`.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Normalize to SQL NULL during silver load** (`CASE WHEN trim(col) = 'None' THEN NULL`, and map inner `None` dict values to NULL booleans) | Gives the data its true meaning (unknown), restores accurate completeness metrics, and keeps "unknown" separate from `False`. This is what TE.1/TE.2 implement, with 0 lossy casts. |
| 2 | **Map `'None'` to `False`** | Simple, and boolean columns end up fully populated, but it invents information (e.g. "no delivery" when the business never answered) and biases every rate and ratio. |
| 3 | **Keep the token and add a `_is_unknown` flag column** | Fully traceable, but it widens tables and every consumer must remember to filter on the flag. That is redundant when NULL already carries the meaning natively. |
| 4 | **Drop the affected rows** | Removes 6.42% of businesses, along with their valid identity, location and rating data, over a minor attribute gap. Disproportionate. |

**Best Logical Solution:** **Option 1: normalize to SQL NULL.** `'None'` is a serialization artefact, not a business value, so NULL is the only faithful representation. Matching case-sensitively on `'None'` (not lowercase `'none'`) keeps the valid `Alcohol = 'none'` category, and treating inner dict `None` as NULL keeps "unknown" apart from an explicit `False`. Downstream SQL (`AVG`, `COUNT(col)`, `LEFT JOIN`) then handles these values correctly by default.

---

---
#### 🚨 DQ-BUSINESS-003 — Inconsistent Python-Repr Quoting in Enum Attributes

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-003` |
| **Severity** | FINDING (format consistency violation) |
| **Detected In** | Phase 6.1.3 — Semantic Data Type Violations (Cell 29) |
| **Also Confirmed In** | Phase 6.1.2 — Data Type Conformance (Cell 27): 7 enum columns flagged "value cleaning needed"; Phase TE.1 — Data Contract & Fix Data Types (Cell 34): 0 out-of-domain and 0 lost values after unquoting |
| **Affected Columns** | `attributes.Alcohol` (33,825 `u'x'` vs 9,326 `'x'`), `WiFi` (42,736 vs 14,128), `NoiseLevel` (31,555 vs 6,399), `RestaurantsAttire` (23,752 vs 15,465), `Smoking` (4,506 vs 47), `BYOBCorkage` (44 vs 1,395), `AgesAllowed` (129 vs 0, consistent but still quoted) |
| **Total Affected Rows** | **27,610** out of 150,346 (18.36%) hold at least one plain-quoted `'x'` enum alongside the dominant `u'x'` form. All 183,307 non-null enum values carry quote wrapping |
| **Invalid Value** | The same category in two spellings, e.g. `u'free'` and `'free'`, or `u'full_bar'` and `'full_bar'` |
| **Domain Rule** | A categorical attribute must use exactly one canonical value per category, from the closed domains documented in the DDL (e.g. WiFi ∈ {free, no, paid}) |

**Description:** Enum attributes were serialized with Python `repr()` in two Python versions: Python 2 unicode literals (`u'free'`) and Python 3 strings (`'free'`). So one category appears as two different strings. A `GROUP BY attr_wifi` gives six buckets instead of three, `WHERE wifi = 'free'` matches nothing, and the counts per category come out split, for example about 21% of WiFi = free businesses (7,385 of 34,414) land in a separate `'free'` bucket. Once the quoting is removed, the 6.1.3 / TE.1 checks show every value falls inside its documented domain, so the content is clean and only the wrapping differs.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Strip the repr wrapper with a regex** (`regexp_replace(col, "^u?'(.*)'$", "$1")`) | One deterministic rule handles both spellings, keeps every value, and lands on the canonical domain. TE.1 confirmed 0 lossy casts and 0 out-of-domain results. |
| 2 | **Explicit mapping table** (`u'free'` → `free`, `'free'` → `free`, ...) | Fully controlled and auditable, but it needs about 40 hand-maintained entries and silently misses any new spelling. |
| 3 | **Normalize only at query/report time** | No pipeline change, but every consumer has to re-implement the fix and results drift between teams. |
| 4 | **Convert enums to integer codes** | Compact storage, but you lose readability, and it contradicts the DDL, which declares these columns as cleaned STRING enums. |

**Best Logical Solution:** **Option 1: regex unquote during silver load, followed by an enum-domain check.** The variation is purely syntactic (Python 2 vs 3 repr), so a single anchored regex resolves it without losing anything. Lowercase `'none'` must stay a valid value (Alcohol = no alcohol); only Python `None` becomes NULL (see DQ-BUSINESS-002). A domain assertion against the DDL value sets then catches any future drift.

---

---
#### 🚨 DQ-BUSINESS-004 — Leading/Trailing Whitespace in Identity Text Fields

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-004` |
| **Severity** | FINDING (format hygiene, low impact) |
| **Detected In** | Phase 6.1.3 — Semantic Data Type Violations (Cell 29) |
| **Also Confirmed In** | **Not remediated in this notebook**: TE.1 (Cell 34) keeps `name`, `address`, `city` as-is so the padding stays visible to later checks. `trim()` is the recommended fix for the Silver load |
| **Affected Columns** | `name` (150 rows), `city` (49 rows), `address` (3 rows) |
| **Total Affected Rows** | **≤ 202** out of 150,346 (≤ 0.13%). Counted per column, so a row with several affected fields counts more than once |
| **Invalid Value** | Trailing spaces, e.g. `'The Mad Griddle '`, `'Temple University Hospital '`, `'Reno '`, `'Audubon '` |
| **Domain Rule** | Business-identity text must be trimmed: the displayed value and the matching key must be identical with no invisible padding |

**Description:** A small number of businesses have trailing spaces in their name, city or address, probably from manual entry on the Yelp business portal. They look the same on screen, but they break exact-match logic: `city = 'Reno'` misses `'Reno '`, `GROUP BY city` splits one city into two buckets, and joins or deduplication on name and address (e.g. entity resolution, geo-matching) silently fail. The `city` case matters most because city is a main grouping dimension in Yelp analytics.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **`trim()` during silver load** | Removes only invisible padding, keeps all real content, can be repeated safely, and costs essentially nothing. Recommended for the Silver loading script (TE.1 deliberately keeps the padding for exploration). |
| 2 | **Full normalization** (trim + collapse internal spaces + case-fold) | Better for matching, but case-folding changes how the business name displays (`McDonald's` → `mcdonald's`). That belongs in a derived match key, not the canonical column. |
| 3 | **Fix only `city`** | Covers the highest-impact grouping column, but leaves the same defect in `name` and `address` for entity-resolution use cases. |
| 4 | **Leave as-is and trim at query time** | No pipeline change, but every consumer has to remember it, and forgotten trims produce silently wrong groupings. |

**Best Logical Solution:** **Option 1: `trim()` on `name`, `address` and `city` during silver load.** The defect affects at most 0.13% of rows, and the fix only removes whitespace that carries no meaning. If case-insensitive matching is needed later, add a separate derived normalized key in Silver Enriched rather than altering the canonical display values.

---

---
### TE) Typing & Restructuring — Plan

*Goal: give every value inside `bronze_final.yelp_business` its real type and split the nested structure into analysable DataFrames, so later checks can detect errors at every nesting level. **Only the structure and the type representation change — no value is moved from one state to another.***

**TE.1 — Data Contract & Fix Data Types (next cell)**

1. **`BUSINESS_CONTRACT`**: a contract built from the full-table value profile. It has one row per output column, with source path, bronze type, decode rule, target type, nullability and allowed domain. A second table, `DICT_KEY_CONTRACT`, lists every key inside the 7 dict strings nested in `attributes`.
2. **`df_typed`**: one row per business, with every value decoded from its bronze encoding:

| Bronze encoding | Example | Target type |
| --- | --- | --- |
| Python bool literal | `'True'` / `'False'` / `'None'` | BOOLEAN TRUE / FALSE / NULL |
| Python int literal | `'2'` | INT |
| Python str literal | `u'free'` / `'free'` | STRING `free` (quote / `u` prefix is encoding, not value) |
| Python dict literal | `{'garage': False, 'lot': True}` | MAP<STRING, BOOLEAN> |
| Time range | `'8:0-17:30'` | STRING kept as-is **+** start `08:00` / end `17:30` (no TIME type on this compute) |
| Comma list | `'Pizza, Italian'` | ARRAY<STRING> |
| DOUBLE star rating / 0-1 flag | `4.5` / `1` | DECIMAL(2,1) / BOOLEAN |

3. **Lossless-decode validation**: every non-null source value must either decode, or be the Python `None` literal. Values that would silently become NULL are counted, as are values outside the contract domain.

**TE.2 — Remodel (cell after TE.1)**: structure `df_typed` into 11 DataFrames, collected in `business_dfs`:

| DataFrame | Shape | Grain / key |
| --- | --- | --- |
| `df_business` | wide | 1 row per business (scalars: `business_id`, `name`, ..., `is_open`) |
| `df_attributes` | wide | 1 row per business, only the 32 single-value attributes (`attributes.AcceptsInsurance`, `attributes.Alcohol`, `attributes.BYOB`, ...) |
| `df_hours` | wide | 1 row per business, 7 days × (raw range, start time, end time) |
| `df_categories` | long | 1 row per category token (`business_id`, `category_position`) |
| 7 nested-attribute DataFrames (`df_ambience`, `df_best_nights`, `df_business_parking`, `df_dietary_restrictions`, `df_good_for_meal`, `df_hair_specializes_in`, `df_music`) | long | 1 row per dict entry: `business_id`, `key`, `value` |

**Deliberately NOT done (this is an exploration notebook)**: no trimming, no deduplication, no dropping or filling of empty `{}` dicts, no case changes. Those stay visible and are reported as DQ findings. An empty `{}` dict or a whole-value `'None'` simply produces no rows in the long DataFrames, because there is no entry to represent. TE.2 counts these explicitly. Everything is in memory: `bronze_final` is **not** modified.

In [0]:
# TE.1) Data Contract & Fix Data Types
# ─────────────────────────────────────────────────────────────────────
# 1) BUSINESS_CONTRACT  : value-driven contract for EVERY bronze leaf, incl. every key inside the
#                         7 Python-dict strings nested in `attributes` (DICT_KEY_CONTRACT).
# 2) df_typed           : 1 row / business, every value DECODED from its bronze encoding:
#      'True'/'False'/'None'  → BOOLEAN TRUE/FALSE/NULL     '1'..'4'          → INT
#      u'x' / 'x'             → STRING x                     "{'k': True, ..}"  → MAP<STRING,BOOLEAN>
#      'H:M-H:M'              → kept + start/end 'HH:mm'     'a, b, c'          → ARRAY<STRING>
#      stars DOUBLE → DECIMAL(2,1)                           is_open 0/1       → BOOLEAN
#    Deliberately NOT done (exploration): no trim, no dedupe, no dropping empty {}, no case changes.
# 3) Lossless-decode validation: a non-null source value must decode OR be the Python None literal.
# In-memory only — bronze_final is NOT modified.
# Reuses ATTR_*_MAP / ENUM_DOMAINS / HOURS_MAP / DICT_TABLE_MAP / bronze_leaves from 6.1.1.
import re
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import types as T

print("=" * 80)
print("TE.1) DATA CONTRACT & FIX DATA TYPES")
print("=" * 80)

DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
_snake = lambda s: re.sub(r"(?<!^)(?=[A-Z])", "_", s).lower()
DICT_COL = {a: f"attr_{_snake(a)}" for a in DICT_TABLE_MAP}            # Ambience → attr_ambience (typed MAP)
DICT_DF  = {a: tbl for a, (tbl, _) in DICT_TABLE_MAP.items()}           # Ambience → yelp_business_ambience
BRONZE_TYPE = {p: dt.simpleString() for p, dt in bronze_leaves(df_bronze.schema)}

HOURS_RE  = r"^(\d{1,2}):(\d{1,2})-(\d{1,2}):(\d{1,2})$"
ENUM_RE   = r"^u?'([^']*)'$"
DICT_RE   = r"^\{.*\}$"
DICT_KEY_RE = r"'[^']+'\s*:"
CAT_DELIM = ", "
PY_BOOL   = ["True", "False", "None"]

# ══ 1) DATASETS — grain / primary key / cardinality vs parent ══
DATASETS = {
    "yelp_business":            {"Shape": "wide", "Grain": "1 row per business",          "PK": ["business_id"], "Cardinality": "parent", "Source": "top-level scalars"},
    "yelp_business_attributes": {"Shape": "wide", "Grain": "1 row per business",          "PK": ["business_id"], "Cardinality": "1:1",    "Source": "attributes.<single-value fields> (32)"},
    "yelp_business_hours":      {"Shape": "wide", "Grain": "1 row per business",          "PK": ["business_id"], "Cardinality": "1:1",    "Source": "hours.<Day> (7)"},
    "yelp_business_categories": {"Shape": "long", "Grain": "1 row per category token",    "PK": ["business_id", "category_position"], "Cardinality": "0:N", "Source": "categories (', ' list)"},
}
for a, tbl in DICT_DF.items():
    DATASETS[tbl] = {"Shape": "long", "Grain": "1 row per dict entry", "PK": ["business_id", "key"],
                     "Cardinality": "0:N", "Source": f"attributes.{a} (Python dict string)"}

# ══ 2) BUSINESS_CONTRACT — one row per output column ══
def _k(ds, column, dtype, source, rule, nullable=True, domain="—", note=""):
    return {"Dataset": ds, "Column": column, "Target Type": dtype, "Nullable": nullable, "Source Path": source,
            "Bronze Type": BRONZE_TYPE.get(source.split("[")[0], "—"), "Decode Rule": rule,
            "Allowed Domain": domain, "Note": note}

def _bid(ds):
    note = "PK" if ds == "yelp_business" else "FK → yelp_business.business_id"
    return _k(ds, "business_id", "string", "business_id", "as-is", False, "22-char Yelp id", note)

C = [_bid("yelp_business"),
     _k("yelp_business", "name",         "string",       "name",         "as-is (not trimmed)"),
     _k("yelp_business", "address",      "string",       "address",      "as-is (not trimmed)"),
     _k("yelp_business", "city",         "string",       "city",         "as-is (not trimmed)"),
     _k("yelp_business", "state",        "string",       "state",        "as-is", domain="state / province code"),
     _k("yelp_business", "postal_code",  "string",       "postal_code",  "as-is", note="STRING keeps leading zeros & alphanumeric (CA) codes"),
     _k("yelp_business", "latitude",     "double",       "latitude",     "as-is", domain="[-90, 90]"),
     _k("yelp_business", "longitude",    "double",       "longitude",    "as-is", domain="[-180, 180]"),
     _k("yelp_business", "stars",        "decimal(2,1)", "stars",        "DOUBLE → DECIMAL(2,1) (exact)", domain="1.0–5.0, step 0.5"),
     _k("yelp_business", "review_count", "bigint",       "review_count", "as-is", domain=">= 0"),
     _k("yelp_business", "is_open",      "boolean",      "is_open",      "0 → FALSE, 1 → TRUE", domain="{0, 1}")]
A = "yelp_business_attributes"
C += [_bid(A)]
C += [_k(A, s, "boolean", f"attributes.{b}", "Python bool literal: 'True'→TRUE, 'False'→FALSE, 'None'→NULL", domain="{'True', 'False', 'None'}")
      for b, s in ATTR_BOOLEAN_MAP.items()]
C += [_k(A, s, "int", f"attributes.{b}", "Python int literal: '1'..'4'→INT, 'None'→NULL", domain="{1, 2, 3, 4}")
      for b, s in ATTR_INT_MAP.items()]
C += [_k(A, s, "string", f"attributes.{b}", "Python str literal: u'x' / 'x' → x, 'None'→NULL",
         domain="{" + ", ".join(sorted(ENUM_DOMAINS[s])) + "}", note="lowercase 'none' is a valid member" if b == "Alcohol" else "")
      for b, s in ATTR_ENUM_MAP.items()]
H = "yelp_business_hours"
C += [_bid(H)]
for d in DAYS:
    hc = HOURS_MAP[d]
    C += [_k(H, hc,                 "string", f"hours.{d}", "as-is", domain="'H:M-H:M'"),
          _k(H, f"{hc}_start_time", "string", f"hours.{d}", "left side of range → 'HH:mm'",  domain="00:00–23:59", note="no TIME type on this compute → zero-padded STRING"),
          _k(H, f"{hc}_end_time",   "string", f"hours.{d}", "right side of range → 'HH:mm'", domain="00:00–23:59", note="may be <= start (overnight / 0:0-0:0)")]
CT = "yelp_business_categories"
C += [_bid(CT),
      _k(CT, "category_position", "int",    "categories", "0-based position in the list", False, ">= 0"),
      _k(CT, "category_name",     "string", "categories", "split on ', ' (token kept as-is, duplicates kept)", False, "non-empty label")]
for a, (tbl, keymap) in DICT_TABLE_MAP.items():
    C += [_bid(tbl),
          _k(tbl, "key",   "string",  f"attributes.{a}", "dict key, as-is", False, "{" + ", ".join(keymap) + "}"),
          _k(tbl, "value", "boolean", f"attributes.{a}", "dict value: True→TRUE, False→FALSE, None→NULL", True, "{True, False, None}")]
BUSINESS_CONTRACT = C

# Every key inside every nested dict (the level-2 contract)
DICT_KEY_CONTRACT = [{"Dict Attribute": a, "Dataset": tbl, "Key": k, "Source Path": f"attributes.{a}['{k}']",
                      "Bronze Encoding": "Python bool literal inside dict string", "Target Type": "boolean",
                      "Allowed Values": "{True, False, None}"}
                     for a, (tbl, keymap) in DICT_TABLE_MAP.items() for k in keymap]

contract_pd = pd.DataFrame(BUSINESS_CONTRACT)
print("\n📜 Datasets defined by the contract:")
display(pd.DataFrame([{"Dataset": t, **{k: (", ".join(v) if isinstance(v, list) else v) for k, v in s.items()},
                       "Columns": int((contract_pd["Dataset"] == t).sum())} for t, s in DATASETS.items()]))
print(f"\n📜 BUSINESS_CONTRACT — {len(contract_pd)} column specs across {len(DATASETS)} datasets:")
display(contract_pd)
print(f"\n📜 DICT_KEY_CONTRACT — {len(DICT_KEY_CONTRACT)} keys inside {len(DICT_TABLE_MAP)} nested attributes:")
display(pd.DataFrame(DICT_KEY_CONTRACT))

# ── Contract coverage: every bronze leaf consumed, every contract source exists ──
_srcs = {r["Source Path"].split("[")[0] for r in BUSINESS_CONTRACT}
_unmapped = sorted(set(BRONZE_TYPE) - _srcs)
_missing  = sorted(_srcs - set(BRONZE_TYPE))
print(f"\n{'✅' if not (_unmapped or _missing) else '🚨'} Coverage: {len(BRONZE_TYPE)} bronze leaves, "
      f"{len(_srcs)} consumed by the contract | unmapped: {_unmapped or 'none'} | missing in bronze: {_missing or 'none'}")

# ══ 3) Decoders (no trim, no case change — only the literal encoding is removed) ══
MAP_T = T.MapType(T.StringType(), T.BooleanType())

def dec_bool(c):
    return F.when(c == "True", F.lit(True)).when(c == "False", F.lit(False)).otherwise(F.lit(None).cast("boolean"))

def dec_int(c):
    return F.when(c.rlike(r"^\d+$"), c.cast("int")).otherwise(F.lit(None).cast("int"))

def dec_enum(c):
    return F.when(c.rlike(ENUM_RE), F.regexp_extract(c, ENUM_RE, 1)).otherwise(F.lit(None).cast("string"))

def dec_dict(c):
    """Python dict literal → JSON → MAP<STRING,BOOLEAN>. Whole-value 'None' / non-dict → NULL."""
    j = F.regexp_replace(c, r"\bu'", "'")
    j = F.regexp_replace(j, "'", '"')
    for py, js in (("True", "true"), ("False", "false"), ("None", "null")):
        j = F.regexp_replace(j, rf"\b{py}\b", js)
    return F.when(c.rlike(DICT_RE), F.from_json(j, MAP_T))

def hh_mm(c, i):
    """i=1 → start, i=3 → end of 'H:M-H:M', zero-padded to 'HH:mm'."""
    return F.when(c.rlike(HOURS_RE), F.format_string("%02d:%02d", F.regexp_extract(c, HOURS_RE, i).cast("int"),
                                                     F.regexp_extract(c, HOURS_RE, i + 1).cast("int")))

# ══ 4) df_typed ══
is_open_t = F.when(F.col("is_open") == 1, F.lit(True)).when(F.col("is_open") == 0, F.lit(False)).otherwise(F.lit(None).cast("boolean"))
typed_cols = [F.col(c) for c in ["business_id", "name", "address", "city", "state", "postal_code", "latitude", "longitude"]]
typed_cols += [F.col("stars").cast("decimal(2,1)").alias("stars"), F.col("review_count").cast("bigint").alias("review_count"),
               is_open_t.alias("is_open")]
typed_cols += [dec_bool(F.col(f"attributes.{b}")).alias(s) for b, s in ATTR_BOOLEAN_MAP.items()]
typed_cols += [dec_int(F.col(f"attributes.{b}")).alias(s)  for b, s in ATTR_INT_MAP.items()]
typed_cols += [dec_enum(F.col(f"attributes.{b}")).alias(s) for b, s in ATTR_ENUM_MAP.items()]
typed_cols += [dec_dict(F.col(f"attributes.{a}")).alias(DICT_COL[a]) for a in DICT_TABLE_MAP]
for d in DAYS:
    src, hc = F.col(f"hours.{d}"), HOURS_MAP[d]
    typed_cols += [src.alias(hc), hh_mm(src, 1).alias(f"{hc}_start_time"), hh_mm(src, 3).alias(f"{hc}_end_time")]
typed_cols += [F.split(F.col("categories"), CAT_DELIM).alias("categories")]

df_typed = df_bronze.select(*typed_cols)
_typed_schema = df_typed.schema
print(f"\n📋 df_typed: {len(_typed_schema.fields)} columns (bronze: 14 top-level / {len(BRONZE_TYPE)} leaves)")
display(pd.DataFrame([{"Column": f.name, "Type": f.dataType.simpleString()} for f in _typed_schema.fields]))

# ══ 5) Lossless-decode validation (full table, one pass) ══
#   Python None : source is the literal 'None' → NULL (decoded, not lost)
#   Lost        : non-null, non-'None' source value that did NOT decode (would silently become NULL)
#   Out-of-domain: decoded value outside the contract domain (reported, never changed)
def _spec(column, group, src, typed, ood=F.lit(False), lost=None):
    none = src.cast("string") == "None"
    lost = lost if lost is not None else (src.isNotNull() & ~none & typed.isNull())
    return (column, group, src, none, lost, ood)

_stars = F.col("stars")
specs = [
    _spec("business_id", "scalar", F.col("business_id"), F.col("business_id"), F.col("business_id").isNull() | (F.length("business_id") != 22)),
    _spec("latitude", "scalar", F.col("latitude"), F.col("latitude"), ~F.col("latitude").between(-90, 90)),
    _spec("longitude", "scalar", F.col("longitude"), F.col("longitude"), ~F.col("longitude").between(-180, 180)),
    _spec("stars", "scalar", _stars, _stars.cast("decimal(2,1)"),
          ~_stars.between(1, 5) | ((_stars * 2) != F.floor(_stars * 2)),
          lost=_stars.isNotNull() & (_stars.cast("decimal(2,1)").cast("double") != _stars)),
    _spec("review_count", "scalar", F.col("review_count"), F.col("review_count"), F.col("review_count") < 0),
    _spec("is_open", "scalar", F.col("is_open"), is_open_t, ~F.col("is_open").isin(0, 1)),
]
for b, s in ATTR_BOOLEAN_MAP.items():
    src = F.col(f"attributes.{b}")
    specs.append(_spec(s, "attr bool", src, dec_bool(src), ~src.isin(*PY_BOOL)))
for b, s in ATTR_INT_MAP.items():
    src = F.col(f"attributes.{b}"); t = dec_int(src)
    specs.append(_spec(s, "attr int", src, t, t.isNotNull() & ~t.between(1, 4)))
for b, s in ATTR_ENUM_MAP.items():
    src = F.col(f"attributes.{b}"); t = dec_enum(src)
    specs.append(_spec(s, "attr enum", src, t, t.isNotNull() & ~t.isin(*sorted(ENUM_DOMAINS[s]))))
for a, (tbl, keymap) in DICT_TABLE_MAP.items():
    src = F.col(f"attributes.{a}"); m = dec_dict(src)
    raw_keys = F.size(F.regexp_extract_all(src, F.lit(DICT_KEY_RE), F.lit(0)))
    unexpected = F.size(F.array_except(F.map_keys(m), F.array(*[F.lit(k) for k in keymap]))) > 0
    lost = (src.isNotNull() & (src != "None") & m.isNull()) | (m.isNotNull() & (F.size(m) != raw_keys))
    specs.append(_spec(DICT_COL[a], "attr dict", src, m, F.coalesce(unexpected, F.lit(False)), lost=lost))
for d in DAYS:
    src = F.col(f"hours.{d}")
    hm = [F.regexp_extract(src, HOURS_RE, i).cast("int") for i in (1, 2, 3, 4)]
    specs.append(_spec(HOURS_MAP[d], "hours", src, hh_mm(src, 1), (hm[0] > 23) | (hm[1] > 59) | (hm[2] > 23) | (hm[3] > 59)))
_cats = F.split(F.col("categories"), CAT_DELIM)
specs.append(_spec("categories", "list", F.col("categories"), _cats,
                   F.exists(_cats, lambda x: (x == "") | (x != F.trim(x)))))

aggs = []
for i, (_, _, src, none, lost, ood) in enumerate(specs):
    aggs += [F.count(src).alias(f"v{i}_nn"), F.sum(none.cast("int")).alias(f"v{i}_none"),
             F.sum(F.coalesce(lost, F.lit(False)).cast("int")).alias(f"v{i}_lost"),
             F.sum(F.when(src.isNotNull(), F.coalesce(ood, F.lit(False))).cast("int")).alias(f"v{i}_ood")]
V = {k: (v or 0) for k, v in df_bronze.agg(*aggs).collect()[0].asDict().items()}

val_rows = []
for i, (column, group, *_r) in enumerate(specs):
    nn, none, lost, ood = V[f"v{i}_nn"], V[f"v{i}_none"], V[f"v{i}_lost"], V[f"v{i}_ood"]
    val_rows.append({"Column": column, "Group": group, "Source Non-Null": nn, "Python None → NULL": none,
                     "Decoded Non-Null": nn - none - lost, "Lost in Decode": lost, "Out-of-Domain": ood,
                     "Status": "FAIL" if lost else ("FINDING" if ood else "PASS")})
val_df = pd.DataFrame(val_rows)
print("\n📊 Lossless-decode validation (full table):")
display(val_df)

_rows_typed, _rows_bronze = df_typed.count(), df_bronze.count()
print(f"\n{'✅' if _rows_typed == _rows_bronze else '🚨'} Rows preserved: {_rows_typed:,} (bronze {_rows_bronze:,})")
print(f"   Python None literals → NULL: {int(val_df['Python None → NULL'].sum()):,}  |  "
      f"Lost in decode: {int(val_df['Lost in Decode'].sum()):,}  |  Out-of-domain: {int(val_df['Out-of-Domain'].sum()):,}")
print("TE.1 COMPLETE")
print("=" * 80)

In [0]:
# TE.2) Remodel — structure df_typed into the business DataFrames (shape only, no value changes)
# ─────────────────────────────────────────────────────────────────────
#   df_business              wide  parent  1 row / business (11 scalars)
#   df_attributes            wide  1:1     32 single-value attributes
#   df_hours                 wide  1:1     7 days × (raw range, start_time, end_time)
#   df_categories            long  0:N     posexplode of the ARRAY (order + duplicates kept)
#   7 nested-attribute dfs   long  0:N     explode of each MAP → (business_id, key, value)
# Every frame is projected in BUSINESS_CONTRACT order & types. In-memory only — nothing is written.
# Reconciliation uses counts taken straight from the RAW bronze strings (independent of TE.1 decoders).
import pandas as pd
from pyspark.sql import functions as F

print("=" * 80)
print("TE.2) REMODEL DATA  —  structure into business DataFrames (values untouched)")
print("=" * 80)

def project(frame, ds):
    return frame.select(*[F.col(r["Column"]).cast(r["Target Type"]).alias(r["Column"])
                          for r in BUSINESS_CONTRACT if r["Dataset"] == ds])

# ══ 1) Build the DataFrames ══
business_dfs = {ds: project(df_typed, ds) for ds in ["yelp_business", "yelp_business_attributes", "yelp_business_hours"]}
business_dfs["yelp_business_categories"] = project(
    df_typed.where(F.col("categories").isNotNull())
            .select("business_id", F.posexplode("categories").alias("category_position", "category_name")),
    "yelp_business_categories")
for a, tbl in DICT_DF.items():
    business_dfs[tbl] = project(df_typed.where(F.col(DICT_COL[a]).isNotNull())
                                        .select("business_id", F.explode(DICT_COL[a]).alias("key", "value")), tbl)

df_business             = business_dfs["yelp_business"]
df_attributes           = business_dfs["yelp_business_attributes"]
df_hours                = business_dfs["yelp_business_hours"]
df_categories           = business_dfs["yelp_business_categories"]
df_ambience             = business_dfs["yelp_business_ambience"]
df_best_nights          = business_dfs["yelp_business_best_nights"]
df_business_parking     = business_dfs["yelp_business_parking"]
df_dietary_restrictions = business_dfs["yelp_business_dietary_restrictions"]
df_good_for_meal        = business_dfs["yelp_business_good_for_meal"]
df_hair_specializes_in  = business_dfs["yelp_business_hair_specializes_in"]
df_music                = business_dfs["yelp_business_music"]

# ══ 2) Independent expectations from the RAW bronze strings ══
_cnt = lambda s, pat: F.when(s.isNotNull(), F.size(F.regexp_extract_all(s, F.lit(pat), F.lit(0)))).otherwise(F.lit(0))
exp_aggs = [F.count("*").alias("n"),
            F.sum(F.when(F.col("categories").isNotNull(), F.size(F.split("categories", ","))).otherwise(F.lit(0))).alias("cat_rows")]
for a in DICT_DF:
    s = F.when(F.col(f"attributes.{a}").rlike(DICT_RE), F.col(f"attributes.{a}"))
    exp_aggs += [F.sum(_cnt(s, DICT_KEY_RE)).alias(f"{a}_rows"), F.sum(_cnt(s, r":\s*True")).alias(f"{a}_T"),
                 F.sum(_cnt(s, r":\s*False")).alias(f"{a}_F"), F.sum(_cnt(s, r":\s*None")).alias(f"{a}_N")]
E = {k: (v or 0) for k, v in df_bronze.agg(*exp_aggs).collect()[0].asDict().items()}
expected_rows = {"yelp_business": E["n"], "yelp_business_attributes": E["n"], "yelp_business_hours": E["n"],
                 "yelp_business_categories": E["cat_rows"], **{tbl: E[f"{a}_rows"] for a, tbl in DICT_DF.items()}}

# ══ 3) Reconciliation audit per DataFrame ══
parent_keys = df_business.select("business_id")
audit = []
for ds, sdf in business_dfs.items():
    spec = DATASETS[ds]
    nn_cols = [r["Column"] for r in BUSINESS_CONTRACT if r["Dataset"] == ds and not r["Nullable"]]
    long_dict = ds in DICT_DF.values()
    aggs = [F.count("*").alias("rows"), F.countDistinct("business_id").alias("biz"),
            F.sum(F.when(F.greatest(*[F.col(c).isNull() for c in nn_cols]) if len(nn_cols) > 1 else F.col(nn_cols[0]).isNull(), 1).otherwise(0)).alias("nn")]
    if long_dict:
        aggs += [F.sum(F.when(F.col("value") == True, 1).otherwise(0)).alias("T"),
                 F.sum(F.when(F.col("value") == False, 1).otherwise(0)).alias("F"),
                 F.sum(F.when(F.col("value").isNull(), 1).otherwise(0)).alias("N")]
    R = sdf.agg(*aggs).collect()[0].asDict()
    dup = R["rows"] - sdf.select(*spec["PK"]).distinct().count()
    orphans = 0 if ds == "yelp_business" else sdf.join(parent_keys, "business_id", "left_anti").count()
    vals_ok, vals_txt = True, "—"
    if long_dict:
        a = [k for k, v in DICT_DF.items() if v == ds][0]
        vals_ok = (R["T"], R["F"], R["N"]) == (E[f"{a}_T"], E[f"{a}_F"], E[f"{a}_N"])
        vals_txt = f"T {R['T']:,} / F {R['F']:,} / NULL {R['N']:,}" + ("" if vals_ok else f"  ≠ raw T {E[f'{a}_T']:,} / F {E[f'{a}_F']:,} / None {E[f'{a}_N']:,}")
    rows_ok = R["rows"] == expected_rows[ds]
    ok = rows_ok and dup == 0 and R["nn"] == 0 and orphans == 0 and vals_ok
    audit.append({"DataFrame": ds, "Shape": spec["Shape"], "Cardinality": spec["Cardinality"], "Rows": R["rows"],
                  "Expected Rows (raw)": expected_rows[ds], "Columns": len(sdf.columns), "Businesses": R["biz"],
                  "PK Duplicates": dup, "NOT NULL Violations": R["nn"], "FK Orphans": orphans,
                  "Values vs Raw": vals_txt, "Status": "PASS" if ok else "FAIL"})
audit_df = pd.DataFrame(audit)
print("\n📊 Reconciliation: every DataFrame vs counts taken from the raw bronze strings")
display(audit_df)

# ══ 4) Structural findings — observed and reported, NOT fixed ══
findings = []
_cat_dup = df_categories.groupBy("business_id", "category_name").count().where("count > 1")
_cd = _cat_dup.agg(F.sum(F.col("count") - 1).alias("extra"), F.count("*").alias("biz_pairs"),
                   F.countDistinct("business_id").alias("biz")).collect()[0]
findings.append({"Area": "categories", "Check": "Same category repeated within a business",
                 "Count": int(_cd["extra"] or 0), "Businesses": int(_cd["biz"] or 0),
                 "Effect in long DataFrame": "extra rows kept (PK is business_id + category_position)"})
findings.append({"Area": "categories", "Check": "Businesses with NULL categories",
                 "Count": E["n"] - df_categories.select("business_id").distinct().count(), "Businesses": None,
                 "Effect in long DataFrame": "no rows"})

d_aggs = []
for a, (tbl, keymap) in DICT_TABLE_MAP.items():
    m, raw_s = F.col(DICT_COL[a]), F.col(f"attributes.{a}")
    exp_keys = F.array(*[F.lit(k) for k in keymap])
    d_aggs += [F.sum((raw_s == "None").cast("int")).alias(f"{a}_none"),
               F.sum((F.size(m) == 0).cast("int")).alias(f"{a}_empty"),
               F.sum(((F.size(m) > 0) & F.forall(F.map_values(m), lambda v: v.isNull())).cast("int")).alias(f"{a}_allnull"),
               F.sum(((F.size(m) > 0) & (F.size(F.array_except(exp_keys, F.map_keys(m))) > 0)).cast("int")).alias(f"{a}_partial")]
D = {k: (v or 0) for k, v in df_typed.join(df_bronze.select("business_id", "attributes"), "business_id")
                                       .agg(*d_aggs).collect()[0].asDict().items()}
for a in DICT_TABLE_MAP:
    findings += [
        {"Area": f"attributes.{a}", "Check": "Whole value is the Python None literal",    "Count": D[f"{a}_none"],    "Businesses": D[f"{a}_none"],    "Effect in long DataFrame": "no rows (nothing to represent)"},
        {"Area": f"attributes.{a}", "Check": "Empty dict {}",                             "Count": D[f"{a}_empty"],   "Businesses": D[f"{a}_empty"],   "Effect in long DataFrame": "no rows (left as-is)"},
        {"Area": f"attributes.{a}", "Check": "Non-empty dict whose values are ALL None",  "Count": D[f"{a}_allnull"], "Businesses": D[f"{a}_allnull"], "Effect in long DataFrame": "rows with value NULL"},
        {"Area": f"attributes.{a}", "Check": "Dict missing ≥1 contract key (partial key set)", "Count": D[f"{a}_partial"], "Businesses": D[f"{a}_partial"], "Effect in long DataFrame": "fewer rows for that business"},
    ]

h_aggs = []
for d in DAYS:
    hc = HOURS_MAP[d]; st, en = F.col(f"{hc}_start_time"), F.col(f"{hc}_end_time")
    h_aggs += [F.sum((F.col(hc) == "0:0-0:0").cast("int")).alias(f"{d}_zero"),
               F.sum(((en < st)).cast("int")).alias(f"{d}_overnight"),
               F.sum(((en == st) & (F.col(hc) != "0:0-0:0")).cast("int")).alias(f"{d}_same")]
h_aggs += [F.sum(F.when(F.greatest(*[F.col(HOURS_MAP[d]).isNotNull() for d in DAYS]), 0).otherwise(1)).alias("no_hours")]
HH = df_hours.agg(*h_aggs).collect()[0].asDict()
_hs = lambda k: sum(HH[f"{d}_{k}"] or 0 for d in DAYS)
findings += [
    {"Area": "hours", "Check": "Day value '0:0-0:0' (closed vs 24h ambiguous)", "Count": _hs("zero"), "Businesses": None, "Effect in long DataFrame": "start = end = 00:00"},
    {"Area": "hours", "Check": "end_time < start_time (crosses midnight)",       "Count": _hs("overnight"), "Businesses": None, "Effect in long DataFrame": "kept as-is"},
    {"Area": "hours", "Check": "end_time = start_time (other than 0:0-0:0)",      "Count": _hs("same"), "Businesses": None, "Effect in long DataFrame": "kept as-is"},
    {"Area": "hours", "Check": "Business with no opening hours on any day",       "Count": HH["no_hours"] or 0, "Businesses": HH["no_hours"] or 0, "Effect in long DataFrame": "1:1 row with all-NULL day columns"},
]
_attr_cols = [r["Column"] for r in BUSINESS_CONTRACT if r["Dataset"] == "yelp_business_attributes" and r["Column"] != "business_id"]
_no_attr = df_attributes.where(" AND ".join(f"`{c}` IS NULL" for c in _attr_cols)).count()
findings.append({"Area": "attributes", "Check": "Business with all 32 single-value attributes NULL", "Count": _no_attr,
                 "Businesses": _no_attr, "Effect in long DataFrame": "1:1 row with all-NULL attribute columns"})
structural_df = pd.DataFrame(findings)
structural_df["Businesses"] = structural_df["Businesses"].astype("Int64")
structural_df["Status"] = structural_df["Count"].apply(lambda n: "FINDING" if n else "PASS")
print("\n📋 Structural findings (observed, not fixed):")
display(structural_df)

# ══ 5) Key-level profile of every nested attribute (long DataFrames) ══
_dicts_per_attr = {a: business_dfs[tbl].select("business_id").distinct().count() for a, tbl in DICT_DF.items()}
_long_all = None
for a, tbl in DICT_DF.items():
    part = business_dfs[tbl].select(F.lit(a).alias("dict_attribute"), "key", "value")
    _long_all = part if _long_all is None else _long_all.unionByName(part)
key_profile = (_long_all.groupBy("dict_attribute", "key")
               .agg(F.count("*").alias("present"),
                    F.sum(F.when(F.col("value") == True, 1).otherwise(0)).alias("true"),
                    F.sum(F.when(F.col("value") == False, 1).otherwise(0)).alias("false"),
                    F.sum(F.when(F.col("value").isNull(), 1).otherwise(0)).alias("null"))
               .toPandas())
key_profile["businesses with dict"] = key_profile["dict_attribute"].map(_dicts_per_attr)
key_profile["missing key"] = key_profile["businesses with dict"] - key_profile["present"]
key_profile["null %"] = (100 * key_profile["null"] / key_profile["present"]).round(2)
key_profile = key_profile.assign(_o=key_profile.apply(lambda r: (list(DICT_TABLE_MAP).index(r["dict_attribute"]),
                                                                  list(DICT_TABLE_MAP[r["dict_attribute"]][1]).index(r["key"])
                                                                  if r["key"] in DICT_TABLE_MAP[r["dict_attribute"]][1] else 99), axis=1)
                          ).sort_values("_o").drop(columns="_o").reset_index(drop=True)
print(f"\n📋 Key-level profile of the 7 nested attributes ({len(key_profile)} keys):")
display(key_profile)

# ══ 6) Samples ══
for name, sdf in [("df_business", df_business), ("df_attributes", df_attributes), ("df_hours", df_hours),
                  ("df_categories", df_categories), ("df_ambience (long)", df_ambience)]:
    print(f"\n📋 Sample — {name}:")
    display(sdf.limit(5))

# Handles for downstream cells: df = parent; 6.1.4 consumes silver_dfs (same keys as the Silver DDL tables)
silver_dfs = business_dfs
df = df_business
total_rows = E["n"]
_fails = int((audit_df["Status"] == "FAIL").sum())
print(f"\n{'✅' if _fails == 0 else '🚨'} {len(business_dfs)} DataFrames built | reconciliation failures: {_fails} | "
      f"structural findings: {int((structural_df['Status'] == 'FINDING').sum())}")
print("TE.2 COMPLETE")
print("=" * 80)


---
#### 🚨 DQ-BUSINESS-005 — Duplicate Categories Within a Business

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-005` |
| **Severity** | FINDING (uniqueness / primary-key violation in normalized child) |
| **Detected In** | Phase TE.2 — Remodel Data (Cell 35): structural findings (43 repeated rows across 43 businesses) |
| **Also Confirmed In** | **Not remediated in this notebook**: `df_categories` keeps every token (668,592 rows = raw token count), keyed by (`business_id`, `category_position`), so the repeats stay visible. Against the Silver DDL PK (`business_id`, `category_name`) they are 43 duplicates |
| **Affected Columns** | `categories` → `yelp_business_categories.category_name` |
| **Total Affected Rows** | **43** out of 150,346 businesses (0.029%), each with exactly 1 category repeated twice. 43 surplus tokens out of 668,592 exploded tokens |
| **Invalid Value** | A label repeated in the same comma-separated list, e.g. `"Gas Stations, Automotive, ..., Gas Stations"` |
| **Domain Rule** | A category is either assigned to a business or not, so it can appear at most once per business (DDL PK `pk_sc_business_categories` = `business_id, category_name`) |

**Description:** In bronze, `categories` is a single comma-separated string. For 43 businesses the same label appears twice in that string, and the repeated labels seen include `Gas Stations`. This probably comes from Yelp merging category taxonomies (a parent and a child mapping to the same label). In bronze the repetition is invisible, but once exploded into the 1:N `yelp_business_categories` table (rule R3) it breaks the declared composite primary key. It would also double-count those businesses in category-level metrics such as "number of gas stations per city" or category-share analyses.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Deduplicate on (`business_id`, `category_name`), keeping the first position** (`GROUP BY ... MIN(category_position)`) | Satisfies the PK, loses no information (the duplicate adds nothing), and keeps the original ordinal of the first mention. Recommended for the Silver load (TE.2 keeps the repeats for exploration). |
| 2 | **`array_distinct(split(categories, ','))` before exploding** | Same outcome with simpler logic, but positions after the removed item shift, so `category_position` no longer matches the original string. |
| 3 | **Keep duplicates and add a surrogate key** | Avoids the PK conflict technically, but keeps the double-counting and contradicts the DDL's declared composite key. |
| 4 | **Quarantine affected businesses** | Fully auditable, but it removes 43 valid businesses' categories over a harmless repeat. Disproportionate. |

**Best Logical Solution:** **Option 1: deduplicate on the composite PK and keep `MIN(category_position)`.** The duplicate carries no extra meaning, so collapsing it is lossless. Keeping the first position preserves the original ordering semantics, and the result satisfies the informational PK the DDL declares, which the query optimizer and downstream joins rely on.

---

---
#### 🚨 DQ-BUSINESS-006 — Empty (`{}`) and All-`None` Nested Attribute Dicts

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-006` |
| **Severity** | FINDING (empty nested payload, semantic null) |
| **Detected In** | Phase TE.2 — Remodel Data (Cell 35): structural findings "Empty dict {}" and "Non-empty dict whose values are ALL None" |
| **Also Confirmed In** | Phase TE.1 (Cell 34): the 81 empty dicts decode losslessly to empty MAPs (0 lost). The earlier wide-format remodel reported the same 83 businesses as "All-flags-NULL" rows (Ambience 6 = 4 `{}` + 2 all-`None`) |
| **Affected Columns** | `attributes.Music` → `df_music` (69 `{}`), `attributes.BusinessParking` → `df_business_parking` (8 `{}`), `attributes.Ambience` → `df_ambience` (4 `{}` + 2 dicts whose keys are all `None`) |
| **Total Affected Rows** | **83** out of 150,346 businesses (0.055%): 81 empty dicts + 2 all-`None` dicts. For Music this is 69 of 7,486 dict values (0.92%) |
| **Invalid Value** | `'{}'`: syntactically a dict, but with no keys. `{'romantic': None, ...}`: every key present but unknown |
| **Domain Rule** | Parsed-dict tables are 0:1 with `yelp_business` and should contain a row only when the business actually reported at least one flag (DDL: "only businesses with the dict") |

**Description:** For 81 businesses the dict attribute exists but is empty (`{}`). The business owner opened the Music, Parking or Ambience section and saved it without ticking anything. For 2 more businesses the Ambience dict lists keys whose values are all `None`. This notebook leaves both as they are:
- **Long DataFrames (TE.2):** an empty dict has no entries, so it produces **no rows**. The business then looks the same as one that never had the attribute, although bronze tells them apart (`'{}'` vs `NULL` / `'None'`). The 2 all-`None` dicts do produce rows, all with `value = NULL`.
- **Wide Silver DDL shape:** keeping `dict IS NOT NULL` turns all 83 into child rows where every flag is NULL. `COUNT(*)` on `yelp_business_music` would then overstate "businesses with music information" by about 0.9%, the 0:1 meaning "a row exists, so data exists" no longer holds, and `LEFT JOIN` consumers can't tell "no dict" from "empty dict".

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Treat `'{}'` as absent: exclude it from the parsed-dict child tables** (filter `trim(dict) NOT IN ('{}')` alongside the `'None'` handling) | Matches the DDL's 0:1 rule ("only businesses with the dict"). An empty dict and a missing dict carry the same information (none), and the business stays fully present in the parent and other children. |
| 2 | **Keep the row and set all flags to `False`** | Every row then has values, but it asserts "no live music, no DJ, ..." when the owner said nothing. That is invented information. |
| 3 | **Keep all-NULL rows (current behaviour)** | Records that the section was opened, but the meaning is unclear, counts are inflated, and every consumer has to filter them. |
| 4 | **Keep the row with an `is_empty_dict` flag** | Fully traceable, but it adds a column the DDL doesn't define (schema drift vs contract) for 83 rows of marginal analytical value. |

**Best Logical Solution:** **Option 1: treat `'{}'` the same as a missing dict and exclude it from the level-2 tables.** It follows the DDL's 0:1 rule without changing the schema, keeps row counts meaningful (a row means at least one reported flag), and invents nothing. The fix is a one-line filter added to the TE.2 dict loop and to the Silver loading script.

---

---
#### 🚨 DQ-BUSINESS-007 — Ambiguous and Cross-Midnight Opening Hours

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-007` |
| **Severity** | FINDING (semantic ambiguity in time ranges) |
| **Detected In** | Phase TE.2 — Remodel Data (Cell 35): structural findings on the new `df_hours` start / end columns |
| **Also Confirmed In** | Phase TE.1 (Cell 34): all 801,015 non-null day values match `H:M-H:M` with hours 0–23 and minutes 0–59 (0 lost, 0 out of range). The format is clean; the problem is what the values mean |
| **Affected Columns** | `hours.Monday` … `hours.Sunday` → `df_hours.hours_<day>`, `hours_<day>_start_time`, `hours_<day>_end_time` |
| **Total Affected Rows** | **133,901** of 801,015 non-null day values (16.7%): **73,908** `'0:0-0:0'` (9.23%), **58,501** with end < start (7.30%), **1,492** with end = start other than `0:0-0:0` (0.19%). Separately, 23,223 businesses (15.4%) have no hours on any day |
| **Invalid Value** | `'0:0-0:0'` (closed? open 24 h? placeholder?), ranges whose end is before the start (e.g. a bar open 17:00 → 02:00), and zero-length ranges such as `'9:0-9:0'` |
| **Domain Rule** | A daily opening range must have a single interpretation: end after start on the same day, or explicitly marked as overnight, 24 h or closed |

**Description:** Yelp stores each day as a bare `start-end` string, with no flag for overnight, 24-hour or closed days. Once the range is split into `start_time` / `end_time`, three patterns stand out:
- `'0:0-0:0'` is the single most common value. The TE.2 sample shows a business open `8:0-18:30` Tuesday–Friday with Monday `'0:0-0:0'`, which reads as "closed", while 24-hour businesses use the same encoding.
- About 7% of ranges cross midnight, so a naive `end - start` duration comes out negative.
- A small set of ranges have zero length.

Opening-hours analytics (duration, "open now", late-night coverage) are wrong unless each pattern is handled explicitly. `attributes.Open24Hours` can't disambiguate, because only 39 businesses report it.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep the raw string and derive explicit flags** (`is_overnight` = end < start; `is_zero_range` = `0:0-0:0`; duration computed modulo 24 h, NULL for ambiguous ranges) | Nothing is lost or reinterpreted. Overnight ranges become computable, and the `0:0-0:0` ambiguity stays visible instead of being guessed. |
| 2 | **Interpret `0:0-0:0` as open 24 h** | Simple, but it labels businesses that are probably closed that day as open all day, which inflates late-night coverage. |
| 3 | **Interpret `0:0-0:0` as closed** | Matches the sample pattern, but it silently drops genuinely 24-hour businesses. |
| 4 | **Leave as strings only** | No effort, but every consumer re-implements the parsing and the overnight arithmetic differently. |

**Best Logical Solution:** **Option 1: keep the raw range and add derived flags plus a modulo-24 h duration.** The overnight case has one correct interpretation and can be computed. The `0:0-0:0` case has no source rule that resolves it, so it should stay flagged and unknown rather than be guessed, until a reliable signal (e.g. review timestamps or a provider rule) is available.

---

---
#### 🚨 DQ-BUSINESS-008 — Partial Key Sets in Nested Attribute Dicts

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-008` |
| **Severity** | FINDING (structural completeness at nesting level 2) |
| **Detected In** | Phase TE.2 — Remodel Data (Cell 35): structural finding "Dict missing ≥1 contract key" and the key-level profile |
| **Also Confirmed In** | Phase 6.1.3 dict-key inventory (Cell 29): no unexpected keys, so only contract keys are absent. TE.1 (Cell 34): raw key tokens = MAP entries for every dict (0 lost) |
| **Affected Columns** | `attributes.Ambience` (1,337 dicts; `divey` absent 1,222, `upscale` 118, `hipster` 68, the other 6 keys 6 each), `attributes.Music` (559; `no_music` absent 559, `background_music` 349, `video` / `karaoke` 347, `jukebox` 333, `live` 315, `dj` 46), `attributes.HairSpecializesIn` (33; `straightperms`, `africanamerican`, `asian`), `attributes.BusinessParking` (13; `street` / `validated` / `lot` 13, `valet` 11, `garage` 2) |
| **Total Affected Rows** | **1,942** of 175,479 non-empty dicts (1.11%), i.e. at most 1,942 of 150,346 businesses (1.29%). `BestNights`, `DietaryRestrictions` and `GoodForMeal` always carry their full key set |
| **Invalid Value** | A dict that omits a contract key, e.g. a `Music` dict without `no_music` |
| **Domain Rule** | Each nested attribute has a fixed key set (`DICT_KEY_CONTRACT`). Every key should be present, with `True`, `False` or `None` |

**Description:** A nested attribute can hold a key in three states: an explicit `True`/`False`, an explicit `None` (unknown), or absent. In the long DataFrames an absent key is simply a missing row. When the data is pivoted to the wide Silver shape, it becomes NULL and can no longer be told apart from an explicit `None`. The pattern is concentrated in specific keys (`Ambience.divey`, `Music.no_music`), which points to keys added to the Yelp form later and never back-filled. Per-key rates (e.g. "% of businesses with a divey ambience") therefore use different denominators per key unless the analysis accounts for it.

The key-level profile shows two more points:
- `Music.no_music` is **never `True`** (0 of 6,858 present).
- `DietaryRestrictions.kosher` is never `True` (0 of 30).

These keys currently carry no positive signal.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Pivot against the contract key list in Silver, document "absent ≡ unknown" (NULL), and use per-key denominators in reporting** | Matches the existing wide DDL, invents nothing, and states clearly that absent and `None` carry the same information (unknown). |
| 2 | **Long format with a `key_present` flag (contract keys cross-joined, missing ones filled with NULL and flagged)** | Fully traceable at row level, but it adds rows and a column the Silver DDL doesn't define. |
| 3 | **Fill absent keys with `False`** | Every row is complete, but it asserts "not divey" / "has music" when nothing was reported, which biases rates. |
| 4 | **Drop dicts with partial key sets** | Removes the inconsistency, but throws away 1,942 dicts of valid reported flags. |

**Best Logical Solution:** **Option 1: pivot against the contract key list and treat absent keys as unknown (NULL).** Absence carries no more information than `None`, the wide DDL already handles it, and per-key denominators keep the rates honest. The long DataFrames in this notebook keep the distinction for exploration.

---

In [0]:
# 6.1.4) Schema Drift Detection
# ─────────────────────────────────────────────────────────────────────
# Baseline = Silver DDL contract (SILVER_CONTRACT). Compared against:
#   (A) the remodeled DataFrames from TE.2      → is our transformation DDL-conformant?
#   (B) the deployed UC tables in silver_cleaned → has the deployed DDL drifted from the file?
# Flags ADDED / REMOVED / RETYPED / REORDERED columns, and (B only) NOT NULL drift.
print("=" * 80)
print("6.1.4) SCHEMA DRIFT DETECTION  —  Silver DDL contract vs remodel vs deployed UC")
print("=" * 80)

def drift(table, actual_fields, layer, check_nullability):
    exp = [(c["column"], c["type"], c["nullable"]) for c in SILVER_CONTRACT[table]]
    exp_t = {c: t for c, t, _ in exp}; exp_n = {c: n for c, _, n in exp}
    act_t = {c: t for c, t, _ in actual_fields}; act_n = {c: n for c, _, n in actual_fields}
    out = []
    for c in act_t:
        if c not in exp_t: out.append((c, "ADDED", "N/A", act_t[c]))
    for c in exp_t:
        if c not in act_t: out.append((c, "REMOVED", exp_t[c], "N/A"))
        elif exp_t[c] != act_t[c]: out.append((c, "RETYPED", exp_t[c], act_t[c]))
        elif check_nullability and exp_n[c] != act_n[c]:
            out.append((c, "NULLABILITY", "NOT NULL" if not exp_n[c] else "NULL", "NOT NULL" if not act_n[c] else "NULL"))
    common_exp = [c for c, _, _ in exp if c in act_t]; common_act = [c for c, _, _ in actual_fields if c in exp_t]
    if common_exp != common_act: out.append(("<order>", "REORDERED", ",".join(common_exp), ",".join(common_act)))
    rows = [{"Layer": layer, "Table": table, "Column": c, "Drift": d, "Baseline": b, "Current": a, "Status": "FINDING"}
            for c, d, b, a in out]
    return rows, len(exp)

all_rows, summary = [], []
for t in SILVER_CONTRACT:
    # (A) remodeled DataFrame (Spark DataFrame nullability is not meaningful → types/order only)
    fa = [(f.name, f.dataType.simpleString(), f.nullable) for f in silver_dfs[t].schema.fields]
    ra, n_exp = drift(t, fa, "A: remodel (TE.2)", False)
    all_rows += ra
    # (B) deployed UC table
    try:
        fb = [(f.name, f.dataType.simpleString(), f.nullable) for f in spark.table(f"{SILVER_SCHEMA}.{t}").schema.fields]
        rb, _ = drift(t, fb, "B: deployed UC", True)
        b_status = "PASS" if not rb else "FINDING"
    except Exception as e:
        rb = [{"Layer": "B: deployed UC", "Table": t, "Column": "<table>", "Drift": "TABLE MISSING",
               "Baseline": "exists", "Current": type(e).__name__, "Status": "FINDING"}]
        b_status = "FINDING"
    all_rows += rb
    summary.append({"Table": t, "Contract Columns": n_exp,
                    "A: Remodel Drifts": len(ra), "A: Status": "PASS" if not ra else "FINDING",
                    "B: Deployed Drifts": len(rb), "B: Status": b_status})

print("\n📊 Drift summary per table:")
display(pd.DataFrame(summary))

if all_rows:
    print("\n📋 Drift details:")
    findings_df = pd.DataFrame(all_rows)
    display(findings_df)
else:
    findings_df = pd.DataFrame(columns=["Layer", "Table", "Column", "Drift", "Baseline", "Current", "Status"])

drift_count = len(all_rows)
n_cols = sum(len(v) for v in SILVER_CONTRACT.values())
print(f"\n{len(SILVER_CONTRACT)} tables / {n_cols} contract columns compared on 2 layers — {drift_count} drift(s) detected")
print("✅ No schema drift — remodel and deployed tables match the Silver DDL exactly." if drift_count == 0
      else f"⚠️  {drift_count} schema drift(s) detected — review above.")
print("SCHEMA DRIFT DETECTION COMPLETE")
print("=" * 80)

In [0]:
# TE.3) Display all cleaned & modeled DataFrames from TE.1 / TE.2
# ─────────────────────────────────────────────────────────────────────
from pyspark.sql import functions as F

_frames = [
    ("df_business",             df_business),
    ("df_attributes",           df_attributes),
    ("df_hours",                df_hours),
    ("df_categories",           df_categories),
    ("df_ambience",             df_ambience),
    ("df_best_nights",          df_best_nights),
    ("df_business_parking",     df_business_parking),
    ("df_dietary_restrictions", df_dietary_restrictions),
    ("df_good_for_meal",        df_good_for_meal),
    ("df_hair_specializes_in",  df_hair_specializes_in),
    ("df_music",                df_music),
]

for name, sdf in _frames:
    _n = sdf.count()
    _c = len(sdf.columns)
    print(f"{'═' * 80}")
    print(f"📋 {name}  —  {_n:,} rows × {_c} columns   [{', '.join(sdf.columns)}]")
    print(f"{'═' * 80}")
    display(sdf.limit(10))
    print()


### 6.2) DOMAIN & RANGE CHECKS
*Verify values belong to allowed sets and fall within permitted bounds. All scope variants live here.*

- **Allowed Value Conformance (single column)** — Values belong to the permitted domain/category list.
  - *Example:* `sex` allows only `M`, `F`, `Other`; found `Male`, `1`, `Unknown`.
  - *Sub:* Geographic codes — `country_code` must be ISO-3166 alpha-2.
  - *Sub:* Department names — `department` must match organizational list.
  - *Sub:* Status codes — `employee_status` must be in `{Active, Terminated, Leave}`.

- **Range Conformance (single column)** — Values within min/max bounds.
  - *Example:* `age` must be 0–120; found `-3` or `250`.
  - *Sub:* Salary bands — `salary` within role-specific band.
  - *Sub:* Date bounds — `hire_date` after 1980-01-01.

- **Pattern Conformance** — Values match required format (email, phone, postal code, ID patterns).
  - *Example:* `email` must match RFC-5322; `phone` must match E.164.

- **Impossible Value Detection** — Values that cannot exist regardless of bounds.
  - *Example:* `birth_date` in the future; `age` negative; `death_date` before `birth_date` (single-column view).


In [0]:
# 6.2.1) Allowed Value Conformance (single column)  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Every column with a CLOSED domain must only hold permitted values.
# Scope (TE.1 / TE.2 frames + df_bronze for raw encodings):
#   A) df_business   : stars (half-star ordinal), is_open (0/1 flag), state (jurisdiction code)
#   B) df_attributes : 24 BOOLEAN attrs (raw literal set), price range {1..4}, 7 ENUM attrs (ENUM_DOMAINS)
#   C) 7 nested dicts: key ∈ DICT_KEY_CONTRACT key set, raw value token ∈ {True, False, None}
#   D) df_categories : open vocabulary → profiled (no official list), not pass/fail
# Detection only — nothing is changed. Helpers defined here are reused by 6.2.2 – 6.2.4.
import pandas as pd
from pyspark.sql import functions as F

print("=" * 80)
print("6.2.1) ALLOWED VALUE CONFORMANCE  —  Business DataFrames (TE.2)")
print("    Checking values against closed / allowed sets")
print("=" * 80)

N_BUS = df_business.count()
findings_621 = []

# ── shared helpers (reused in 6.2.2 – 6.2.4) ──
_ICON = {"PASS": "✅", "FINDING": "⚠️", "INFO": "ℹ️"}

def dq_add(findings, frame, column, rule, checked, violations, status=None, examples="", note=""):
    checked, violations = int(checked or 0), int(violations or 0)
    status = status or ("PASS" if violations == 0 else "FINDING")
    findings.append({"DataFrame": frame, "Column": column, "Rule": rule, "Checked": checked,
                     "Violations": violations, "Pct": round(violations / checked * 100, 3) if checked else 0.0,
                     "Status": status, "Examples": examples, "Note": note})
    print(f"   {_ICON[status]} {column:<40} {violations:>9,} / {checked:>9,}  — {status}  [{rule}]")
    if examples and status != "PASS":
        print(f"      e.g. {examples}")

def dq_examples(frame, expr_col, cond, n=8):
    return ", ".join(repr(r[0]) for r in frame.where(cond).select(expr_col).distinct().limit(n).collect())

def dq_summary(findings, title):
    print(f"\n{'=' * 80}\nSUMMARY — {title}\n{'=' * 80}")
    out = pd.DataFrame(findings)
    display(out)
    n_f = int((out["Status"] == "FINDING").sum()); n_i = int((out["Status"] == "INFO").sum())
    print(f"\n{len(out)} checks | {n_f} FINDING | {n_i} INFO | {len(out) - n_f - n_i} PASS")
    return out

# ══════════════════════════════════════════════════════════════════════
# A) df_business — scalar closed domains
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─' * 60}\n📌 A) df_business — scalar closed domains")

# A1) stars — Yelp business rating is a rounded mean on a half-star scale
STAR_DOMAIN = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
# A3) state — USPS (50 states + DC + territories) ∪ Canadian provinces/territories
US_STATES = {"AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA","KS","KY","LA","ME","MD",
             "MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC",
             "SD","TN","TX","UT","VT","VA","WA","WV","WI","WY","DC","PR","VI","GU","AS","MP"}
CA_PROVINCES = {"AB","BC","MB","NB","NL","NS","NT","NU","ON","PE","QC","SK","YT"}
VALID_JURISDICTIONS = US_STATES | CA_PROVINCES
# Yelp Open Dataset footprint: 11 metro areas (Philadelphia incl. NJ/DE suburbs, Tampa, Indianapolis, Nashville,
# Tucson, New Orleans, Reno, Edmonton, Santa Barbara, Boise, St. Louis incl. IL side)
YELP_FOOTPRINT_STATES = {"PA","NJ","DE","FL","IN","TN","AZ","LA","NV","AB","CA","ID","MO","IL"}

raw_is_open = df_bronze.select(F.col("is_open").alias("raw_is_open"))
r = df_business.agg(
    F.sum(F.col("stars").isNotNull().cast("int")).alias("stars_nn"),
    F.sum((F.col("stars").isNotNull() & ~F.col("stars").cast("double").isin(STAR_DOMAIN)).cast("int")).alias("stars_bad"),
    F.sum(F.col("stars").isNull().cast("int")).alias("stars_null"),
    F.sum(F.col("is_open").isNull().cast("int")).alias("is_open_null"),
    F.sum(F.col("state").isNotNull().cast("int")).alias("state_nn"),
    F.sum((F.col("state").isNotNull() & ~F.col("state").isin(list(VALID_JURISDICTIONS))).cast("int")).alias("state_bad"),
    F.sum((F.col("state").isin(list(VALID_JURISDICTIONS)) & ~F.col("state").isin(list(YELP_FOOTPRINT_STATES))).cast("int")).alias("state_off"),
).first()
raw_open_bad = raw_is_open.where(F.col("raw_is_open").isNull() | ~F.col("raw_is_open").isin(0, 1)).count()

dq_add(findings_621, "df_business", "stars", "∈ {1.0, 1.5, …, 5.0} (half-star)", r["stars_nn"], r["stars_bad"] + r["stars_null"],
       examples=dq_examples(df_business, "stars", ~F.col("stars").cast("double").isin(STAR_DOMAIN)))
dq_add(findings_621, "df_business", "is_open (raw bronze)", "∈ {0, 1}", N_BUS, raw_open_bad,
       note=f"typed BOOLEAN NULLs: {r['is_open_null']}")
dq_add(findings_621, "df_business", "state", "valid USPS / Canadian jurisdiction code", r["state_nn"], r["state_bad"],
       examples=dq_examples(df_business, "state", ~F.col("state").isin(list(VALID_JURISDICTIONS))))
_off = (df_business.where(F.col("state").isin(list(VALID_JURISDICTIONS)) & ~F.col("state").isin(list(YELP_FOOTPRINT_STATES)))
        .groupBy("state").count().orderBy(F.desc("count")).collect())
dq_add(findings_621, "df_business", "state (Yelp footprint)", "∈ the 14 states/provinces of the 11 Yelp metros",
       r["state_nn"], r["state_off"], status="INFO" if r["state_off"] else "PASS",
       examples=", ".join(f"{x['state']}={x['count']}" for x in _off),
       note="real codes but outside the dataset's metros → likely mislabeled state; coordinates cross-checked in 6.6")

print("\n   📊 stars distribution:")
for row in df_business.groupBy("stars").count().orderBy("stars").collect():
    print(f"      ⭐ {row['stars']}: {row['count']:>8,}  ({row['count'] / N_BUS * 100:5.2f}%)")

# ══════════════════════════════════════════════════════════════════════
# B) df_attributes — 24 BOOLEAN, 1 INT, 7 ENUM single-value attributes
#    BOOLEAN: decoded values are TRUE/FALSE/NULL by construction, so the closed set is
#    verified on the RAW literal ('True' / 'False' / 'None') — any other token would
#    silently turn into NULL during decoding.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─' * 60}\n📌 B) df_attributes — single-value attribute domains")

PRICE_DOMAIN = [1, 2, 3, 4]
agg_exprs = []
for b, s in ATTR_BOOLEAN_MAP.items():
    raw = F.col(f"attributes.{b}")
    agg_exprs += [F.sum(raw.isNotNull().cast("int")).alias(f"{s}__nn"),
                  F.sum((raw.isNotNull() & ~raw.isin("True", "False", "None")).cast("int")).alias(f"{s}__bad")]
for b, s in {**ATTR_INT_MAP, **ATTR_ENUM_MAP}.items():
    raw = F.col(f"attributes.{b}")
    agg_exprs += [F.sum(raw.isNotNull().cast("int")).alias(f"{s}__nn"),
                  F.sum((raw == "None").cast("int")).alias(f"{s}__none")]
raw_attr = df_bronze.agg(*agg_exprs).first()

typed_exprs = []
for s in ATTR_INT_MAP.values():
    typed_exprs += [F.sum((F.col(s).isNotNull() & ~F.col(s).isin(PRICE_DOMAIN)).cast("int")).alias(f"{s}__bad")]
for s in ATTR_ENUM_MAP.values():
    typed_exprs += [F.sum((F.col(s).isNotNull() & ~F.col(s).isin(sorted(ENUM_DOMAINS[s]))).cast("int")).alias(f"{s}__bad")]
typed_attr = df_attributes.agg(*typed_exprs).first()

print("   BOOLEAN attributes — raw literal ∈ {'True', 'False', 'None'}:")
for b, s in ATTR_BOOLEAN_MAP.items():
    dq_add(findings_621, "df_attributes", s, "raw ∈ {'True','False','None'}", raw_attr[f"{s}__nn"], raw_attr[f"{s}__bad"],
           examples=dq_examples(df_bronze, F.col(f"attributes.{b}"), ~F.col(f"attributes.{b}").isin("True", "False", "None")) if raw_attr[f"{s}__bad"] else "")
print("\n   INT / ENUM attributes — decoded value ∈ contract domain:")
for b, s in ATTR_INT_MAP.items():
    dq_add(findings_621, "df_attributes", s, "∈ {1, 2, 3, 4} ($ … $$$$)", raw_attr[f"{s}__nn"] - raw_attr[f"{s}__none"], typed_attr[f"{s}__bad"],
           examples=dq_examples(df_attributes, s, ~F.col(s).isin(PRICE_DOMAIN)) if typed_attr[f"{s}__bad"] else "",
           note=f"'None' literal: {raw_attr[f'{s}__none']} (DQ-BUSINESS-002)")
for b, s in ATTR_ENUM_MAP.items():
    dom = sorted(ENUM_DOMAINS[s])
    dq_add(findings_621, "df_attributes", s, "∈ {" + ", ".join(dom) + "}", raw_attr[f"{s}__nn"] - raw_attr[f"{s}__none"], typed_attr[f"{s}__bad"],
           examples=dq_examples(df_attributes, s, ~F.col(s).isin(dom)) if typed_attr[f"{s}__bad"] else "",
           note=f"'None' literal: {raw_attr[f'{s}__none']} (DQ-BUSINESS-002); u'x' vs 'x' quoting = DQ-BUSINESS-003")

# ══════════════════════════════════════════════════════════════════════
# C) 7 nested dict attributes (level-2 domain)
#    key   : must belong to the DICT_KEY_CONTRACT key set of that attribute
#    value : RAW token after ':' must be True / False / None (checked on bronze text,
#            because from_json would turn any other token into NULL)
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─' * 60}\n📌 C) Nested dict attributes — key set & value domain")
DICT_FRAMES = {"Ambience": df_ambience, "BestNights": df_best_nights, "BusinessParking": df_business_parking,
               "DietaryRestrictions": df_dietary_restrictions, "GoodForMeal": df_good_for_meal,
               "HairSpecializesIn": df_hair_specializes_in, "Music": df_music}
DICT_FRAME_NAMES = {"Ambience": "df_ambience", "BestNights": "df_best_nights", "BusinessParking": "df_business_parking",
                    "DietaryRestrictions": "df_dietary_restrictions", "GoodForMeal": "df_good_for_meal",
                    "HairSpecializesIn": "df_hair_specializes_in", "Music": "df_music"}
PAIR_RE = r"(u?'[^']+'\s*:\s*[^,}]+)"
for a, (tbl, keymap) in DICT_TABLE_MAP.items():
    fr = DICT_FRAMES[a]
    keys = list(keymap.keys())
    k = fr.agg(F.count("*").alias("n"), F.sum((~F.col("key").isin(keys)).cast("int")).alias("bad")).first()
    raw_tokens = (df_bronze.where(F.col(f"attributes.{a}").rlike(r"^\{.*\}$"))
                  .select(F.explode(F.regexp_extract_all(F.col(f"attributes.{a}"), F.lit(PAIR_RE), F.lit(1))).alias("pair"))
                  .select(F.trim(F.regexp_extract("pair", r":\s*(.*)$", 1)).alias("tok")))
    v = raw_tokens.agg(F.count("*").alias("n"), F.sum((~F.col("tok").isin("True", "False", "None")).cast("int")).alias("bad")).first()
    _name = DICT_FRAME_NAMES[a]
    dq_add(findings_621, _name, f"{a}.key", f"∈ {len(keys)} contract keys", k["n"], k["bad"],
           examples=dq_examples(fr, "key", ~F.col("key").isin(keys)) if k["bad"] else "")
    dq_add(findings_621, _name, f"{a}.value (raw token)", "∈ {True, False, None}", v["n"], v["bad"],
           examples=dq_examples(raw_tokens, "tok", ~F.col("tok").isin("True", "False", "None")) if v["bad"] else "")

# ══════════════════════════════════════════════════════════════════════
# D) df_categories — OPEN vocabulary (Yelp taxonomy not shipped with the dataset)
#    Not a pass/fail domain: profile the vocabulary and surface long-tail labels
#    that a reviewer should confirm against the official Yelp category list.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─' * 60}\n📌 D) df_categories — category vocabulary profile (open domain)")
cat_freq = df_categories.groupBy("category_name").agg(F.countDistinct("business_id").alias("businesses"))
c = cat_freq.agg(F.count("*").alias("vocab"), F.sum((F.col("businesses") == 1).cast("int")).alias("singletons")).first()
print(f"   Vocabulary size: {c['vocab']:,} distinct labels | used by a single business: {c['singletons']:,}")
print("   Top 10 labels:", ", ".join(f"{x['category_name']} ({x['businesses']:,})"
                                   for x in cat_freq.orderBy(F.desc("businesses")).limit(10).collect()))
dq_add(findings_621, "df_categories", "category_name (long tail)", "label used by ≥ 2 businesses", c["vocab"], c["singletons"],
       status="INFO" if c["singletons"] else "PASS",
       examples=", ".join(repr(x[0]) for x in cat_freq.where("businesses = 1").orderBy("category_name").limit(8).collect()),
       note="open taxonomy — confirm singletons against the official Yelp category list")

print("\n   ℹ️  df_hours has no closed domain (time ranges) → covered by range (6.2.2) / pattern (6.2.3) checks.")

# ══ SUMMARY ══
findings_621_df = dq_summary(findings_621, "6.2.1 Allowed Value Conformance")
print("ALLOWED VALUE CONFORMANCE CHECK COMPLETE")
print("=" * 80)

---
#### 🚨 DQ-BUSINESS-009 — Non-Existent State Code `XMS` (UK Business in a US/CA Dataset)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-009` |
| **Severity** | FINDING (domain violation, record outside dataset scope) |
| **Detected In** | Phase 6.2 — Domain & Range Checks → **6.2.1 Allowed Value Conformance** (Cell 43): `state` not in the USPS / Canadian jurisdiction list |
| **Also Confirmed In** | 6.2.3 Pattern Conformance (Cell 49): `state` fails `^[A-Z]{2}$`, `postal_code` classified MALFORMED. 6.2.4 Impossible Value Detection (Cell 57): non-existent jurisdiction + foreign (UK-style) postcode |
| **Affected Columns** | `state`, `postal_code`, `address`, `city` (and, by contradiction, `latitude` / `longitude`) |
| **Total Affected Rows** | **1** out of 150,346 businesses (0.001%) |
| **Invalid Value** | `Pumpernickles Catering`, `95a Mount Pleasant`, `Liverpool`, state `XMS`, postal `L3 5TB`, coordinates `38.673, -90.423` |
| **Domain Rule** | `state` must be a real US state / territory or Canadian province code, and the business must lie inside the dataset's 11 metro areas |

**Description:** `XMS` is not a USPS or Canada Post jurisdiction code. Read together with the rest of the record (Liverpool, `L3 5TB`, "95a Mount Pleasant"), it is a UK business in Merseyside: the state field holds a county code and the postal code is a valid UK postcode. The coordinates, however, point to the St. Louis metro. The address fields and the coordinates contradict each other, so at least one of them is wrong, and nothing in the record tells us which. The record also falls outside the dataset's US/Canada scope. It is harmless in volume, but it would create a phantom `XMS` member in any state-level dimension and break a FK/CHECK constraint on a state lookup table.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Quarantine the business** (`quarantine.yelp_business_out_of_scope`, with reason `NON_EXISTENT_STATE`), and exclude it and its child rows from Silver | The record is internally contradictory and out of scope. Quarantine keeps it auditable without letting a fake jurisdiction into the model. |
| 2 | **Re-assign `state = 'MO'` from the coordinates** | Fits the geo dimension, but it trusts the coordinates over four consistent address fields and the city would still read "Liverpool". |
| 3 | **Keep the row, set `state` to NULL** | Satisfies the domain check, but keeps a UK address inside a US/CA dataset and hides the problem from consumers. |
| 4 | **Keep as-is** | No effort, but the invalid code flows into every state-level aggregate and violates the domain. |

**Best Logical Solution:** **Option 1: quarantine.** Correcting any field would mean guessing which half of a self-contradictory record is true. One business is a negligible loss, and quarantine keeps the evidence for audit.

---

---
#### 🚨 DQ-BUSINESS-010 — Real State Codes Outside the Yelp Footprint (Mislabeled or Out-of-Scope Businesses)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-010` |
| **Severity** | FINDING (domain / accuracy, record-level inconsistency) |
| **Detected In** | Phase 6.2 — Domain & Range Checks → **6.2.1 Allowed Value Conformance** (Cell 43): INFO check "state ∈ the 14 states/provinces of the 11 Yelp metros" |
| **Also Confirmed In** | 6.2.2 Range Conformance (Cell 46): 1 of these businesses (Bennington, VT) also lies > 100 km from every metro (see DQ-BUSINESS-011). The state-vs-coordinates relation is a cross-column check and should be re-tested in 6.6 |
| **Affected Columns** | `state` (and `city`, `address`, `postal_code` vs `latitude` / `longitude`) |
| **Total Affected Rows** | **20** out of 150,346 businesses (0.013%): TX 4, CO 3, WA 2, HI 2, MA 2, NC 1, UT 1, MT 1, MI 1, SD 1, VI 1, VT 1 |
| **Invalid Value** | Three patterns (see below), e.g. `K Soho Salon, Tampa, SD, 33606` or `Wildlife Wonders, Austin, TX` placed at Tampa coordinates |
| **Domain Rule** | The Yelp Open Dataset covers 11 metros in 14 states/provinces (PA, NJ, DE, FL, IN, TN, AZ, LA, NV, AB, CA, ID, MO, IL). A valid but out-of-footprint state must be explained by the rest of the record |

**Description:** The 20 codes are real jurisdictions, so a plain domain check passes them. Looking at each full record shows three different problems:

| Pattern | Rows | Examples | Meaning |
| --- | --- | --- | --- |
| **(a) Only the state is wrong** (city, ZIP and coordinates agree on a footprint metro) | 6 | `Tampa, SD, 33606` · `Mount Juliet, TX, 37122` · `Nashville, TX, 37217` · `Tucson, HI, 85641` · `Eagle, UT, 83616` (Eagle ID) · `Boston, MA, 19103` (Philadelphia ZIP + coords) | data-entry error in `state` |
| **(b) Address is out of footprint, coordinates are inside a metro** | 13 | `Houston, TX` at Nashville coords · `Vail, CO` at Tucson · `Spokane, WA` at Wilmington DE · `Kalispell, MT` at Tampa · `Tortola, VI` at Tampa | address and coordinates contradict, so one of them is wrong |
| **(c) Fully outside the footprint** | 1 | `All Service Citgo, Bennington, VT` (coords also in Vermont) | consistent but out of dataset scope |

Left as-is, pattern (a) assigns businesses to the wrong state in every state-level aggregate, and patterns (b) and (c) bring addresses the dataset doesn't cover into geo analytics.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Rule-based by pattern**: (a) correct `state` from the USPS ZIP-prefix → state mapping when ZIP, city and coordinates agree; (b) and (c) quarantine with reasons `ADDRESS_COORD_CONFLICT` / `OUT_OF_FOOTPRINT` | Fixes only what three independent fields agree on, and isolates records where no single truth can be established. |
| 2 | **Re-derive `state` for all 20 from the coordinates (reverse geocoding)** | Uniform, but for pattern (b) it would label a Houston address as Tennessee. It trusts the field we can't verify. |
| 3 | **Quarantine all 20** | Simple and safe, but it throws away 6 businesses whose only defect is a typo with a provable correction. |
| 4 | **Keep as-is** | Valid codes, but wrong state attribution and contradictory geography remain. |

**Best Logical Solution:** **Option 1: correct pattern (a) from the ZIP prefix, quarantine patterns (b) and (c).** A correction is applied only when ZIP, city and coordinates agree on the answer. Everything else is contradictory or out of scope, so it is isolated rather than guessed.

---

In [0]:
# 6.2.2) Range Conformance (single column)  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Numeric / temporal / length values must fall inside physically or business-valid bounds.
#   A) df_business   : stars [1, 5], review_count [0, ∞) + dataset floor, latitude / longitude at 3 levels
#                      (global → North America → Yelp metro footprint)
#   B) df_hours      : clock components, daily opening duration
#   C) df_categories : labels per business, label length
#   D) 7 nested dicts: entries per business ≤ contract key count
#   E) text lengths  : name / address / city
# Requires dq_add / dq_examples / dq_summary from 6.2.1.
import math
import pandas as pd
from pyspark.sql import functions as F

print("=" * 80)
print("6.2.2) RANGE CONFORMANCE  —  Business DataFrames (TE.2)")
print("    Checking numeric / temporal / length values fall within valid bounds")
print("=" * 80)

N_BUS = df_business.count()
findings_622 = []

# ══════════════════════════════════════════════════════════════════════
# A) df_business — numeric ranges
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─' * 60}\n📌 A) df_business — numeric ranges")

# 11 Yelp Open Dataset metro centres (lat, lon)
YELP_METROS = {"Philadelphia": (39.9526, -75.1652), "Tampa": (27.9506, -82.4572), "Indianapolis": (39.7684, -86.1581),
               "Nashville": (36.1627, -86.7816), "Tucson": (32.2226, -110.9747), "New Orleans": (29.9511, -90.0715),
               "Reno": (39.5296, -119.8138), "Edmonton": (53.5461, -113.4938), "Santa Barbara": (34.4208, -119.6982),
               "Boise": (43.6150, -116.2023), "St. Louis": (38.6270, -90.1994)}
METRO_RADIUS_KM = 100      # p99.9 of distance-to-nearest-metro is ~60 km; 100 km leaves a wide margin

def _haversine_km(lat0, lon0):
    dlat = F.radians(F.col("latitude") - lat0) / 2
    dlon = F.radians(F.col("longitude") - lon0) / 2
    a = F.pow(F.sin(dlat), 2) + math.cos(math.radians(lat0)) * F.cos(F.radians("latitude")) * F.pow(F.sin(dlon), 2)
    return F.lit(2 * 6371.0) * F.asin(F.sqrt(a))

_dist = [_haversine_km(*xy) for xy in YELP_METROS.values()]
_names = list(YELP_METROS.keys())
geo = df_business.select("business_id", "name", "city", "state", "latitude", "longitude", "stars", "review_count",
                         F.least(*_dist).alias("km_to_metro"),
                         F.element_at(F.array(*[F.lit(n) for n in _names]),
                                      F.array_position(F.array(*_dist), F.least(*_dist)).cast("int")).alias("nearest_metro"))

r = geo.agg(
    F.sum(F.col("stars").isNotNull().cast("int")).alias("stars_nn"),
    F.sum((~F.col("stars").between(1, 5)).cast("int")).alias("stars_oor"),
    F.sum(F.col("review_count").isNotNull().cast("int")).alias("rc_nn"),
    F.sum((F.col("review_count") < 0).cast("int")).alias("rc_neg"),
    F.sum((F.col("review_count") < 5).cast("int")).alias("rc_lt5"),
    F.min("review_count").alias("rc_min"), F.max("review_count").alias("rc_max"),
    F.expr("percentile(review_count, array(0.25, 0.5, 0.75, 0.99, 0.999))").alias("rc_q"),
    F.sum((F.col("latitude").isNotNull() & F.col("longitude").isNotNull()).cast("int")).alias("geo_nn"),
    F.sum((~F.col("latitude").between(-90, 90)).cast("int")).alias("lat_oor"),
    F.sum((~F.col("longitude").between(-180, 180)).cast("int")).alias("lon_oor"),
    F.sum((~(F.col("latitude").between(14, 72) & F.col("longitude").between(-170, -50))).cast("int")).alias("na_oor"),
    F.sum((F.col("km_to_metro") > METRO_RADIUS_KM).cast("int")).alias("far"),
    F.min("latitude").alias("lat_min"), F.max("latitude").alias("lat_max"),
    F.min("longitude").alias("lon_min"), F.max("longitude").alias("lon_max"),
    F.expr("percentile(km_to_metro, array(0.5, 0.99, 0.999))").alias("km_q"),
).first()

dq_add(findings_622, "df_business", "stars", "[1.0 – 5.0]", r["stars_nn"], r["stars_oor"])
dq_add(findings_622, "df_business", "review_count", "[≥ 0]", r["rc_nn"], r["rc_neg"])
dq_add(findings_622, "df_business", "review_count (dataset floor)", "[≥ 5] — Yelp Open Dataset curation floor", r["rc_nn"], r["rc_lt5"],
       status="INFO" if r["rc_lt5"] else "PASS", note=f"observed min = {r['rc_min']}")
q = r["rc_q"]
log_iqr_upper = math.exp(math.log(q[2]) + 1.5 * (math.log(q[2]) - math.log(q[0])))   # IQR fence on log scale (count data)
rc_extreme = df_business.where(F.col("review_count") > log_iqr_upper).count()
print(f"      review_count: min={r['rc_min']}  Q1={q[0]:.0f}  median={q[1]:.0f}  Q3={q[2]:.0f}  p99={q[3]:.0f}  p99.9={q[4]:.0f}  max={r['rc_max']:,}")
dq_add(findings_622, "df_business", "review_count (heavy tail)", f"≤ {log_iqr_upper:,.0f} (log-scale IQR fence)", r["rc_nn"], rc_extreme,
       status="INFO" if rc_extreme else "PASS",
       examples=", ".join(f"{x['name']} ({x['city']}) = {x['review_count']:,}" for x in
                          df_business.orderBy(F.desc("review_count")).limit(3).collect()),
       note="popular venues, not errors — consider log transform for analytics")

print(f"\n      latitude  range: {r['lat_min']:.4f} – {r['lat_max']:.4f}   longitude range: {r['lon_min']:.4f} – {r['lon_max']:.4f}")
print(f"      distance to nearest Yelp metro: median={r['km_q'][0]:.1f} km  p99={r['km_q'][1]:.1f} km  p99.9={r['km_q'][2]:.1f} km")
dq_add(findings_622, "df_business", "latitude", "[-90, 90]", r["geo_nn"], r["lat_oor"])
dq_add(findings_622, "df_business", "longitude", "[-180, 180]", r["geo_nn"], r["lon_oor"])
dq_add(findings_622, "df_business", "latitude/longitude (N. America)", "lat [14, 72] & lon [-170, -50]", r["geo_nn"], r["na_oor"])
_far = geo.where(F.col("km_to_metro") > METRO_RADIUS_KM).orderBy(F.desc("km_to_metro"))
dq_add(findings_622, "df_business", "latitude/longitude (Yelp footprint)", f"≤ {METRO_RADIUS_KM} km from one of the 11 Yelp metros",
       r["geo_nn"], r["far"],
       examples="; ".join(f"{x['name']} ({x['city']}, {x['state']}) @ {x['latitude']:.3f},{x['longitude']:.3f} → {x['km_to_metro']:.0f} km from {x['nearest_metro']}"
                          for x in _far.limit(5).collect()),
       note="coordinates outside every metro the dataset covers → likely geocoding error")
if r["far"]:
    display(_far.select("business_id", "name", "city", "state", "latitude", "longitude", F.round("km_to_metro", 1).alias("km_to_metro"), "nearest_metro"))

# ══════════════════════════════════════════════════════════════════════
# B) df_hours — clock components & daily opening duration (long form: 1 row per business-day)
#    Raw ranges 'H:M-H:M' are parsed back to integers so out-of-range components
#    (e.g. 25:00, 9:75) cannot hide behind the zero-padded start/end strings.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─' * 60}\n📌 B) df_hours — clock components & daily opening duration")
H_RE = r"^(\d{1,2}):(\d{1,2})-(\d{1,2}):(\d{1,2})$"
hours_long = (df_hours.select("business_id", F.explode(F.array(*[F.struct(F.lit(d).alias("day"), F.col(HOURS_MAP[d]).alias("raw"))
                                                                for d in DAYS])).alias("h"))
              .select("business_id", "h.day", "h.raw").where(F.col("raw").isNotNull()))
hours_long = hours_long.select("*", *[F.regexp_extract("raw", H_RE, i).cast("int").alias(n)
                                      for i, n in [(1, "sh"), (2, "sm"), (3, "eh"), (4, "em")]])
hours_long = hours_long.withColumn("open_min", F.pmod((F.col("eh") * 60 + F.col("em")) - (F.col("sh") * 60 + F.col("sm")), F.lit(1440)))

h = hours_long.agg(
    F.count("*").alias("n"),
    F.sum((~F.col("sh").between(0, 23) | ~F.col("eh").between(0, 23)).cast("int")).alias("hour_oor"),
    F.sum((~F.col("sm").between(0, 59) | ~F.col("em").between(0, 59)).cast("int")).alias("min_oor"),
    F.sum(((F.col("raw") != "0:0-0:0") & (F.col("open_min") > 0)).cast("int")).alias("n_dur"),
    F.sum(((F.col("raw") != "0:0-0:0") & F.col("open_min").between(1, 30)).cast("int")).alias("short"),
    F.expr("percentile(CASE WHEN raw != '0:0-0:0' AND open_min > 0 THEN open_min END, array(0.01, 0.5, 0.99))").alias("dur_q"),
).first()
print(f"      business-day ranges: {h['n']:,} | open duration (excl. 0:0-0:0 & zero-length): "
      f"p1={h['dur_q'][0] / 60:.1f} h  median={h['dur_q'][1] / 60:.1f} h  p99={h['dur_q'][2] / 60:.1f} h")
dq_add(findings_622, "df_hours", "hours_<day> hour component", "start/end hour ∈ [0, 23]", h["n"], h["hour_oor"])
dq_add(findings_622, "df_hours", "hours_<day> minute component", "start/end minute ∈ [0, 59]", h["n"], h["min_oor"])
dq_add(findings_622, "df_hours", "hours_<day> open duration", "> 30 min for a real opening window", h["n_dur"], h["short"],
       status="INFO" if h["short"] else "PASS",
       examples=", ".join(f"{x['raw']} ({x['count']})" for x in hours_long.where((F.col("raw") != "0:0-0:0") & F.col("open_min").between(1, 30))
                          .groupBy("raw").count().orderBy(F.desc("count")).limit(6).collect()),
       note="implausibly short windows (e.g. 9:0-9:15) — likely entry errors; '0:0-0:0' & zero-length ranges are DQ-BUSINESS-007")

# ══════════════════════════════════════════════════════════════════════
# C) df_categories — labels per business & label length
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─' * 60}\n📌 C) df_categories — labels per business & label length")
MAX_CATS = 20
per_bus = df_categories.groupBy("business_id").agg(F.count("*").alias("n_cat"))
c = per_bus.agg(F.count("*").alias("n"), F.min("n_cat").alias("mn"), F.max("n_cat").alias("mx"),
                F.expr("percentile(n_cat, array(0.5, 0.99))").alias("q"),
                F.sum((F.col("n_cat") > MAX_CATS).cast("int")).alias("many")).first()
l = df_categories.agg(F.count("*").alias("n"), F.min(F.length("category_name")).alias("mn"), F.max(F.length("category_name")).alias("mx"),
                      F.sum((~F.length("category_name").between(2, 60)).cast("int")).alias("bad")).first()
print(f"      labels per business: min={c['mn']}  median={c['q'][0]:.0f}  p99={c['q'][1]:.0f}  max={c['mx']}  "
      f"| label length: {l['mn']}–{l['mx']} chars")
dq_add(findings_622, "df_categories", "labels per business", "[1, ∞) for businesses with categories", c["n"], 0 if c["mn"] >= 1 else c["n"])
dq_add(findings_622, "df_categories", "labels per business (over-tagging)", f"≤ {MAX_CATS} labels", c["n"], c["many"],
       status="INFO" if c["many"] else "PASS",
       examples=", ".join(f"{x['name']} = {x['n_cat']}" for x in per_bus.where(F.col("n_cat") > MAX_CATS)
                          .join(df_business.select("business_id", "name"), "business_id").orderBy(F.desc("n_cat")).limit(4).collect()),
       note="category stuffing dilutes category-level metrics")
dq_add(findings_622, "df_categories", "category_name length", "[2, 60] chars", l["n"], l["bad"])

# ══════════════════════════════════════════════════════════════════════
# D) 7 nested dicts — entries per business must not exceed the contract key count
#    (> key count would mean repeated keys inside one dict string)
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─' * 60}\n📌 D) Nested dicts — entries per business ≤ contract key count")
for a, (tbl, keymap) in DICT_TABLE_MAP.items():
    k_max = len(keymap)
    pb = DICT_FRAMES[a].groupBy("business_id").agg(F.count("*").alias("n"))
    d = pb.agg(F.count("*").alias("bus"), F.sum((F.col("n") > k_max).cast("int")).alias("bad"), F.max("n").alias("mx")).first()
    dq_add(findings_622, DICT_FRAME_NAMES[a], f"{a} entries / business", f"[1, {k_max}] (observed max {d['mx']})", d["bus"], d["bad"])

# ══════════════════════════════════════════════════════════════════════
# E) Identity text lengths (raw, untrimmed)
#    Empty strings ('' address) are a completeness / masked-null issue → reported in 6.3.6 / 6.4,
#    so length bounds here apply to non-empty values only.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─' * 60}\n📌 E) Identity text lengths (non-empty values)")
TEXT_BOUNDS = {"name": (2, 100), "address": (3, 120), "city": (2, 50)}
for cname, (lo, hi) in TEXT_BOUNDS.items():
    t = df_business.where(F.col(cname).isNotNull() & (F.col(cname) != "")).agg(
        F.count("*").alias("n"), F.min(F.length(cname)).alias("mn"), F.max(F.length(cname)).alias("mx"),
        F.sum((~F.length(cname).between(lo, hi)).cast("int")).alias("bad")).first()
    dq_add(findings_622, "df_business", f"{cname} length", f"[{lo}, {hi}] chars (observed {t['mn']}–{t['mx']})", t["n"], t["bad"],
           examples=dq_examples(df_business, cname, (F.col(cname) != "") & ~F.length(cname).between(lo, hi)) if t["bad"] else "")
short_names = df_business.where(F.length("name") <= 2)
n_short = short_names.count()
print(f"      ℹ️  2-char names (valid brands such as 'BP', 'TA', 'Za'): {n_short:,} — "
      + ", ".join(sorted({x[0] for x in short_names.select('name').collect()})))

# ══ SUMMARY ══
findings_622_df = dq_summary(findings_622, "6.2.2 Range Conformance")
print("RANGE CONFORMANCE CHECK COMPLETE")
print("=" * 80)

---
#### 🚨 DQ-BUSINESS-011 — Coordinates Outside Every Yelp Metro (Geocoding Error)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-011` |
| **Severity** | FINDING (accuracy, geolocation) |
| **Detected In** | Phase 6.2 — Domain & Range Checks → **6.2.2 Range Conformance** (Cell 46): `latitude/longitude` more than 100 km (haversine) from all 11 Yelp metro centres |
| **Also Confirmed In** | 6.2.2 global [-90, 90] / [-180, 180] and North-America envelope checks: 0 violations, so these are plausible-looking coordinates in the wrong place, not malformed values. 6.2.4 (Cell 57): 0 Null-Island / zero-axis coordinates |
| **Affected Columns** | `latitude`, `longitude` |
| **Total Affected Rows** | **3** out of 150,346 flagged (0.002%). After review: **1 genuine error**, 1 out-of-scope business, 1 false positive |
| **Invalid Value** | `Broadway Veterinary Hospital, 350 East Linden St, Boise, ID 83706` at `43.994, -92.462` (~Rochester, Minnesota, 626 km from Boise) |
| **Domain Rule** | A business's coordinates must locate its address and fall inside the dataset footprint. Median distance to the nearest metro is 13 km, p99.9 is 60 km |

**Description:** The three businesses beyond the 100 km threshold have different causes:
- **Broadway Veterinary Hospital (Boise, ID):** the address, city, state and ZIP are all consistent Boise values, but the coordinates are in south-east Minnesota. This is a **geocoding error**. Any map, distance or "nearby" analysis puts this vet ~1,000 km away.
- **All Service Citgo (Bennington, VT, 365 km):** the coordinates match its Vermont address. The record is consistent but out of scope, and it is already covered by DQ-BUSINESS-010 pattern (c).
- **Champion Exteriors (Stone Harbor, NJ 08247, 106 km):** Stone Harbor is a real Cape May County shore town in the Philadelphia market. The coordinates are correct, so this is a **false positive** of the 100 km threshold, not a data error.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Fix the genuine error only**: replace the Boise vet's coordinates with the centroid of ZIP 83706 from a reference table (or set them to NULL if none is available); handle Bennington via DQ-BUSINESS-010; keep Stone Harbor unchanged | Treats each case by its real cause. A ZIP centroid is a defensible approximation, and NULL is honest when no reference exists. |
| 2 | **Set coordinates to NULL for all 3** | Simple, but it deletes two sets of correct coordinates. |
| 3 | **Quarantine all 3 businesses** | Over-reaction: two of the three have a valid address and an otherwise complete record. |
| 4 | **Raise the threshold until nothing is flagged** | Hides the real Minnesota geocode, which is 626 km out. The rule should be reviewed per case, not tuned to pass. |

**Best Logical Solution:** **Option 1: re-geocode the Boise vet to its ZIP centroid (NULL if no reference), route Bennington to the DQ-BUSINESS-010 quarantine, and keep Stone Harbor.** Only provably wrong coordinates change, and the threshold stays as a review trigger.

---

---
#### 🚨 DQ-BUSINESS-012 — Implausibly Short Daily Opening Windows (15 / 30 minutes)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-012` |
| **Severity** | WARNING (plausibility, probable data-entry error) |
| **Detected In** | Phase 6.2 — Domain & Range Checks → **6.2.2 Range Conformance** (Cell 46): INFO check "open duration > 30 min for a real opening window" |
| **Also Confirmed In** | 6.2.2 hour / minute component checks and 6.2.3 format checks (Cell 49): 0 violations, so the values are well-formed. 6.2.4 (Cell 57) covers the separate zero-length case (DQ-BUSINESS-007) |
| **Affected Columns** | `hours.<Day>` → `df_hours.hours_<day>`, `hours_<day>_start_time`, `hours_<day>_end_time` |
| **Total Affected Rows** | **321** of 725,615 measurable business-day ranges (0.044%), across **238** businesses. Exactly two durations occur: 15 min (147) and 30 min (174) |
| **Invalid Value** | `Arrow Electric` Saturday `8:0-8:15`, `Melt` Sunday `11:0-11:15`, `Oriental Kitchen` `12:0-12:30` Wednesday–Sunday, `Jaramillo Hair salon` Tuesday `17:30-18:0` |
| **Domain Rule** | A daily opening window for a restaurant, salon or service business is expected to last hours, not minutes. Median duration is ~10 h, p1 is 3 h |

**Description:** All other opening ranges are at least about 3 hours long, so these windows sit far outside the normal distribution. Only two durations occur, exactly 15 and 30 minutes, which points to the Yelp hours picker: the owner selected the same (or the next) time slot in both the "from" and "to" dropdowns. A restaurant open 12:00–12:30 every day is not credible. A few could be genuine (a pick-up slot, a short service window), so the values can't be proven wrong one by one. Used as-is, they drag down average opening hours and "open now" coverage for those businesses.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep the raw range and treat it as low-confidence in analytics** (exclude windows of 30 min or less from duration and "open now" metrics, in the same derived-flag framework as DQ-BUSINESS-007) | Invents nothing and keeps the lineage, while stopping a probable entry error from distorting metrics. |
| 2 | **Set those day values to NULL** | Removes the noise, but it destroys information that is occasionally genuine and can't be told apart from a missing day. |
| 3 | **Replace with the business's modal opening hours from other days** | Plausible-looking output, but fabricated: the owner never reported those hours. |
| 4 | **Keep and use as-is** | No effort, but known-implausible values bias opening-hours analytics. |

**Best Logical Solution:** **Option 1: keep the raw value and exclude it from duration-based metrics.** Without a ground-truth source, the honest choice is to keep what was reported and stop it from skewing aggregates.

---

In [0]:
# 6.2.3) Pattern Conformance  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# String values must match their required structural format.
#   A) business_id  : 22-char Yelp hash in ALL 11 frames (PK side and FK side)
#   B) state        : 2 upper-case letters
#   C) postal_code  : US ZIP (NNNNN) or full Canadian postal code (A1A 1A1, valid letter set)
#   D) hours        : raw 'H:M-H:M', derived 'HH:mm', 15-minute schedule grid
#   E) name / address / city : printable, no invisible chars, city free of embedded state/punctuation
#   F) category_name: clean label token
#   G) attributes   : raw Python-literal encodings per attribute type (pre-decode)
# Casing / whitespace normalisation is profiled in 6.3; this cell checks structure only.
# Requires dq_add / dq_examples / dq_summary from 6.2.1.
import pandas as pd
from pyspark.sql import functions as F

print("=" * 80)
print("6.2.3) PATTERN CONFORMANCE  —  Business DataFrames (TE.2)")
print("    Checking structural format of string-valued columns")
print("=" * 80)

findings_623 = []
INVISIBLE_RE = "[\u200B-\u200D\u2060\uFEFF\u00A0]"     # zero-width space/joiners, word joiner, BOM, NBSP
CONTROL_RE = r"[\x00-\x1F\x7F]"

# ══ A) business_id across every frame ══
print(f"\n{'─' * 60}\n📌 A) business_id — ^[A-Za-z0-9_-]{{22}}$ in all 11 DataFrames")
ID_RE = r"^[A-Za-z0-9_-]{22}$"
for fname, fr in [("df_business", df_business), ("df_attributes", df_attributes), ("df_hours", df_hours),
                  ("df_categories", df_categories)] + [(DICT_FRAME_NAMES[a], DICT_FRAMES[a]) for a in DICT_TABLE_MAP]:
    x = fr.agg(F.count("*").alias("n"), F.sum((F.col("business_id").isNull() | ~F.col("business_id").rlike(ID_RE)).cast("int")).alias("bad")).first()
    dq_add(findings_623, fname, "business_id", "^[A-Za-z0-9_-]{22}$", x["n"], x["bad"],
           examples=dq_examples(fr, "business_id", ~F.col("business_id").rlike(ID_RE)) if x["bad"] else "")

# ══ B) state ══
print(f"\n{'─' * 60}\n📌 B) state — two upper-case letters")
x = df_business.agg(F.count("state").alias("n"), F.sum((~F.col("state").rlike(r"^[A-Z]{2}$")).cast("int")).alias("bad")).first()
dq_add(findings_623, "df_business", "state", "^[A-Z]{2}$", x["n"], x["bad"],
       examples=dq_examples(df_business, "state", ~F.col("state").rlike(r"^[A-Z]{2}$")))

# ══ C) postal_code — classify every value into a format class ══
#   Canada Post: letters D F I O Q U never used; W and Z never used as the first letter.
print(f"\n{'─' * 60}\n📌 C) postal_code — US ZIP or full Canadian postal code")
CA_FULL = r"^[ABCEGHJ-NPRSTVXY][0-9][ABCEGHJ-NPRSTV-Z] [0-9][ABCEGHJ-NPRSTV-Z][0-9]$"
CA_FSA = r"^[ABCEGHJ-NPRSTVXY][0-9][ABCEGHJ-NPRSTV-Z]$"
pc_class = (F.when(F.col("postal_code").isNull(), "NULL")
             .when(F.col("postal_code") == "", "EMPTY_STRING")
             .when(F.col("postal_code").rlike(r"^\d{5}$"), "US_ZIP5")
             .when(F.col("postal_code").rlike(r"^\d{5}-\d{4}$"), "US_ZIP+4")
             .when(F.col("postal_code").rlike(CA_FULL), "CA_FULL")
             .when(F.col("postal_code").rlike(CA_FSA), "CA_FSA_ONLY (truncated)")
             .otherwise("MALFORMED"))
pc = df_business.withColumn("pc_class", pc_class)
pc_dist = pc.groupBy("pc_class").agg(F.count("*").alias("rows"), F.array_sort(F.collect_set("state")).alias("states"),
                                     F.slice(F.collect_set("postal_code"), 1, 5).alias("samples")).orderBy(F.desc("rows"))
display(pc_dist)
pc_counts = {x["pc_class"]: x["rows"] for x in pc_dist.collect()}
n_pc = sum(pc_counts.values())
dq_add(findings_623, "df_business", "postal_code (empty string)", "non-empty postal code", n_pc, pc_counts.get("EMPTY_STRING", 0),
       note="'' masquerading as a value — masked-null handling in 6.3.6")
dq_add(findings_623, "df_business", "postal_code (truncated CA FSA)", "full 'A1A 1A1' (not 3-char FSA only)", n_pc, pc_counts.get("CA_FSA_ONLY (truncated)", 0),
       examples=dq_examples(pc, "postal_code", F.col("pc_class").startswith("CA_FSA")),
       note="Edmonton codes cut to the Forward Sortation Area — loses delivery-unit precision")
dq_add(findings_623, "df_business", "postal_code (malformed)", "US ZIP5 / ZIP+4 / CA_FULL", n_pc, pc_counts.get("MALFORMED", 0),
       examples=dq_examples(pc, "postal_code", F.col("pc_class") == "MALFORMED"))

# ══ D) hours — raw & derived formats, schedule grid ══
print(f"\n{'─' * 60}\n📌 D) hours — raw 'H:M-H:M', derived 'HH:mm', 15-minute grid")
hl = (df_hours.select("business_id", F.explode(F.array(*[F.struct(F.lit(d).alias("day"), F.col(HOURS_MAP[d]).alias("raw"),
                                                                  F.col(f"{HOURS_MAP[d]}_start_time").alias("st"),
                                                                  F.col(f"{HOURS_MAP[d]}_end_time").alias("et")) for d in DAYS])).alias("h"))
      .select("business_id", "h.*").where(F.col("raw").isNotNull()))
HHMM = r"^([01][0-9]|2[0-3]):[0-5][0-9]$"
x = hl.agg(F.count("*").alias("n"),
           F.sum((~F.col("raw").rlike(r"^\d{1,2}:\d{1,2}-\d{1,2}:\d{1,2}$")).cast("int")).alias("raw_bad"),
           F.sum((~F.col("st").rlike(HHMM) | ~F.col("et").rlike(HHMM) | F.col("st").isNull() | F.col("et").isNull()).cast("int")).alias("hhmm_bad"),
           F.sum((F.col("raw").rlike(r"(^|[-:])\d(?=$|[-:])") ).cast("int")).alias("unpadded"),
           F.sum((~F.substring("st", 4, 2).isin("00", "15", "30", "45") | ~F.substring("et", 4, 2).isin("00", "15", "30", "45")).cast("int")).alias("off_grid")).first()
dq_add(findings_623, "df_hours", "hours_<day> (raw)", "^H{1,2}:M{1,2}-H{1,2}:M{1,2}$", x["n"], x["raw_bad"],
       examples=dq_examples(hl, "raw", ~F.col("raw").rlike(r"^\d{1,2}:\d{1,2}-\d{1,2}:\d{1,2}$")) if x["raw_bad"] else "")
dq_add(findings_623, "df_hours", "hours_<day>_start/end_time", "^HH:mm$ (00:00–23:59)", x["n"], x["hhmm_bad"])
dq_add(findings_623, "df_hours", "hours_<day> (raw, non zero-padded)", "zero-padded HH:mm in source", x["n"], x["unpadded"], status="INFO",
       examples="'8:0-17:30', '0:0-0:0'", note="source convention (minutes as '0'); TE.1 derives padded HH:mm — informational")
dq_add(findings_623, "df_hours", "hours_<day> (schedule grid)", "minutes ∈ {00, 15, 30, 45}", x["n"], x["off_grid"],
       status="INFO" if x["off_grid"] else "PASS",
       examples=", ".join(f"{r_['raw']} ({r_['count']})" for r_ in hl.where(~F.substring("st", 4, 2).isin("00", "15", "30", "45")
                          | ~F.substring("et", 4, 2).isin("00", "15", "30", "45")).groupBy("raw").count().orderBy(F.desc("count")).limit(6).collect()),
       note="'x:59' style ends are a common 'until midnight' idiom; odd minutes (x:01, x:29) suggest typos")

# ══ E) name / address / city ══
print(f"\n{'─' * 60}\n📌 E) name / address / city — printable structure")
x = df_business.agg(
    F.sum(F.col("name").rlike(CONTROL_RE).cast("int")).alias("name_ctrl"),
    F.sum(F.col("name").rlike(INVISIBLE_RE).cast("int")).alias("name_inv"),
    F.sum((~F.col("name").rlike("[A-Za-z0-9]")).cast("int")).alias("name_noalnum"),
    F.sum(F.col("name").rlike("[^\\x00-\\x7F]").cast("int")).alias("name_nonascii"),
    F.sum((F.col("address") != "").cast("int")).alias("addr_n"),
    F.sum(F.col("address").rlike(CONTROL_RE).cast("int")).alias("addr_ctrl"),
    F.sum(F.col("address").rlike(INVISIBLE_RE).cast("int")).alias("addr_inv"),
    F.sum(((F.col("address") != "") & ~F.col("address").rlike("[A-Za-z]")).cast("int")).alias("addr_noalpha"),
    F.sum(((F.col("address") != "") & ~F.col("address").rlike(r"\d")).cast("int")).alias("addr_nonum"),
    F.sum(F.col("city").rlike(INVISIBLE_RE).cast("int")).alias("city_inv"),
    F.sum(F.col("city").rlike(r"[,;]\s*$").cast("int")).alias("city_trail"),
    F.sum(F.col("city").rlike(r",\s*\S").cast("int")).alias("city_suffix"),
    F.sum(F.col("city").rlike(r"\(.*\)").cast("int")).alias("city_paren"),
    F.sum(F.col("city").rlike(r"\d").cast("int")).alias("city_digit"),
).first()
N_BUS = df_business.count()
dq_add(findings_623, "df_business", "name (control chars)", "no \\x00–\\x1F, \\x7F", N_BUS, x["name_ctrl"])
dq_add(findings_623, "df_business", "name (invisible chars)", "no zero-width / NBSP / BOM", N_BUS, x["name_inv"],
       examples=dq_examples(df_business, "name", F.col("name").rlike(INVISIBLE_RE)) if x["name_inv"] else "")
dq_add(findings_623, "df_business", "name (no alphanumeric)", "contains ≥ 1 letter/digit", N_BUS, x["name_noalnum"])
dq_add(findings_623, "df_business", "name (non-ASCII)", "ASCII only", N_BUS, x["name_nonascii"], status="INFO",
       examples=dq_examples(df_business, "name", F.col("name").rlike("[^\\x00-\\x7F]"), 6),
       note="mostly legitimate accents/curly quotes/emoji — keep UTF-8; encoding review in 6.3.5")
dq_add(findings_623, "df_business", "address (control chars)", "no \\x00–\\x1F, \\x7F", x["addr_n"], x["addr_ctrl"])
dq_add(findings_623, "df_business", "address (invisible chars)", "no zero-width / NBSP / BOM", x["addr_n"], x["addr_inv"],
       examples=dq_examples(df_business, "address", F.col("address").rlike(INVISIBLE_RE)) if x["addr_inv"] else "")
dq_add(findings_623, "df_business", "address (no letters)", "street name present (≥ 1 letter)", x["addr_n"], x["addr_noalpha"],
       examples=dq_examples(df_business, "address", (F.col("address") != "") & ~F.col("address").rlike("[A-Za-z]")))
dq_add(findings_623, "df_business", "address (no house number)", "street address starts with a number", x["addr_n"], x["addr_nonum"],
       status="INFO", examples=dq_examples(df_business, "address", (F.col("address") != "") & ~F.col("address").rlike(r"\d"), 6),
       note="landmarks / malls / intersections — not geocodable to a rooftop")
dq_add(findings_623, "df_business", "city (invisible chars)", "no zero-width / NBSP / BOM", N_BUS, x["city_inv"],
       examples=", ".join(repr(r_[0]) for r_ in df_business.where(F.col("city").rlike(INVISIBLE_RE)).select("city").distinct().limit(6).collect()),
       note="leading U+200B makes '\u200bClayton' ≠ 'Clayton' in GROUP BY / joins")
dq_add(findings_623, "df_business", "city (trailing punctuation)", "no trailing ',' or ';'", N_BUS, x["city_trail"],
       examples=dq_examples(df_business, "city", F.col("city").rlike(r"[,;]\s*$")))
dq_add(findings_623, "df_business", "city (embedded state/region)", "city name only (no ', PA' / ', Arizona')", N_BUS, x["city_suffix"],
       examples=dq_examples(df_business, "city", F.col("city").rlike(r",\s*\S")))
dq_add(findings_623, "df_business", "city (parenthetical qualifier)", "no '(...)' neighbourhood qualifiers", N_BUS, x["city_paren"],
       examples=dq_examples(df_business, "city", F.col("city").rlike(r"\(.*\)")))
dq_add(findings_623, "df_business", "city (digits)", "no digits in a city name", N_BUS, x["city_digit"])

# ══ F) category_name token ══
print(f"\n{'─' * 60}\n📌 F) category_name — clean label token")
CAT_RE = r"^\S(.*\S)?$"
x = df_categories.agg(F.count("*").alias("n"),
                      F.sum((~F.col("category_name").rlike(CAT_RE)).cast("int")).alias("ws"),
                      F.sum(F.col("category_name").rlike(r"[,;|]").cast("int")).alias("delim"),
                      F.sum((~F.col("category_name").rlike(r"^[A-Z0-9]")).cast("int")).alias("case"),
                      F.sum(F.col("category_name").rlike(INVISIBLE_RE + "|" + CONTROL_RE).cast("int")).alias("inv")).first()
dq_add(findings_623, "df_categories", "category_name (edge whitespace)", "no leading/trailing whitespace, non-empty", x["n"], x["ws"])
dq_add(findings_623, "df_categories", "category_name (residual delimiter)", "no ',', ';', '|' inside a token", x["n"], x["delim"])
dq_add(findings_623, "df_categories", "category_name (initial capital)", "starts with upper-case letter or digit", x["n"], x["case"],
       examples=dq_examples(df_categories, "category_name", ~F.col("category_name").rlike(r"^[A-Z0-9]")) if x["case"] else "",
       note="'& Probates' is a fragment of 'Wills, Trusts, & Probates' — see the fused-label check below")

# Fused-label detection (generic): a Yelp label that itself contains ', ' is split by the list delimiter into
# fragments that ALWAYS appear together, adjacent and in the same order. A pair (A, B) is a split label when
# every occurrence of A is immediately followed by B and every occurrence of B is immediately preceded by A.
from pyspark.sql import Window as W
_w = W.partitionBy("business_id").orderBy("category_position")
_pairs = (df_categories.withColumn("nxt", F.lead("category_name").over(_w)).where(F.col("nxt").isNotNull())
          .groupBy("category_name", "nxt").agg(F.count("*").alias("n_pair")))
_tok = df_categories.groupBy("category_name").agg(F.count("*").alias("n_tok"))
_fused = (_pairs.join(_tok, "category_name")
          .join(_tok.withColumnRenamed("category_name", "nxt").withColumnRenamed("n_tok", "n_nxt"), "nxt")
          .where((F.col("n_pair") == F.col("n_tok")) & (F.col("n_pair") == F.col("n_nxt")) & (F.col("n_tok") >= 5))
          .select("category_name", "nxt", "n_pair").collect())
# chain adjacent pairs into full labels (e.g. Wills → Trusts → & Probates)
_nxt = {r_["category_name"]: r_["nxt"] for r_ in _fused}
_heads = [a for a in _nxt if a not in set(_nxt.values())]
fused_labels = {}
for hd in _heads:
    parts = [hd]
    while parts[-1] in _nxt:
        parts.append(_nxt[parts[-1]])
    fused_labels[", ".join(parts)] = parts
fused_tokens = sorted({t for p in fused_labels.values() for t in p})
_fused_bus = df_categories.where(F.col("category_name").isin(fused_tokens)).select("business_id").distinct().count()
_fused_rows = df_categories.where(F.col("category_name").isin(fused_tokens)).count()
print(f"      Fused labels reconstructed: {len(fused_labels)} → " + "; ".join(
    f"'{lbl}' = {len(p)} tokens" for lbl, p in fused_labels.items()))
dq_add(findings_623, "df_categories", "category_name (label split by ', ' delimiter)", "each token is a complete Yelp label",
       df_categories.select("business_id").distinct().count(), _fused_bus,
       examples=", ".join(f"'{lbl}'" for lbl in fused_labels),
       note=f"{_fused_rows:,} fragment rows; fragments ('Beer', 'Used', 'Mags', 'Wills', ...) are not real categories")
dq_add(findings_623, "df_categories", "category_name (hidden chars)", "no control / zero-width chars", x["n"], x["inv"])

# ══ G) attributes — raw encoding pattern per attribute type (bronze, pre-decode) ══
print(f"\n{'─' * 60}\n📌 G) attributes — raw Python-literal encoding per type")
ATTR_PATTERNS = {"BOOLEAN": (ATTR_BOOLEAN_MAP, r"^(True|False|None)$"),
                 "INT":     (ATTR_INT_MAP,     r"^([0-9]+|None)$"),
                 "ENUM":    (ATTR_ENUM_MAP,    r"^(u?'[a-z0-9_]+'|None)$"),
                 "DICT":    ({a: a for a in DICT_TABLE_MAP}, r"^(\{.*\}|None)$")}
exprs = []
for kind, (amap, rx) in ATTR_PATTERNS.items():
    for b in amap:
        c_ = F.col(f"attributes.{b}")
        exprs += [F.sum(c_.isNotNull().cast("int")).alias(f"{b}__n"), F.sum((c_.isNotNull() & ~c_.rlike(rx)).cast("int")).alias(f"{b}__bad")]
raw = df_bronze.agg(*exprs).first()
for kind, (amap, rx) in ATTR_PATTERNS.items():
    n_tot = sum(raw[f"{b}__n"] for b in amap); bad_tot = sum(raw[f"{b}__bad"] for b in amap)
    bad_cols = [b for b in amap if raw[f"{b}__bad"]]
    dq_add(findings_623, "df_bronze.attributes", f"{len(amap)} {kind} attributes", rx, n_tot, bad_tot,
           examples=", ".join(f"{b}: {dq_examples(df_bronze, F.col(f'attributes.{b}'), ~F.col(f'attributes.{b}').rlike(rx), 3)}" for b in bad_cols))
print("      ℹ️  u'x' vs 'x' quoting variants inside ENUM values are documented as DQ-BUSINESS-003.")

# ══ SUMMARY ══
findings_623_df = dq_summary(findings_623, "6.2.3 Pattern Conformance")
print("PATTERN CONFORMANCE CHECK COMPLETE")
print("=" * 80)

---
#### 🚨 DQ-BUSINESS-013 — Category Labels Containing `, ` Split into Fake Categories

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-013` |
| **Severity** | FINDING (parsing / delimiter collision, semantic corruption of a dimension) |
| **Detected In** | Phase 6.2 — Domain & Range Checks → **6.2.3 Pattern Conformance** (Cell 49): "category_name (label split by ', ' delimiter)" fused-label detection, plus the "initial capital" check (`'& Probates'`, 38 rows) |
| **Also Confirmed In** | 6.2.1 (Cell 43): the vocabulary of 1,311 labels includes the 10 fragments. TE.2 (Cell 35): `split(categories, ', ')` is the source of the split |
| **Affected Columns** | `categories` → `df_categories.category_name` / `yelp_business_categories.category_name` |
| **Total Affected Rows** | **4,150** of 150,243 businesses with categories (2.76%); **9,371** of 668,592 category rows (1.40%) are fragments |
| **Invalid Value** | 4 Yelp labels that contain `, `: `Beer, Wine & Spirits` (2,413 businesses) → `Beer` + `Wine & Spirits`; `Books, Mags, Music & Video` (975) → 3 tokens; `Used, Vintage & Consignment` (753) → 2 tokens; `Wills, Trusts, & Probates` (38) → `Wills` + `Trusts` + `& Probates` |
| **Domain Rule** | Each `category_name` must be one complete Yelp taxonomy label |

**Description:** Bronze `categories` is a `, `-separated list, but four official Yelp labels contain `, ` themselves, so the separator is ambiguous. The split was detected generically: a pair of tokens is a split label when every occurrence of the first is immediately followed by the second, and vice versa. All 10 fragments meet this 100% (none ever appears on its own), which proves they are pieces of one label and not independent categories. The effects:
- 10 fake labels (`Beer`, `Mags`, `Used`, `Wills`, ...) replace 4 real ones in the category dimension.
- "Number of categories per business" is inflated by 1–2 for 4,150 businesses.
- Category searches give wrong results. A liquor store tagged `Beer, Wine & Spirits` shows up under a `Beer` category that doesn't exist on Yelp.
- After the split, `category_position` no longer reflects the label order.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Protect known comma-bearing labels before splitting**: maintain a whitelist (the 4 labels found here, regenerated by the adjacency rule on each load), replace them with a placeholder, split on `, `, then restore | Deterministic and cheap. It fixes the root cause in the parser, the result is lossless, and positions stay consistent. |
| 2 | **Split first, then merge adjacent fragment pairs** using the pair table from 6.2.3 | Same outcome, but it needs a window and a recursive merge for the 3-token labels, and positions have to be renumbered afterwards. |
| 3 | **Tokenize against the official Yelp `categories.json` taxonomy (longest match)** | The most authoritative source, but it adds an external dependency that isn't in the landing zone today. |
| 4 | **Keep the fragments** | No effort, but the category dimension stays corrupted for 2.8% of businesses. |

**Best Logical Solution:** **Option 1: whitelist-protected splitting, with the whitelist re-validated by the adjacency rule (and by the official taxonomy if it is ever ingested).** It removes the ambiguity at the source and loses nothing. Apply the DQ-BUSINESS-005 deduplication after the reassembly.

---

---
#### 🚨 DQ-BUSINESS-014 — Truncated Canadian Postal Codes (FSA Only)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-014` |
| **Severity** | FINDING (pattern / precision loss) |
| **Detected In** | Phase 6.2 — Domain & Range Checks → **6.2.3 Pattern Conformance** (Cell 49): `postal_code` class `CA_FSA_ONLY (truncated)` |
| **Also Confirmed In** | 6.2.4 (Cell 57): none of the 196 contains a letter Canada Post never issues, so the 3-character prefixes are valid Forward Sortation Areas |
| **Affected Columns** | `postal_code` |
| **Total Affected Rows** | **196** of 5,573 Alberta businesses (3.5%), 0.13% of all businesses. 38 distinct FSAs (`T5J`, `T6E`, `T5T`, ...), 7 of them also have an empty `address` |
| **Invalid Value** | `T5J` instead of a full `A1A 1A1` code such as `T5J 1B9` |
| **Domain Rule** | A Canadian postal code is 6 characters, `A1A 1A1` (FSA + Local Delivery Unit) |

**Description:** Every one of the 196 truncated codes is in the Edmonton metro, where 5,356 other businesses have the full code. The 3 characters present are the **Forward Sortation Area**, a real and correct prefix that identifies a district. The 3-character Local Delivery Unit (block or building level) is missing. The value isn't wrong, just imprecise. A strict `A1A 1A1` CHECK constraint would reject these rows, and joins to a full postal-code reference table would fail for them.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep the FSA as-is and document that Canadian codes can be FSA-level** (validate with `^[A-Z]\d[A-Z]( \d[A-Z]\d)?$`) | The value is true and useful for district-level analysis. Nothing is invented or thrown away. |
| 2 | **Complete the code by reverse-geocoding the coordinates** | Restores precision, but needs an external geocoder, and the inferred code might not be the business's mailing code. |
| 3 | **Set to NULL** | Satisfies a strict pattern, but deletes valid district information. |
| 4 | **Pad to 6 characters (`T5J 0A0`)** | Passes the format check with a fake code. |

**Best Logical Solution:** **Option 1: keep the FSA and widen the pattern rule to accept FSA-only codes.** It is accurate partial information, and Option 2 can enrich it later if a geocoder becomes available.

---

---
#### 🚨 DQ-BUSINESS-015 — Empty-String Postal Codes (Masked NULL)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-015` |
| **Severity** | FINDING (masked null / completeness) |
| **Detected In** | Phase 6.2 — Domain & Range Checks → **6.2.3 Pattern Conformance** (Cell 49): `postal_code` class `EMPTY_STRING` |
| **Also Confirmed In** | 4.6 Missing values summary: `postal_code` reports 0 NULLs, because `''` is not counted as missing. Re-test in 6.3.6 Null Representation and 6.4 Completeness |
| **Affected Columns** | `postal_code` |
| **Total Affected Rows** | **73** of 150,346 businesses (0.049%), spread over 11 states. **56** of the 73 also have an empty `address` |
| **Invalid Value** | `''` |
| **Domain Rule** | A missing postal code must be NULL, and a present one must match a postal format |

**Description:** The 73 values are blank strings, not NULLs, so `IS NULL`-based completeness metrics report `postal_code` as 100% populated. Most of them (56) also have a blank address. That is the signature of **service-area businesses** (mobile cleaners, photographers, contractors) that hide their street address on Yelp. They still have coordinates, at a district or city centroid. The value means "unknown", but it is stored as a value.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Convert `''` → NULL in Silver** (`NULLIF(trim(postal_code), '')`) | Standard masked-null handling. Completeness metrics become correct and nothing is invented. |
| 2 | **Infer the ZIP from the coordinates** | Fills the gap, but for service-area businesses the coordinates are a centroid, so the ZIP would be a guess. |
| 3 | **Quarantine the 73 businesses** | Over-reaction: the rest of each record is valid. |
| 4 | **Keep `''`** | Breaks completeness metrics and pattern constraints. |

**Best Logical Solution:** **Option 1: `''` → NULL.** An empty string carries no information beyond "unknown", and NULL is how the model states that. Apply the same rule to the 5,127 empty `address` values (6.3.6).

---

---
#### 🚨 DQ-BUSINESS-016 — Malformed Postal Codes (Typos in US / Canadian Codes)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-016` |
| **Severity** | FINDING (pattern violation, data-entry error) |
| **Detected In** | Phase 6.2 — Domain & Range Checks → **6.2.3 Pattern Conformance** (Cell 49): `postal_code` class `MALFORMED` (5 rows) |
| **Also Confirmed In** | 6.2.4 Impossible Value Detection (Cell 57): `336140` flagged as a 6-digit ZIP that no authority issues. The 5th malformed value, `L3 5TB`, is the UK postcode of DQ-BUSINESS-009 and is handled there |
| **Affected Columns** | `postal_code` |
| **Total Affected Rows** | **4** of 150,346 businesses (excluding the DQ-BUSINESS-009 record) |
| **Invalid Value** | `336140` (Apna Kabab House, Tampa FL: 6 digits); `T6x 2H4` (Oodle Noodle, Edmonton: lower-case letter); `T6T 0G` (Q Nails, Edmonton: missing last digit); `T8A 5WA` (Hunter's Social Grill, Sherwood Park: letter where a digit belongs) |
| **Domain Rule** | US: `^\d{5}(-\d{4})?$`. Canada: `^[A-Z]\d[A-Z] \d[A-Z]\d$` |

**Description:** Four isolated typos, each with a different cause:
- `T6x 2H4` only differs in case, and upper-casing gives a valid code.
- `T6T 0G` and `T8A 5WA` have a valid FSA (`T6T`, `T8A`) but a broken Local Delivery Unit.
- `336140` has one digit too many, and we can't tell which digit is the extra one.

They fail any strict pattern constraint and won't join to a postal reference table.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Per-case deterministic rules**: upper-case Canadian codes (`T6X 2H4`); truncate codes with a valid FSA but a broken LDU to the FSA (`T6T`, `T8A`, consistent with DQ-BUSINESS-014); set non-recoverable US codes (`336140`) to NULL | Each fix is provably lossless or only discards the broken part. No digit is guessed. |
| 2 | **Set all 4 to NULL** | Simple, but it throws away a fully recoverable code and two valid FSAs. |
| 3 | **Correct `336140` → `33614`** (drop the last digit) | Plausible, since 33614 is a Tampa ZIP, but it is a guess: the address (Bruce B Downs Blvd) suggests another ZIP. |
| 4 | **Keep as-is** | Pattern violations reach Silver. |

**Best Logical Solution:** **Option 1: upper-case, truncate to the valid FSA, NULL what can't be recovered.** It keeps every provably correct character and invents none.

---

---
#### 🚨 DQ-BUSINESS-017 — Invisible Unicode Characters (Zero-Width Space / NBSP) in Name, Address and City

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-017` |
| **Severity** | FINDING (encoding / hidden characters) |
| **Detected In** | Phase 6.2 — Domain & Range Checks → **6.2.3 Pattern Conformance** (Cell 49): "invisible chars" checks on `name`, `address`, `city` |
| **Also Confirmed In** | The control-character checks in the same cell return 0, so these are Unicode format/space characters, not ASCII control codes. Related to DQ-BUSINESS-004 (edge whitespace) |
| **Affected Columns** | `name`, `address`, `city` |
| **Total Affected Rows** | **29** values: `name` 15 (5 U+200B zero-width space, 10 U+00A0 no-break space), `address` 10 (1 U+200B, 9 U+00A0), `city` 4 (3 U+200B, 1 U+00A0) |
| **Invalid Value** | `'\u200bClayton'`, `'\u200bLargo'`, `'\u200bLithia'`, `'Saint\xa0Louis'`, `'Club Philadelphia\u200b'`, `'8438 N Lindbergh\xa0Blvd'` |
| **Domain Rule** | Identity text may contain any printable Unicode but no invisible format characters. Spaces must be U+0020 |

**Description:** These characters come from copy-paste out of web pages and word processors. They are invisible when displayed but change the value. `'\u200bClayton'` and `'Clayton'` are two different cities in a `GROUP BY`, a join to a city reference, or a search, and `'Saint\xa0Louis'` doesn't match `'Saint Louis'`. Tokenizers and string-matching (deduplication, entity resolution) treat NBSP and ZWSP inconsistently. This is separate from the 1,381 names with legitimate non-ASCII characters (accents, `’`, `®`), which must be kept.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Targeted normalization**: replace U+00A0 with a regular space, remove U+200B–U+200D, U+2060 and U+FEFF, then trim and collapse repeated spaces | Fixes exactly the invisible characters, keeps legitimate Unicode, and is deterministic and lossless in meaning. |
| 2 | **Apply Unicode NFKC normalization** | Also maps NBSP to a space, but leaves U+200B in place and changes other characters (ligatures, full-width forms), which is more than needed. |
| 3 | **Strip all non-ASCII** | Removes the problem, but destroys 1,381 correct names (`Café`, `Joe’s`). |
| 4 | **Keep as-is** | Silent grouping and join mismatches stay. |

**Best Logical Solution:** **Option 1: targeted removal / replacement of invisible characters.** It is the smallest change that fixes the problem, and it touches nothing a human reader would see.

---

---
#### 🚨 DQ-BUSINESS-018 — Polluted City Values (Trailing Punctuation, Embedded State, Qualifiers, Census Names)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-018` |
| **Severity** | FINDING (pattern / standardization) |
| **Detected In** | Phase 6.2 — Domain & Range Checks → **6.2.3 Pattern Conformance** (Cell 49): `city` trailing punctuation, embedded state/region and parenthetical-qualifier checks |
| **Also Confirmed In** | 6.2.2 Range Conformance (Cell 46): `city` length > 50 for the 2 `Nashville-Davidson metropolitan government (balance)` rows. Case variants (1,416 raw vs 1,238 case-folded distinct cities) belong to 6.3 |
| **Affected Columns** | `city` |
| **Total Affected Rows** | **32** of 150,346 businesses: **17** trailing comma (`Nashville,`, `Saint Louis,` ×2, `Tampa,`, ...), **9** embedded state/region (`Tucson, Arizona` ×2, `Palm Harbor, FL`, `Havertown, PA`, `Mt Laurel Twp, NJ`, `Maryland Height, MO`, `Tampa,Fl`, ...), **6** parenthetical / census names (`Nashville-Davidson metropolitan government (balance)` ×2, `Indianapolis city (balance)`, `Boise (Meridian)`, `Largo (Walsingham)`, `Philadelphia (Northeast Philly)`) |
| **Invalid Value** | See above |
| **Domain Rule** | `city` holds only the locality name. State, punctuation and neighbourhood qualifiers go elsewhere or nowhere |

**Description:** Owners type the city in free text, so it picks up the state (`, PA`), stray punctuation and neighbourhood notes. A few rows carry U.S. Census place names (`... (balance)`), probably from a geocoder. Each variant creates its own city member: Nashville appears as `Nashville`, `Nashville,` and `Nashville-Davidson metropolitan government (balance)`. That fragments city-level counts and breaks joins to a city/metro dimension. `Maryland Height` is also misspelled (Maryland Heights).

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Deterministic rules plus a small curated map**: strip trailing `,;`; remove a `, <state>` / `, <state name>` suffix when it equals the record's state; map census `(balance)` names to the city (`Nashville`, `Indianapolis`); resolve the parenthetical rows and the misspelling through a reviewed mapping table (the ZIP decides `Boise` vs `Meridian`) | Fixes all 32 transparently, with every change traceable to a rule or a reviewed mapping entry. |
| 2 | **Standardize against a gazetteer (USPS / GeoNames) by ZIP** | The most authoritative option, but it adds an external dependency and may rename legitimate local names. |
| 3 | **Keep the raw value and add a cleaned `city_std` column** | Keeps the original, but adds a column the Silver DDL doesn't define (contract drift). |
| 4 | **Keep as-is** | Fragmented city dimension. |

**Best Logical Solution:** **Option 1: rule-based cleanup plus a reviewed mapping for the 7 non-rule cases.** At this volume the curated map is cheap and auditable, and it gives one city member per locality without an external dependency.

---

---
#### 🚨 DQ-BUSINESS-019 — Degenerate Street Addresses (No Street Name)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-019` |
| **Severity** | FINDING (validity, placeholder / misplaced value) |
| **Detected In** | Phase 6.2 — Domain & Range Checks → **6.2.3 Pattern Conformance** (Cell 49): "address (no letters)" |
| **Also Confirmed In** | 6.2.2 Range Conformance (Cell 46): `address` length < 3 for `'.'` and `'54'` |
| **Affected Columns** | `address` |
| **Total Affected Rows** | **11** of 145,219 non-empty addresses (0.008%) |
| **Invalid Value** | **House number only (6):** `3070`, `5348` (Coast Dental), `850`, `22930`, `11964`, `54`. **ZIP+4 typed into the address (4):** `63088-1828`, `08080-4554`, `08096-3558`, `62025-2812`, each equal to the record's own `postal_code` + 4. **Placeholder (1):** `.` (Anthony David Photography, Nashville) |
| **Domain Rule** | A street address must contain a street name. A bare number, a postal code or punctuation is not an address |

**Description:** None of the 11 values can locate the business. The ZIP+4 values are the postal code entered in the wrong field, so they only repeat `postal_code`. The `.` is a placeholder used to get past a required field. The bare house numbers lack the street they belong to. Geocoding, address matching and deduplication by address all fail or mismatch on them. They are different from the 620 addresses without a house number (malls, landmarks, intersections such as `Citizens Bank Park`), which are legitimate and are kept.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Set all 11 to NULL** (missing street address), keep postal code and coordinates | None of them carries usable street information: the ZIP+4 duplicates `postal_code`, and a house number without a street can't be used. NULL states the truth. |
| 2 | **Move the ZIP+4 extension into `postal_code`** (`63088` → `63088-1828`), NULL the others | Keeps the +4 precision, but changes the postal-code format for 4 rows only, which is inconsistent with the other 144,716 five-digit ZIPs. |
| 3 | **Reverse-geocode an address from the coordinates** | Fills the field, but the inferred address may not be the business's real one. |
| 4 | **Keep as-is** | Invalid addresses reach Silver and break geocoding / matching. |

**Best Logical Solution:** **Option 1: NULL all 11.** It turns misleading values into explicit unknowns, loses no information that isn't already in `postal_code` / coordinates, and matches the masked-null handling of DQ-BUSINESS-015.

---

In [0]:
# 6.2.4) Impossible Value Detection  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Values that CANNOT exist in the real world, independent of any configurable bound.
# Single-column view: each rule reads ONE bronze field (a nested dict / time range counts as one field).
#   A) df_business : Null Island (0,0) / zero coordinates, rating off the half-star grid, negative counts
#   B) postal_code : codes that no postal authority issues (US 00000 / 6-digit ZIP, forbidden CA letters, foreign format)
#   C) state       : code of a jurisdiction that does not exist
#   D) df_hours    : clock values ≥ 24 h / ≥ 60 min, zero-length non-midnight windows
#   E) nested dicts: mutually-exclusive / implied flags inside one dict (Music, DietaryRestrictions)
#   F) df_categories: the same category twice for one business
# Ends with the consolidated 6.2 summary. Requires dq_* helpers + findings_621..623 from 6.2.1 – 6.2.3.
import pandas as pd
from pyspark.sql import functions as F

print("=" * 80)
print("6.2.4) IMPOSSIBLE VALUE DETECTION  —  Business DataFrames (TE.2)")
print("    Checking for values that cannot physically / logically exist")
print("=" * 80)

N_BUS = df_business.count()
findings_624 = []

# ══ A) df_business ══
print(f"\n{'─' * 60}\n📌 A) df_business — coordinates, rating grid, counts")
x = df_business.agg(
    F.sum(((F.col("latitude") == 0) & (F.col("longitude") == 0)).cast("int")).alias("null_island"),
    F.sum(((F.col("latitude") == 0) | (F.col("longitude") == 0)).cast("int")).alias("zero_axis"),
    F.sum((F.col("stars") * 2 != F.floor(F.col("stars") * 2)).cast("int")).alias("off_grid"),
    F.sum((F.col("review_count") < 0).cast("int")).alias("neg_rc"),
).first()
dq_add(findings_624, "df_business", "latitude/longitude (Null Island)", "(0, 0) is a geocoding default, not a location", N_BUS, x["null_island"])
dq_add(findings_624, "df_business", "latitude/longitude (zero axis)", "no exact 0 on either axis (equator / prime meridian)", N_BUS, x["zero_axis"])
dq_add(findings_624, "df_business", "stars (half-star grid)", "stars × 2 is an integer", N_BUS, x["off_grid"])
dq_add(findings_624, "df_business", "review_count (negative)", "a count cannot be negative", N_BUS, x["neg_rc"])

# ══ B) postal_code values that are never issued ══
print(f"\n{'─' * 60}\n📌 B) postal_code — codes no postal authority issues")
pcol = F.col("postal_code")
IMPOSSIBLE_PC = {
    "US ZIP 00000":                       pcol == "00000",
    "6+ digit 'ZIP'":                     pcol.rlike(r"^\d{6,}$"),
    "CA code with forbidden letter (D F I O Q U, or W/Z first)":
        pcol.rlike(r"^[A-Za-z]\d[A-Za-z]") & (F.upper(pcol).rlike("[DFIOQU]") | F.upper(pcol).rlike("^[WZ]")),
    "Foreign (UK-style) postcode":        pcol.rlike(r"^[A-Z]{1,2}\d[A-Z\d]? \d[A-Z]{2}$") & ~pcol.rlike(r"^[A-Z]\d[A-Z] "),   # exclude (mistyped) Canadian A1A-prefixed codes
}
x = df_business.agg(*[F.sum(c_.cast("int")).alias(f"c{i}") for i, c_ in enumerate(IMPOSSIBLE_PC.values())]).first()
for i, (lbl, c_) in enumerate(IMPOSSIBLE_PC.items()):
    dq_add(findings_624, "df_business", f"postal_code ({lbl})", "issued postal code", N_BUS, x[f"c{i}"],
           examples="; ".join(f"{r_['postal_code']} → {r_['name']} ({r_['city']}, {r_['state']})" for r_ in
                              df_business.where(c_).select("postal_code", "name", "city", "state").limit(5).collect()) if x[f"c{i}"] else "")

# ══ C) state code of a non-existent jurisdiction (VALID_JURISDICTIONS from 6.2.1) ══
print(f"\n{'─' * 60}\n📌 C) state — jurisdiction must exist")
_bad_state = df_business.where(~F.col("state").isin(list(VALID_JURISDICTIONS)))
dq_add(findings_624, "df_business", "state (non-existent jurisdiction)", "real US state/territory or Canadian province", N_BUS, _bad_state.count(),
       examples="; ".join(f"{r_['state']} → {r_['name']}, {r_['address']}, {r_['city']} {r_['postal_code']}" for r_ in _bad_state.limit(5).collect()),
       note="'XMS' + Liverpool + 'L3 5TB' = a UK (Merseyside) business inside a US/CA dataset")

# ══ D) df_hours ══
print(f"\n{'─' * 60}\n📌 D) df_hours — impossible clock values & zero-length windows")
H_RE = r"^(\d{1,2}):(\d{1,2})-(\d{1,2}):(\d{1,2})$"
hl4 = (df_hours.select("business_id", F.explode(F.array(*[F.struct(F.lit(d).alias("day"), F.col(HOURS_MAP[d]).alias("raw"))
                                                          for d in DAYS])).alias("h"))
       .select("business_id", "h.*").where(F.col("raw").isNotNull())
       .select("*", *[F.regexp_extract("raw", H_RE, i).cast("int").alias(n) for i, n in [(1, "sh"), (2, "sm"), (3, "eh"), (4, "em")]]))
x = hl4.agg(F.count("*").alias("n"),
            F.sum(((F.col("sh") >= 24) | (F.col("eh") >= 24) | (F.col("sm") >= 60) | (F.col("em") >= 60)).cast("int")).alias("clock"),
            F.sum(((F.col("sh") == F.col("eh")) & (F.col("sm") == F.col("em")) & (F.col("raw") != "0:0-0:0")).cast("int")).alias("zero_len")).first()
dq_add(findings_624, "df_hours", "hours_<day> (impossible clock)", "hour < 24 and minute < 60", x["n"], x["clock"])
dq_add(findings_624, "df_hours", "hours_<day> (zero-length window)", "open time ≠ close time (except '0:0-0:0')", x["n"], x["zero_len"],
       examples=", ".join(f"{r_['raw']} ({r_['count']})" for r_ in hl4.where((F.col("sh") == F.col("eh")) & (F.col("sm") == F.col("em")) & (F.col("raw") != "0:0-0:0"))
                          .groupBy("raw").count().orderBy(F.desc("count")).limit(6).collect()),
       note="already documented as DQ-BUSINESS-007 (open 0 minutes, or an unflagged 24 h day)")

# ══ E) nested dicts — contradictory flags inside ONE dict ══
#   Music: no_music = TRUE together with any other music flag = TRUE is a contradiction.
#   DietaryRestrictions: every vegan option is vegetarian → vegan TRUE and vegetarian FALSE is a contradiction.
print(f"\n{'─' * 60}\n📌 E) Nested dicts — contradictory flags within one dict")
def _wide(fr, keys):
    return fr.groupBy("business_id").agg(*[F.max(F.when(F.col("key") == k, F.col("value").cast("int"))).alias(k.replace("-", "_")) for k in keys])

mus = _wide(df_music, list(DICT_TABLE_MAP["Music"][1]))
_other_music = [k for k in DICT_TABLE_MAP["Music"][1] if k != "no_music"]
mus_bad = mus.where((F.col("no_music") == 1) & (F.greatest(*[F.coalesce(F.col(k), F.lit(0)) for k in _other_music]) == 1))
dq_add(findings_624, "df_music", "no_music vs other music flags", "no_music=TRUE ⇒ every other music flag ≠ TRUE", mus.count(), mus_bad.count())

diet = _wide(df_dietary_restrictions, list(DICT_TABLE_MAP["DietaryRestrictions"][1]))
diet_bad = diet.where((F.col("vegan") == 1) & (F.col("vegetarian") == 0))
n_diet_bad = diet_bad.count()
dq_add(findings_624, "df_dietary_restrictions", "vegan vs vegetarian", "vegan=TRUE ⇒ vegetarian ≠ FALSE (vegan ⊂ vegetarian)", diet.count(), n_diet_bad,
       note="tiny base (30 businesses) — owner-entered flags; treat as unreliable rather than correct automatically")
if n_diet_bad:
    display(diet_bad.join(df_business.select("business_id", "name", "city"), "business_id"))

# ══ F) df_categories — same label twice for one business (membership is a set) ══
print(f"\n{'─' * 60}\n📌 F) df_categories — duplicate membership")
dup = df_categories.groupBy("business_id", "category_name").count().where("count > 1")
dq_add(findings_624, "df_categories", "(business_id, category_name) repeated", "a business holds a category at most once",
       df_categories.select("business_id").distinct().count(), dup.select("business_id").distinct().count(),
       examples=dq_examples(dup, "category_name", F.lit(True), 5), note="already documented as DQ-BUSINESS-005")

findings_624_df = dq_summary(findings_624, "6.2.4 Impossible Value Detection")

# ══════════════════════════════════════════════════════════════════════
# 6.2 CONSOLIDATED — every non-PASS check from 6.2.1 – 6.2.4
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 80}\n6.2) DOMAIN & RANGE CHECKS — CONSOLIDATED FINDINGS\n{'=' * 80}")
all_62 = pd.concat([findings_621_df.assign(Check="6.2.1 Allowed Value"), findings_622_df.assign(Check="6.2.2 Range"),
                    findings_623_df.assign(Check="6.2.3 Pattern"), findings_624_df.assign(Check="6.2.4 Impossible")], ignore_index=True)
print(all_62.groupby(["Check", "Status"]).size().unstack(fill_value=0).to_string())
issues_62 = (all_62[all_62["Status"] != "PASS"]
             .sort_values(["Status", "Check", "Violations"], ascending=[True, True, False])
             [["Check", "Status", "DataFrame", "Column", "Rule", "Violations", "Checked", "Pct", "Examples", "Note"]])
display(issues_62)
print(f"\n{len(all_62)} checks in 6.2 | {int((all_62['Status'] == 'FINDING').sum())} FINDING | {int((all_62['Status'] == 'INFO').sum())} INFO")
print("IMPOSSIBLE VALUE DETECTION COMPLETE")
print("=" * 80)

---
#### 🚨 DQ-BUSINESS-020 — Contradictory Dietary Flags (`vegan = True`, `vegetarian = False`)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-020` |
| **Severity** | WARNING (logical implication violated inside one nested attribute) |
| **Detected In** | Phase 6.2 — Domain & Range Checks → **6.2.4 Impossible Value Detection** (Cell 57): "vegan vs vegetarian" (vegan ⊂ vegetarian) |
| **Also Confirmed In** | 6.2.1 (Cell 43): every key and raw token of `DietaryRestrictions` is in the contract domain, so the values are well-formed and only their combination is contradictory. The other intra-dict rule, `Music.no_music` vs other music flags, has 0 violations |
| **Affected Columns** | `attributes.DietaryRestrictions` → `df_dietary_restrictions` (`vegan`, `vegetarian`) |
| **Total Affected Rows** | **3** of 30 businesses with the dict (10%): Rolled Mountain Creamery (Reno; also `dairy-free = True`), Greenheart Bakery (Tucson), The DeLuxe (Reno) |
| **Invalid Value** | `{'vegan': True, ..., 'vegetarian': False}` |
| **Domain Rule** | Vegan food is a strict subset of vegetarian food, so offering vegan options implies offering vegetarian options |

**Description:** Read as a taxonomy, the combination is impossible. Read as owner-entered checkboxes, it is understandable: an owner may tick only the most specific option ("vegan") and leave "vegetarian" unticked, which `False` then records. The base is tiny (30 businesses), so 3 rows make a 10% contradiction rate for this attribute. Any "vegetarian-friendly" metric built on `vegetarian` alone would miss these businesses.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep the flags as reported in Silver, document them as independent self-reported options, and derive `vegetarian_friendly = vegan OR vegetarian` in Gold** | Silver stays faithful to the source. The domain logic is applied once, explicitly, where metrics are built. |
| 2 | **Set `vegetarian = TRUE` when `vegan = TRUE` in Silver** | Logically sound, but it overwrites a reported value with an inferred one, so lineage to the source is lost. |
| 3 | **Set both flags to NULL** | Removes the contradiction by deleting a probably-true signal (the vegan option). |
| 4 | **Drop the 3 dicts** | Loses all 7 dietary flags for 10% of the attribute's population. |

**Best Logical Solution:** **Option 1: keep the reported values in Silver and resolve the implication in a Gold derived measure.** The source is preserved, the business rule is explicit and reusable, and metrics come out correct.

---

> **Cross-references from 6.2.4 (already documented, no new ID):** zero-length opening windows (1,492) → **DQ-BUSINESS-007**. Repeated (`business_id`, `category_name`) (43) → **DQ-BUSINESS-005**. Non-existent state `XMS` and UK postcode → **DQ-BUSINESS-009**. 6-digit ZIP `336140` → **DQ-BUSINESS-016**.
>
> **Reviewed and not raised as errors (INFO only):** `review_count` heavy tail (2,362 above the log-IQR fence: popular venues, which are valid); 56 single-business category labels (an open taxonomy, valid long tail); 12 businesses with more than 20 categories (large multi-service venues, max 36); 1,381 names with legitimate non-ASCII characters; 620 landmark / intersection addresses without a house number; unpadded `H:M` source convention and `x:59` closing times (format conventions, not errors).

---


### 6.3) FORMAT & ENCODING STANDARDIZATION
*Detect and normalize inconsistent representations. Detection only — normalization is a separate remediation step.*

- **Categorical Representation Consistency** — Same concept, different encodings.
  - *Example:* `Male`, `male`, `M`, `1` all representing the same category.
  - *Sub:* Category mapping consistency across related columns — `country_code = 'US'` but `country_name = 'Canada'`.

- **Date Format Consistency** — Dates stored in non-uniform formats.
  - *Example:* `2024-01-15`, `01/15/2024`, `15-Jan-2024` co-existing in one column.

- **Numeric Format Consistency** — Numbers stored with embedded symbols, separators, or units.
  - *Example:* `"$1,200.50"`, `"1.200,50 EUR"`, `"5 kg"` in a numeric column.

- **Boolean Representation Consistency** — Mixed boolean encodings co-existing.
  - *Example:* `Yes/No`, `True/False`, `1/0`, `Y/N` all present in one column.

- **Text Normalization Issues**
  - *Sub:* Leading/trailing whitespace — `" John "` vs `"John"`.
  - *Sub:* Multiple consecutive whitespaces — `"John    Doe"`.
  - *Sub:* Inconsistent capitalization — `"New York"`, `"new york"`, `"NEW YORK"`.
  - *Sub:* Escape/control characters — embedded `\n`, `\t`, `\r`, `\0`, non-printable characters that corrupt parsing.

- **Null Representation Consistency** — Non-standard nulls masquerading as data.
  - *Example:* `"N/A"`, `"NULL"`, `"None"`, `""`, `" "`, `"—"`, `"unknown"`, `"TBD"`, `"."`, `"NIL"`.
  - *Note:* This check only **detects and normalizes**. Counting missingness happens in 6.4.

In [0]:
# 6.3.1) Categorical Representation Consistency  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Same concept, different encodings. Detection only — nothing is changed.
#   A) Spelling variants of ONE real-world category (case / punctuation / whitespace /
#      Saint↔St, Mount↔Mt, Fort↔Ft folded into a key; the modal spelling is the canonical one)
#        A1 city (within state)   A2 name (chain / brand)   A3 category_name
#   B) Same enum value written in several raw encodings (u'x' vs 'x'), and a valid category
#      that differs from the Python null literal only by case (Alcohol 'none' vs None)
#   C) Category-mapping consistency across related columns (the "country_code vs country_name" rule):
#        C1 state ↔ postal_code country     C2 state ↔ USPS ZIP prefix     C3 province ↔ Canada Post letter
#        C4 city ↔ state (minority state)   C5 BYOB ↔ BYOBCorkage ↔ Corkage (one concept, three columns)
# Requires dq_add / dq_examples / dq_summary + US_STATES / CA_PROVINCES from 6.2.1.
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import Window

print("=" * 80)
print("6.3.1) CATEGORICAL REPRESENTATION CONSISTENCY  —  Business DataFrames (TE.2)")
print("    Same concept, different encodings")
print("=" * 80)

N_BUS = df_business.count()
findings_631 = []

def norm_key(c):
    """Spelling-insensitive key: NBSP→space, case, curly quotes, Saint/Mount/Fort, then strip non-alphanumerics."""
    k = F.lower(F.regexp_replace(c, "\u00A0", " "))
    k = F.regexp_replace(k, "[\u2018\u2019`´]", "'")
    for full, abbr in (("saint", "st"), ("mount", "mt"), ("fort", "ft")):
        k = F.regexp_replace(k, rf"\b{full}\b", abbr)
    return F.regexp_replace(k, "[^a-z0-9]", "")

def variant_clusters(frame, col_name, scope=()):
    """Clusters of ≥2 raw spellings sharing one key (within `scope`). canonical = most frequent spelling."""
    g = list(scope)
    sp = (frame.groupBy(*g, col_name).count().withColumn("_k", norm_key(F.col(col_name))).where(F.col("_k") != "")
          .withColumn("_rk", F.row_number().over(Window.partitionBy(*g, "_k").orderBy(F.desc("count"), F.col(col_name)))))
    return (sp.groupBy(*g, "_k")
              .agg(F.count("*").alias("spellings"), F.sum("count").alias("rows"),
                   F.max(F.when(F.col("_rk") == 1, F.col(col_name))).alias("canonical"),
                   F.sum(F.when(F.col("_rk") > 1, F.col("count")).otherwise(0)).alias("non_canonical_rows"),
                   F.sort_array(F.collect_list(F.struct(F.col("_rk"), F.format_string("'%s' (%d)", F.col(col_name), F.col("count")).alias("v")))).alias("_v"))
              .where("spellings > 1")
              .select(*g, "canonical", "spellings", "rows", "non_canonical_rows",
                      F.concat_ws(" | ", F.transform("_v", lambda x: x["v"])).alias("variants"))
              .orderBy(F.desc("non_canonical_rows"), F.desc("rows")))

def cluster_examples(cl, n=4):
    return "; ".join(r_["variants"] for r_ in cl.limit(n).collect())

# ══ A) Spelling variants of the same category ══
print(f"\n{'─' * 60}\n📌 A) Spelling variants of one category (key = case/punctuation/space/Saint-St folded)")
city_cl = variant_clusters(df_business, "city", ["state"])
x = city_cl.agg(F.count("*").alias("n"), F.sum("non_canonical_rows").alias("bad")).first()
dq_add(findings_631, "df_business", "city (spelling variants, per state)", "one spelling per city", N_BUS, x["bad"],
       examples=cluster_examples(city_cl),
       note=f"{x['n']} city clusters with >1 spelling; 'Saint Louis' / 'St. Louis' / 'St Louis' / 'St.Louis' and 'St. Petersburg' / 'Saint Petersburg' split one metro into several groups")
print("   Top city clusters:")
display(city_cl.limit(20))

name_cl = variant_clusters(df_business, "name")
x = name_cl.agg(F.count("*").alias("n"), F.sum("non_canonical_rows").alias("bad")).first()
dq_add(findings_631, "df_business", "name (chain / brand spelling variants)", "one spelling per brand", N_BUS, x["bad"], status="INFO",
       examples=cluster_examples(name_cl),
       note=f"{x['n']} brand clusters (McDonald's / McDonalds / McDonald’s, US Post Office / U.S. Post Office). Owner-entered names — "
            "keep raw `name`, add a normalised brand key in Silver for chain analytics")
print("   Top brand clusters:")
display(name_cl.limit(20))

cat_cl = variant_clusters(df_categories, "category_name")
x = cat_cl.agg(F.count("*").alias("n"), F.sum("non_canonical_rows").alias("bad")).first()
dq_add(findings_631, "df_categories", "category_name (spelling variants)", "one spelling per category label", df_categories.count(), x["bad"],
       examples=cluster_examples(cat_cl), note="Yelp-controlled taxonomy — expected clean")

# ══ B) Enum attributes — same value, several raw encodings ══
print(f"\n{'─' * 60}\n📌 B) Enum attributes — one value, several raw encodings")
for b_, s_ in ATTR_ENUM_MAP.items():
    raw = F.col(f"attributes.{b_}")
    x = df_bronze.agg(F.sum(raw.rlike(r"^u'").cast("int")).alias("u"), F.sum(raw.rlike(r"^'").cast("int")).alias("p")).first()
    u_, p_ = int(x["u"] or 0), int(x["p"] or 0)
    dq_add(findings_631, "df_attributes", f"{s_} (raw quoting style)", "one encoding per value (u'x' or 'x', not both)", u_ + p_, min(u_, p_),
           examples=f"u'x' = {u_:,} | 'x' = {p_:,}", note="minority style counted; DQ-BUSINESS-003 — TE.1 dec_enum() already folds both")

alc = df_bronze.groupBy(F.col("attributes.Alcohol").alias("raw")).count().where(F.lower(F.col("raw")).rlike("none")).collect()
alc_valid = sum(r_["count"] for r_ in alc if r_["raw"] != "None")
dq_add(findings_631, "df_attributes", "attr_alcohol ('none' category vs None literal)", "a valid category must not differ from the null literal only by case",
       N_BUS, alc_valid, status="INFO", examples=", ".join(f"{r_['raw']} = {r_['count']:,}" for r_ in alc),
       note="'none' = serves no alcohol (valid); None = unknown. Any case-folding null-cleaner would silently turn 'none' into NULL")

# ══ C) Cross-column category mapping ══
print(f"\n{'─' * 60}\n📌 C) Cross-column mapping consistency")
pc_, st_ = F.col("postal_code"), F.col("state")
pc_country = F.when(pc_.rlike(r"^\d{5}(-\d{4})?$"), "US").when(pc_.rlike(r"^[A-Za-z]\d[A-Za-z]"), "CA")
st_country = F.when(st_.isin(list(US_STATES)), "US").when(st_.isin(list(CA_PROVINCES)), "CA")
geo = df_business.select("business_id", "name", "city", "state", "postal_code", pc_country.alias("pc_country"), st_country.alias("st_country"))

# C1) state country vs postal-code country
c1 = geo.where(F.col("pc_country").isNotNull() & F.col("st_country").isNotNull())
c1_bad = c1.where(F.col("pc_country") != F.col("st_country"))
dq_add(findings_631, "df_business", "state ↔ postal_code (country)", "US state ⇒ US ZIP, CA province ⇒ CA code", c1.count(), c1_bad.count(),
       examples="; ".join(f"{r_['state']} / {r_['postal_code']} → {r_['name']}" for r_ in c1_bad.limit(5).collect()))

# C2) state vs USPS 3-digit ZIP prefix ranges (USPS National ZIP prefix allocation)
ZIP3_RANGES = {
    "AL": [(350, 369)], "AK": [(995, 999)], "AZ": [(850, 865)], "AR": [(716, 729)], "CA": [(900, 961)], "CO": [(800, 816)],
    "CT": [(60, 69)], "DE": [(197, 199)], "DC": [(200, 205)], "FL": [(320, 349)], "GA": [(300, 319), (398, 399)], "HI": [(967, 968)],
    "ID": [(832, 838)], "IL": [(600, 629)], "IN": [(460, 479)], "IA": [(500, 528)], "KS": [(660, 679)], "KY": [(400, 427)],
    "LA": [(700, 714)], "ME": [(39, 49)], "MD": [(206, 219)], "MA": [(10, 27), (55, 55)], "MI": [(480, 499)], "MN": [(550, 567)],
    "MS": [(386, 397)], "MO": [(630, 658)], "MT": [(590, 599)], "NE": [(680, 693)], "NV": [(889, 898)], "NH": [(30, 38)],
    "NJ": [(70, 89)], "NM": [(870, 884)], "NY": [(100, 149), (5, 5), (63, 63)], "NC": [(270, 289)], "ND": [(580, 588)],
    "OH": [(430, 459)], "OK": [(730, 749)], "OR": [(970, 979)], "PA": [(150, 196)], "RI": [(28, 29)], "SC": [(290, 299)],
    "SD": [(570, 577)], "TN": [(370, 385)], "TX": [(750, 799), (733, 733), (885, 885)], "UT": [(840, 847)], "VT": [(50, 59)],
    "VA": [(220, 246), (201, 201)], "WA": [(980, 994)], "WV": [(247, 268)], "WI": [(530, 549)], "WY": [(820, 831)],
    "PR": [(6, 9)], "VI": [(8, 8)], "GU": [(969, 969)],
}
zip3 = F.substring("postal_code", 1, 3).cast("int")
zip_ok = F.lit(False)
for s_, rngs in ZIP3_RANGES.items():
    in_rng = F.lit(False)
    for lo, hi in rngs:
        in_rng = in_rng | zip3.between(lo, hi)
    zip_ok = zip_ok | ((F.col("state") == s_) & in_rng)
c2 = geo.where((F.col("pc_country") == "US") & F.col("state").isin(list(ZIP3_RANGES))).withColumn("zip_ok", zip_ok)
c2_bad = c2.where(~F.col("zip_ok"))
n_c2_bad = c2_bad.count()
dq_add(findings_631, "df_business", "state ↔ postal_code (USPS ZIP prefix)", "ZIP3 prefix belongs to the stated state", c2.count(), n_c2_bad,
       examples="; ".join(f"{r_['state']} / {r_['postal_code']} → {r_['name']} ({r_['city']})" for r_ in c2_bad.limit(6).collect()),
       note="either the state or the ZIP is wrong; resolve with coordinates in 6.6 (cross-field)")
if n_c2_bad:
    display(c2_bad.groupBy("state", F.substring("postal_code", 1, 3).alias("zip3")).agg(F.count("*").alias("rows"),
            F.slice(F.collect_set("city"), 1, 5).alias("cities")).orderBy(F.desc("rows")))

# C3) Canadian province vs Canada Post first letter
CA_POST_LETTER = {"NL": "A", "NS": "B", "PE": "C", "NB": "E", "QC": "GHJ", "ON": "KLMNP", "MB": "R", "SK": "S",
                  "AB": "T", "BC": "V", "NU": "X", "NT": "X", "YT": "Y"}
let_ok = F.lit(False)
for s_, letters in CA_POST_LETTER.items():
    let_ok = let_ok | ((F.col("state") == s_) & F.upper(F.substring("postal_code", 1, 1)).isin(list(letters)))
c3 = geo.where((F.col("pc_country") == "CA") & F.col("state").isin(list(CA_POST_LETTER)))
c3_bad = c3.where(~let_ok)
dq_add(findings_631, "df_business", "province ↔ postal_code (Canada Post letter)", "AB ⇒ 'T…', BC ⇒ 'V…', …", c3.count(), c3_bad.count(),
       examples=dq_examples(c3_bad, "postal_code", F.lit(True)))

# C4) city ↔ state — rows filed under a minority state for a city that clearly belongs elsewhere
city_state = (df_business.withColumn("_k", norm_key(F.col("city"))).groupBy("_k", "state").count()
              .withColumn("city_rows", F.sum("count").over(Window.partitionBy("_k")))
              .withColumn("_rk", F.row_number().over(Window.partitionBy("_k").orderBy(F.desc("count")))))
modal = city_state.where("_rk = 1").select("_k", F.col("state").alias("modal_state"))
minority = (city_state.where((F.col("_rk") > 1) & (F.col("city_rows") >= 20) & (F.col("count") / F.col("city_rows") < 0.05))
            .join(modal, "_k"))
minority_rows = df_business.withColumn("_k", norm_key(F.col("city"))).join(minority.select("_k", "state", "modal_state"), ["_k", "state"])
dq_add(findings_631, "df_business", "city ↔ state (minority state)", "a city (≥20 rows) is filed under its dominant state; <5% minority states are suspect",
       N_BUS, minority_rows.count(), status="INFO",
       examples="; ".join(f"{r_['city']} filed as {r_['state']} (usually {r_['modal_state']}) → {r_['name']}" for r_ in minority_rows.limit(5).collect()),
       note="border metros (Philadelphia PA/NJ/DE, St. Louis MO/IL) make some splits legitimate — confirm with coordinates in 6.6")

# C5) BYOB / BYOBCorkage / Corkage — one policy encoded in three columns
#   BYOBCorkage: 'no' = BYOB not allowed | 'yes_free' = BYOB, no fee | 'yes_corkage' = BYOB with a corkage fee
byob = df_attributes.select("business_id", "attr_byob", "attr_byob_corkage", "attr_corkage").where(F.col("attr_byob_corkage").isNotNull())
BYOB_RULES = {
    "BYOBCorkage yes_* ⇒ BYOB ≠ FALSE":       (F.col("attr_byob_corkage").isin("yes_free", "yes_corkage") & (F.col("attr_byob") == False), "attr_byob"),
    "BYOBCorkage 'no' ⇒ BYOB ≠ TRUE":         ((F.col("attr_byob_corkage") == "no") & (F.col("attr_byob") == True), "attr_byob"),
    "BYOBCorkage 'yes_free' ⇒ Corkage ≠ TRUE": ((F.col("attr_byob_corkage") == "yes_free") & (F.col("attr_corkage") == True), "attr_corkage"),
    "BYOBCorkage 'yes_corkage' ⇒ Corkage ≠ FALSE": ((F.col("attr_byob_corkage") == "yes_corkage") & (F.col("attr_corkage") == False), "attr_corkage"),
    "BYOBCorkage 'no' ⇒ Corkage ≠ TRUE (no fee without BYOB)": ((F.col("attr_byob_corkage") == "no") & (F.col("attr_corkage") == True), "attr_corkage"),
}
x = byob.agg(*[F.sum(F.col(other).isNotNull().cast("int")).alias(other) for other in ("attr_byob", "attr_corkage")],
             *[F.sum(F.coalesce(cond, F.lit(False)).cast("int")).alias(f"r{i}") for i, (cond, _) in enumerate(BYOB_RULES.values())]).first()
for i, (lbl, (cond, other)) in enumerate(BYOB_RULES.items()):
    dq_add(findings_631, "df_attributes", f"BYOB policy: {lbl}", "three BYOB columns must describe the same policy", x[other], x[f"r{i}"])
print("\n   BYOB policy contingency (typed values, BYOBCorkage known):")
display(byob.groupBy("attr_byob_corkage", "attr_byob", "attr_corkage").count().orderBy("attr_byob_corkage", "attr_byob", "attr_corkage"))

findings_631_df = dq_summary(findings_631, "6.3.1 Categorical Representation Consistency")
print("CATEGORICAL REPRESENTATION CONSISTENCY CHECK COMPLETE")
print("=" * 80)

---
#### 🚨 DQ-BUSINESS-021 — One City Written in Many Spellings (`Saint Louis` / `St. Louis` / `St Louis` / `ST LOUIS` …)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-021` |
| **Severity** | FINDING (representation consistency; directly breaks GROUP BY city / metro analytics) |
| **Detected In** | Phase 6.3 — Format & Encoding Standardization → **6.3.1 Categorical Representation Consistency** (Cell 60), check A1 "city (spelling variants, per state)" |
| **Also Confirmed In** | 6.3.5 (Cell 72): 176 ALL-CAPS / all-lower cities and 3 odd interior capitals (DQ-BUSINESS-030), 9 double-spaced cities (DQ-BUSINESS-029). Trailing-comma / embedded-state cities are already DQ-BUSINESS-018; trailing spaces DQ-BUSINESS-004; NBSP / zero-width DQ-BUSINESS-017 |
| **Affected Columns** | `city` (grouped within `state`) |
| **Total Affected Rows** | **4,576** of 150,346 businesses (3.04%) carry a non-canonical spelling, across **153** city clusters with more than one spelling |
| **Invalid Value** | `Saint Louis` (4,827) vs `St. Louis` (1,254) / `St Louis` (356) / `St.Louis` (9) / `SAINT LOUIS` (3) … 14 spellings; `Saint Petersburg` (1,663) vs `St. Petersburg` (1,185) / `St Petersburg` (387) … 8; `Mount Juliet` vs `Mt. Juliet` / `Mt Juliet`; `King of Prussia` vs `King Of Prussia` (134); `O'Fallon` vs `O Fallon` / `OFallon`; `Feasterville Trevose` vs `Feasterville-Trevose`; `Land O Lakes` vs `Land O' Lakes` |
| **Domain Rule** | A city is a controlled geographic entity: one place = one canonical name per state (USPS / Canada Post preferred place name) |

**Description:** `city` is free text typed by owners, so the same municipality is spelled with and without the `Saint→St.` / `Mount→Mt.` / `Fort→Ft.` abbreviation, with or without the period, with different casing, hyphens and apostrophes. The two largest clusters alone (St. Louis 1,634 and St. Petersburg 1,586 non-canonical rows) split two of the 11 Yelp metros into 8-14 groups each. Any city-level KPI (business density, avg stars per city, top cities) is wrong until this is resolved. Note that the *modal* spelling is not always the official USPS form (USPS uses `SAINT LOUIS`, `SAINT PETERSBURG`, `MOUNT JULIET`), so a reference list beats a frequency vote.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep raw `city`; add `city_normalized` in Silver built from (a) trim + whitespace collapse + invisible-char removal, (b) the folding key used in 6.3.1 (case, punctuation, Saint/Mount/Fort), (c) a reviewed `city_key → canonical name` mapping table per state** | Fixes every cluster deterministically, keeps lineage to the source value, and the mapping table (153 rows) is small enough to review once and reuse for checkin/tip joins |
| 2 | **Overwrite `city` with the most frequent spelling of its cluster** | Fast, but loses the original value and can pick a non-official form (`St. Pete Beach` vs `Saint Pete Beach`); a frequency vote may also merge genuinely different places with similar keys |
| 3 | **Reverse-geocode `latitude`/`longitude` to an official place name** | Most authoritative, but needs an external gazetteer / API, and coordinates have their own issues (DQ-BUSINESS-011, DQ-BUSINESS-027) |
| 4 | **Leave as-is and normalise only in Gold queries** | No Silver change, but every consumer must re-implement the same logic, and results will drift between teams |

**Best Logical Solution:** **Option 1 — raw `city` + derived `city_normalized` from a rule-based key and a curated per-state mapping.** It resolves all 153 clusters reproducibly, uses official spellings rather than the modal one, and keeps the raw value for audit. Option 3 can later validate the mapping for the few ambiguous clusters.

---

---
#### 🚨 DQ-BUSINESS-022 — Same Brand / Chain Written in Several Spellings (`AutoZone` / `Autozone` / `Auto Zone`)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-022` |
| **Severity** | LOW / INFO (owner-entered display name; impacts chain-level analytics and entity resolution, not record validity) |
| **Detected In** | Phase 6.3 — Format & Encoding Standardization → **6.3.1 Categorical Representation Consistency** (Cell 60), check A2 "name (chain / brand spelling variants)" |
| **Also Confirmed In** | 6.3.5 (Cell 72): 701 names use a curly apostrophe `’` vs 21k+ straight `'` (`David's Bridal` 12 vs `David’s Bridal` 10), 855 ALL-CAPS / all-lower names (`SEPHORA` vs `Sephora`, `ALDI` vs `Aldi`), 263 names with double spaces |
| **Affected Columns** | `name` |
| **Total Affected Rows** | **1,821** of 150,346 businesses (1.21%) carry a non-modal spelling, across **1,252** brand clusters |
| **Invalid Value** | `Autozone` (35) / `AutoZone` (28) / `Auto Zone` (5) / `autozone` (1); `Winn-Dixie` (33) / `Winn Dixie` (32); `TJ Maxx` / `T J Maxx` / `Tj Maxx` / `T.J. Maxx` / `T.J.Maxx`; `Little Caesars Pizza` / `Little Caesar's Pizza`; `HH Gregg` / `hhgregg` / `h h  gregg` |
| **Domain Rule** | One brand should map to one brand key so chain counts, market share and chain-vs-independent features are correct |

**Description:** Unlike `city`, a business name is not a controlled vocabulary; stylisation is sometimes intentional (`lululemon`, `hhgregg`, `SEPHORA` are official brand marks). The issue is therefore not that individual names are "wrong" but that the same chain cannot be grouped reliably: e.g. Winn-Dixie is split exactly 50/50 between two spellings. This affects chain detection, duplicate-listing detection and any "top chains" analysis.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep raw `name`; add a derived `name_match_key` in Silver (NFC, curly→straight quotes, case-fold, strip punctuation/whitespace, `&`→`and`)**, and build `brand` in Gold from that key | Display names stay as the owner wrote them; the key groups all 1,252 clusters without editing any value; cheap and deterministic |
| 2 | **Overwrite `name` with the modal spelling of its cluster** | Destroys legitimate stylisation and may merge distinct businesses whose keys collide (e.g. two different `Joe's` shops) |
| 3 | **Fuzzy matching (Jaro-Winkler / embeddings) + category + geography to build a brand dimension** | Catches deeper variants (`Little Caesars` vs `Little Caesars Pizza`), but is costly, probabilistic, and belongs in Gold enrichment rather than Silver cleaning |
| 4 | **Do nothing** | Valid for record display, but chain metrics stay fragmented |

**Best Logical Solution:** **Option 1 — raw `name` + deterministic `name_match_key` in Silver**, with Option 3 as a later Gold enhancement. It keeps source fidelity and brand stylisation while making chain grouping correct.

---

---
#### 🚨 DQ-BUSINESS-023 — `state` Contradicts the USPS ZIP Prefix of `postal_code`

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-023` |
| **Severity** | FINDING (cross-column category mapping; one of the two geographic fields is wrong) |
| **Detected In** | Phase 6.3 — Format & Encoding Standardization → **6.3.1 Categorical Representation Consistency** (Cell 60), check C2 "state ↔ postal_code (USPS ZIP prefix)" |
| **Also Confirmed In** | 6.3.1 C4 (Cell 60): 48 businesses filed under a minority state for their city (`Eagle` UT/ID, `Franklin` IN/TN). Partial overlap with DQ-BUSINESS-010 (out-of-footprint states such as SD/TX/HI/UT). Country-level checks pass: 0 US-state / CA-postcode mismatches (C1) and 0 Canada Post letter mismatches (C3, 5,555 rows) |
| **Affected Columns** | `state`, `postal_code` (and `city`) |
| **Total Affected Rows** | **71** of 144,716 US businesses with a 5-digit ZIP (0.049%), in 62 (state, ZIP3) groups |
| **Invalid Value** | `CA / 836xx Eagle` (Idaho ZIP), `SD / 336xx Tampa` (Florida ZIP), `TX / 372xx Nashville` (Tennessee ZIP), `TN / 28801 White Duck Taco Shop, Nashville` (Asheville NC ZIP), `PA / 08102 Benjamin Franklin Bridge` (Camden NJ ZIP), `NV / 88431 Sparks` (NM ZIP) |
| **Domain Rule** | Every USPS 3-digit ZIP prefix is allocated to exactly one state (USPS National ZIP prefix allocation); `state` must match it |

**Description:** Three patterns explain the 71 rows:

| Pattern | Examples | Likely truth |
| --- | --- | --- |
| Wrong `state`, ZIP and city agree | `CA/UT · 836 · Eagle`, `SD · 336 · Tampa`, `TX · 372 · Nashville`, `HI · 856 · Tucson` | State typo / wrong drop-down value (overlaps DQ-BUSINESS-010) |
| Border metro, ZIP of the neighbouring state | `NJ · 191 · Philadelphia`, `PA · 080-083 · Cherry Hill`, `DE · 190/193 · Media / Glen Mills`, `IL · 631 · St. Louis`, `MO · 622`, `CA · 895 · Reno` | Business sits near the state line; either state or ZIP was taken from the wrong side |
| ZIP typo, state and city agree | `PA · 432 · Philadelphia`, `TN · 928 · Nashville`, `IN · 922 · Indianapolis`, `AZ · 875 · Tuscon`, `PA · 104/117 · Collegeville` | Digits transposed / mistyped ZIP |

Wrong state assignments mis-route businesses to the wrong metro, tax region and time zone (DQ-BUSINESS-025 note), and wrong ZIPs break joins to census / demographic data.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Flag the 71 rows (`geo_conflict_flag`) and resolve with coordinates in 6.6: point-in-polygon of `latitude`/`longitude` against state and ZIP boundaries decides which field is wrong; correct only that field in Silver, record the rule in lineage** | Coordinates are an independent third witness; it fixes the field that is actually wrong instead of guessing, and is auditable |
| 2 | **Always trust the ZIP and overwrite `state` from the ZIP prefix** | Correct for pattern 1, but wrong for pattern 3 (ZIP typos) — would move a Philadelphia business to Ohio |
| 3 | **Always trust `state` and NULL the ZIP** | Safe for pattern 3, wrong for pattern 1; loses a likely-correct ZIP |
| 4 | **Quarantine the 71 rows** | Clean, but removes valid businesses (with reviews) from all downstream analysis for a fixable error |

**Best Logical Solution:** **Option 1 — flag now, adjudicate with coordinates in 6.6, correct only the losing field.** At 71 rows the cost is negligible and it is the only option that is correct for all three patterns.

---

---
#### 🚨 DQ-BUSINESS-024 — One BYOB Policy Encoded in Three Columns That Contradict Each Other

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-024` |
| **Severity** | FINDING (cross-column category mapping; the same fact has two different values) |
| **Detected In** | Phase 6.3 — Format & Encoding Standardization → **6.3.1 Categorical Representation Consistency** (Cell 60), check C5 "BYOB policy" (5 rules + contingency table) |
| **Affected Columns** | `attributes.BYOB` (BOOLEAN), `attributes.BYOBCorkage` (enum `no` / `yes_free` / `yes_corkage`), `attributes.Corkage` (BOOLEAN) → `df_attributes.attr_byob`, `attr_byob_corkage`, `attr_corkage` |
| **Total Affected Rows** | **94** distinct businesses of 1,439 with a known `BYOBCorkage` (6.5%), **100** rule violations (validated with an independent SQL query) |
| **Invalid Value** | Rule breakdown below |
| **Domain Rule** | `BYOBCorkage` is the most specific field: `no` ⇒ BYOB = FALSE and no corkage fee; `yes_free` ⇒ BYOB = TRUE, Corkage = FALSE; `yes_corkage` ⇒ BYOB = TRUE, Corkage = TRUE |

| Rule | Violations | Of (both known) |
| --- | --- | --- |
| `BYOBCorkage yes_*` but `BYOB = False` | 49 | 163 |
| `BYOBCorkage no` but `BYOB = True` | 10 | 163 |
| `BYOBCorkage yes_free` but `Corkage = True` | 11 | 600 |
| `BYOBCorkage yes_corkage` but `Corkage = False` | 1 | 600 |
| `BYOBCorkage no` but `Corkage = True` (fee without BYOB) | 29 | 600 |

**Description:** Yelp collected the BYOB policy through three separate questions at different times, and owners answered them independently. The largest group (38 rows: `yes_free` + `BYOB = False` + `Corkage = False`) suggests `BYOB = False` was read as "we don't *require* BYOB". `no` + `Corkage = True` (29) is semantically impossible: a corkage fee exists only when outside bottles are allowed. Filters such as "BYOB restaurants" return different sets depending on which column is used.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep the 3 raw columns in Silver; derive one `byob_policy` (`not_allowed` / `free` / `with_fee` / `unknown`) with precedence `BYOBCorkage` > `BYOB` + `Corkage`, and add `byob_conflict_flag`** | `BYOBCorkage` is the only field that encodes the full policy, so it is the most reliable witness; raw values stay auditable; analysts get one consistent field |
| 2 | **Overwrite `BYOB` / `Corkage` in Silver to agree with `BYOBCorkage`** | Consistent, but replaces reported values with inferred ones and hides the conflict |
| 3 | **NULL all three fields for the 94 conflicting businesses** | Removes contradictions but discards mostly-correct information (e.g. `BYOBCorkage` itself) |
| 4 | **Majority vote among the three columns** | Only two columns are comparable per rule; a vote is undefined when one is NULL (the most common case) |

**Best Logical Solution:** **Option 1 — derived `byob_policy` with `BYOBCorkage` precedence plus a conflict flag.** It produces one trustworthy answer, documents the precedence rule, and keeps the source data intact.

---

> **Cross-references from 6.3.1 (already documented, no new ID):** enum quoting style `u'x'` vs `'x'` (Alcohol 9,326, WiFi 14,128, NoiseLevel 6,399, RestaurantsAttire 15,465, Smoking 47, BYOBCorkage 44 minority values) → **DQ-BUSINESS-003**. `Alcohol 'none'` (20,910) is a valid category and differs from the `None` literal (38) only by case → null-cleaning must be case-sensitive (**DQ-BUSINESS-002**). City ↔ state minority filings (48) → **DQ-BUSINESS-023** / **DQ-BUSINESS-010**.
>
> **Reviewed and not raised:** `category_name` has 0 spelling variants (Yelp-controlled taxonomy); 0 US/CA country mismatches between `state` and `postal_code`; 0 Canada Post first-letter mismatches; `AgesAllowed` uses a single quoting style.

---

In [0]:
# 6.3.2) Date & Time Format Consistency  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# The business table has NO calendar-date column. Its only temporal data is the weekly
# opening schedule (`hours.<Day>` = 'H:M-H:M', local wall-clock time). So "date format
# consistency" here means: is the SAME instant / schedule always written the SAME way?
#   A) Schema scan    : no DATE/TIMESTAMP columns; no date-like strings hidden in text columns
#   B) Time notation  : one notation only (no AM/PM, '.', 'h', en-dash, spaces, words, 24:00)
#   C) Zero-padding   : one padding convention ('8:0' vs '08:00' not mixed)
#   D) Midnight close : '…-0:0' vs '…-23:59' (vs '24:00') — two spellings of the same instant
#   E) Open all day   : '0:0-0:0' vs '0:0-23:59' / '0:0-23:45', and vs attribute Open24Hours
#   F) Derived HH:mm  : TE.1 start/end columns uniform
#   G) Day naming / order / time zone — representation conventions to carry into Silver
# Value ranges & ambiguity semantics are in 6.2.2 / 6.2.4 / DQ-BUSINESS-007; here = representation only.
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import types as T

print("=" * 80)
print("6.3.2) DATE & TIME FORMAT CONSISTENCY  —  Business DataFrames (TE.2)")
print("    One representation per instant / schedule")
print("=" * 80)

N_BUS = df_business.count()
findings_632 = []

# ══ A) Schema scan ══
print(f"\n{'─' * 60}\n📌 A) Schema scan — date/time-typed columns & date-like strings")
_dt_cols = [p for p, dt in bronze_leaves(df_bronze.schema) if isinstance(dt, (T.DateType, T.TimestampType))]
dq_add(findings_632, "df_bronze", "DATE / TIMESTAMP columns", "business table carries no calendar date (informational)",
       len(BRONZE_TYPE), len(_dt_cols), status="INFO" if _dt_cols else "PASS", examples=", ".join(_dt_cols),
       note="temporal facts live in review / tip / checkin; business only has a weekly schedule")
DATE_LIKE = r"\b(19|20)\d{2}[-/.](0?[1-9]|1[0-2])[-/.](0?[1-9]|[12]\d|3[01])\b|\b(0?[1-9]|1[0-2])/(0?[1-9]|[12]\d|3[01])/(\d{2}|\d{4})\b"
x = df_business.agg(*[F.sum(F.col(c).rlike(DATE_LIKE).cast("int")).alias(c) for c in ("name", "address", "city", "postal_code")]).first()
for c in ("name", "address", "city", "postal_code"):
    dq_add(findings_632, "df_business", f"{c} (embedded date string)", "no date literal stored in a text column", N_BUS, x[c],
           examples=dq_examples(df_business, c, F.col(c).rlike(DATE_LIKE)) if x[c] else "")

# ══ Long view of the schedule: one row per (business, day) slot ══
hl = (df_hours.select("business_id", F.explode(F.array(*[F.struct(F.lit(d).alias("day"), F.col(HOURS_MAP[d]).alias("raw"),
                                                                  F.col(f"{HOURS_MAP[d]}_start_time").alias("st"),
                                                                  F.col(f"{HOURS_MAP[d]}_end_time").alias("et")) for d in DAYS])).alias("h"))
      .select("business_id", "h.*").where(F.col("raw").isNotNull()))
N_SLOTS = hl.count()

# ══ B) Notation — every slot must use the 24-h 'H:M-H:M' notation ══
print(f"\n{'─' * 60}\n📌 B) Time notation — single 24-hour 'H:M-H:M' notation ({N_SLOTS:,} day-slots)")
r_ = F.col("raw")
NOTATIONS = {
    "AM/PM (12-hour clock)":           r_.rlike("(?i)[ap]\\.?m\\b"),
    "'.' or 'h' time separator":       r_.rlike(r"\d[.h]\d"),
    "en/em dash or 'to' as range sep": r_.rlike("[\u2013\u2014]|(?i)\\bto\\b"),
    "embedded whitespace":             r_.rlike(r"\s"),
    "words (closed / open / 24 hours)": r_.rlike("(?i)[a-z]{3,}"),
    "'24:00' end-of-day notation":     r_.rlike(r"(^|-)24:"),
    "seconds component (H:M:S)":       r_.rlike(r"\d+:\d+:\d+"),
    "multiple ranges in one day":      r_.rlike(r"[,;/]") | (F.size(F.split(r_, "-")) > 2),
}
x = hl.agg(*[F.sum(c_.cast("int")).alias(f"n{i}") for i, c_ in enumerate(NOTATIONS.values())]).first()
for i, (lbl, c_) in enumerate(NOTATIONS.items()):
    dq_add(findings_632, "df_hours", f"hours_<day> ({lbl})", "24-h 'H:M-H:M' only", N_SLOTS, x[f"n{i}"],
           examples=dq_examples(hl, "raw", c_) if x[f"n{i}"] else "",
           note="multiple ranges would mean split shifts were flattened" if "multiple" in lbl else "")

shape = F.regexp_replace(F.regexp_replace("raw", r"\d\d", "NN"), r"\d", "N")
print("\n   Raw shape distribution (N = one digit):")
display(hl.groupBy(shape.alias("shape")).agg(F.count("*").alias("slots"), F.slice(F.collect_set("raw"), 1, 4).alias("samples"))
          .withColumn("pct", F.round(F.col("slots") / N_SLOTS * 100, 2)).orderBy(F.desc("slots")))

# ══ C) Zero-padding convention ══
print(f"\n{'─' * 60}\n📌 C) Zero-padding convention ('8:0' vs '08:00')")
x = hl.agg(F.sum(r_.rlike(r"(^|[-:])0\d").cast("int")).alias("padded"),
           F.sum(r_.rlike(r"(^|[-:])\d(?=$|[-:])").cast("int")).alias("unpadded")).first()
padded, unpadded = int(x["padded"] or 0), int(x["unpadded"] or 0)
dq_add(findings_632, "df_hours", "hours_<day> (mixed padding conventions)", "one convention: all-unpadded OR all-padded",
       N_SLOTS, min(padded, unpadded) if padded and unpadded else 0,
       examples=f"padded components: {padded:,} | unpadded components: {unpadded:,}",
       note="source consistently uses the unpadded Python-style '8:0' / '17:30' — one convention, parse with H:m")

# ══ D) Midnight close — one instant, several spellings ══
print(f"\n{'─' * 60}\n📌 D) Midnight close — '…-0:0' vs '…-23:59' vs '…-24:0'")
not_allday = r_ != "0:0-0:0"
MIDNIGHT = {"-0:0": r_.rlike(r"-0:0$") & not_allday, "-23:59": r_.rlike(r"-23:59$"), "-24:0": r_.rlike(r"-24:0+$")}
x = hl.agg(*[F.sum(c_.cast("int")).alias(k.replace("-", "m").replace(":", "_")) for k, c_ in MIDNIGHT.items()]).first()
mid = {k: int(x[k.replace("-", "m").replace(":", "_")] or 0) for k in MIDNIGHT}
dominant = max(mid, key=mid.get)
dq_add(findings_632, "df_hours", "hours_<day> (midnight close spelling)", f"one spelling of midnight (dominant '{dominant}')",
       sum(mid.values()), sum(v for k, v in mid.items() if k != dominant),
       examples=" | ".join(f"'…{k}' = {v:,}" for k, v in mid.items()),
       note="'23:59' loses one minute and makes duration maths inconsistent; normalise to end = 00:00 (+1 day) in Silver")

# ══ E) Open all day — several spellings + attribute ══
print(f"\n{'─' * 60}\n📌 E) 'Open all day' — '0:0-0:0' vs '0:0-23:59' / '0:0-23:45' vs attr Open24Hours")
ALLDAY = {"0:0-0:0": r_ == "0:0-0:0", "0:0-23:59": r_ == "0:0-23:59", "0:0-23:45": r_ == "0:0-23:45", "0:0-24:0": r_.rlike(r"^0:0-24:0+$")}
x = hl.agg(*[F.sum(c_.cast("int")).alias(f"a{i}") for i, c_ in enumerate(ALLDAY.values())]).first()
allday = {k: int(x[f"a{i}"] or 0) for i, k in enumerate(ALLDAY)}
dq_add(findings_632, "df_hours", "hours_<day> ('open all day' spelling)", "one spelling for a 24-hour day",
       sum(allday.values()), sum(v for k, v in allday.items() if k != "0:0-0:0"),
       examples=" | ".join(f"'{k}' = {v:,}" for k, v in allday.items()),
       note="'0:0-0:0' itself is ambiguous (24 h vs closed vs unknown) — DQ-BUSINESS-007")

n_allday_days = F.size(F.filter(F.array(*[F.col(HOURS_MAP[d]) for d in DAYS]), lambda v: v == "0:0-0:0"))
o24 = (df_attributes.where(F.col("attr_open_24_hours").isNotNull()).select("business_id", "attr_open_24_hours")
       .join(df_hours.select("business_id", n_allday_days.alias("allday_days"),
                             F.size(F.filter(F.array(*[F.col(HOURS_MAP[d]) for d in DAYS]), lambda v: v.isNotNull())).alias("days_with_hours")), "business_id"))
o24_bad = o24.where((F.col("attr_open_24_hours") & (F.col("days_with_hours") > 0) & (F.col("allday_days") < F.col("days_with_hours")))
                    | (~F.col("attr_open_24_hours") & (F.col("days_with_hours") > 0) & (F.col("allday_days") == F.col("days_with_hours"))))
dq_add(findings_632, "df_attributes", "attr_open_24_hours ↔ hours", "Open24Hours=TRUE ⇔ every listed day is all-day", o24.count(), o24_bad.count(),
       status="INFO" if o24_bad.count() else "PASS",
       note="the same fact ('open 24 h') is represented twice; attribute only filled for a handful of businesses")
if o24_bad.count():
    display(o24_bad.join(df_hours, "business_id").select("business_id", "attr_open_24_hours", "allday_days", "days_with_hours", *[HOURS_MAP[d] for d in DAYS]))

# ══ F) Derived TE.1 start/end columns ══
print(f"\n{'─' * 60}\n📌 F) Derived start/end columns (TE.1) — uniform 'HH:mm'")
HHMM = r"^([01]\d|2[0-3]):[0-5]\d$"
x = hl.agg(F.sum((~F.coalesce(F.col("st").rlike(HHMM), F.lit(False)) | ~F.coalesce(F.col("et").rlike(HHMM), F.lit(False))).cast("int")).alias("bad")).first()
dq_add(findings_632, "df_hours", "hours_<day>_start/end_time", "every derived value is 'HH:mm'", N_SLOTS, x["bad"])

# ══ G) Conventions: day naming, day order, time zone ══
print(f"\n{'─' * 60}\n📌 G) Day naming / day order / time zone conventions")
hours_days = [f.name for f in df_bronze.schema["hours"].dataType.fields]
best_keys = list(DICT_TABLE_MAP["BestNights"][1])
dq_add(findings_632, "df_bronze", "day naming (hours vs BestNights)", "one day-name convention across structures", 7,
       sum(1 for d in DAYS if d not in best_keys), status="INFO",
       examples=f"hours: {hours_days[:2]}… | BestNights keys: {best_keys[:2]}…",
       note="Title-case struct fields vs lower-case dict keys — map both to ISO day number (1=Mon) in Silver")
dq_add(findings_632, "df_bronze", "hours struct field order", "calendar order Monday → Sunday", 7,
       sum(1 for a, b_ in zip(hours_days, DAYS) if a != b_), status="INFO", examples=", ".join(hours_days),
       note="alphabetical order from JSON schema inference; TE.1 / HOURS_MAP already re-orders to calendar order")
TZ_BY_STATE = {"PA": "America/New_York", "NJ": "America/New_York", "DE": "America/New_York", "FL": "America/New_York",
               "IN": "America/Indiana/Indianapolis", "TN": "America/Chicago", "MO": "America/Chicago", "IL": "America/Chicago",
               "LA": "America/Chicago", "AZ": "America/Phoenix", "ID": "America/Boise", "AB": "America/Edmonton",
               "NV": "America/Los_Angeles", "CA": "America/Los_Angeles"}
biz_with_hours = hl.select("business_id").distinct().join(df_business.select("business_id", "state"), "business_id")
tz_mix = biz_with_hours.groupBy(F.coalesce(F.create_map(*[F.lit(v) for kv in TZ_BY_STATE.items() for v in kv])[F.col("state")], F.lit("unmapped")).alias("tz")).count().orderBy(F.desc("count"))
tz_rows = tz_mix.collect()
dq_add(findings_632, "df_hours", "hours (time zone)", "local wall-clock time; zone not stored", biz_with_hours.count(), len(tz_rows), status="INFO",
       examples=", ".join(f"{r_['tz']}={r_['count']:,}" for r_ in tz_rows),
       note="violations = distinct zones the schedules are expressed in; derive a tz column from state before comparing hours across metros (AZ has no DST)")

findings_632_df = dq_summary(findings_632, "6.3.2 Date & Time Format Consistency")
print("DATE & TIME FORMAT CONSISTENCY CHECK COMPLETE")
print("=" * 80)

---
#### 🚨 DQ-BUSINESS-025 — Midnight Close and "Open All Day" Written in Several Spellings (`-0:0` vs `-23:59`, `0:0-0:0` vs `0:0-23:59`)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-025` |
| **Severity** | FINDING (time-format consistency; same instant / schedule has several representations) |
| **Detected In** | Phase 6.3 — Format & Encoding Standardization → **6.3.2 Date & Time Format Consistency** (Cell 65), checks D "midnight close spelling" and E "open all day spelling" |
| **Affected Columns** | `hours.Monday` … `hours.Sunday` → `df_hours.hours_<day>` and derived `hours_<day>_end_time` |
| **Total Affected Rows** | **1,384** day-slots of 801,015 (0.17%): **959** closing at `23:59` instead of `0:0` (3.9% of 24,746 midnight closes), **425** all-day slots written `0:0-23:59` (412) or `0:0-23:45` (13) instead of the dominant `0:0-0:0` (73,908) |
| **Invalid Value** | `11:0-23:59`, `0:0-23:59`, `0:0-23:45` |
| **Domain Rule** | One instant = one representation: midnight close = `00:00` of the next day; a 24-hour day = one canonical encoding |

**Description:** Everything else in the schedule format is clean: 16 raw shapes, all 24-hour `H:M-H:M`, no AM/PM, no `24:00`, no seconds, no split shifts, one consistent unpadded convention. The only representation conflict is at the end of the day. `23:59` silently drops one minute and makes duration maths and "open now at 23:59:30" checks disagree with `-0:0` rows; `0:0-23:59` is a second spelling of "open 24 h" alongside `0:0-0:0`, which itself is ambiguous (DQ-BUSINESS-007). `0:0-23:45` (13) may be genuine (closes 15 min before midnight) and should not be force-mapped to 24 h.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep raw `hours_<day>`; in the Silver derived columns map `end = 23:59` → `00:00` with `closes_next_day = TRUE`, map `0:0-23:59` → `is_open_24h = TRUE`, keep `0:0-23:45` as a real window, and compute `open_minutes` with modulo-24 h arithmetic** | Unifies the two spellings of the same instant without inventing data; duration and "open now" logic become consistent; consistent with the DQ-BUSINESS-007 derived-flag design |
| 2 | **Rewrite the raw strings to `-0:0` / `0:0-0:0`** | Simple, but alters source values and pushes the 425 rows into the already-ambiguous `0:0-0:0` bucket |
| 3 | **Treat `23:59` as a separate closing time** | No change needed, but every downstream duration is 1 minute short and midnight-close counts are under-reported by 959 |
| 4 | **NULL the non-canonical slots** | Loses valid opening information for 1,384 slots |

**Best Logical Solution:** **Option 1 — canonicalise in derived Silver columns (`end_time`, `closes_next_day`, `is_open_24h`, `open_minutes`) and keep the raw string.** It resolves the representation conflict at the semantic level while preserving the source.

---

---
#### 🚨 DQ-BUSINESS-026 — `Open24Hours = True` Contradicts the Weekly Schedule

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-026` |
| **Severity** | LOW (redundant representation of one fact disagrees; tiny population) |
| **Detected In** | Phase 6.3 — Format & Encoding Standardization → **6.3.2 Date & Time Format Consistency** (Cell 65), check E "attr_open_24_hours ↔ hours" |
| **Also Confirmed In** | 6.3.4 (Cell 70): `Open24Hours` is the sparsest boolean attribute (19 True / 20 False / 150,307 NULL) |
| **Affected Columns** | `attributes.Open24Hours` → `df_attributes.attr_open_24_hours`; `hours.<Day>` |
| **Total Affected Rows** | **15** of 39 businesses with a known `Open24Hours` (38.5%); all 15 are `True` with fewer than 7 all-day slots (10 have none, 5 have only Monday = `0:0-0:0`) |
| **Invalid Value** | `Open24Hours = True` with `hours` = `9:0-17:0` every day (`aG-CtKPYEJ8ZR5xdnqg5Qw`), or `0:0-0:0` Monday + `9:0-0:0` other days (`mGVZZXjH1SyLIuxpcNNIqw`) |
| **Domain Rule** | "Open 24 hours" is stated twice (attribute and schedule); both must agree |

**Description:** The attribute was answered once and never maintained, while the schedule is edited more often (seasonal hours). A pattern of `Monday = 0:0-0:0` with normal hours on other days also shows that `0:0-0:0` is sometimes a placeholder rather than "24 h" — which is exactly the DQ-BUSINESS-007 ambiguity. With only 39 populated values the attribute is too sparse to be used to resolve `0:0-0:0` for the other 73,908 slots.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Treat `hours` as the source of truth; keep `attr_open_24_hours` raw but derive `is_open_24h` from the schedule (DQ-BUSINESS-025), and add `open24_conflict_flag` for the 15** | The schedule is richer and more recently maintained; one authoritative derived field avoids two answers to the same question |
| 2 | **Trust the attribute and overwrite hours to `0:0-0:0`** | Would erase detailed and probably correct opening hours for 15 businesses |
| 3 | **NULL `attr_open_24_hours` where it conflicts** | Removes the contradiction but also the signal; acceptable but loses lineage |
| 4 | **Drop the attribute from Silver** | 99.97% NULL already; defensible, but the DDL contract includes it, so it must be kept for schema conformance |

**Best Logical Solution:** **Option 1 — schedule wins, attribute kept raw with a conflict flag.** It is the most reliable evidence, needs no destructive change and keeps the DDL contract.

---

> **Reviewed and not raised (INFO only):** no DATE/TIMESTAMP column in the business table and no date literals in `name` / `address` / `city` / `postal_code`; the unpadded `H:M` convention is used consistently (0 padded components) and TE.1 derives uniform `HH:mm`; `hours` struct fields come in alphabetical order and use Title-case names vs lower-case `BestNights` keys (map both to ISO day numbers in Silver). **Time zone:** schedules are local wall-clock time and the zone is not stored — 7 IANA zones are implied by `state` (AZ without DST), 13 businesses map to no zone (out-of-footprint states, DQ-BUSINESS-010). Recommendation: derive a `time_zone` column from `state` in Silver before comparing hours across metros.

---

In [0]:
# 6.3.3) Numeric Format Consistency  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Numbers stored with embedded symbols / separators / units, float artefacts, or in
# several notations. Numeric-bearing fields of the business table:
#   A) stars          DOUBLE → DECIMAL(2,1): float residue, storage type (DQ-BUSINESS-001)
#   B) latitude/long. DOUBLE: precision (decimal places), sign convention, swapped axes, sci-notation
#   C) review_count   integer type, no sentinel encodings
#   D) RestaurantsPriceRange2  numeric tier stored as a STRING literal: '2' vs u'2' / '2.0' / '$$'
#   E) postal_code    numeric-looking STRING: float artefact '19103.0', lost leading zero, ZIP+4 mix,
#                     Canadian spacing 'T6E 1A1' vs 'T6E1A1'
#   F) address        house-number formats (ranges, fractions, alphanumeric), ordinal street numbers
#   G) name           purely numeric names (informational)
# Value ranges are in 6.2.2; here = representation only.
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import types as T

print("=" * 80)
print("6.3.3) NUMERIC FORMAT CONSISTENCY  —  Business DataFrames (TE.2)")
print("    Numbers with symbols, artefacts or mixed notations")
print("=" * 80)

N_BUS = df_business.count()
findings_633 = []

# ══ A) stars ══
print(f"\n{'─' * 60}\n📌 A) stars — storage type & float residue")
raw_stars = F.col("stars")
x = df_bronze.agg(F.sum((raw_stars * 10 != F.round(raw_stars * 10)).cast("int")).alias("residue"),
                  F.sum((F.length(raw_stars.cast("string")) > 3).cast("int")).alias("long_repr")).first()
dq_add(findings_633, "df_bronze", "stars (storage type)", "exact decimal type for a half-star rating", N_BUS,
       N_BUS if isinstance(df_bronze.schema["stars"].dataType, T.DoubleType) else 0, status="INFO",
       examples=f"bronze {df_bronze.schema['stars'].dataType.simpleString()} → contract {df_business.schema['stars'].dataType.simpleString()}",
       note="DQ-BUSINESS-001; TE.1 casts exactly (lossless-decode validated)")
dq_add(findings_633, "df_bronze", "stars (binary float residue)", "value × 10 is an integer; repr ≤ 3 chars ('4.5')", N_BUS,
       max(int(x["residue"] or 0), int(x["long_repr"] or 0)))

# ══ B) latitude / longitude ══
print(f"\n{'─' * 60}\n📌 B) latitude / longitude — precision & notation")
dp = lambda c: F.length(F.regexp_extract(F.col(c).cast("string"), r"\.(\d+)$", 1))
coord = df_business.select("business_id", "name", "city", "latitude", "longitude",
                           F.least(dp("latitude"), dp("longitude")).alias("dp"),
                           F.col("latitude").cast("string").alias("lat_s"), F.col("longitude").cast("string").alias("lon_s"))
print("   Decimal places (min of lat/long) — 1 dp ≈ 11 km, 3 dp ≈ 110 m, 5 dp ≈ 1 m, 7 dp ≈ 1 cm:")
display(coord.groupBy("dp").agg(F.count("*").alias("rows")).withColumn("pct", F.round(F.col("rows") / N_BUS * 100, 3)).orderBy("dp"))
x = coord.agg(F.sum((F.col("dp") <= 3).cast("int")).alias("low"), F.sum((F.col("dp") >= 9).cast("int")).alias("spurious"),
              F.sum((F.col("lat_s").rlike("[eE]") | F.col("lon_s").rlike("[eE]")).cast("int")).alias("sci"),
              F.sum(((F.col("latitude") < 0) | (F.col("longitude") > 0)).cast("int")).alias("sign"),
              F.sum((F.col("latitude").between(-180, -50) & F.col("longitude").between(10, 75)).cast("int")).alias("swap")).first()
dq_add(findings_633, "df_business", "latitude/longitude (≤ 3 decimals)", "rooftop-level precision (≥ 4 dp ≈ 11 m)", N_BUS, x["low"], status="INFO",
       examples="; ".join(f"{r_['name']} ({r_['city']}) {r_['latitude']}, {r_['longitude']}" for r_ in coord.where("dp <= 3").limit(4).collect()),
       note="coarse coordinates = city / ZIP-centroid geocodes; lower weight in distance-based features")
dq_add(findings_633, "df_business", "latitude/longitude (≥ 9 decimals)", "no spurious sub-millimetre precision", N_BUS, x["spurious"], status="INFO",
       note="float-to-string artefact of a different geocoder; harmless, but shows mixed coordinate sources")
dq_add(findings_633, "df_business", "latitude/longitude (scientific notation)", "plain decimal notation", N_BUS, x["sci"])
dq_add(findings_633, "df_business", "latitude/longitude (sign convention)", "North America: lat > 0, lon < 0", N_BUS, x["sign"])
dq_add(findings_633, "df_business", "latitude/longitude (swapped axes)", "lat not in the longitude band and vice versa", N_BUS, x["swap"])

# ══ C) review_count ══
print(f"\n{'─' * 60}\n📌 C) review_count — integer type & sentinel encodings")
rc_type = df_bronze.schema["review_count"].dataType
x = df_business.agg(F.sum(((F.col("review_count") < 0) | F.col("review_count").isin(99999, 999999, 2147483647)).cast("int")).alias("sent"),
                    F.min("review_count").alias("mn"), F.max("review_count").alias("mx")).first()
dq_add(findings_633, "df_bronze", "review_count (storage type)", "integral type", N_BUS,
       0 if isinstance(rc_type, (T.IntegerType, T.LongType)) else N_BUS, examples=rc_type.simpleString())
dq_add(findings_633, "df_business", "review_count (sentinel values)", "no negative / 99999-style placeholder counts", N_BUS, x["sent"],
       note=f"observed range {x['mn']:,} – {x['mx']:,} (dataset only contains businesses with ≥ 5 reviews)")

# ══ D) RestaurantsPriceRange2 — numeric tier stored as a Python literal string ══
print(f"\n{'─' * 60}\n📌 D) RestaurantsPriceRange2 — numeric tier encodings")
prc = F.col("attributes.RestaurantsPriceRange2")
PRICE_FORMS = {"bare digit '2'": prc.rlike(r"^\d+$"), "quoted u'2' / '2'": prc.rlike(r"^u?'\d+'$"),
               "decimal '2.0'": prc.rlike(r"^\d+\.\d+$"), "currency '$$'": prc.rlike(r"^\$+$"),
               "padded ' 2'": prc.rlike(r"^\s|\s$"), "Python None": prc == "None"}
x = df_bronze.agg(F.sum(prc.isNotNull().cast("int")).alias("nn"), *[F.sum(c_.cast("int")).alias(f"p{i}") for i, c_ in enumerate(PRICE_FORMS.values())]).first()
forms = {k: int(x[f"p{i}"] or 0) for i, k in enumerate(PRICE_FORMS)}
print("   " + " | ".join(f"{k}: {v:,}" for k, v in forms.items()))
dq_add(findings_633, "df_attributes", "attr_restaurants_price_range_2 (raw notation)", "one numeric notation (bare digit) besides None",
       int(x["nn"] or 0) - forms["Python None"], sum(v for k, v in forms.items() if k not in ("bare digit '2'", "Python None")),
       note="'None' literal handled as masked NULL in 6.3.6 / DQ-BUSINESS-002")

# ══ E) postal_code — numeric-looking string ══
print(f"\n{'─' * 60}\n📌 E) postal_code — numeric artefacts & notation variants")
pc_ = F.col("postal_code")
us_state = F.col("state").isin(list(US_STATES))
ca_prov = F.col("state").isin(list(CA_PROVINCES))
PC_FORMS = {
    "float artefact ('19103.0')":            (pc_.rlike(r"^\d+\.\d+$"), "FINDING"),
    "scientific notation ('1.9E4')":        (pc_.rlike(r"^\d(\.\d+)?[eE]\d+$"), "FINDING"),
    "lost leading zero (4-digit US ZIP)":   (us_state & pc_.rlike(r"^\d{4}$"), "FINDING"),
    "ZIP+4 mixed with ZIP5":                (pc_.rlike(r"^\d{5}-?\d{4}$"), "INFO"),
    "thousands separator / space in ZIP":   (us_state & pc_.rlike(r"^\d{1,3}[ ,]\d{2,3}$"), "FINDING"),
    "CA code without middle space":         (ca_prov & pc_.rlike(r"^[A-Za-z]\d[A-Za-z]\d[A-Za-z]\d$"), "FINDING"),
    "CA code with lower-case letters":      (ca_prov & pc_.rlike("[a-z]"), "FINDING"),
    "CA code with '-' instead of space":    (ca_prov & pc_.rlike(r"^[A-Za-z]\d[A-Za-z]-\d[A-Za-z]\d$"), "FINDING"),
}
x = df_business.agg(*[F.sum(F.coalesce(c_, F.lit(False)).cast("int")).alias(f"e{i}") for i, (c_, _) in enumerate(PC_FORMS.values())]).first()
for i, (lbl, (c_, sev)) in enumerate(PC_FORMS.items()):
    n_ = int(x[f"e{i}"] or 0)
    dq_add(findings_633, "df_business", f"postal_code ({lbl})", "one canonical notation per country", N_BUS, n_,
           status=(sev if n_ else "PASS"), examples=dq_examples(df_business, "postal_code", c_) if n_ else "")

# ══ F) address — house-number representation ══
print(f"\n{'─' * 60}\n📌 F) address — house-number & street-number formats")
ad = F.col("address")
ORDINAL_WORDS = "First|Second|Third|Fourth|Fifth|Sixth|Seventh|Eighth|Ninth|Tenth|Eleventh|Twelfth"
ADDR_FORMS = {
    "numeric house number ('1616 Chapala St')": ad.rlike(r"^\d+\s"),
    "house-number range ('100-102 Main St')":    ad.rlike(r"^\d+\s*-\s*\d+\s"),
    "fractional house number ('12 1/2', '½')":  ad.rlike(r"^\d+\s+1/2\b|½"),
    "alphanumeric house number ('123A', 'W123')": ad.rlike(r"^(\d+[A-Za-z]\b|[NSEW]\d+\b)"),
    "PO Box":                                    ad.rlike(r"(?i)\bp\.?\s*o\.?\s*box\b"),
    "ordinal street as digits ('5th Ave')":      ad.rlike(r"(?i)\b\d+(st|nd|rd|th)\b"),
    "ordinal street as words ('Fifth Ave')":     ad.rlike(rf"(?i)\b({ORDINAL_WORDS})\s+(St|Street|Ave|Avenue)\b"),
    "no number at all (non-empty)":              (ad != "") & ~ad.rlike(r"\d"),
}
x = df_business.agg(*[F.sum(c_.cast("int")).alias(f"a{i}") for i, c_ in enumerate(ADDR_FORMS.values())]).first()
addr_tbl = [{"Format": k, "Rows": int(x[f"a{i}"] or 0), "Pct": round(int(x[f"a{i}"] or 0) / N_BUS * 100, 3),
             "Examples": dq_examples(df_business, "address", c_, 3)} for i, (k, c_) in enumerate(ADDR_FORMS.items())]
display(pd.DataFrame(addr_tbl))
n_word_ord = addr_tbl[6]["Rows"]
dq_add(findings_633, "df_business", "address (ordinal street notation)", "one notation for numbered streets ('5th' vs 'Fifth')",
       addr_tbl[5]["Rows"] + n_word_ord, min(addr_tbl[5]["Rows"], n_word_ord), status="INFO",
       examples=f"digits: {addr_tbl[5]['Rows']:,} | words: {n_word_ord:,}",
       note="USPS Pub. 28 keeps the name as written; matters for address matching / dedup, not for storage")
dq_add(findings_633, "df_business", "address (non-plain house numbers)", "plain integer house number", N_BUS,
       addr_tbl[1]["Rows"] + addr_tbl[2]["Rows"] + addr_tbl[3]["Rows"], status="INFO",
       examples=f"ranges {addr_tbl[1]['Rows']:,}, fractions {addr_tbl[2]['Rows']:,}, alphanumeric {addr_tbl[3]['Rows']:,}",
       note="valid real-world forms — do not cast a house-number column to INT in Silver")

# ══ G) name — purely numeric ══
n_num = df_business.where(F.col("name").rlike(r"^[\d\s.,#-]+$")).count()
dq_add(findings_633, "df_business", "name (purely numeric)", "a business name contains letters", N_BUS, n_num, status="INFO" if n_num else "PASS",
       examples=dq_examples(df_business, "name", F.col("name").rlike(r"^[\d\s.,#-]+$")) if n_num else "",
       note="usually genuine brand names (e.g. '24', '1912') — confirm against categories")

findings_633_df = dq_summary(findings_633, "6.3.3 Numeric Format Consistency")
print("NUMERIC FORMAT CONSISTENCY CHECK COMPLETE")
print("=" * 80)

---
#### 🚨 DQ-BUSINESS-027 — Coarse (City / ZIP-Centroid) and Mixed-Precision Coordinates

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-027` |
| **Severity** | LOW (numeric format / accuracy; values are valid but not equally precise) |
| **Detected In** | Phase 6.3 — Format & Encoding Standardization → **6.3.3 Numeric Format Consistency** (Cell 68), check B "latitude/longitude precision" (decimal-places distribution) |
| **Also Confirmed In** | 6.2.2 (Cell 46): 3 coordinates > 100 km from every Yelp metro (DQ-BUSINESS-011). Sign convention, swapped axes and scientific notation all PASS |
| **Affected Columns** | `latitude`, `longitude` |
| **Total Affected Rows** | **348** of 150,346 businesses (0.23%) with ≤ 3 decimals on the least precise axis (1 dp: 41, 2 dp: 119, 3 dp: 188); in addition **29,229** (19.4%) carry ≥ 9 decimals |
| **Invalid Value** | `Nothing But Family (Nashville) 36.06, -86.67`; `Reno Oriental Rug Wash (Sparks) 39.54, -119.75`; `CVS Pharmacy (Tampa) 28.0604, -82.541` |
| **Domain Rule** | A storefront needs ≥ 4 decimals (≈ 11 m); 2 decimals ≈ 1.1 km, 1 decimal ≈ 11 km |

**Description:** The decimal-place profile (mode 7 dp = 46.8%, 6 dp = 26.8%, 10 dp = 16.1%) shows coordinates coming from several geocoders. The 348 coarse points are city- or ZIP-centroid geocodes (134 of them belong to businesses with an empty `address`, i.e. service-area businesses that hide their street location): they plot several unrelated businesses on the same spot and can be off by up to ~1-11 km, which distorts distance features, nearest-competitor counts and map clustering. The 9-10 dp values are a float-to-string artefact of another source; they are harmless but confirm the mixed provenance.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep values; add `geo_precision_dp` and `geo_quality` (`rooftop` ≥ 4 dp / `approximate` ≤ 3 dp) in Silver** | No value is invented; downstream models can down-weight or exclude approximate points; cheap and transparent |
| 2 | **Re-geocode the 348 rows from `address` + `city` + `postal_code`** | Gives true precision, but needs an external geocoder, and 134 of the 348 (38.5%) have an empty address (DQ-BUSINESS-032), so they cannot be re-geocoded |
| 3 | **Round all coordinates to a common precision (e.g. 6 dp)** | Removes the format inconsistency for the 9-10 dp rows, but cannot add precision to coarse ones |
| 4 | **NULL the coarse coordinates** | Loses metro-level location that is still useful for city-level analytics |

**Best Logical Solution:** **Option 1 — keep values and expose a precision / quality flag**, optionally followed by Option 2 for the rows that do have a full address. Precision is metadata about the value, so it should be recorded, not hidden.

---

> **Cross-references from 6.3.3 (already documented, no new ID):** `stars` stored as DOUBLE instead of DECIMAL(2,1) → **DQ-BUSINESS-001** (no float residue found). Lower-case Canadian postal code `T6x 2H4` (1) → **DQ-BUSINESS-016**. `RestaurantsPriceRange2` `'None'` → **DQ-BUSINESS-002**. Purely numeric name with trailing space (`'2300 '`) → **DQ-BUSINESS-004**.
>
> **Reviewed and not raised (valid real-world forms):** `review_count` is BIGINT with no sentinels (range 5 - 7,568); `RestaurantsPriceRange2` uses only bare digits `1`-`4`; no float / scientific / lost-leading-zero / ZIP+4 / separator artefacts in `postal_code`; house-number ranges (1,007), fractions (61), alphanumeric (463) and PO Boxes (4) are legitimate USPS forms (never cast house numbers to INT); ordinal street words vs digits (132 vs 12,975) is part of address standardisation (DQ-BUSINESS-031); 10 purely numeric names are genuine brand names (`24`, `1984`, `360`).

---

In [0]:
# 6.3.4) Boolean Representation Consistency  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Bronze has NO native BOOLEAN column, yet ~80 yes/no facts. They are encoded 4 different ways:
#   is_open        BIGINT 0/1
#   24 attributes  STRING Python literal 'True' / 'False' / 'None'
#   7 nested dicts Python tokens True / False / None inside a dict string (keys u'k' or 'k')
#   enums          negation as a category word: WiFi 'no', Smoking 'no', BYOBCorkage 'no', Alcohol 'none'
# Checks:
#   A) Encoding inventory across columns (same concept, several physical encodings)
#   B) 24 boolean attributes — raw token matrix; alternative spellings (true/TRUE/1/0/yes/Y/u'True')
#   C) Nested dicts — value tokens ∈ {True, False, None}; key quoting style mixed within a dict
#   D) Boolean semantics hidden in enums
#   E) Typed frames (TE.1) — BOOLEAN NULL count == raw NULL + raw 'None' (no token silently lost)
# The Python 'None' literal as a masked NULL is profiled in 6.3.6 (DQ-BUSINESS-002).
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import types as T

print("=" * 80)
print("6.3.4) BOOLEAN REPRESENTATION CONSISTENCY  —  Business DataFrames (TE.2)")
print("    One encoding per yes/no fact")
print("=" * 80)

N_BUS = df_business.count()
findings_634 = []
CANON = ["True", "False", "None"]

# ══ A) Encoding inventory ══
print(f"\n{'─' * 60}\n📌 A) Encoding inventory — how yes/no facts are physically stored in bronze")
n_dict_keys = sum(len(km) for _, km in DICT_TABLE_MAP.values())
inv = pd.DataFrame([
    {"Encoding": "BIGINT 0 / 1", "Fields": 1, "Where": "is_open", "Example": "1"},
    {"Encoding": "STRING 'True' / 'False' / 'None'", "Fields": len(ATTR_BOOLEAN_MAP), "Where": "attributes.<24 boolean attrs>", "Example": "'True'"},
    {"Encoding": "Python token inside dict string", "Fields": n_dict_keys, "Where": f"{len(DICT_TABLE_MAP)} nested dicts", "Example": "{'garage': False, ...}"},
    {"Encoding": "Negative category word in an enum", "Fields": 4, "Where": "WiFi / Smoking / BYOBCorkage 'no', Alcohol 'none'", "Example": "u'no'"},
])
display(inv)
native_bool = [p for p, dt in bronze_leaves(df_bronze.schema) if isinstance(dt, T.BooleanType)]
dq_add(findings_634, "df_bronze", "yes/no facts (physical encodings)", "one boolean encoding across the table",
       int(inv["Fields"].sum()), len(inv) - 1, status="INFO",
       examples="; ".join(f"{r['Encoding']} × {r['Fields']}" for _, r in inv.iterrows()),
       note=f"native BOOLEAN leaves in bronze: {len(native_bool)}; TE.1 unifies to BOOLEAN (is_open, 24 attrs, 7 MAP<STRING,BOOLEAN>)")

# ══ B) 24 boolean attributes — raw token matrix ══
print(f"\n{'─' * 60}\n📌 B) 24 boolean attributes — raw token matrix")
ALT_BOOL = r"(?i)^\s*(u?'(true|false|none)'|true|false|t|f|y|n|yes|no|1|0|on|off|null|nan)\s*$"
aggs = []
for b_, s_ in ATTR_BOOLEAN_MAP.items():
    raw = F.col(f"attributes.{b_}")
    aggs += [F.sum((raw == "True").cast("int")).alias(f"{s_}__T"), F.sum((raw == "False").cast("int")).alias(f"{s_}__F"),
             F.sum((raw == "None").cast("int")).alias(f"{s_}__N"), F.sum(raw.isNull().cast("int")).alias(f"{s_}__null"),
             F.sum((raw.isNotNull() & ~raw.isin(*CANON)).cast("int")).alias(f"{s_}__other"),
             F.sum((raw.isNotNull() & ~raw.isin(*CANON) & raw.rlike(ALT_BOOL)).cast("int")).alias(f"{s_}__alt")]
bm = df_bronze.agg(*aggs).first()
matrix = pd.DataFrame([{"Attribute": s_, "'True'": bm[f"{s_}__T"], "'False'": bm[f"{s_}__F"], "'None'": bm[f"{s_}__N"],
                        "SQL NULL": bm[f"{s_}__null"], "Other token": bm[f"{s_}__other"], "Alt boolean spelling": bm[f"{s_}__alt"]}
                       for s_ in ATTR_BOOLEAN_MAP.values()])
display(matrix)
n_bool_nn = int((matrix["'True'"] + matrix["'False'"] + matrix["'None'"] + matrix["Other token"]).sum())
dq_add(findings_634, "df_attributes", "24 boolean attrs (alternative spellings)", "only 'True' / 'False' / 'None' (no true/TRUE/1/0/yes/Y/u'True')",
       n_bool_nn, int(matrix["Other token"].sum()),
       examples=", ".join(f"{r['Attribute']}={r['Other token']}" for _, r in matrix[matrix["Other token"] > 0].iterrows()))
dq_add(findings_634, "df_attributes", "24 boolean attrs (case variants)", "exact case 'True' / 'False'",
       n_bool_nn, int(matrix["Alt boolean spelling"].sum()))
both_null_forms = matrix[(matrix["'None'"] > 0) & (matrix["SQL NULL"] > 0)]
dq_add(findings_634, "df_attributes", "24 boolean attrs (two 'unknown' encodings)", "one encoding for unknown (SQL NULL vs 'None')",
       len(matrix), len(both_null_forms), status="INFO" if len(both_null_forms) else "PASS",
       examples=", ".join(f"{a_} ('None'={n_:,})" for a_, n_ in both_null_forms.sort_values("'None'", ascending=False).head(5)[["Attribute", "'None'"]].itertuples(index=False)),
       note="attribute columns where SQL NULL and the 'None' literal co-exist — DQ-BUSINESS-002; detail in 6.3.6")
only_true = matrix[(matrix["'False'"] == 0) & (matrix["'True'"] > 0)]
only_false = matrix[(matrix["'True'"] == 0) & (matrix["'False'"] > 0)]
dq_add(findings_634, "df_attributes", "24 boolean attrs (one-sided columns)", "both TRUE and FALSE observed",
       len(matrix), len(only_true) + len(only_false), status="INFO" if len(only_true) + len(only_false) else "PASS",
       examples=", ".join(list(only_true["Attribute"]) + list(only_false["Attribute"])),
       note="a one-sided flag means FALSE is never recorded and absence (NULL) is the implicit 'no'")

# ══ C) Nested dicts — value tokens and key quoting ══
print(f"\n{'─' * 60}\n📌 C) Nested dicts — value tokens & key quoting")
for a_, (tbl, keymap) in DICT_TABLE_MAP.items():
    s = F.col(f"attributes.{a_}")
    d = df_bronze.select(s.alias("s")).where(F.col("s").rlike(r"^\{"))
    toks = d.select(F.explode(F.regexp_extract_all("s", F.lit(r":\s*([^,}]+)"), F.lit(1))).alias("tok")).withColumn("tok", F.trim("tok"))
    t = {r_["tok"]: r_["count"] for r_ in toks.groupBy("tok").count().collect()}
    n_tok = sum(t.values()); bad = {k: v for k, v in t.items() if k not in CANON}
    dq_add(findings_634, DICT_FRAME_NAMES[a_] if "DICT_FRAME_NAMES" in globals() else tbl, f"{a_} (value tokens)",
           "every dict value ∈ {True, False, None}", n_tok, sum(bad.values()),
           examples=", ".join(f"{k}={v:,}" for k, v in bad.items()))
    k = d.agg(F.sum(F.col("s").rlike(r"u'[^']+'\s*:").cast("int")).alias("u"),
              F.sum(F.col("s").rlike(r"(^\{|,\s*)'[^']+'\s*:").cast("int")).alias("p"),
              F.sum((F.col("s").rlike(r"u'[^']+'\s*:") & F.col("s").rlike(r"(^\{|,\s*)'[^']+'\s*:")).cast("int")).alias("both")).first()
    u_, p_, both_ = int(k["u"] or 0), int(k["p"] or 0), int(k["both"] or 0)
    dq_add(findings_634, DICT_FRAME_NAMES[a_] if "DICT_FRAME_NAMES" in globals() else tbl, f"{a_} (key quoting style)",
           "one key quoting style (u'k' or 'k') per dict", d.count(), min(u_, p_) + both_, status="INFO" if (u_ and p_) else "PASS",
           examples=f"u'k' dicts = {u_:,} | 'k' dicts = {p_:,} | mixed inside one dict = {both_:,}",
           note="Python-2 vs Python-3 repr leaking from the source export (same root cause as DQ-BUSINESS-003); TE.1 dec_dict() strips the u-prefix")

# ══ D) Boolean semantics hidden in enum categories ══
print(f"\n{'─' * 60}\n📌 D) Boolean semantics hidden in enum categories")
NEG_ENUM = {"attr_wifi": "no", "attr_smoking": "no", "attr_byob_corkage": "no", "attr_alcohol": "none"}
x = df_attributes.agg(*[F.sum((F.col(c) == v).cast("int")).alias(c) for c, v in NEG_ENUM.items()],
                      *[F.sum(F.col(c).isNotNull().cast("int")).alias(f"{c}__nn") for c in NEG_ENUM]).first()
for c, v in NEG_ENUM.items():
    dq_add(findings_634, "df_attributes", f"{c} = '{v}'", "negation stored as a category word, not FALSE", x[f"{c}__nn"], x[c], status="INFO",
           note="valid enum member; when deriving has_wifi / allows_smoking / byob / serves_alcohol flags map this value to FALSE")

# ══ E) Typed frames — lossless boolean decode ══
print(f"\n{'─' * 60}\n📌 E) Typed df_attributes — BOOLEAN NULLs == raw NULL + raw 'None'")
typed_nulls = df_attributes.agg(*[F.sum(F.col(s_).isNull().cast("int")).alias(s_) for s_ in ATTR_BOOLEAN_MAP.values()]).first()
diff = {s_: int(typed_nulls[s_] or 0) - int(bm[f"{s_}__null"] or 0) - int(bm[f"{s_}__N"] or 0) for s_ in ATTR_BOOLEAN_MAP.values()}
wrong_type = [s_ for s_ in ATTR_BOOLEAN_MAP.values() if not isinstance(df_attributes.schema[s_].dataType, T.BooleanType)]
dq_add(findings_634, "df_attributes", "24 boolean attrs (typed BOOLEAN)", "BooleanType and NULLs = raw NULL + 'None'", len(diff),
       sum(1 for v in diff.values() if v != 0) + len(wrong_type),
       examples=", ".join(f"{k}: {v:+,}" for k, v in diff.items() if v) + (f" | non-boolean: {wrong_type}" if wrong_type else ""))
x = df_business.agg(F.sum(F.col("is_open").isNull().cast("int")).alias("n")).first()
dq_add(findings_634, "df_business", "is_open (BIGINT 0/1 → BOOLEAN)", "every 0/1 decoded to FALSE/TRUE", N_BUS, x["n"],
       note=f"typed as {df_business.schema['is_open'].dataType.simpleString()}")

findings_634_df = dq_summary(findings_634, "6.3.4 Boolean Representation Consistency")
print("BOOLEAN REPRESENTATION CONSISTENCY CHECK COMPLETE")
print("=" * 80)

---
#### 🚨 DQ-BUSINESS-028 — Yes/No Facts Stored in Four Physical Encodings (BIGINT 0/1, `'True'`/`'False'` Strings, Dict Tokens with `u'k'` / `'k'` Keys, Negative Enum Words)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-028` |
| **Severity** | MEDIUM (format / encoding; no wrong values, but no native BOOLEAN anywhere and parsing is mandatory before any use) |
| **Detected In** | Phase 6.3 — Format & Encoding Standardization → **6.3.4 Boolean Representation Consistency** (Cell 70), checks A "encoding inventory", C "nested-dict key quoting" and D "negation as category word" |
| **Affected Columns** | `is_open` (BIGINT), 24 boolean `attributes.*` (STRING), 7 dict-string attributes (`Ambience`, `BestNights`, `BusinessParking`, `DietaryRestrictions`, `GoodForMeal`, `HairSpecializesIn`, `Music`), 4 enums (`WiFi`, `Smoking`, `BYOBCorkage`, `Alcohol`) |
| **Total Affected Rows** | 78 yes/no facts in **0** native BOOLEAN columns. Dict key-quoting split: **12,255** dicts use Python-2 `u'k'` keys vs `'k'` in the rest (Ambience 4,713 / 43,728; BusinessParking 4,314 / 88,814; GoodForMeal 1,816 / 28,784; Music 771 / 7,486; BestNights 506 / 5,683; HairSpecializesIn 135 / 1,035; DietaryRestrictions 0 / 30). Negation as category word: WiFi `no` 21,831, Alcohol `none` 20,910, Smoking `no` 2,405, BYOBCorkage `no` 747 |
| **Invalid Value** | `is_open = 1`; `RestaurantsTakeOut = 'True'`; `BusinessParking = "{u'garage': False, u'street': True, ...}"` vs `"{'garage': False, 'street': True, ...}"` |
| **Domain Rule** | A yes/no fact is stored as BOOLEAN (TRUE / FALSE / NULL), with one encoding for the whole table (R5 Semantic Data Types of the Silver DDL) |

**Description:** Inside each encoding the data is clean: the 24 boolean attributes use only exact-case `'True'` / `'False'` / `'None'` (0 alternative spellings, 0 case variants, both TRUE and FALSE observed in every column); every dict value is a Python `True` / `False` / `None` token, and no dict mixes `u'k'` and `'k'` keys internally. The problem is the *mix across columns*: four parsers are required, the dict strings are Python `repr()` rather than JSON (so `from_json` fails on `u'` prefixes and on `True` / `None`), and the 2/3 key-quoting split shows two source export versions (same root cause as DQ-BUSINESS-003). Negation-as-enum (`WiFi = 'no'`) is a valid category but must be mapped explicitly when a boolean flag (`has_wifi`) is derived.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Decode once in Silver: `is_open` → BOOLEAN; 24 attrs via an exact map `'True'→TRUE`, `'False'→FALSE`, `'None'`/NULL→NULL; dict strings via one anchored parser that strips the `u` prefix and maps Python tokens, landing in the 7 child tables (BOOLEAN flags); enums kept as enums, with derived boolean flags documented (`has_wifi = WiFi IN ('free','paid')`)** | Matches the Silver DDL (R3/R5), one deterministic rule per encoding, already prototyped in TE.1 (`dec_bool`, `dec_dict`); no value changes |
| 2 | **Convert dict strings to JSON (`replace("'", '"')`, `True→true`) and use `from_json`** | Standard tooling, but naive quote replacement breaks on apostrophes inside keys/values and is fragile |
| 3 | **Python `ast.literal_eval` UDF** | Exact Python semantics, but row-by-row UDF is slow on 150k × 7 dicts and not Spark-Connect friendly at scale |
| 4 | **Keep STRING columns in Silver** | No work, but violates the contract and pushes parsing to every consumer |

**Best Logical Solution:** **Option 1 — encoding-specific deterministic decoders into native BOOLEAN columns / child tables**, validated with the lossless-decode check from TE.1. It is exact, fast (native Spark functions) and satisfies the Silver contract.

---

> **Cross-references from 6.3.4 (already documented, no new ID):** 21 of 24 boolean attributes mix SQL NULL and the `'None'` literal (largest: RestaurantsDelivery 3,948, RestaurantsTakeOut 2,105, OutdoorSeating 1,882) → **DQ-BUSINESS-002**. `Open24Hours` sparsity / conflict → **DQ-BUSINESS-026**. BYOB policy conflicts → **DQ-BUSINESS-024**.
>
> **Reviewed and not raised:** no alternative boolean spellings (`true`, `TRUE`, `1`, `yes`, `Y`, `u'True'`); no one-sided boolean columns; all dict value tokens in the {True, False, None} domain; `is_open` 0/1 decodes losslessly.

---

In [0]:
# 6.3.5) Text Normalization Issues  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Free-text identity fields of a business: name, address, city (+ state, postal_code, category_name).
# These drive entity resolution (chain detection, dedup), geocoding and joins to external data, so
# every invisible difference matters. Detection only — TE.1/TE.2 keep values untouched.
#   A) Hygiene         : leading/trailing ws, multi-space, control chars (\t \n \r \x00), NBSP,
#                        zero-width / BOM, empty-after-trim
#   B) Capitalisation  : ALL CAPS / all lower / odd interior caps ('EdMonton') per column
#   C) Encoding        : curly vs straight quotes, HTML entities, mojibake, decomposed (NFD) accents, non-ASCII
#   D) Punctuation     : trailing ',' ';' '.' on name / address / city
#   E) Address tokens  : USPS Pub. 28 suffix / unit / directional variants (Street–St–St., Suite–Ste–#, North–N)
# Spelling-variant clusters were measured in 6.3.1; structural patterns in 6.2.3.
import pandas as pd
from pyspark.sql import functions as F

print("=" * 80)
print("6.3.5) TEXT NORMALIZATION ISSUES  —  Business DataFrames (TE.2)")
print("    Whitespace, casing, encoding and abbreviation consistency")
print("=" * 80)

N_BUS = df_business.count()
N_CAT = df_categories.count()
findings_635 = []
TEXT_COLS = [("df_business", df_business, c, N_BUS) for c in ("name", "address", "city", "state", "postal_code")] + \
            [("df_categories", df_categories, "category_name", N_CAT)]

# ══ A) Hygiene — one aggregation pass per frame ══
print(f"\n{'─' * 60}\n📌 A) Whitespace & invisible characters")
HYGIENE = {
    "leading/trailing whitespace": (lambda c: F.col(c) != F.trim(F.col(c)), "DQ-BUSINESS-004"),
    "multiple consecutive spaces": (lambda c: F.col(c).rlike("  +"), ""),
    "tab / newline / carriage return": (lambda c: F.col(c).rlike("[\t\n\r]"), ""),
    "other control chars (\\x00–\\x1F, \\x7F)": (lambda c: F.col(c).rlike("[\\x00-\\x08\\x0B\\x0C\\x0E-\\x1F\\x7F]"), ""),
    "non-breaking space (U+00A0)": (lambda c: F.col(c).rlike("\u00A0"), "DQ-BUSINESS-017"),
    "zero-width / BOM (U+200B–200D, 2060, FEFF)": (lambda c: F.col(c).rlike("[\u200B-\u200D\u2060\uFEFF]"), "DQ-BUSINESS-017"),
    "empty after trim": (lambda c: (F.trim(F.col(c)) == ""), "masked-null view in 6.3.6"),
}
hyg_rows = []
for fname, fr, c, n in TEXT_COLS:
    x = fr.agg(*[F.sum(F.coalesce(fn(c), F.lit(False)).cast("int")).alias(f"h{i}") for i, (fn, _) in enumerate(HYGIENE.values())]).first()
    row = {"DataFrame": fname, "Column": c}
    for i, (lbl, (fn, ref)) in enumerate(HYGIENE.items()):
        v = int(x[f"h{i}"] or 0); row[lbl] = v
        if v:
            dq_add(findings_635, fname, f"{c} ({lbl})", "no invisible / redundant whitespace", n, v,
                   examples=", ".join(repr(r_[0]) for r_ in fr.where(fn(c)).select(c).distinct().limit(5).collect()), note=ref)
    if not any(row[l] for l in HYGIENE):
        dq_add(findings_635, fname, f"{c} (whitespace & invisible chars)", "no invisible / redundant whitespace", n, 0)
    hyg_rows.append(row)
print("\n   Hygiene matrix (rows affected):")
display(pd.DataFrame(hyg_rows))

# ══ B) Capitalisation ══
print(f"\n{'─' * 60}\n📌 B) Capitalisation consistency")
def case_class(c):
    v = F.col(c)
    return (F.when(~v.rlike("[A-Za-z]"), "no letters")
             .when(v == F.upper(v), "ALL CAPS")
             .when(v == F.lower(v), "all lower")
             .when(v.rlike("[a-z][A-Z]") & ~v.rlike("(Mc|Mac|O')[A-Z]"), "interior caps")
             .otherwise("Mixed / Title"))
case_tbl = (df_business.select(*[case_class(c).alias(c) for c in ("name", "address", "city")])
            .select(F.explode(F.array(*[F.struct(F.lit(c).alias("column"), F.col(c).alias("case_class")) for c in ("name", "address", "city")])).alias("x"))
            .groupBy("x.column", "x.case_class").count().orderBy("column", F.desc("count")))
display(case_tbl)
cc = {(r_["column"], r_["case_class"]): r_["count"] for r_ in case_tbl.collect()}
# city: proper nouns — anything but Title case is a normalisation defect
dq_add(findings_635, "df_business", "city (ALL CAPS / all lower)", "Title-case place name", N_BUS,
       cc.get(("city", "ALL CAPS"), 0) + cc.get(("city", "all lower"), 0),
       examples=dq_examples(df_business, "city", case_class("city").isin("ALL CAPS", "all lower")))
dq_add(findings_635, "df_business", "city (odd interior capitals)", "no 'EdMonton' / 'ClearWater'", N_BUS, cc.get(("city", "interior caps"), 0),
       examples=dq_examples(df_business, "city", case_class("city") == "interior caps"))
# address: an ALL CAPS / all lower address is a data-entry artefact (USPS uses upper-case only for mail pieces)
dq_add(findings_635, "df_business", "address (ALL CAPS / all lower)", "consistent mixed-case street address", N_BUS,
       df_business.where((F.col("address") != "") & case_class("address").isin("ALL CAPS", "all lower") & F.col("address").rlike("[A-Za-z]{3}")).count(),
       examples=dq_examples(df_business, "address", (F.col("address") != "") & case_class("address").isin("ALL CAPS", "all lower") & F.col("address").rlike("[A-Za-z]{3}")))
# name: brand stylisation is legitimate ('ALDI', 'sweetgreen') → INFO only
n_name_case = df_business.where(case_class("name").isin("ALL CAPS", "all lower") & F.col("name").rlike("[A-Za-z]{4}")).count()
dq_add(findings_635, "df_business", "name (ALL CAPS / all lower)", "mixed-case name (brand stylisation allowed)", N_BUS, n_name_case, status="INFO",
       examples=dq_examples(df_business, "name", case_class("name").isin("ALL CAPS", "all lower") & F.col("name").rlike("[A-Za-z]{4}")),
       note="keep display name raw; use a case-folded key for matching")
x = df_categories.agg(F.sum((~F.col("category_name").rlike("^[A-Z0-9]")).cast("int")).alias("n")).first()
dq_add(findings_635, "df_categories", "category_name (initial capital)", "Yelp taxonomy labels start upper-case", N_CAT, x["n"],
       examples=dq_examples(df_categories, "category_name", ~F.col("category_name").rlike("^[A-Z0-9]")),
       note="fragments of labels split on the ', ' delimiter — same root cause as DQ-BUSINESS-013")

# ══ C) Encoding / Unicode ══
print(f"\n{'─' * 60}\n📌 C) Encoding & Unicode representation")
MOJIBAKE = "Ã[\u0080-\u00BF]|Â[\u0080-\u00BF]|â€[\u0080-\u00BF™œ˜šž]|\uFFFD"
ENC = {
    "curly apostrophe / quotes (’ ‘ “ ”)": ("[\u2018\u2019\u201C\u201D]", "INFO", "same brand split by ' vs ’ (McDonald's / McDonald’s); fold to ASCII in the match key"),
    "HTML entities (&amp; &#39; &quot;)":   ("&(amp|lt|gt|quot|apos|nbsp|#\\d{1,5}|#x[0-9a-fA-F]{1,4});", "FINDING", ""),
    "mojibake (Ã©, â€™, U+FFFD)":            (MOJIBAKE, "FINDING", "UTF-8 decoded as Latin-1 upstream"),
    "decomposed accents (NFD combining marks)": ("[\u0300-\u036F]", "FINDING", "'e\u0301' ≠ 'é' byte-wise — normalise to NFC"),
    "non-ASCII letters (accents, CJK)":       ("[\\p{L}&&[^\\x00-\\x7F]]", "INFO", "legitimate (Café, Edmonton French names); keep, but NFC-normalise"),
}
for c in ("name", "address", "city"):
    x = df_business.agg(*[F.sum(F.col(c).rlike(p).cast("int")).alias(f"e{i}") for i, (p, _, _) in enumerate(ENC.values())]).first()
    for i, (lbl, (p, sev, note)) in enumerate(ENC.items()):
        v = int(x[f"e{i}"] or 0)
        dq_add(findings_635, "df_business", f"{c} ({lbl})", "one canonical character representation", N_BUS, v,
               status=(sev if v else "PASS"), examples=dq_examples(df_business, c, F.col(c).rlike(p), 4) if v else "", note=note if v else "")

# ══ D) Trailing punctuation ══
print(f"\n{'─' * 60}\n📌 D) Trailing punctuation")
for c, pat, note in (("name", r"[,;:]\s*$", ""), ("address", r"[,;]\s*$", ""), ("city", r"[,;]\s*$", "DQ-BUSINESS-018 (abbreviations such as 'Twp.' / 'Hts.' are not counted)")):
    v = df_business.where(F.col(c).rlike(pat)).count()
    dq_add(findings_635, "df_business", f"{c} (trailing , ; punctuation)", "no dangling separator at the end", N_BUS, v,
           examples=dq_examples(df_business, c, F.col(c).rlike(pat)) if v else "", note=note)

# ══ E) Address abbreviation standard (USPS Publication 28) ══
print(f"\n{'─' * 60}\n📌 E) Address tokens vs USPS Pub. 28 standard abbreviations")
ADDR_STD = {   # concept : (USPS standard, [variants that mean the same thing])
    "Street":    ("st",   ["street", "st.", "str"]),
    "Avenue":    ("ave",  ["avenue", "ave.", "av"]),
    "Boulevard": ("blvd", ["boulevard", "blvd."]),
    "Road":      ("rd",   ["road", "rd."]),
    "Drive":     ("dr",   ["drive", "dr."]),
    "Highway":   ("hwy",  ["highway", "hwy."]),
    "Parkway":   ("pkwy", ["parkway", "pkwy."]),
    "Lane":      ("ln",   ["lane", "ln."]),
    "Suite":     ("ste",  ["suite", "ste."]),
    "Unit":      ("unit", ["#"]),
    "North":     ("n",    ["north", "n."]),
    "South":     ("s",    ["south", "s."]),
    "East":      ("e",    ["east", "e."]),
    "West":      ("w",    ["west", "w."]),
}
tok = (df_business.where(F.col("address") != "")
       .select("business_id", F.posexplode(F.split(F.lower(F.regexp_replace("address", "#", " # ")), r"[\s,]+")).alias("pos", "tok"))
       .where(F.col("pos") > 0))   # skip the house-number position: a leading 'N' / 'E' is part of a grid number
tok_counts = {r_["tok"]: r_["count"] for r_ in tok.groupBy("tok").count()
              .where(F.col("tok").isin([t for std, var in ADDR_STD.values() for t in [std] + var])).collect()}
std_rows = []
for concept, (std, var) in ADDR_STD.items():
    std_rows.append({"Concept": concept, "USPS standard": std.upper(), "Standard uses": tok_counts.get(std, 0),
                     **{f"variant {i + 1}": f"{v_} = {tok_counts.get(v_, 0):,}" for i, v_ in enumerate(var)},
                     "Non-standard uses": sum(tok_counts.get(v_, 0) for v_ in var)})
std_df = pd.DataFrame(std_rows)
display(std_df)
all_var = [v_ for _, var in ADDR_STD.values() for v_ in var]
n_addr_nonstd = tok.where(F.col("tok").isin(all_var)).select("business_id").distinct().count()
dq_add(findings_635, "df_business", "address (non-standard suffix / unit / directional)", "USPS Pub. 28 abbreviations (ST, AVE, STE, N …)",
       df_business.where(F.col("address") != "").count(), n_addr_nonstd, status="INFO",
       examples=", ".join(f"{r['Concept']}: {r['Non-standard uses']:,}" for _, r in std_df.sort_values("Non-standard uses", ascending=False).head(6).iterrows()),
       note="same street written several ways breaks address-level dedup / geocoder joins; standardise into a separate match column, keep raw")

findings_635_df = dq_summary(findings_635, "6.3.5 Text Normalization Issues")
print("TEXT NORMALIZATION CHECK COMPLETE")
print("=" * 80)

---
#### 🚨 DQ-BUSINESS-029 — Multiple Consecutive Spaces and Dangling Separators in `name`, `address`, `city`

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-029` |
| **Severity** | LOW (text normalization; invisible to readers, breaks exact matching and dedup) |
| **Detected In** | Phase 6.3 — Format & Encoding Standardization → **6.3.5 Text Normalization Issues** (Cell 72), checks A "multiple consecutive spaces" and D "trailing , ; punctuation" |
| **Also Confirmed In** | 6.3.1 (Cell 60): double-spaced spellings appear inside city clusters (`St  Louis`, `Mt  Juliet`, `Santa  Barbara`) and brand clusters (`h h  gregg`) |
| **Affected Columns** | `name`, `address`, `city` |
| **Total Affected Rows** | Multiple spaces: **name 263**, **address 245**, **city 9**. Dangling trailing separator: **name 1**, **address 6** (city 17 already in DQ-BUSINESS-018). ≈ **524** values in total (0.35% of rows) |
| **Invalid Value** | `'PAWS  Plaquemines Animal Welfare Society'`, `'David Gower, Jr.  - Coldwell Banker Preferred'`, `'10303  108 Street NW'`, `'14350 Mundy Dr, Ste  700'`, `'West  Deptford'`, `"Mom's Kitchen,"`, `'2810 Limekiln Pike,'`, `'150 2nd Ave N, 970,'` |
| **Domain Rule** | Words are separated by exactly one space; a value does not end with a list separator |

**Description:** These are keying artefacts (double-tap of the space bar, copy/paste of a multi-line address, a removed suite number leaving its comma). No control characters (`\t \n \r \x00`) were found in any field and `state` / `category_name` are clean. The impact is on equality-based logic: `'8330  82 Avenue'` and `'8330 82 Avenue'` are two addresses for an exact-match dedup or a join to a geocoder cache.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **In Silver apply `regexp_replace(x, '\\s+', ' ')` then strip a trailing `[,;]` (after the trim of DQ-BUSINESS-004 and invisible-char removal of DQ-BUSINESS-017)** | The change is meaning-preserving (whitespace / a dangling comma carry no information), deterministic and cheap; it is the industry-standard canonical text form |
| 2 | **Normalise only in derived match keys (`name_match_key`, `address_match_key`)** | Keeps raw text byte-identical, but displays and exports still show the artefacts |
| 3 | **Leave as-is** | No effort, but dedup and joins stay unreliable |

**Best Logical Solution:** **Option 1 — collapse whitespace and strip dangling separators in the cleaned Silver columns**, applied in one ordered text-hygiene step together with DQ-BUSINESS-004 and DQ-BUSINESS-017. It is lossless in meaning and benefits every consumer.

---

---
#### 🚨 DQ-BUSINESS-030 — Inconsistent Capitalisation in `city` and `address` (`TAMPA`, `tampa`, `EdMonton`, `4774 PARIS AVE`)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-030` |
| **Severity** | LOW (text normalization; feeds DQ-BUSINESS-021 city fragmentation) |
| **Detected In** | Phase 6.3 — Format & Encoding Standardization → **6.3.5 Text Normalization Issues** (Cell 72), check B "capitalisation consistency" |
| **Also Confirmed In** | 6.3.1 (Cell 60): case variants inside city clusters (`Tampa` 9,048 vs `TAMPA` 16 / `tampa` 2; `Tucson` vs `TUCSON` 6; `Philadelphia` vs `PHILADELPHIA` 4) |
| **Affected Columns** | `city`, `address` (and `name`, INFO only) |
| **Total Affected Rows** | **city 179** (120 ALL CAPS, 56 all lower, 3 odd interior capitals); **address 62** ALL CAPS / all lower containing at least one 3-letter word (the raw case table shows 564 ALL CAPS + 29 all lower, but most of those contain only numbers plus 1-2-letter tokens such as directionals or unit letters, which have no case to normalise); **name** 855 ALL CAPS / all lower and 4,018 interior caps — *informational only* |
| **Invalid Value** | `'TAMPA'`, `'NEW PORT RICHEY'`, `'abington'`, `'phoenixville'`, `'EdMonton'`, `'ClearWater'`, `'20 mullanphy gardens'`, `'593 s lowry st'`, `'4774 PARIS AVE'` |
| **Domain Rule** | Place names and street addresses use one case convention (Title case for display, upper case for USPS matching) |

**Description:** City and address casing is an entry habit, not information. For `name`, casing is frequently a deliberate brand mark (`ALDI`, `YMCA`, `CAVA`, `west elm`, `lululemon`, `GameStop`), so changing it would corrupt the display name; it is therefore not counted as an error here and is handled by the case-folded match key of DQ-BUSINESS-022.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **`city`: resolve through the canonical mapping of DQ-BUSINESS-021 (case-folded key → official name). `address`: case-fold only in `address_match_key`; keep raw display. `name`: never re-case** | The mapping gives correct official casing (`McKees Rocks`, `O'Fallon`) that `initcap()` cannot; addresses and names keep the owner's text while matching becomes case-insensitive |
| 2 | **`initcap()` on city and address** | Simple, but produces errors: `Mckees Rocks`, `O'fallon`, `Nw`, `Ste 2b`, `Us-19` |
| 3 | **`upper()` everything (USPS style)** | Good for postal matching, poor for display, and destroys brand casing if applied to `name` |
| 4 | **Leave as-is** | City groupings stay split (DQ-BUSINESS-021) |

**Best Logical Solution:** **Option 1 — dictionary-based canonical city, case-insensitive match keys for address/name.** It fixes the analytical impact without introducing new casing errors.

---

---
#### 🚨 DQ-BUSINESS-031 — Address Tokens Not in USPS Pub. 28 Standard Form (`Street` / `St.` vs `St`, `Suite` vs `Ste`, `North` vs `N`)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-031` |
| **Severity** | LOW / INFO (address standardisation; affects dedup and geocoding joins, not validity) |
| **Detected In** | Phase 6.3 — Format & Encoding Standardization → **6.3.5 Text Normalization Issues** (Cell 72), check E "non-standard suffix / unit / directional" |
| **Also Confirmed In** | 6.3.3 (Cell 68): ordinal street names written as words vs digits (`Second Ave` 132 vs `2nd Ave` 12,975) |
| **Affected Columns** | `address` |
| **Total Affected Rows** | **11,165** of 145,219 non-empty addresses (7.69%) contain at least one non-standard token |
| **Invalid Value** | Street: `street` 2,670 / `st.` 181 / `str` 6; Avenue: `avenue` 1,882 / `ave.` 99 / `av` 18; Road 1,283; West 1,157; South 1,078; East 896; North 852; Highway 696; Suite 564 (`suite` 533 / `ste.` 31); Drive 430; Boulevard 370; Unit `#` 189 |
| **Domain Rule** | USPS Publication 28 standard abbreviations (ST, AVE, BLVD, RD, DR, HWY, PKWY, LN, STE, UNIT, N/S/E/W); Canada Post equivalents for AB |

**Description:** The same physical address can be written `1616 North Chapala Street, Suite 2` or `1616 N Chapala St, Ste 2`. This is legitimate free-text variation, but it defeats exact-match deduplication (two listings of the same business), joins to external address / census datasets, and geocoding caches. Standard forms already dominate (e.g. `ST` 36,053 vs 2,857 variants).

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep raw `address`; add `address_standardized` (and `address_match_key`) in Silver using a token-level USPS Pub. 28 dictionary applied to whole words only (suffix in final street position, directional as prefix/suffix, unit designators), plus `5th`↔`Fifth` ordinal mapping** | Deterministic and auditable; only whole-word tokens are touched (avoids `Main St` → `Main Saint` errors); raw text preserved for display |
| 2 | **Use an address-parsing library (libpostal / usaddress) to split into components** | Most robust, gives house number / street / unit columns, but adds a native dependency and UDF cost |
| 3 | **Overwrite `address` with the standardised form** | Consistent, but loses what the owner entered and makes mistakes irreversible |
| 4 | **Leave as-is** | Valid for display; dedup stays weak |

**Best Logical Solution:** **Option 1 — derived standardised address column with whole-word USPS dictionary**, with Option 2 as a later enhancement if address-level matching becomes a core use case.

---

> **Cross-references from 6.3.5 (already documented, no new ID):** leading / trailing whitespace (name 150, address 3, city 49) → **DQ-BUSINESS-004**. NBSP / zero-width characters (name 15, address 10, city 4) → **DQ-BUSINESS-017**. Empty-after-trim `address` (5,127) → **DQ-BUSINESS-032**; `postal_code` (73) → **DQ-BUSINESS-015**. City trailing commas (17) → **DQ-BUSINESS-018**. `category_name` fragment `& Probates` (38) → **DQ-BUSINESS-013**. Curly apostrophes in names (701) → **DQ-BUSINESS-022**.
>
> **Reviewed and not raised:** 0 tab / newline / carriage-return / other control characters in any field; 0 HTML entities, 0 mojibake, 0 decomposed (NFD) accents; 621 names and 8 addresses with non-ASCII letters are legitimate (`Café`, `Poké`, `La Cañada Dr`) — keep, NFC-normalise.

---

In [0]:
# 6.3.6) Null Representation Consistency  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Non-standard nulls masquerading as data. Detection only — COUNTING missingness is 6.4.
# "Unknown" can be written in many ways in this table:
#   A) Top-level strings : SQL NULL vs '' vs whitespace-only vs 'None' vs masked tokens (n/a, '.', '-', unknown…)
#   B) Placeholder values: business names / addresses that are placeholders, not data
#   C) attributes        : NULL struct vs struct of all-NULL fields; per attribute SQL NULL vs Python 'None'
#                          literal (DQ-BUSINESS-002); quoted u'None' as an enum category; Alcohol 'none' = valid value
#   D) nested dicts      : SQL NULL vs 'None' vs '{}' vs dict whose every value is None (DQ-BUSINESS-006)
#   E) hours             : NULL struct vs NULL day inside a listed week vs '0:0-0:0' (DQ-BUSINESS-007)
#   F) categories        : NULL vs '' vs empty tokens
#   G) numeric sentinels : NaN, 0 / 0.0 used as "unknown" in stars / coordinates / review_count
# Ends with the consolidated 6.3 summary (findings_631 – findings_636).
import pandas as pd
from pyspark.sql import functions as F

print("=" * 80)
print("6.3.6) NULL REPRESENTATION CONSISTENCY  —  Business DataFrames (TE.2)")
print("    One representation for 'unknown'")
print("=" * 80)

N_BUS = df_business.count()
findings_636 = []
MASKED_TOKENS = ["n/a", "na", "n.a.", "null", "nil", "nan", "unknown", "unk", "tbd", "tba", "missing", "undefined",
                 "not available", "not applicable", "no address", "none listed", "empty", "blank", "#n/a", "#value!"]
PUNCT_ONLY = r"^[\s.\-_*?/#,\u2013\u2014]+$"

# ══ A) Top-level string columns — representation matrix ══
print(f"\n{'─' * 60}\n📌 A) Top-level strings — how 'missing' is written")
def null_forms(c):
    v = F.col(c); t = F.lower(F.trim(v))
    return {"SQL NULL": v.isNull(), "'' (empty)": v == "", "whitespace-only": (v != "") & (F.trim(v) == ""),
            "'None' literal": t == "none", "masked token": t.isin(MASKED_TOKENS),
            "punctuation-only": (F.trim(v) != "") & v.rlike(PUNCT_ONLY)}
STR_COLS = [("df_business", df_business, c) for c in ("name", "address", "city", "state", "postal_code")] + \
           [("df_bronze", df_bronze, "categories"), ("df_categories", df_categories, "category_name")]
mat_rows = []
for fname, fr, c in STR_COLS:
    forms = null_forms(c)
    x = fr.agg(F.count("*").alias("n"), *[F.sum(F.coalesce(e, F.lit(False)).cast("int")).alias(f"f{i}") for i, e in enumerate(forms.values())]).first()
    cnt = {k: int(x[f"f{i}"] or 0) for i, k in enumerate(forms)}
    masked = sum(v for k, v in cnt.items() if k != "SQL NULL")
    kinds = [k for k, v in cnt.items() if v]
    mat_rows.append({"DataFrame": fname, "Column": c, "Rows": x["n"], **cnt, "Representations in use": len(kinds)})
    ex_cond = F.lit(False)
    for k, e in forms.items():
        if k != "SQL NULL":
            ex_cond = ex_cond | F.coalesce(e, F.lit(False))
    dq_add(findings_636, fname, f"{c} (masked nulls)", "missing = SQL NULL only (no '', 'None', 'n/a', '.', …)", x["n"], masked,
           examples=(", ".join(f"{k}={v:,}" for k, v in cnt.items() if v and k != "SQL NULL") + " | " +
                     dq_examples(fr, c, ex_cond, 6)) if masked else "",
           note=f"forms in use: {', '.join(kinds) if kinds else 'none'}")
print("\n   Null-representation matrix:")
display(pd.DataFrame(mat_rows))

# ══ B) Placeholder values ══
print(f"\n{'─' * 60}\n📌 B) Placeholder names / addresses")
NAME_PH = r"(?i)^\s*(test|testing|test business|tbd|tba|n/?a|none|unknown|closed|permanently closed|coming soon|new business|business|untitled|placeholder|xxx+|asdf)\s*$"
ADDR_PH = r"(?i)^\s*(n/?a|none|unknown|tbd|address withheld|not provided|no address|private|mobile|various locations|serving .* area|by appointment( only)?|call for (address|location))\s*$"
for c, pat, note in (("name", NAME_PH, ""), ("address", ADDR_PH, "service-area businesses legitimately have no street address — should be NULL + a flag, not text")):
    v = df_business.where(F.col(c).rlike(pat)).count()
    dq_add(findings_636, "df_business", f"{c} (placeholder text)", "no placeholder values standing in for data", N_BUS, v,
           examples=dq_examples(df_business, c, F.col(c).rlike(pat)) if v else "", note=note if v else "")
empty_addr = df_business.where(F.col("address") == "")
n_empty_addr = empty_addr.count()
if n_empty_addr:
    print("\n   Businesses with address = '' — top categories (service-area vs genuinely missing):")
    display(empty_addr.join(df_categories, "business_id").groupBy("category_name").agg(F.countDistinct("business_id").alias("businesses"))
            .withColumn("pct_of_empty_addr", F.round(F.col("businesses") / n_empty_addr * 100, 1)).orderBy(F.desc("businesses")).limit(15))

# ══ C) attributes ══
print(f"\n{'─' * 60}\n📌 C) attributes — struct-level and field-level null forms")
attr_fields = [f.name for f in df_bronze.schema["attributes"].dataType.fields]
all_fields_null = F.greatest(*[F.col(f"attributes.{a}").isNotNull().cast("int") for a in attr_fields]) == 0
x = df_bronze.agg(F.sum(F.col("attributes").isNull().cast("int")).alias("struct_null"),
                  F.sum((F.col("attributes").isNotNull() & all_fields_null).cast("int")).alias("struct_all_null"),
                  F.sum(all_fields_null.cast("int")).alias("no_attr_info")).first()
dq_add(findings_636, "df_bronze", "attributes (whole struct)", "no-attribute business = one representation", N_BUS,
       min(int(x["struct_null"] or 0), int(x["struct_all_null"] or 0)),
       status="INFO" if (x["struct_null"] and x["struct_all_null"]) else "PASS",
       examples=f"NULL struct = {int(x['struct_null'] or 0):,} | struct with every field NULL = {int(x['struct_all_null'] or 0):,} | no attribute info at all = {int(x['no_attr_info'] or 0):,}",
       note="TE.2 always emits a df_attributes row (1:1), so these businesses appear as an all-NULL attribute row")

scalar_attrs = {**ATTR_BOOLEAN_MAP, **ATTR_INT_MAP, **ATTR_ENUM_MAP}
aggs = []
for b_, s_ in scalar_attrs.items():
    raw = F.col(f"attributes.{b_}")
    aggs += [F.sum(raw.isNull().cast("int")).alias(f"{s_}__sql"), F.sum((raw == "None").cast("int")).alias(f"{s_}__py"),
             F.sum(raw.rlike(r"^u?'(None|null|)'$").cast("int")).alias(f"{s_}__qnone"), F.sum((raw == "").cast("int")).alias(f"{s_}__empty")]
am = df_bronze.agg(*aggs).first()
attr_null = pd.DataFrame([{"Attribute": s_, "SQL NULL": am[f"{s_}__sql"], "'None' literal": am[f"{s_}__py"],
                           "quoted 'None' / ''": am[f"{s_}__qnone"], "'' (empty)": am[f"{s_}__empty"]} for s_ in scalar_attrs.values()])
attr_null["Masked total"] = attr_null["'None' literal"] + attr_null["quoted 'None' / ''"] + attr_null["'' (empty)"]
display(attr_null.sort_values("Masked total", ascending=False))
n_py_none = int(attr_null["'None' literal"].sum())
dq_add(findings_636, "df_attributes", f"{len(scalar_attrs)} scalar attrs ('None' literal)", "missing = SQL NULL, not the string 'None'",
       int((N_BUS - attr_null["SQL NULL"]).sum()), n_py_none,
       examples=", ".join(f"{a_}={n_:,}" for a_, n_ in attr_null.sort_values("'None' literal", ascending=False).head(5)[["Attribute", "'None' literal"]].itertuples(index=False)),
       note="DQ-BUSINESS-002 — TE.1 decodes 'None' → NULL; columns therefore mix two 'unknown' encodings in bronze")
dq_add(findings_636, "df_attributes", "enum attrs (quoted 'None' / '' as a category)", "no u'None' / '' category value",
       int((N_BUS - attr_null["SQL NULL"]).sum()), int((attr_null["quoted 'None' / ''"] + attr_null["'' (empty)"]).sum()))
alc_none = df_attributes.where(F.col("attr_alcohol") == "none").count()
dq_add(findings_636, "df_attributes", "attr_alcohol = 'none' (NOT a null)", "valid category 'serves no alcohol' must survive null-cleaning", N_BUS, alc_none,
       status="INFO", note="do NOT apply a case-insensitive 'none' → NULL rule to enums (see 6.3.1)")

# ══ D) Nested dicts — four ways to say 'no information' ══
print(f"\n{'─' * 60}\n📌 D) Nested dicts — SQL NULL vs 'None' vs '{{}}' vs all-None dict")
d_rows = []
for a_, (tbl_, _) in DICT_TABLE_MAP.items():
    s = F.col(f"attributes.{a_}")
    is_dict = s.rlike(r"^\{")
    has_val = s.rlike(r":\s*(True|False)")
    x = df_bronze.agg(F.sum(s.isNull().cast("int")).alias("sql"), F.sum((s == "None").cast("int")).alias("py"),
                      F.sum((F.regexp_replace(s, r"\s", "") == "{}").cast("int")).alias("empty"),
                      F.sum((is_dict & (F.regexp_replace(s, r"\s", "") != "{}") & ~has_val).cast("int")).alias("all_none"),
                      F.sum((is_dict & has_val).cast("int")).alias("informative")).first()
    n_sql, n_py, n_empty, n_alln = (int(x[k] or 0) for k in ("sql", "py", "empty", "all_none"))
    d_rows.append({"Dict": a_, "SQL NULL": n_sql, "'None' literal": n_py, "'{}' empty": n_empty,
                   "all values None": n_alln, "informative dicts": int(x["informative"] or 0)})
    dq_add(findings_636, tbl_, f"{a_} (masked 'no information')", "no info = SQL NULL only",
           N_BUS - n_sql, n_py + n_empty + n_alln,
           examples=f"'None'={n_py:,} | '{{}}'={n_empty:,} | all-None={n_alln:,}",
           note="DQ-BUSINESS-006 — in the long frames these become 0 rows (absent) or rows with value NULL")
display(pd.DataFrame(d_rows))

# ══ E) hours ══
print(f"\n{'─' * 60}\n📌 E) hours — NULL struct vs NULL day vs '0:0-0:0'")
day_cols = [F.col(f"hours.{d}") for d in DAYS]
n_days = F.size(F.filter(F.array(*day_cols), lambda v: v.isNotNull()))
x = df_bronze.agg(F.sum(F.col("hours").isNull().cast("int")).alias("struct_null"),
                  F.sum((F.col("hours").isNotNull() & (n_days == 0)).cast("int")).alias("all_days_null"),
                  F.sum((F.col("hours").isNotNull() & n_days.between(1, 6)).cast("int")).alias("partial"),
                  F.sum((F.col("hours").isNotNull() & (n_days == 7)).cast("int")).alias("full"),
                  F.sum(F.size(F.filter(F.array(*day_cols), lambda v: v == "0:0-0:0"))).alias("zz")).first()
dq_add(findings_636, "df_hours", "hours (whole struct)", "no schedule = one representation", N_BUS,
       min(int(x["struct_null"] or 0), int(x["all_days_null"] or 0)),
       examples=f"NULL struct = {int(x['struct_null'] or 0):,} | struct with 7 NULL days = {int(x['all_days_null'] or 0):,}")
dq_add(findings_636, "df_hours", "hours_<day> (NULL inside a listed week)", "a missing day means 'closed' — must be stated, not implied",
       int(x["partial"] or 0) + int(x["full"] or 0), int(x["partial"] or 0), status="INFO",
       note="NULL day = closed (Yelp UI convention) OR not entered — indistinguishable; make 'is_closed' explicit in Silver")
dq_add(findings_636, "df_hours", "hours_<day> = '0:0-0:0'", "no value that can mean 24 h, closed, or unknown", N_SLOTS if "N_SLOTS" in globals() else N_BUS * 7,
       int(x["zz"] or 0), status="INFO", note="DQ-BUSINESS-007 — ambiguous sentinel; resolved with attr_open_24_hours where present (6.3.2)")

# ══ F) categories ══
print(f"\n{'─' * 60}\n📌 F) categories — empty list vs NULL vs empty tokens")
x = df_bronze.agg(F.sum(F.col("categories").isNull().cast("int")).alias("sql"),
                  F.sum((F.trim(F.col("categories")) == "").cast("int")).alias("empty"),
                  F.sum(F.col("categories").rlike(r"(^|,)\s*(,|$)").cast("int")).alias("empty_tok")).first()
dq_add(findings_636, "df_bronze", "categories (empty list / empty tokens)", "no categories = SQL NULL; no ', ,' gaps", N_BUS,
       int(x["empty"] or 0) + int(x["empty_tok"] or 0), note=f"SQL NULL = {int(x['sql'] or 0):,} businesses (single representation)")

# ══ G) Numeric sentinels ══
print(f"\n{'─' * 60}\n📌 G) Numeric sentinels used as 'unknown'")
x = df_bronze.agg(*[F.sum(F.isnan(c).cast("int")).alias(f"nan_{c}") for c in ("latitude", "longitude", "stars")],
                  F.sum(((F.col("latitude") == 0) | (F.col("longitude") == 0)).cast("int")).alias("zero_coord"),
                  F.sum((F.col("stars") == 0).cast("int")).alias("zero_stars"),
                  F.sum((F.col("review_count") == 0).cast("int")).alias("zero_rc")).first()
dq_add(findings_636, "df_bronze", "latitude / longitude / stars (NaN)", "no NaN in DOUBLE columns", N_BUS,
       sum(int(x[f"nan_{c}"] or 0) for c in ("latitude", "longitude", "stars")))
dq_add(findings_636, "df_bronze", "latitude / longitude (0 as unknown)", "no 0.0 geocode placeholder", N_BUS, x["zero_coord"], note="cross-check of 6.2.4 Null Island")
dq_add(findings_636, "df_bronze", "stars (0 as unknown)", "no 0-star placeholder (scale starts at 1.0)", N_BUS, x["zero_stars"])
dq_add(findings_636, "df_bronze", "review_count (0 as unknown)", "0 would mean 'not counted' in a ≥ 5-review dataset", N_BUS, x["zero_rc"])

findings_636_df = dq_summary(findings_636, "6.3.6 Null Representation Consistency")

# ══════════════════════════════════════════════════════════════════════
# 6.3 CONSOLIDATED — every non-PASS check from 6.3.1 – 6.3.6
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 80}\n6.3) FORMAT & ENCODING STANDARDIZATION — CONSOLIDATED FINDINGS\n{'=' * 80}")
all_63 = pd.concat([findings_631_df.assign(Check="6.3.1 Categorical"), findings_632_df.assign(Check="6.3.2 Date/Time"),
                    findings_633_df.assign(Check="6.3.3 Numeric"), findings_634_df.assign(Check="6.3.4 Boolean"),
                    findings_635_df.assign(Check="6.3.5 Text"), findings_636_df.assign(Check="6.3.6 Null repr.")], ignore_index=True)
print(all_63.groupby(["Check", "Status"]).size().unstack(fill_value=0).to_string())
issues_63 = (all_63[all_63["Status"] != "PASS"]
             .sort_values(["Status", "Check", "Violations"], ascending=[True, True, False])
             [["Check", "Status", "DataFrame", "Column", "Rule", "Violations", "Checked", "Pct", "Examples", "Note"]])
display(issues_63)
print(f"\n{len(all_63)} checks in 6.3 | {int((all_63['Status'] == 'FINDING').sum())} FINDING | {int((all_63['Status'] == 'INFO').sum())} INFO")
print("NULL REPRESENTATION CONSISTENCY CHECK COMPLETE")
print("=" * 80)

---
#### 🚨 DQ-BUSINESS-032 — Empty-String `address` Masquerading as a Value (Masked NULL, Not Missing at Random)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-032` |
| **Severity** | FINDING (null representation; 0 SQL NULLs exist, so every completeness metric under-reports missing addresses) |
| **Detected In** | Phase 6.3 — Format & Encoding Standardization → **6.3.6 Null Representation Consistency** (Cell 76), check A "address (masked nulls)" and the category profile of empty addresses |
| **Also Confirmed In** | 6.3.5 (Cell 72): "address (empty after trim)" = 5,127. 6.3.3 (Cell 68): 134 of the 348 coarse coordinates belong to these businesses (DQ-BUSINESS-027). Same pattern as empty `postal_code` (DQ-BUSINESS-015); 56 businesses have both empty. The single `'.'` address is part of DQ-BUSINESS-019 |
| **Affected Columns** | `address` |
| **Total Affected Rows** | **5,128** of 150,346 businesses (3.41%): **5,127** `''` + **1** punctuation-only `'.'`; **0** SQL NULL |
| **Invalid Value** | `''`, `'.'` |
| **Domain Rule** | Unknown / not disclosed = SQL NULL only |

**Description:** `address` has two representations of "no address" (`''`, `'.'`) and none of them is NULL, so `count(address)` and `isNull()` report 100% completeness. The empty values are **not random**: Home Services make up 45.6% of them, Local Services 20.1%, Event Planning 14.7%, Food Trucks 7.8%, Contractors 8.2%, Home Cleaning 5.8%, Landscaping 5.1%. These are **service-area businesses** (plumbers, cleaners, caterers, food trucks) that travel to the customer and deliberately hide their street address on Yelp. So the missingness is MNAR / structurally explained, and the empty value actually carries a business meaning ("no public storefront").

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Convert `''` and `'.'` → SQL NULL in Silver and add a derived `is_service_area_business` (address NULL) flag; document the MNAR reason in the column comment** | One null representation (consistent with DQ-BUSINESS-002 / -015); completeness metrics in 6.4 become correct; the business meaning of the missing address is kept explicitly instead of hidden in an empty string |
| 2 | **Convert to NULL only** | Correct representation, but loses the easy signal that these are mobile / service-area businesses |
| 3 | **Impute the address by reverse-geocoding coordinates** | Fabricates a storefront address that the business intentionally does not have, and coordinates are often coarse for these rows |
| 4 | **Drop the 5,128 businesses** | Removes a whole business segment (home / local services) and biases every category analysis |

**Best Logical Solution:** **Option 1 — `''` / `'.'` → NULL plus an explicit service-area flag.** It standardises the null representation and preserves the real-world meaning of the missingness for 6.4 pattern analysis.

---

---
#### 🚨 DQ-BUSINESS-033 — "Closed" Days Encoded Implicitly as NULL Inside a Listed Week

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-033` |
| **Severity** | MEDIUM (null representation; one NULL carries two meanings, "closed" and "not entered") |
| **Detected In** | Phase 6.3 — Format & Encoding Standardization → **6.3.6 Null Representation Consistency** (Cell 76), check E "hours_<day> (NULL inside a listed week)" |
| **Also Confirmed In** | Independent SQL check on `bronze_final.yelp_business` (same count). 6.3.2 (Cell 65): `0:0-0:0` (73,908 slots) is the other ambiguous schedule value (DQ-BUSINESS-007) |
| **Affected Columns** | `hours.Monday` … `hours.Sunday` → `df_hours.hours_<day>` |
| **Total Affected Rows** | **55,837** of 127,123 businesses with a schedule (43.9%) have at least one NULL day; 88,846 NULL day-slots in total (889,861 possible slots - 801,015 filled). Separately, 23,223 businesses have no `hours` struct at all (single, consistent representation) |
| **Invalid Value** | `hours = {Monday: '9:0-17:0', ..., Friday: '9:0-17:0', Saturday: NULL, Sunday: NULL}` |
| **Domain Rule** | Within a published schedule every day should state its status explicitly: open window, closed, or unknown |

**Description:** Yelp shows a missing day as "Closed", so in most cases a NULL day inside a listed week means *closed* (weekend closures of offices, Monday closures of restaurants). But the same NULL is also produced when an owner simply never entered a day, and the data cannot distinguish the two. Downstream, a naive `hours_saturday IS NULL` is read as "unknown" by SQL semantics but as "closed" by Yelp semantics; "open on weekends" or "days open per week" features will differ depending on which reading is used.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep raw NULLs; add per-day `is_closed_<day>` = (hours struct present AND day NULL), `days_open_per_week`, and a table-level comment stating the Yelp convention "NULL day in a listed week = closed"** | Applies the source platform's own documented semantics explicitly, leaves NULL for the truly unknown case (no struct), and makes features reproducible |
| 2 | **Replace NULL days with a sentinel string `'closed'`** | Explicit, but mixes text into a time-range column and breaks the `H:M-H:M` parser |
| 3 | **Treat NULL days as unknown everywhere** | SQL-pure, but contradicts what Yelp users see and under-counts closures for 55,837 businesses |
| 4 | **Impute the day from neighbouring days** | Invents opening hours that the business never published |

**Best Logical Solution:** **Option 1 — derived explicit `is_closed_<day>` flags following the Yelp convention, raw NULL preserved.** It removes the ambiguity for analytics without altering source values, and pairs with the DQ-BUSINESS-007 / -025 derived schedule columns.

---

> **Cross-references from 6.3.6 (already documented, no new ID):** 9,625 `'None'` literals across 32 scalar attributes (RestaurantsDelivery 3,948, RestaurantsTakeOut 2,105, OutdoorSeating 1,882, DriveThru 755 …) → **DQ-BUSINESS-002**. Nested dicts with masked "no information" (`'None'`, `'{}'`, all-None): BusinessParking 2,279, Ambience 557, GoodForMeal 303, Music 104, HairSpecializesIn 30, BestNights 11, DietaryRestrictions 1 → **DQ-BUSINESS-002** / **DQ-BUSINESS-006**. Empty `postal_code` (73) → **DQ-BUSINESS-015**. `'0:0-0:0'` (73,908) → **DQ-BUSINESS-007**. `Alcohol 'none'` (20,910) must survive null-cleaning (valid category, see 6.3.1).
>
> **Reviewed and not raised:** `name`, `city`, `state`, `category_name` have no masked nulls; no placeholder names / addresses (`TBD`, `N/A`, `Unknown`); `attributes` has one representation for "no attributes" (13,744 NULL structs, 0 all-NULL structs); `hours` has one representation for "no schedule" (23,223 NULL structs); `categories` uses SQL NULL only (103); no NaN in DOUBLE columns; no 0 placeholders in coordinates, `stars` or `review_count`. Missingness volumes and MCAR / MAR / MNAR classification continue in 6.4.

---


### 6.4) COMPLETENESS CHECKS
*Measure missingness after null normalization. Understand patterns and impact.*

- **Missingness Measurement**
  - *Sub:* Per column — null percentage per column.
  - *Sub:* Per row — rows with excessive missing values (e.g., >50% null).
  - *Sub:* Per cell — total null rate after normalizing variants from 6.3.

- **Missingness Pattern Analysis**
  - *Sub:* Randomness classification — MCAR / MAR / MNAR.
  - *Sub:* Missing value correlation — missingness in `text` correlated with missingness in `stars` (e.g., are reviews missing text when they also lack a rating?).
  - *Sub:* Temporal gaps — missing time periods in time-series data (e.g., no reviews in certain months during 2005–2022).

- **Critical Field Coverage** — Required fields (per business rules) must be non-null.
  - *Example:* `review_id`, `user_id`, `business_id`, `stars`, `text`, and `date` must never be null for a valid review record.

In [0]:
# 6.4.1) Per-Column Missingness  —  Business DataFrames (after null normalization)
# ─────────────────────────────────────────────────────────────────────
# 6.3.6 showed that "unknown" is written in many ways, so SQL-NULL counts UNDER-REPORT missingness.
# Here missingness is MEASURED on the TE.2 frames with every 6.3 null form normalised to "missing":
#   strings    : SQL NULL | '' | whitespace / invisible-only (NBSP, U+200B) | 'None' | masked token | punctuation-only ('.')
#   numerics   : SQL NULL | NaN
#   attributes : SQL NULL | Python 'None' literal (TE.1 already decodes 'None' → NULL in df_attributes)
#   dicts      : SQL NULL | 'None' | '{}' | dict whose every value is None   → "no information" (DQ-BUSINESS-006)
#   hours      : NULL struct = no schedule.  NULL day inside a listed week = closed (DQ-BUSINESS-033) → profiled, not counted as missing
#   categories : SQL NULL (single representation, 6.3.6)
# Applicability: Yelp attributes are category-specific (a dentist has no 'Alcohol'), so a NULL mixes
# "not applicable" with "not provided". Coverage is therefore ALSO measured inside the segment the
# attribute applies to; attributes with >= 50 % in-segment coverage form EXPECTED_ATTRS (reused in 6.4.2 – 6.4.5).
# Builds df_miss (1 row / business, m_<field> = 1 if effectively missing) reused by 6.4.2 – 6.4.7.
# Detection only — nothing is changed.
import re
import pandas as pd
from pyspark.sql import functions as F

print("=" * 80)
print("6.4.1) PER-COLUMN MISSINGNESS  —  Business DataFrames (TE.2), after null normalization")
print("=" * 80)

N_BUS = df_business.count()
findings_641 = []
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
_snake = lambda s: re.sub(r"(?<!^)(?=[A-Z])", "_", s).lower()
DICT_COL = {a: f"attr_{_snake(a)}" for a in DICT_TABLE_MAP}
DICT_DF = {a: tbl for a, (tbl, _) in DICT_TABLE_MAP.items()}
SCALAR_ATTRS = {**ATTR_BOOLEAN_MAP, **ATTR_INT_MAP, **ATTR_ENUM_MAP}          # bronze name → df_attributes column

# ══ 0) Effective-missing rules (all null forms found in 6.3.6) ══
MASKED_64 = ["none", "n/a", "na", "n.a.", "null", "nil", "nan", "unknown", "unk", "tbd", "tba", "missing", "undefined",
             "not available", "not applicable", "no address", "none listed", "empty", "blank", "#n/a", "#value!"]
INVISIBLE_64 = r"[\s\u00a0\u200b\u200c\u200d\ufeff]"
PUNCT_ONLY_64 = r"^[\s.\-_*?/#,\u2013\u2014]+$"
def str_missing(c):
    v = F.col(c)
    return (v.isNull() | (F.regexp_replace(v, INVISIBLE_64, "") == "") | F.lower(F.trim(v)).isin(MASKED_64) | v.rlike(PUNCT_ONLY_64))
def num_missing(c):
    return F.col(c).isNull() | F.isnan(F.col(c).cast("double"))

# ══ 1) Segments (applicability of category-specific attributes), from df_categories ══
SEGMENTS = {
    "food":         ["Restaurants", "Food", "Bars", "Nightlife", "Coffee & Tea", "Cafes"],
    "nightlife":    ["Nightlife", "Bars", "Pubs", "Lounges", "Dance Clubs", "Music Venues", "Cocktail Bars", "Wine Bars"],
    "hair":         ["Hair Salons", "Barbers", "Hair Stylists"],
    "health":       ["Health & Medical", "Doctors", "Dentists", "General Dentistry", "Chiropractors", "Optometrists", "Physical Therapy"],
    "service_area": ["Home Services", "Local Services", "Event Planning & Services", "Food Trucks", "Contractors",
                     "Home Cleaning", "Landscaping", "Movers", "Plumbing", "Caterers"],   # travel to the customer (DQ-BUSINESS-032)
}
ATTR_SEGMENT = {a: "all" for a in list(SCALAR_ATTRS) + list(DICT_TABLE_MAP)}
ATTR_SEGMENT.update({a: "food" for a in ["Alcohol", "Ambience", "BYOB", "BYOBCorkage", "Caters", "Corkage", "DietaryRestrictions",
                                         "DriveThru", "GoodForMeal", "HappyHour", "HasTV", "NoiseLevel", "OutdoorSeating",
                                         "RestaurantsAttire", "RestaurantsCounterService", "RestaurantsDelivery",
                                         "RestaurantsGoodForGroups", "RestaurantsReservations", "RestaurantsTableService",
                                         "RestaurantsTakeOut"]})
ATTR_SEGMENT.update({a: "nightlife" for a in ["AgesAllowed", "BestNights", "CoatCheck", "GoodForDancing", "Music", "Smoking"]})
ATTR_SEGMENT.update({"HairSpecializesIn": "hair", "AcceptsInsurance": "health"})

seg = df_categories.groupBy("business_id").agg(
    *[F.max(F.trim("category_name").isin(v).cast("int")).alias(f"seg_{k}") for k, v in SEGMENTS.items()])

# ══ 2) df_miss — one row per business, effective (m_) and raw SQL-NULL (r_) indicators ══
CORE_STR, CORE_NUM = ["name", "address", "city", "state", "postal_code"], ["latitude", "longitude", "stars", "review_count"]
core = df_business.select(
    "business_id", "state", "city", "is_open", F.col("stars").cast("double").alias("stars_v"), "review_count",
    *[str_missing(c).cast("int").alias(f"m_{c}") for c in CORE_STR],
    *[num_missing(c).cast("int").alias(f"m_{c}") for c in CORE_NUM],
    F.col("is_open").isNull().cast("int").alias("m_is_open"),
    *[F.col(c).isNull().cast("int").alias(f"r_{c}") for c in CORE_STR + CORE_NUM + ["is_open"]])
raw_nested = df_bronze.select(
    "business_id",
    F.col("attributes").isNull().cast("int").alias("r_attributes"),
    F.col("hours").isNull().cast("int").alias("r_schedule"),
    F.col("categories").isNull().cast("int").alias("m_categories"),
    F.col("categories").isNull().cast("int").alias("r_categories"),
    *[F.col(f"attributes.{b}").isNull().cast("int").alias(f"r_{s}") for b, s in SCALAR_ATTRS.items()],
    *[F.col(f"attributes.{a}").isNull().cast("int").alias(f"r_{DICT_COL[a]}") for a in DICT_TABLE_MAP])
attr_m = df_attributes.select("business_id", *[F.col(s).isNull().cast("int").alias(f"m_{s}") for s in SCALAR_ATTRS.values()])
hours_m = df_hours.select("business_id", *[F.col(HOURS_MAP[d]).isNull().cast("int").alias(f"m_{HOURS_MAP[d]}") for d in DAYS])

df_miss = core.join(raw_nested, "business_id", "left").join(attr_m, "business_id", "left").join(hours_m, "business_id", "left") \
              .join(seg, "business_id", "left")
for a, tbl in DICT_DF.items():          # informative dict = at least one key with a non-NULL value in the long frame
    info = business_dfs[tbl].where(F.col("value").isNotNull()).select("business_id").distinct().withColumn(f"i_{DICT_COL[a]}", F.lit(1))
    df_miss = df_miss.join(info, "business_id", "left")
df_miss = df_miss.fillna(0, subset=[f"seg_{k}" for k in SEGMENTS] + [f"i_{DICT_COL[a]}" for a in DICT_TABLE_MAP])
df_miss = df_miss.select("*", *[(1 - F.col(f"i_{DICT_COL[a]}")).alias(f"m_{DICT_COL[a]}") for a in DICT_TABLE_MAP])
df_miss = df_miss.withColumn("seg_all", F.lit(1)) \
    .withColumn("m_schedule", F.least(*[F.col(f"m_{HOURS_MAP[d]}") for d in DAYS])) \
    .withColumn("m_attributes", F.least(*[F.col(f"m_{s}") for s in SCALAR_ATTRS.values()], *[F.col(f"m_{DICT_COL[a]}") for a in DICT_TABLE_MAP]))

# ══ 3) Field registry — requirement level, raw vs effective indicator, applicability ══
def _f(field, frame, group, req, seg_="all", ref="", raw=None, app=None):
    return {"field": field, "frame": frame, "group": group, "req": req, "seg": seg_, "ref": ref,
            "raw": raw or f"r_{field}", "app": app if app is not None else F.col(f"seg_{seg_}")}
FIELDS64 = [
    _f("name", "df_business", "identity", "REQUIRED"),
    _f("address", "df_business", "location", "CONDITIONAL", ref="DQ-BUSINESS-032 (service-area MNAR)"),
    _f("city", "df_business", "location", "REQUIRED"),
    _f("state", "df_business", "location", "REQUIRED"),
    _f("postal_code", "df_business", "location", "CONDITIONAL", ref="DQ-BUSINESS-015"),
    _f("latitude", "df_business", "location", "REQUIRED"),
    _f("longitude", "df_business", "location", "REQUIRED"),
    _f("stars", "df_business", "rating", "REQUIRED"),
    _f("review_count", "df_business", "rating", "REQUIRED"),
    _f("is_open", "df_business", "status", "REQUIRED"),
    _f("attributes", "df_attributes + 7 dict dfs", "block", "EXPECTED", ref="any informative attribute (13,744 NULL structs + all-masked)"),
    _f("schedule", "df_hours", "block", "EXPECTED", ref="no hours struct"),
    _f("categories", "df_categories", "block", "EXPECTED", ref="0 category rows"),
]
FIELDS64 += [_f(s, "df_attributes", "attribute", "OPTIONAL", ATTR_SEGMENT[b], "DQ-BUSINESS-002 ('None' literal)") for b, s in SCALAR_ATTRS.items()]
FIELDS64 += [_f(DICT_COL[a], DICT_DF[a], "attribute (dict)", "OPTIONAL", ATTR_SEGMENT[a], "DQ-BUSINESS-006 (masked dicts)") for a in DICT_TABLE_MAP]
FIELDS64 += [_f(HOURS_MAP[d], "df_hours", "schedule day", "PROFILED", ref="DQ-BUSINESS-033 (NULL day = closed)",
                raw=f"m_{HOURS_MAP[d]}", app=(1 - F.col("r_schedule"))) for d in DAYS]

aggs = [F.count("*").alias("n")]
for i, f in enumerate(FIELDS64):
    m, app = F.col(f"m_{f['field']}"), f["app"]
    aggs += [F.sum(m).alias(f"e{i}"), F.sum(F.col(f["raw"])).alias(f"r{i}"), F.sum(app).alias(f"a{i}"), F.sum(m * app).alias(f"ea{i}")]
A = df_miss.agg(*aggs).first()
assert A["n"] == N_BUS, f"df_miss rows {A['n']} != {N_BUS} (join fan-out)"

def _tier(cov):
    return "NEAR-EMPTY" if cov < 5 else ("SPARSE" if cov < 50 else ("PARTIAL" if cov < 90 else "WELL-COVERED"))
rows = []
for i, f in enumerate(FIELDS64):
    e, r, a, ea = (int(A[k] or 0) for k in (f"e{i}", f"r{i}", f"a{i}", f"ea{i}"))
    if f["group"] == "schedule day":
        e, r = ea, ea                                    # only days inside a listed week are meaningful
    cov_seg = round((a - ea) / a * 100, 2) if a else 0.0
    rows.append({"Field": f["field"], "Frame": f["frame"], "Group": f["group"], "Requirement": f["req"],
                 "Raw SQL NULL": r, "Hidden (masked)": max(e - r, 0), "Effective Missing": e,
                 "Effective %": round(e / N_BUS * 100, 2), "Applies To": f["seg"] if f["group"] != "schedule day" else "has schedule",
                 "Applicable Rows": a, "Missing in Segment": ea, "Coverage in Segment %": cov_seg,
                 "Tier": _tier(cov_seg), "Ref": f["ref"]})
col_null_df = pd.DataFrame(rows)

# ══ A) Required / conditional / expected fields ══
print(f"\n{'─' * 60}\n📌 A) Core fields & content blocks  (N = {N_BUS:,} businesses)")
for r_ in col_null_df[col_null_df["Requirement"].isin(["REQUIRED", "CONDITIONAL", "EXPECTED"])].to_dict("records"):
    dq_add(findings_641, r_["Frame"], r_["Field"], f"{r_['Requirement']} — effectively non-missing", N_BUS, r_["Effective Missing"],
           examples=f"raw SQL NULL={r_['Raw SQL NULL']:,} | hidden by masking={r_['Hidden (masked)']:,}" if r_["Effective Missing"] else "",
           note=r_["Ref"])
hidden_total = int(col_null_df.loc[col_null_df["Group"] != "schedule day", "Hidden (masked)"].sum())
hidden_cols = col_null_df[(col_null_df["Hidden (masked)"] > 0) & (col_null_df["Group"] != "schedule day")]
dq_add(findings_641, "all business frames", "SQL-NULL count vs effective missing", "raw isNull() must not under-report missingness",
       int(col_null_df.loc[col_null_df["Group"] != "schedule day", "Effective Missing"].sum()), hidden_total,
       examples=", ".join(f"{x['Field']}={x['Hidden (masked)']:,}" for x in hidden_cols.sort_values("Hidden (masked)", ascending=False).head(8).to_dict("records")),
       note="masked nulls (''/'.'/'None'/'{}'/all-None) are invisible to count(col) — DQ-BUSINESS-002/-006/-015/-032")

# ══ B) Optional attributes — overall vs in-segment coverage ══
print(f"\n{'─' * 60}\n📌 B) Optional attributes — applicability-aware coverage")
attr_df = col_null_df[col_null_df["Group"].str.startswith("attribute")].sort_values("Coverage in Segment %")
near_empty = attr_df[attr_df["Tier"] == "NEAR-EMPTY"]
dq_add(findings_641, "df_attributes + dict dfs", "near-empty attributes (< 5 % in segment)", "an attribute column should carry information for its own segment",
       len(attr_df), len(near_empty),
       examples=", ".join(f"{x['Field']}={x['Coverage in Segment %']}% of {x['Applies To']}" for x in near_empty.to_dict("records")),
       note="candidates to drop / archive in Silver rather than carry as all-NULL columns")
out_seg_null = int((attr_df["Effective Missing"] - attr_df["Missing in Segment"]).sum())
dq_add(findings_641, "df_attributes + dict dfs", "NULL = 'not applicable' OR 'not provided'", "one NULL must not carry two meanings",
       int(attr_df["Effective Missing"].sum()), out_seg_null, status="INFO",
       note="NULL cells OUTSIDE the attribute's segment are structural (not applicable); inside the segment they are true gaps")
EXPECTED_ATTRS = [(x["Field"], x["Applies To"]) for x in attr_df.to_dict("records") if x["Coverage in Segment %"] >= 50]
print(f"\n   Coverage tiers: {attr_df['Tier'].value_counts().to_dict()}")
print(f"   EXPECTED_ATTRS (>= 50 % in-segment coverage, reused in 6.4.2): {len(EXPECTED_ATTRS)} → {[a for a, _ in EXPECTED_ATTRS]}")

# ══ C) Schedule days inside a listed week ══
day_df = col_null_df[col_null_df["Group"] == "schedule day"]
dq_add(findings_641, "df_hours", "hours_<day> NULL inside a listed week", "implicit 'closed' (Yelp convention) — profiled, not missing",
       int(day_df["Applicable Rows"].sum()), int(day_df["Missing in Segment"].sum()), status="INFO",
       examples=", ".join(f"{x['Field']}={x['Missing in Segment']:,}" for x in day_df.sort_values("Missing in Segment", ascending=False).to_dict("records")),
       note="DQ-BUSINESS-033 — weekly gap patterns analysed in 6.4.6")

# ══ D) Nested-dict key completeness (dicts that DO carry information) ══
print(f"\n{'─' * 60}\n📌 D) Nested dicts — key completeness of informative dicts (DQ-BUSINESS-008)")
key_rows = []
for a, (tbl, keymap) in DICT_TABLE_MAP.items():
    k = business_dfs[tbl].groupBy("business_id").agg(F.count("*").alias("keys"), F.sum(F.col("value").isNull().cast("int")).alias("null_vals"))
    k = k.where(F.col("keys") > F.col("null_vals"))                 # informative dicts only
    x = k.agg(F.count("*").alias("dicts"), F.sum((F.col("keys") >= len(keymap)).cast("int")).alias("full"),
              F.avg(F.col("keys") / len(keymap)).alias("avg_cov"), F.sum("null_vals").alias("null_vals"), F.sum("keys").alias("cells")).first()
    d_, full = int(x["dicts"] or 0), int(x["full"] or 0)
    key_rows.append({"Dict": a, "Expected Keys": len(keymap), "Informative Dicts": d_, "Full Key Set": full,
                     "Partial Key Set": d_ - full, "Avg Key Coverage %": round((x["avg_cov"] or 0) * 100, 1),
                     "None-valued keys": int(x["null_vals"] or 0), "Key cells": int(x["cells"] or 0)})
key_df = pd.DataFrame(key_rows)
display(key_df)
dq_add(findings_641, "7 dict dfs", "informative dicts with a partial key set", "each dict lists every contract key",
       int(key_df["Informative Dicts"].sum()), int(key_df["Partial Key Set"].sum()), status="INFO",
       examples=", ".join(f"{x['Dict']}={x['Partial Key Set']:,}" for x in key_rows if x["Partial Key Set"]),
       note="DQ-BUSINESS-008 — absent key = unknown, not False")

print(f"\n{'─' * 60}\n📋 Per-column missingness matrix ({len(col_null_df)} fields):")
display(col_null_df.sort_values(["Group", "Effective %"], ascending=[True, False]))

cols_with_missing = col_null_df.loc[(col_null_df["Effective Missing"] > 0) & (col_null_df["Group"] != "schedule day"), "Field"].tolist()
print(f"\n   Fields with effective missingness: {len(cols_with_missing)} / {len(col_null_df) - len(day_df)}"
      f"  (raw SQL NULL would report {int((col_null_df.loc[col_null_df['Group'] != 'schedule day', 'Raw SQL NULL'] > 0).sum())})")
findings_641_df = dq_summary(findings_641, "6.4.1 Per-Column Missingness")
print("PER-COLUMN MISSINGNESS CHECK COMPLETE")
print("=" * 80)

In [0]:
# 6.4.2) Per-Row Missingness  —  Business profile completeness
# ─────────────────────────────────────────────────────────────────────
# A raw "null count over all 90 columns" is meaningless here: a dentist is never expected to have 'Alcohol'.
# Each business is scored only on the fields it is EXPECTED to have (its own applicable profile):
#   core (10)   : name, city, state, latitude, longitude, stars, review_count, is_open  (+ address, postal_code
#                 unless the business is a service-area business — DQ-BUSINESS-032)
#   blocks (3)  : any attribute information, a weekly schedule, at least one category
#   EXPECTED_ATTRS from 6.4.1 : attributes that >= 50 % of the business's own segment fills (e.g. price range for all,
#                 Alcohol / NoiseLevel for food, Music for nightlife). Missing what most peers have = a real gap.
# Thresholds: > 25 % and > 50 % of the applicable profile missing. Also flags skeleton profiles
# (no attributes AND no schedule AND no categories) and core-incomplete rows.
import pandas as pd
from pyspark.sql import functions as F

print("=" * 80)
print("6.4.2) PER-ROW MISSINGNESS  —  applicability-aware business profile completeness")
print("=" * 80)
findings_642 = []

CORE_ALWAYS = ["name", "city", "state", "latitude", "longitude", "stars", "review_count", "is_open"]
CORE_COND = ["address", "postal_code"]
BLOCKS = ["attributes", "schedule", "categories"]
not_sa = 1 - F.col("seg_service_area")
PROFILE = [(c, F.lit(1)) for c in CORE_ALWAYS] + [(c, not_sa) for c in CORE_COND] + [(b, F.lit(1)) for b in BLOCKS] + \
          [(a, F.col(f"seg_{s}")) for a, s in EXPECTED_ATTRS]
print(f"   Profile fields: {len(PROFILE)}  (core {len(CORE_ALWAYS) + len(CORE_COND)} | blocks {len(BLOCKS)} | expected attrs {len(EXPECTED_ATTRS)})")

n_app = sum(app for _, app in PROFILE)
n_mis = sum(F.col(f"m_{c}") * app for c, app in PROFILE)
core_mis = sum(F.col(f"m_{c}") for c in CORE_ALWAYS) + sum(F.col(f"m_{c}") * not_sa for c in CORE_COND)
df_row64 = df_miss.select(
    "business_id", "city", "state", "is_open", "review_count", "seg_food", "seg_service_area",
    n_app.alias("n_app"), n_mis.alias("n_mis"), (n_mis / n_app * 100).alias("pct_mis"), core_mis.alias("core_mis"),
    sum(F.col(f"m_{c}") for c in CORE_ALWAYS + CORE_COND).alias("core_mis_all"),
    (F.col("m_attributes") * F.col("m_schedule") * F.col("m_categories")).alias("skeleton"),
    (F.col("m_attributes") * F.col("m_schedule")).alias("no_attr_no_hours"),
    (F.col("m_address") * F.col("m_postal_code")).alias("no_addr_no_zip"))

R = df_row64.agg(
    F.count("*").alias("n"), F.sum((F.col("n_mis") > 0).cast("int")).alias("any"),
    F.sum((F.col("pct_mis") > 25).cast("int")).alias("gt25"), F.sum((F.col("pct_mis") > 50).cast("int")).alias("gt50"),
    F.max("pct_mis").alias("max_pct"), F.avg("pct_mis").alias("avg_pct"), F.avg("n_app").alias("avg_app"),
    F.expr("percentile(pct_mis, 0.5)").alias("p50"), F.expr("percentile(pct_mis, 0.9)").alias("p90"),
    F.sum((F.col("core_mis_all") > 0).cast("int")).alias("core_any"), F.sum("skeleton").alias("skeleton"),
    F.sum("no_attr_no_hours").alias("no_attr_no_hours"), F.sum("no_addr_no_zip").alias("no_addr_no_zip"),
    F.sum(((F.col("core_mis") > 0) & (F.col("seg_service_area") == 0)).cast("int")).alias("core_storefront")).first()
N_ = int(R["n"])
print(f"\n   Businesses scored:                     {N_:,}   (avg applicable fields / row: {R['avg_app']:.1f})")
print(f"   Rows with >= 1 expected field missing: {R['any']:,}  ({R['any'] / N_ * 100:.2f}%)")
print(f"   Rows > 25 % of profile missing:        {R['gt25']:,}  ({R['gt25'] / N_ * 100:.2f}%)")
print(f"   Rows > 50 % of profile missing:        {R['gt50']:,}  ({R['gt50'] / N_ * 100:.2f}%)")
print(f"   Missing % per row — mean {R['avg_pct']:.2f} | median {R['p50']:.2f} | p90 {R['p90']:.2f} | max {R['max_pct']:.2f}")

# ── distribution of profile-missing % ──
bucket = (F.when(F.col("pct_mis") == 0, "0 %").when(F.col("pct_mis") <= 10, "(0, 10]").when(F.col("pct_mis") <= 25, "(10, 25]")
           .when(F.col("pct_mis") <= 50, "(25, 50]").when(F.col("pct_mis") <= 75, "(50, 75]").otherwise("(75, 100]"))
print("\n   ── Distribution of % expected fields missing per business ──")
dist = (df_row64.groupBy(bucket.alias("pct_missing_bucket")).agg(F.count("*").alias("businesses"),
        F.round(F.avg("review_count"), 1).alias("avg_review_count"), F.round(F.avg(F.col("is_open").cast("int")) * 100, 1).alias("pct_open"))
        .withColumn("pct_of_businesses", F.round(F.col("businesses") / N_ * 100, 2)).toPandas())
_order = ["0 %", "(0, 10]", "(10, 25]", "(25, 50]", "(50, 75]", "(75, 100]"]
dist["_o"] = dist["pct_missing_bucket"].map(_order.index)
display(dist.sort_values("_o").drop(columns="_o"))

# ── which fields drive incomplete rows ──
hi = df_miss.join(df_row64.where(F.col("pct_mis") > 25).select("business_id"), "business_id")
hi_n = R["gt25"] or 0
if hi_n:
    drv = hi.agg(*[F.sum(F.col(f"m_{c}") * app).alias(c) for c, app in PROFILE]).first().asDict()
    print(f"\n   ── Fields most often missing in the {hi_n:,} rows with > 25 % missing ──")
    display(pd.DataFrame([{"Field": k, "Missing in high-gap rows": int(v or 0), "% of high-gap rows": round((v or 0) / hi_n * 100, 1)}
                          for k, v in drv.items()]).sort_values("Missing in high-gap rows", ascending=False).head(12))

# ── worst profiles (examples) ──
print("\n   ── Most incomplete profiles (highest % missing, then most reviews) ──")
display(df_row64.where(F.col("pct_mis") > 50).join(df_business.select("business_id", "name", "address"), "business_id")
        .select("business_id", "name", "address", "city", "state", "is_open", "review_count", "n_app", "n_mis", F.round("pct_mis", 1).alias("pct_mis"), "skeleton")
        .orderBy(F.desc("pct_mis"), F.desc("review_count")).limit(15))

# ── skeleton profiles ──
if R["skeleton"]:
    print(f"\n   ── Skeleton profiles (no attributes, no schedule, no categories): {R['skeleton']:,} ──")
    display(df_row64.where(F.col("skeleton") == 1).join(df_business.select("business_id", "name", "address", "stars"), "business_id")
            .select("business_id", "name", "address", "city", "state", "stars", "review_count", "is_open").orderBy(F.desc("review_count")).limit(10))

dq_add(findings_642, "df_miss", "rows with >= 1 expected field missing", "profile complete for its segment", N_, R["any"], status="INFO",
       note="expected-attribute gaps are normal on Yelp (owner / crowd-sourced) — informative only")
dq_add(findings_642, "df_miss", "rows > 25 % of profile missing", "<= 25 % of applicable fields missing", N_, R["gt25"],
       note="thin profiles — weak features for recommendation / segmentation")
dq_add(findings_642, "df_miss", "rows > 50 % of profile missing", "<= 50 % of applicable fields missing", N_, R["gt50"],
       note="candidates for a 'low-information' flag in Silver, not for deletion (stars / reviews still valid)")
dq_add(findings_642, "df_business", "core identity incomplete (>= 1 core field)", "all core fields present (address/ZIP exempt for service-area)",
       N_, R["core_storefront"], examples=f"all rows incl. exempt service-area = {R['core_any']:,} | address AND postal_code missing = {R['no_addr_no_zip']:,}",
       note="storefront businesses without address / ZIP — DQ-BUSINESS-015 / -032")
dq_add(findings_642, "df_miss", "skeleton profile (no attributes + no hours + no categories)", "a listed business has at least one descriptive block",
       N_, R["skeleton"], examples=dq_examples(df_row64.join(df_business.select("business_id", "name"), "business_id"), "name", F.col("skeleton") == 1, 6) if R["skeleton"] else "")
dq_add(findings_642, "df_miss", "no attributes AND no schedule", "descriptive blocks present", N_, R["no_attr_no_hours"], status="INFO",
       note="see 6.4.5 for the co-missing pattern")
row_metrics_642 = dict(R.asDict())
findings_642_df = dq_summary(findings_642, "6.4.2 Per-Row Missingness")
print("PER-ROW MISSINGNESS CHECK COMPLETE")
print("=" * 80)

In [0]:
# 6.4.3) Per-Cell Null Rate  —  per DataFrame, raw vs effective vs applicability-adjusted
# ─────────────────────────────────────────────────────────────────────
# Null rate of the whole cell matrix (rows × value columns), computed three ways:
#   Raw          : SQL NULL only (what count(col) / isNull() reports on bronze)
#   Effective    : after normalising every 6.3 null form ('', '.', 'None', '{}', all-None dict) → the TRUE gap rate
#   Applicable   : effective nulls only in cells the business is expected to have (6.4.1 segments) → actionable gap rate
# Wide frames: df_business (10 value cols), df_attributes (32), df_hours (7 raw day cols), 7 nested dicts at business level.
# Long frames: df_categories (category_name), 7 dict dfs (value) — a NULL value cell there = key listed with None.
import pandas as pd
from pyspark.sql import functions as F

print("=" * 80)
print("6.4.3) PER-CELL NULL RATE  —  raw vs effective vs applicability-adjusted")
print("=" * 80)
findings_643 = []

BUS_COLS = ["name", "address", "city", "state", "postal_code", "latitude", "longitude", "stars", "review_count", "is_open"]
ATTR_COLS = list(SCALAR_ATTRS.values())
DAY_COLS = [HOURS_MAP[d] for d in DAYS]
DICT_COLS = [DICT_COL[a] for a in DICT_TABLE_MAP]
seg_of = {f["field"]: f["seg"] for f in FIELDS64}

def _block(cols, raw_prefix="r_", app=None):
    aggs = [F.sum(sum(F.col(f"m_{c}") for c in cols)).alias("eff"),
            F.sum(sum(F.col(f"{raw_prefix}{c}" if raw_prefix else f"m_{c}") for c in cols)).alias("raw")]
    if app:
        aggs += [F.sum(sum(app(c) for c in cols)).alias("app"), F.sum(sum(F.col(f"m_{c}") * app(c) for c in cols)).alias("eff_app")]
    return aggs
_seg_app = lambda c: F.col(f"seg_{seg_of[c]}")
blocks = {
    "df_business":                 (BUS_COLS, "r_", None),
    "df_attributes":               (ATTR_COLS, "r_", _seg_app),
    "7 nested dicts (business-level)": (DICT_COLS, "r_", _seg_app),
    "df_hours (raw day columns)":  (DAY_COLS, None, lambda c: 1 - F.col("r_schedule")),
}
exprs = []
for i, (name, (cols, rp, app)) in enumerate(blocks.items()):
    for e, nm in zip(_block(cols, rp or "", app), ["eff", "raw", "app", "eff_app"]):
        exprs.append(e.alias(f"b{i}_{nm}"))
X = df_miss.agg(*exprs).first()

rows = []
for i, (name, (cols, rp, app)) in enumerate(blocks.items()):
    cells = N_BUS * len(cols)
    eff, raw = int(X[f"b{i}_eff"] or 0), int(X[f"b{i}_raw"] or 0)
    app_cells = int(X[f"b{i}_app"]) if app else cells
    eff_app = int(X[f"b{i}_eff_app"]) if app else eff
    rows.append({"DataFrame": name, "Shape": "wide", "Value Cols": len(cols), "Total Cells": cells,
                 "Raw NULL Cells": raw, "Raw NULL %": round(raw / cells * 100, 3),
                 "Effective NULL Cells": eff, "Effective NULL %": round(eff / cells * 100, 3),
                 "Hidden by Masking": eff - raw, "Applicable Cells": app_cells,
                 "Applicable NULL Cells": eff_app, "Applicable NULL %": round(eff_app / app_cells * 100, 3) if app_cells else 0.0})
# df_hours note: raw = effective (no masked day values); 'applicable' = days inside a listed week (NULL there = closed)

# ── Long frames ──
cat = df_categories.agg(F.count("*").alias("n"), F.sum(F.col("category_name").isNull().cast("int")).alias("raw"),
                        F.sum((F.col("category_name").isNull() | (F.regexp_replace("category_name", INVISIBLE_64, "") == "")).cast("int")).alias("eff")).first()
rows.append({"DataFrame": "df_categories (category_name)", "Shape": "long", "Value Cols": 1, "Total Cells": int(cat["n"]),
             "Raw NULL Cells": int(cat["raw"] or 0), "Raw NULL %": round((cat["raw"] or 0) / cat["n"] * 100, 3),
             "Effective NULL Cells": int(cat["eff"] or 0), "Effective NULL %": round((cat["eff"] or 0) / cat["n"] * 100, 3),
             "Hidden by Masking": int((cat["eff"] or 0) - (cat["raw"] or 0)), "Applicable Cells": int(cat["n"]),
             "Applicable NULL Cells": int(cat["eff"] or 0), "Applicable NULL %": round((cat["eff"] or 0) / cat["n"] * 100, 3)})
long_dict = []
for a, tbl in DICT_DF.items():
    x = business_dfs[tbl].agg(F.count("*").alias("n"), F.sum(F.col("value").isNull().cast("int")).alias("nulls")).first()
    long_dict.append((a, int(x["n"]), int(x["nulls"] or 0)))
ld_n, ld_null = sum(n for _, n, _ in long_dict), sum(v for _, _, v in long_dict)
rows.append({"DataFrame": "7 dict dfs (value)", "Shape": "long", "Value Cols": 1, "Total Cells": ld_n,
             "Raw NULL Cells": ld_null, "Raw NULL %": round(ld_null / ld_n * 100, 3), "Effective NULL Cells": ld_null,
             "Effective NULL %": round(ld_null / ld_n * 100, 3), "Hidden by Masking": 0, "Applicable Cells": ld_n,
             "Applicable NULL Cells": ld_null, "Applicable NULL %": round(ld_null / ld_n * 100, 3)})

cell_df = pd.DataFrame(rows)
wide = cell_df[cell_df["Shape"] == "wide"]
tot = {k: int(wide[k].sum()) for k in ["Total Cells", "Raw NULL Cells", "Effective NULL Cells", "Hidden by Masking", "Applicable Cells", "Applicable NULL Cells"]}
cell_df = pd.concat([cell_df, pd.DataFrame([{"DataFrame": "WIDE BUSINESS MODEL (total)", "Shape": "wide", "Value Cols": int(wide["Value Cols"].sum()), **tot,
    "Raw NULL %": round(tot["Raw NULL Cells"] / tot["Total Cells"] * 100, 3), "Effective NULL %": round(tot["Effective NULL Cells"] / tot["Total Cells"] * 100, 3),
    "Applicable NULL %": round(tot["Applicable NULL Cells"] / tot["Applicable Cells"] * 100, 3)}])], ignore_index=True)
print("\n📋 Cell-level null rates:")
display(cell_df)
print("\n   Per-dict long-frame NULL values (key listed as None):")
display(pd.DataFrame([{"Dict": a, "Key-value cells": n, "NULL values": v, "NULL %": round(v / n * 100, 2) if n else 0.0} for a, n, v in long_dict]))

for r_ in cell_df.to_dict("records"):
    if r_["DataFrame"].startswith("WIDE"):
        continue
    st = None if r_["Hidden by Masking"] else ("INFO" if r_["Effective NULL Cells"] else "PASS")
    dq_add(findings_643, r_["DataFrame"], "cell null rate (raw vs effective)", "raw SQL-NULL rate = true missing rate (no hidden nulls)",
           r_["Total Cells"], r_["Hidden by Masking"] if r_["Hidden by Masking"] else r_["Effective NULL Cells"], status=st,
           examples=f"raw {r_['Raw NULL %']}% → effective {r_['Effective NULL %']}% → applicable {r_['Applicable NULL %']}%",
           note=("masked nulls under-report the cell null rate" if r_["Hidden by Masking"] else
                 ("DQ-BUSINESS-006: key listed with None" if "dict dfs" in r_["DataFrame"] else
                  ("NULL day = closed (DQ-BUSINESS-033)" if "hours" in r_["DataFrame"] else ""))))
w = cell_df.iloc[-1]
dq_add(findings_643, "wide business model", "overall effective cell null rate", "informative: size of the gap Silver inherits",
       w["Total Cells"], w["Effective NULL Cells"], status="INFO",
       examples=f"raw {w['Raw NULL %']}% | effective {w['Effective NULL %']}% | applicable-only {w['Applicable NULL %']}%",
       note="the gap between effective and applicable = structural 'not applicable' NULLs (DQ-BUSINESS-034)")
completeness_643 = {"raw_null_pct": float(w["Raw NULL %"]), "effective_null_pct": float(w["Effective NULL %"]),
                    "applicable_null_pct": float(w["Applicable NULL %"])}
print(f"\n   Wide model: {w['Total Cells']:,} cells | raw NULL {w['Raw NULL %']}% | effective {w['Effective NULL %']}% | "
      f"applicable-only {w['Applicable NULL %']}%  → completeness of applicable cells = {100 - w['Applicable NULL %']:.2f}%")
findings_643_df = dq_summary(findings_643, "6.4.3 Per-Cell Null Rate")
print("PER-CELL NULL RATE CHECK COMPLETE")
print("=" * 80)

---
#### 🚨 DQ-BUSINESS-034 — Structural "Not Applicable" NULLs Dominate the Wide Attribute Model

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-034` |
| **Severity** | MEDIUM (modelling / completeness-semantics issue; no value is wrong, but NULL carries two meanings) |
| **Detected In** | Phase 6.4 — Completeness → **6.4.3 Per-Cell Null Rate** (Cell 82), "effective vs applicable" breakdown |
| **Also Confirmed In** | 6.4.1 (Cell 80): 5 near-empty attributes (< 5 % in-segment coverage: `OpenTwentyFourHours` 0.03 %, `RestaurantsCounterService` 0.03 %, `DietaryRestrictions` 0.04 %, `AgesAllowed` 0.5 %, `BYOBCorkage` 2.13 %). 6.4.4 (Cell 84): 0 MCAR-consistent attributes outside these near-empty ones |
| **Affected Columns** | 32 scalar attributes in `df_attributes`, 7 nested dicts |
| **Total Affected Rows** | Wide model: **8,419,376** cells; raw NULL **58.30 %**, effective **58.51 %**, but only **36.47 %** once cells that do not apply to the business type are excluded. `df_attributes` alone: 78.8 % effective vs 62.7 % applicable |
| **Invalid Value** | `attr_restaurants_reservations = NULL` for a dentist (not applicable) is stored identically to `NULL` for a restaurant (unknown) |
| **Domain Rule** | Missingness metrics and Silver columns must distinguish *unknown* from *not applicable to this business type* |

**Description:** Yelp asks a different questionnaire per business type (restaurants, nightlife, salons, health). Flattening every attribute into one wide row gives every business ~40 attribute columns, most of which do not apply to it. Around 22 percentage points of the wide-model null rate are this structural "not applicable" NULL, not missing data. A completeness KPI or an ML feature matrix built on the raw wide table overstates missingness, and imputation would invent answers to questions that were never asked. The near-empty attributes add almost-all-NULL columns with no analytic value.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Store attributes in the long (business_id, attribute, value) Silver table, plus an applicability map (attribute → segment) used by completeness metrics; archive the 5 near-empty attributes** | Long form stores only answered questions; applicability makes "unknown" measurable; removes dead columns |
| 2 | **Keep wide table but add a `<attr>_applicable` flag per attribute** | Correct semantics, but doubles column count |
| 3 | **Fill non-applicable cells with a sentinel (`'N/A'`)** | Breaks boolean / enum types |
| 4 | **Impute all NULLs** | Fabricates answers for non-applicable questions |

**Best Logical Solution:** **Option 1 — long attribute model plus an explicit applicability map; drop / archive near-empty attributes.**

---

In [0]:
# 6.4.4) Missingness Pattern Analysis (MCAR / MAR / MNAR)  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
#   MCAR — missingness independent of everything (a random 50/50 split test cannot detect dependence, so it is not used)
#   MAR  — missingness depends on OBSERVED variables  → tested: missing rate vs 5 fully-observed business covariates
#   MNAR — missingness depends on the unobserved value itself → not testable from data; assigned from domain evidence
# Test: for each field with effective missingness, a 2 × k contingency (missing / present × covariate level) per covariate →
#   chi² and Cramér's V (effect size, robust to N = 150 k where every p-value is ~0).  Covariates (0 missing, 6.4.1):
#   is_open | review_count bucket (popularity) | stars bucket | state (Yelp metro) | segment (food / service-area / health / hair / other)
# Rule: V >= 0.10 → MAR (driver = covariate with max V); 0.05 – 0.10 → weak MAR; < 0.05 → MCAR-consistent.
# Optional attributes are tested INSIDE their applicable segment (so "not applicable" is not mistaken for MAR on segment).
import math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F

print("=" * 80)
print("6.4.4) MISSINGNESS PATTERN ANALYSIS (MCAR / MAR / MNAR)  —  Business DataFrames")
print("=" * 80)
findings_644 = []

_fp = list(YELP_FOOTPRINT_STATES) if "YELP_FOOTPRINT_STATES" in globals() else ["PA", "FL", "TN", "IN", "MO", "LA", "AZ", "NJ", "NV", "AB", "CA", "ID", "DE", "IL"]
df_cov = df_miss.select("*",
    F.when(F.col("is_open"), "open").otherwise("closed").alias("cv_is_open"),
    F.when(F.col("review_count") < 10, "a) 5-9").when(F.col("review_count") < 25, "b) 10-24")
     .when(F.col("review_count") < 100, "c) 25-99").when(F.col("review_count") < 500, "d) 100-499").otherwise("e) 500+").alias("cv_reviews"),
    F.when(F.col("stars_v") <= 2.5, "a) 1.0-2.5").when(F.col("stars_v") <= 3.5, "b) 3.0-3.5")
     .when(F.col("stars_v") <= 4.5, "c) 4.0-4.5").otherwise("d) 5.0").alias("cv_stars"),
    F.when(F.col("state").isin(_fp), F.col("state")).otherwise("OTHER").alias("cv_state"),
    F.when(F.col("seg_food") == 1, "food").when(F.col("seg_service_area") == 1, "service_area")
     .when(F.col("seg_health") == 1, "health").when(F.col("seg_hair") == 1, "hair")
     .when(F.col("m_categories") == 1, F.lit(None)).otherwise("other").alias("cv_segment"))   # no category → excluded (segment is derived FROM categories)
COVARS = ["cv_is_open", "cv_reviews", "cv_stars", "cv_state", "cv_segment"]

# Fields tested: core / blocks with any missingness + every optional attribute with >= 30 in-segment gaps
TEST = [f for f in FIELDS64 if f["group"] != "schedule day"]
_cnt = dict(zip(col_null_df["Field"], col_null_df["Missing in Segment"]))
_app_n = dict(zip(col_null_df["Field"], col_null_df["Applicable Rows"]))
TEST = [f for f in TEST if 30 <= _cnt.get(f["field"], 0) < _app_n.get(f["field"], 0)]
print(f"   Fields tested: {len(TEST)} | covariates: {COVARS}")

cont = {}
for cv in COVARS:
    aggs = [F.count("*").alias("n")]
    for i, f in enumerate(TEST):
        aggs += [F.sum(F.col(f"m_{f['field']}") * f["app"]).alias(f"m{i}"), F.sum(f["app"]).alias(f"a{i}")]
    cont[cv] = df_cov.where(F.col(cv).isNotNull()).groupBy(cv).agg(*aggs).toPandas()

def cramers_v(miss, app):
    """2 x k table: rows missing/present, columns = covariate levels."""
    pairs = [(m, a) for m, a in zip(miss, app) if a > 0]
    n = sum(a for _, a in pairs); M = sum(m for m, _ in pairs)
    if n == 0 or M in (0, n) or len(pairs) < 2:
        return 0.0, 0.0
    p = M / n; chi2 = 0.0
    for m, a in pairs:
        e1, e0 = a * p, a * (1 - p)
        chi2 += (m - e1) ** 2 / e1 + ((a - m) - e0) ** 2 / e0
    return chi2, math.sqrt(chi2 / n)

DOMAIN_MNAR = {
    "address":     "service-area businesses deliberately hide their street address (DQ-BUSINESS-032) — missing BECAUSE there is no storefront",
    "postal_code": "same mechanism as address for 56 rows; rest are entry gaps (DQ-BUSINESS-015)",
    "schedule":    "owners of low-traffic / closed businesses rarely publish hours — absence partly reflects the (unseen) operating reality",
    "attributes":  "attributes are owner / crowd answers — an un-answered attribute is often one the business has no positive answer for",
}
res = []
for i, f in enumerate(TEST):
    row = {"Field": f["field"], "Group": f["group"], "Requirement": f["req"], "Applies To": f["seg"],
           "Missing": int(_cnt[f["field"]]), "Missing % (segment)": round(_cnt[f["field"]] / _app_n[f["field"]] * 100, 2)}
    best, best_v, spread = None, -1.0, ""
    for cv in COVARS:
        t = cont[cv]
        if cv == "cv_segment" and f["field"] == "categories":      # tautological covariate for this field
            row[f"V {cv[3:]}"] = None
            continue
        _, v = cramers_v(t[f"m{i}"].tolist(), t[f"a{i}"].tolist())
        row[f"V {cv[3:]}"] = round(v, 3)
        if v > best_v:
            ok = t[t[f"a{i}"] >= 500]
            rates = (ok[f"m{i}"] / ok[f"a{i}"] * 100).round(1)
            lv = dict(zip(ok[cv], rates))
            best, best_v = cv[3:], v
            spread = (f"{min(lv, key=lv.get)}={min(lv.values())}% → {max(lv, key=lv.get)}={max(lv.values())}%" if lv else "")
    row["Strongest Driver"], row["Max V"], row["Missing-rate Range (driver)"] = best, round(best_v, 3), spread
    cls = "MAR" if best_v >= 0.10 else ("MAR (weak)" if best_v >= 0.05 else "MCAR-consistent")
    if f["field"] == "address":
        cls = f"MNAR (structural) + {cls}"
    elif f["field"] in ("schedule", "attributes") and best_v >= 0.05:
        cls = f"{cls} + suspected MNAR"
    if f["field"] in DOMAIN_MNAR:
        why = DOMAIN_MNAR[f["field"]]
    elif best == "reviews":
        why = "crowd-sourced attribute: answers accumulate with review volume → low-traffic businesses stay blank"
    elif best == "segment":
        why = "asked of every business, but answered mainly by certain business types → de-facto category-specific"
    elif best == "state":
        why = "regional sourcing gap — the attribute is rarely captured in one metro / market"
    elif best == "stars":
        why = "missingness varies with rating level"
    elif best == "is_open":
        why = "closed businesses stopped being enriched"
    else:
        why = ""
    row["Classification"] = cls
    near_empty = row["Missing % (segment)"] >= 95
    row["Rationale"] = (why if best_v >= 0.05 or f["field"] in DOMAIN_MNAR else
                        ("near-empty column — too little signal to model the mechanism" if near_empty else
                         "no dependence on any observed covariate — isolated entry gaps"))
    res.append(row)
mech_df = pd.DataFrame(res).sort_values(["Group", "Max V"], ascending=[True, False])
mech_df = pd.concat([mech_df, pd.DataFrame([{
    "Field": "hours_<day> (NULL inside a listed week)", "Group": "schedule day", "Requirement": "PROFILED", "Applies To": "has schedule",
    "Missing": int(day_df["Missing in Segment"].sum()), "Missing % (segment)": round(day_df["Missing in Segment"].sum() / day_df["Applicable Rows"].sum() * 100, 2),
    "Classification": "MNAR (by design)", "Rationale": "the day is NULL because the business is closed that day — the missing value IS the information (DQ-BUSINESS-033)"}])],
    ignore_index=True)
print("\n📋 Missingness mechanism per field (Cramér's V per covariate):")
display(mech_df)

# ── Missing-rate profile of the core blocks by the main drivers (readable evidence) ──
profile_fields = [(i, f) for i, f in enumerate(TEST) if f["field"] in ("address", "postal_code", "attributes", "schedule", "categories")]
for cv in ["cv_reviews", "cv_is_open", "cv_segment"]:
    t = cont[cv]; out = pd.DataFrame({cv[3:]: t[cv], "businesses": t["n"]})
    for i, f in profile_fields:
        out[f"{f['field']} missing %"] = (t[f"m{i}"] / t[f"a{i}"].where(t[f"a{i}"] > 0) * 100).round(2)
    print(f"\n   ── Missing rate of core blocks by {cv[3:]} ──")
    display(out.sort_values(cv[3:]))

# ── Heatmap of Cramér's V ──
vcols = [c for c in mech_df.columns if c.startswith("V ")]
hm = mech_df.dropna(subset=["Max V"]).set_index("Field")[vcols].astype(float)
hm = hm.loc[hm.max(axis=1).sort_values(ascending=False).index]
fig, ax = plt.subplots(figsize=(9, max(4, 0.32 * len(hm))))
sns.heatmap(hm, annot=True, fmt=".2f", cmap="Reds", vmin=0, vmax=0.4, cbar_kws={"label": "Cramér's V"}, ax=ax)
ax.set_title("6.4.4 Missingness dependence on observed covariates (MAR evidence)")
plt.tight_layout(); plt.show()

# ── Findings ──
for r_ in mech_df.to_dict("records"):
    if r_["Group"] == "schedule day":
        dq_add(findings_644, "df_hours", r_["Field"], "mechanism documented", int(day_df["Applicable Rows"].sum()), r_["Missing"],
               status="INFO", examples=r_["Classification"], note=r_["Rationale"])
        continue
    impactful = r_["Requirement"] in ("CONDITIONAL", "EXPECTED") and r_["Max V"] >= 0.10
    dq_add(findings_644, "df_miss", r_["Field"], "missingness independent of observed covariates (MCAR)",
           _app_n[r_["Field"]], r_["Missing"], status="FINDING" if impactful else "INFO",
           examples=f"{r_['Classification']} | driver={r_['Strongest Driver']} (V={r_['Max V']}) | {r_['Missing-rate Range (driver)']}",
           note=("non-random gap in a core block → complete-case analysis is biased toward popular / open businesses" if impactful else r_["Rationale"]))
mar_attrs = mech_df[(mech_df["Group"].str.startswith("attribute")) & (mech_df["Max V"] >= 0.10)]
dq_add(findings_644, "df_attributes + dict dfs", "optional attributes with MAR missingness", "in-segment missingness independent of covariates",
       int((mech_df["Group"].str.startswith("attribute")).sum()), len(mar_attrs), status="INFO" if len(mar_attrs) else "PASS",
       examples=", ".join(f"{x['Field']}({x['Strongest Driver']}, V={x['Max V']})" for x in mar_attrs.head(8).to_dict("records")),
       note="impute only conditionally on the driver (e.g. by review-count band), or keep NULL + 'unknown' flag")
mech_644_df = mech_df
findings_644_df = dq_summary(findings_644, "6.4.4 Missingness Pattern Analysis")
print("MISSINGNESS PATTERN ANALYSIS COMPLETE")
print("=" * 80)

In [0]:
# 6.4.5) Missing Value Correlation  —  do gaps in one field predict gaps in another?
# ─────────────────────────────────────────────────────────────────────
# Business-table equivalent of "is text missing when stars is missing?":
#   - Are businesses with no schedule also the ones with no attributes / no categories (thin listings)?
#   - Do address and postal_code go missing together (service-area pattern)?
#   - Do the expected attributes vanish as a block (never enriched) or independently (single un-answered questions)?
# Method: phi coefficient (Pearson on 0/1 indicators) computed ONLY on rows where BOTH fields are applicable
# (so 'not applicable' does not create fake correlation), plus lift = P(B missing | A missing) / P(B missing).
# Also the co-missing pattern table of the 5 core fields / blocks.
import math
import itertools
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F

print("=" * 80)
print("6.4.5) MISSING VALUE CORRELATION  —  co-missingness between business fields")
print("=" * 80)
findings_645 = []

CORE_SET = ["address", "postal_code", "attributes", "schedule", "categories"]
exp_sorted = sorted(EXPECTED_ATTRS, key=lambda x: -_cnt.get(x[0], 0))[:10]
IND = [(c, F.lit(1)) for c in CORE_SET] + [(a, F.col(f"seg_{s}")) for a, s in exp_sorted]
names = [c for c, _ in IND]
print(f"   Fields: {len(IND)} ({', '.join(names)}) → {len(IND) * (len(IND) - 1) // 2} pairs")

aggs = []
for (i, (a, appA)), (j, (b, appB)) in itertools.combinations(enumerate(IND), 2):
    w = appA * appB; mA, mB = F.col(f"m_{a}") * w, F.col(f"m_{b}") * w
    aggs += [F.sum(w).alias(f"n_{i}_{j}"), F.sum(mA).alias(f"a_{i}_{j}"), F.sum(mB).alias(f"b_{i}_{j}"),
             F.sum(F.col(f"m_{a}") * F.col(f"m_{b}") * w).alias(f"ab_{i}_{j}")]
P = df_miss.agg(*aggs).first()

pairs, mat = [], pd.DataFrame(0.0, index=names, columns=names)
for (i, (a, _)), (j, (b, _)) in itertools.combinations(enumerate(IND), 2):
    n, A_, B_, AB = (int(P[f"{k}_{i}_{j}"] or 0) for k in ("n", "a", "b", "ab"))
    den = math.sqrt(A_ * (n - A_) * B_ * (n - B_)) if n else 0
    phi = (n * AB - A_ * B_) / den if den else 0.0
    p_b, p_b_a = (B_ / n if n else 0), (AB / A_ if A_ else 0)
    p_a, p_a_b = (A_ / n if n else 0), (AB / B_ if B_ else 0)
    mat.loc[a, b] = mat.loc[b, a] = phi
    pairs.append({"Field A": a, "Field B": b, "Rows (both applicable)": n, "A missing": A_, "B missing": B_, "Both missing": AB,
                  "phi": round(phi, 3), "P(B miss)": round(p_b * 100, 2), "P(B miss | A miss)": round(p_b_a * 100, 2),
                  "Lift B|A": round(p_b_a / p_b, 2) if p_b else None, "P(A miss | B miss)": round(p_a_b * 100, 2),
                  "Lift A|B": round(p_a_b / p_a, 2) if p_a else None})
for c in names:
    mat.loc[c, c] = 1.0
pairs_df = pd.DataFrame(pairs).sort_values("phi", key=lambda s: s.abs(), ascending=False)
# 'attributes' block missing ⇒ every attribute missing (by construction) → those pairs are tautological, not evidence
_taut = (pairs_df["Field A"] == "attributes") & pairs_df["Field B"].str.startswith("attr_")
pairs_df = pairs_df.assign(Tautological=_taut)

fig, ax = plt.subplots(figsize=(11, 8.5))
sns.heatmap(mat, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax, cbar_kws={"label": "phi (co-missingness)"})
ax.set_title("6.4.5 Missingness correlation (phi on jointly-applicable rows)")
plt.tight_layout(); plt.show()

strong = pairs_df[(pairs_df["phi"].abs() >= 0.3) & ~pairs_df["Tautological"]]
print(f"\n📋 Correlated missingness (|phi| >= 0.3): {len(strong)} of {len(pairs_df)} pairs")
display(strong.assign(Strength=lambda d: d["phi"].abs().map(lambda v: "strong" if v >= 0.7 else "moderate")))
print("\n   Core-block pairs (always shown):")
display(pairs_df[pairs_df["Field A"].isin(CORE_SET) & pairs_df["Field B"].isin(CORE_SET)])

# ── The headline question, answered explicitly ──
sch_attr = pairs_df[((pairs_df["Field A"] == "attributes") & (pairs_df["Field B"] == "schedule")) |
                    ((pairs_df["Field A"] == "schedule") & (pairs_df["Field B"] == "attributes"))].iloc[0]
q = df_miss.groupBy("m_schedule").agg(F.count("*").alias("n"), F.avg("m_attributes").alias("p_attr"), F.avg("m_categories").alias("p_cat")).toPandas().set_index("m_schedule")
print(f"\n   ❓ Are businesses without a schedule also missing attributes / categories?")
print(f"      P(no attributes | no schedule)  = {q.loc[1, 'p_attr'] * 100:.2f}%   vs   P(no attributes | has schedule) = {q.loc[0, 'p_attr'] * 100:.2f}%")
print(f"      P(no categories | no schedule)  = {q.loc[1, 'p_cat'] * 100:.3f}%  vs   P(no categories | has schedule) = {q.loc[0, 'p_cat'] * 100:.3f}%")

# ── Co-missing pattern of the core blocks ──
print("\n📋 Co-missing patterns of the core blocks (1 = missing):")
pat = (df_miss.groupBy("m_address", "m_postal_code", "m_attributes", "m_schedule", "m_categories")
       .agg(F.count("*").alias("businesses"), F.round(F.avg("review_count"), 1).alias("avg_review_count"),
            F.round(F.avg(F.col("is_open").cast("int")) * 100, 1).alias("pct_open"))
       .withColumn("pct", F.round(F.col("businesses") / N_BUS * 100, 3)).orderBy(F.desc("businesses")).toPandas())
pat["blocks missing"] = pat[["m_address", "m_postal_code", "m_attributes", "m_schedule", "m_categories"]].sum(axis=1)
display(pat)
multi = pat[pat["blocks missing"] >= 2]

# ── Findings ──
attr_pairs = pairs_df[pairs_df["Field A"].str.startswith("attr_") & pairs_df["Field B"].str.startswith("attr_")]
attr_strong = attr_pairs[attr_pairs["phi"].abs() >= 0.3]
dq_add(findings_645, "df_attributes + dict dfs", "expected attributes co-missing as a block", "attribute gaps are independent questions",
       len(attr_pairs), len(attr_strong), status="FINDING" if len(attr_strong) else "PASS",
       examples=" | ".join(f"{x['Field A'][5:]}↔{x['Field B'][5:]} phi={x['phi']} lift={x['Lift B|A']}" for x in attr_strong.head(5).to_dict("records")),
       note="listings are enriched (or not) as a whole questionnaire — imputing one attribute from another missing one is impossible")
ap = pairs_df[pairs_df["Field A"].isin(["address", "postal_code"]) & pairs_df["Field B"].isin(["address", "postal_code"])].iloc[0]
dq_add(findings_645, "df_business", "address ↔ postal_code", "missingness of the two fields is independent",
       ap["Rows (both applicable)"], ap["Both missing"], status="INFO",
       examples=f"phi={ap['phi']} | P(addr miss | ZIP miss)={ap['P(A miss | B miss)']}% vs P(addr miss)={round(ap['A missing'] / ap['Rows (both applicable)'] * 100, 2)}% (lift {ap['Lift A|B']})",
       note=f"{ap['P(A miss | B miss)']:.0f} % of empty ZIPs also have no address — same withheld-location mechanism (DQ-BUSINESS-015 / -032); phi is low only because ZIP gaps are rare")
_lift = max(sch_attr["Lift B|A"] or 0, sch_attr["Lift A|B"] or 0)
dq_add(findings_645, "df_miss", "schedule ↔ attributes", "missingness of the two blocks is independent",
       sch_attr["Rows (both applicable)"], sch_attr["Both missing"], status="FINDING" if _lift >= 1.5 else "INFO",
       examples=f"phi={sch_attr['phi']} | P(no attr | no schedule)={q.loc[1, 'p_attr'] * 100:.1f}% vs {q.loc[0, 'p_attr'] * 100:.1f}% (lift {sch_attr['Lift B|A']})",
       note="thin listings lack several blocks at once — 6.4.2 skeleton / low-information flag")
dq_add(findings_645, "df_miss", "businesses missing >= 2 core blocks", "gaps are isolated (one block per business)",
       N_BUS, int(multi["businesses"].sum()), status="INFO",
       examples=", ".join(f"{int(x['businesses']):,}×[addr={x['m_address']},zip={x['m_postal_code']},attr={x['m_attributes']},hours={x['m_schedule']},cat={x['m_categories']}]"
                          for x in multi.head(5).to_dict("records")))
pairs_645_df = pairs_df
findings_645_df = dq_summary(findings_645, "6.4.5 Missing Value Correlation")
print("MISSING VALUE CORRELATION CHECK COMPLETE")
print("=" * 80)

---
#### 🚨 DQ-BUSINESS-035 — Missingness Is Not Random: It Follows Popularity and Business Type, and Attributes Go Missing as a Block

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-035` |
| **Severity** | HIGH for analytics (complete-case analysis and naive imputation are biased) |
| **Detected In** | Phase 6.4 — Completeness → **6.4.4 MCAR / MAR / MNAR** (Cell 84) and **6.4.5 Missing Value Correlation** (Cell 85) |
| **Also Confirmed In** | 6.4.2 (Cell 81): 28,570 rows > 25 % profile missing, 102 skeleton listings. 6.4.7 (Cell 89): 1,543 businesses not analytics-ready |
| **Affected Columns** | `attributes` block, `hours` block, 38 of 44 tested fields classified MAR (Cramér's V >= 0.10) |
| **Total Affected Rows** | attributes 13,960 / schedule 23,223 missing; **4,298** businesses miss both; 36 of 45 expected-attribute pairs co-missing with phi >= 0.3 |
| **Invalid Value** | n/a (pattern issue) |
| **Domain Rule** | Silver features and KPIs must not assume missing attributes / hours are a random sample |

**Description:**
* **Popularity driven (MAR):** missing schedule falls from **23.7 %** (5–9 reviews) to **0.8 %** (500+); missing attributes from 17.4 % to 0 %. Crowd-sourced attributes such as `NoiseLevel` are missing for 76.9 % of 5–9-review restaurants against 1.3 % of 500+.
* **Business-type driven:** "all-business" attributes are in practice category-specific (`GoodForKids` missing 36.7 % for food against 95.4 % for service-area; `ByAppointmentOnly` 9.1 % for hair against 92.1 % for food).
* **Regional sourcing gap:** `BusinessAcceptsCreditCards` is missing for **95.8 %** of Alberta (Edmonton) businesses against 15.5 % in Missouri. This is a market-level collection difference, not business behaviour.
* **Block-wise:** restaurant attributes vanish together (`RestaurantsAttire` ↔ `RestaurantsGoodForGroups` phi 0.80; P(GoodForGroups missing | Attire missing) = 96.5 %). A business without hours is 2.4× as likely to also lack attributes (18.5 % vs 7.6 %). A listing is enriched as a whole questionnaire or not at all.
* **Structural MNAR:** missing address is concentrated in service-area businesses (11.6 % against 0.9–1.4 % for storefront segments), see DQ-BUSINESS-032.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep NULLs, add per-business `profile_completeness_pct`, `has_attributes`, `has_hours`, `is_low_information` flags and document MAR drivers; analyses stratify / weight by `review_count` band and segment** | Preserves all rows, makes the bias visible and controllable downstream |
| 2 | **Impute attributes from other attributes / similar businesses** | Block-wise missingness means there is usually no observed attribute to impute from; would fabricate answers mostly for low-traffic businesses |
| 3 | **Drop incomplete businesses (complete-case)** | Removes 24 % of businesses, skewed towards small / new / non-restaurant businesses |
| 4 | **Treat missing as `False`** | Wrong for crowd-sourced attributes: absence means "not asked / not answered", not "no" |

**Best Logical Solution:** **Option 1 — flag and stratify, never complete-case or blanket-impute.** For the Alberta credit-card gap, report the metric per market or exclude AB from that attribute's KPIs.

---

In [0]:
# 6.4.6) Temporal Gap Analysis  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# The business table is a SNAPSHOT (no event timestamp), so it has two time series of its own:
#   A) the WEEKLY SCHEDULE (df_hours) — a 7-slot cycle. Gaps here are missing days inside a published week.
#      Weekend / Monday closures are normal (DQ-BUSINESS-033); an isolated MID-WEEK hole (Tue/Wed/Thu NULL while
#      both neighbouring days are open) or a 1–2-day schedule is far more likely an incomplete entry.
#   B) the ACTIVITY TIMELINE implied by the business's reviews (bronze_final.yelp_review, read-only, joined on
#      business_id). Completeness questions for the business entity:
#        - does every business have review history, and is it as long as its own review_count claims?
#        - is the dataset-level month grid continuous between the first and last review?
#        - how long are the silent periods per business, and which "open" businesses went silent before the snapshot?
#      (Full review-vs-review_count reconciliation is 6.6 consistency; here only the *missing history* side is measured.)
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("=" * 80)
print("6.4.6) TEMPORAL GAP ANALYSIS  —  weekly schedule + review activity timeline")
print("=" * 80)
findings_646 = []
REVIEW_TABLE = "yelp_poc_uc.bronze_final.yelp_review"

# ══ A) Weekly schedule gaps ══
print(f"\n{'─' * 60}\n📌 A) Weekly schedule — missing days inside a published week")
listed = [F.col(HOURS_MAP[d]).isNotNull().cast("int") for d in DAYS]
mask = F.concat(*[F.when(F.col(HOURS_MAP[d]).isNotNull(), F.lit("1")).otherwise(F.lit("0")) for d in DAYS])
midweek_hole = F.lit(False)
for i in (1, 2, 3):                                          # Tue, Wed, Thu with both neighbours listed
    midweek_hole = midweek_hole | (F.col(HOURS_MAP[DAYS[i]]).isNull() & F.col(HOURS_MAP[DAYS[i - 1]]).isNotNull() & F.col(HOURS_MAP[DAYS[i + 1]]).isNotNull())
hw = df_hours.select("business_id", sum(listed).alias("days_listed"), mask.alias("week_mask"), midweek_hole.alias("midweek_hole")) \
             .where(F.col("days_listed") > 0)
H = hw.agg(F.count("*").alias("n"), F.sum(F.col("midweek_hole").cast("int")).alias("hole"),
           F.sum((F.col("days_listed") <= 2).cast("int")).alias("sparse"), F.sum((F.col("days_listed") == 7).cast("int")).alias("full")).first()
print(f"   Businesses with a schedule: {H['n']:,} | full 7-day week: {H['full']:,} ({H['full'] / H['n'] * 100:.1f}%)")
print("\n   Days listed per week:")
display(hw.groupBy("days_listed").agg(F.count("*").alias("businesses")).withColumn("pct", F.round(F.col("businesses") / H["n"] * 100, 2)).orderBy("days_listed"))
print("\n   NULL frequency by weekday (inside listed weeks):")
display(day_df[["Field", "Missing in Segment", "Applicable Rows"]].rename(columns={"Missing in Segment": "NULL days", "Applicable Rows": "Schedules"})
        .assign(**{"NULL %": lambda d: (d["NULL days"] / d["Schedules"] * 100).round(2)}))
print("\n   Most frequent week patterns (Mon→Sun, 1 = listed):")
display(hw.groupBy("week_mask").agg(F.count("*").alias("businesses"), F.max("midweek_hole").alias("midweek_hole"))
        .withColumn("pct", F.round(F.col("businesses") / H["n"] * 100, 2)).orderBy(F.desc("businesses")).limit(15))
hole_ex = (hw.where("midweek_hole").join(df_business.select("business_id", "name", "city"), "business_id")
           .join(df_hours, "business_id").select("name", "city", "week_mask", *DAY_COLS).limit(8))
print("\n   Examples of isolated mid-week holes:")
display(hole_ex)
dq_add(findings_646, "df_hours", "isolated mid-week hole (Tue/Wed/Thu)", "a single closed mid-week day between two open days is unusual → possible missing entry",
       H["n"], H["hole"], status="INFO",
       note="mostly Tuesday (restaurants / salons often close Tue); Wed/Thu holes are the suspicious ones — indistinguishable from closure, flag only (DQ-BUSINESS-033)")
dq_add(findings_646, "df_hours", "sparse schedule (1–2 days listed)", "a published schedule covers the operating week",
       H["n"], H["sparse"], status="INFO", note="seasonal / appointment / event businesses OR partially entered hours")

# ══ B) Review activity timeline ══
print(f"\n{'─' * 60}\n📌 B) Review activity timeline  (source: {REVIEW_TABLE}, read-only)")
rv = spark.table(REVIEW_TABLE).select("business_id", F.to_date(F.substring("date", 1, 10)).alias("rd"))
per = rv.groupBy("business_id").agg(F.count("*").alias("n_loaded"), F.min("rd").alias("first_review"), F.max("rd").alias("last_review"))
w = Window.partitionBy("business_id").orderBy("rd")
gap = rv.withColumn("g", F.datediff("rd", F.lag("rd").over(w))).groupBy("business_id").agg(F.max("g").alias("max_gap_days"))
tl = df_business.select("business_id", "name", "city", "is_open", "review_count").join(per, "business_id", "left").join(gap, "business_id", "left")
snap = rv.agg(F.min("rd").alias("lo"), F.max("rd").alias("hi")).first()
SNAP_END, SNAP_START = snap["hi"], snap["lo"]
print(f"   Review window: {SNAP_START} → {SNAP_END} (snapshot end = latest review)")

T = tl.agg(F.count("*").alias("n"), F.sum(F.col("n_loaded").isNull().cast("int")).alias("no_hist"),
           F.sum((F.col("n_loaded") < F.col("review_count")).cast("int")).alias("short"),
           F.sum(F.when(F.col("n_loaded") < F.col("review_count"), F.col("review_count") - F.col("n_loaded")).otherwise(0)).alias("short_reviews"),
           F.sum((F.col("max_gap_days") > 730).cast("int")).alias("gap2y"), F.sum((F.col("max_gap_days") > 1095).cast("int")).alias("gap3y"),
           F.expr("percentile(max_gap_days, array(0.5, 0.9, 0.99))").alias("gap_p"),
           F.sum((F.col("is_open") & (F.datediff(F.lit(SNAP_END), F.col("last_review")) > 365)).cast("int")).alias("open_silent_1y"),
           F.sum((F.col("is_open") & (F.datediff(F.lit(SNAP_END), F.col("last_review")) > 1095)).cast("int")).alias("open_silent_3y"),
           F.sum(F.col("is_open").cast("int")).alias("n_open")).first()
print(f"   Businesses with NO review history loaded: {T['no_hist']:,}")
print(f"   Businesses with fewer loaded reviews than review_count: {T['short']:,} (missing {T['short_reviews']:,} reviews)")
print(f"   Longest silent period per business (days) — median {T['gap_p'][0]:.0f} | p90 {T['gap_p'][1]:.0f} | p99 {T['gap_p'][2]:.0f}")
print(f"   Open businesses silent > 1 year before snapshot end: {T['open_silent_1y']:,} of {T['n_open']:,}  | > 3 years: {T['open_silent_3y']:,}")

# dataset-level month grid between first and last review (no calendar-boundary artefacts)
months = rv.groupBy(F.trunc("rd", "month").alias("m")).agg(F.count("*").alias("reviews"), F.countDistinct("business_id").alias("active_businesses")).toPandas()
grid = pd.date_range(pd.Timestamp(SNAP_START).to_period("M").to_timestamp(), pd.Timestamp(SNAP_END).to_period("M").to_timestamp(), freq="MS")
months["m"] = pd.to_datetime(months["m"])
missing_months = sorted(set(grid) - set(months["m"]))
months = months.sort_values("m")
months["yoy_drop_pct"] = ((months["reviews"] / months["reviews"].shift(12) - 1) * 100).round(1)
print(f"   Month grid {grid[0]:%Y-%m} → {grid[-1]:%Y-%m}: {len(grid)} expected, {len(months)} present, {len(missing_months)} missing")
yearly = months.assign(year=months["m"].dt.year).groupby("year").agg(reviews=("reviews", "sum"), months=("m", "count"),
                                                                   peak_active_businesses=("active_businesses", "max")).reset_index()
print("\n   Yearly review volume for the 150 k businesses (partial first / last years flagged by months < 12):")
display(yearly)
collapse = months[(months["yoy_drop_pct"] <= -40)]
if len(collapse):
    print("\n   Months with a >= 40 % year-over-year volume drop (collection or real-world shock):")
    display(collapse[["m", "reviews", "active_businesses", "yoy_drop_pct"]])

print("\n   Examples — businesses whose loaded history is shorter than review_count:")
display(tl.where(F.col("n_loaded") < F.col("review_count")).select("business_id", "name", "city", "review_count", "n_loaded",
        (F.col("review_count") - F.col("n_loaded")).alias("missing_reviews"), "first_review", "last_review").orderBy(F.desc("missing_reviews")))
print("\n   Examples — open businesses with the longest silence before the snapshot:")
display(tl.where(F.col("is_open")).select("business_id", "name", "city", "review_count", "first_review", "last_review",
        F.datediff(F.lit(SNAP_END), F.col("last_review")).alias("days_silent")).orderBy(F.desc("days_silent")).limit(10))

dq_add(findings_646, "df_business × yelp_review", "business without any review history", "every business has >= 1 review (dataset rule: >= 5)", T["n"], T["no_hist"])
dq_add(findings_646, "df_business × yelp_review", "loaded history shorter than review_count", "no reviews missing from the business's history",
       T["n"], T["short"], examples=f"{T['short_reviews']:,} reviews missing in total",
       note="genuinely incomplete history (removed / filtered reviews); the reverse case (more loaded than review_count) is 6.6 consistency")
dq_add(findings_646, "yelp_review (business timeline)", "month grid continuity", "no empty month between first and last review", len(grid), len(missing_months),
       examples=", ".join(f"{m:%Y-%m}" for m in missing_months[:12]))
dq_add(findings_646, "yelp_review (business timeline)", "months with >= 40 % YoY volume drop", "no collapse in review volume", len(months), len(collapse),
       status="INFO", examples=", ".join(f"{x:%Y-%m}" for x in collapse["m"].head(10)),
       note="2020 COVID dip is a real-world effect, not a collection gap" if any(collapse["m"].dt.year == 2020) else "")
dq_add(findings_646, "df_business × yelp_review", "silent period > 2 years inside a business's history", "continuous activity",
       T["n"], T["gap2y"], status="INFO", examples=f"> 3 years: {T['gap3y']:,}", note="low-volume businesses — long gaps are expected, not data loss")
dq_add(findings_646, "df_business × yelp_review", "is_open = TRUE but silent > 1 year before snapshot", "open businesses show recent activity",
       T["n_open"], T["open_silent_1y"], status="INFO", examples=f"> 3 years silent: {T['open_silent_3y']:,}",
       note="stale is_open is likely — is_open is the only status signal and has no as-of date")
timeline_646 = dict(T.asDict()); timeline_646.update({"snap_start": SNAP_START, "snap_end": SNAP_END, "missing_months": len(missing_months),
                                                      "midweek_hole": H["hole"], "sparse_schedule": H["sparse"]})
findings_646_df = dq_summary(findings_646, "6.4.6 Temporal Gap Analysis")
print("TEMPORAL GAP ANALYSIS COMPLETE")
print("=" * 80)

---
#### 🚨 DQ-BUSINESS-036 — `is_open = TRUE` for Businesses Silent for Years (Status Without an As-Of Date)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-036` |
| **Severity** | MEDIUM (temporal completeness; the status field is probably out of date for a large share of "open" businesses) |
| **Detected In** | Phase 6.4 — Completeness → **6.4.6 Temporal Gap Analysis** (Cell 87), part B, business review timeline |
| **Also Confirmed In** | Independent SQL check (same counts). Review month grid 2005-02 → 2022-01 has **0** missing months (204 present); the 2020-03 → 2020-07 drop (-46 % to -72 % YoY) matches COVID lockdowns and is not a collection gap |
| **Affected Columns** | `is_open` (with `yelp_review.date` as evidence) |
| **Total Affected Rows** | **34,047** of 119,698 open businesses (28.4 %) have no review in the last 12 months before the snapshot end (2022-01-19); **7,608** are silent > 3 years (longest: 4,536 days, last review 2009) |
| **Invalid Value** | `is_open = 1`, `last_review = 2009-08-19` |
| **Domain Rule** | An activity status should have an as-of date, and "open" should be supported by recent activity |

**Description:** `is_open` is the only lifecycle signal in the business table and it has no timestamp. Many "open" businesses have had no customer activity for years, which on Yelp usually means the business closed without anyone flagging it. Long silent periods *inside* a history (46,784 businesses with a > 2-year gap) are expected for low-volume businesses and are not data loss. The review history itself is complete: every business has reviews, and only **9** businesses have one review fewer loaded than their `review_count`. The opposite direction (more reviews loaded than `review_count`) is a consistency issue for 6.6.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep `is_open` as published; derive `last_review_date`, `days_since_last_activity` (reviews + tips + check-ins) and `is_active_12m` in Silver; document `is_open` as "Yelp status at snapshot, not verified"** | Source value preserved; analysts get an evidence-based activity flag |
| 2 | **Overwrite `is_open = 0` when silent > N years** | Changes source data on a heuristic; some businesses are open but simply not reviewed |
| 3 | **Drop silent businesses** | Loses valid history for time-series analysis |

**Best Logical Solution:** **Option 1 — derived activity recency flags next to the unchanged `is_open`.**

---

In [0]:
# 6.4.7) Critical Field Coverage  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Business-record contract: which fields MUST be present (after null normalization) for a row to be a usable business.
#   TIER 1  ALWAYS REQUIRED  — identity, location anchor, rating & status; without them the row cannot be joined,
#           mapped, ranked or filtered:  business_id (PK / FK target of review, tip, checkin), name, city, state,
#           latitude, longitude, stars, review_count, is_open
#           Coverage is measured twice: PRESENT (not NULL / not masked) and USABLE (present AND passes the basic
#           domain rule from 6.2) — a present-but-impossible value is as useless as a NULL.
#   TIER 2  CONDITIONALLY REQUIRED — address for storefront businesses (service-area businesses legitimately hide it,
#           DQ-BUSINESS-032); postal_code whenever an address is published.
#   TIER 3  EXPECTED, NOT CRITICAL — categories (needed for every segment / category analysis), attributes, schedule.
#           Missing them does not invalidate the record; it lowers its analytical value (6.4.2 profile score).
# Record level: VALID = all Tier 1 usable; ANALYTICS-READY = VALID + Tier 2 satisfied + categories present.
import pandas as pd
from pyspark.sql import functions as F

print("=" * 80)
print("6.4.7) CRITICAL FIELD COVERAGE  —  business-record contract")
print("=" * 80)
findings_647 = []

STAR_SET = [x / 2 for x in range(2, 11)]                       # 1.0, 1.5 … 5.0
TIER1 = [
    ("business_id",  "PK; join key for review / tip / checkin", F.col("business_id").rlike(r"^[A-Za-z0-9_-]{22}$")),
    ("name",         "display / entity resolution",            F.length(F.trim(F.col("name"))) > 0),
    ("city",         "geographic grouping",                    F.length(F.trim(F.col("city"))) > 0),
    ("state",        "geographic grouping",                    F.col("state").rlike(r"^[A-Z]{2,3}$")),
    ("latitude",     "mapping / distance",                     F.col("latitude").between(-90, 90) & (F.col("latitude") != 0)),
    ("longitude",    "mapping / distance",                     F.col("longitude").between(-180, 180) & (F.col("longitude") != 0)),
    ("stars",        "core rating metric",                     F.col("stars").isin(STAR_SET)),
    ("review_count", "popularity / weighting",                 F.col("review_count") >= 0),
    ("is_open",      "status filter",                          F.col("is_open").isNotNull()),
]
m_cols = [f"m_{c}" for c, _, _ in TIER1 if c != "business_id"] + ["m_address", "m_postal_code", "m_categories", "m_attributes", "m_schedule", "seg_service_area"]
cf = (df_business.join(df_miss.select("business_id", *m_cols), "business_id", "left")
      .withColumn("m_business_id", (F.col("business_id").isNull() | (F.trim(F.col("business_id")) == "")).cast("int")))   # PK not in df_miss
usable = {c: ((F.col(f"m_{c}") == 0) & F.coalesce(rule, F.lit(False))) for c, _, rule in TIER1}
valid_rec = F.lit(True)
for c in usable:
    valid_rec = valid_rec & usable[c]
storefront = F.col("seg_service_area") == 0
tier2_ok = ~((F.col("m_address") == 1) & storefront) & ~((F.col("m_postal_code") == 1) & (F.col("m_address") == 0))
ready = valid_rec & tier2_ok & (F.col("m_categories") == 0)

aggs = [F.count("*").alias("n")]
for c, _, _ in TIER1:
    aggs += [F.sum(f"m_{c}").alias(f"miss_{c}"), F.sum((~usable[c]).cast("int")).alias(f"bad_{c}")]
aggs += [F.sum(((F.col("m_address") == 1) & storefront).cast("int")).alias("addr_store"), F.sum(storefront.cast("int")).alias("n_store"),
         F.sum(((F.col("m_address") == 1) & ~storefront).cast("int")).alias("addr_sa"), F.sum((~storefront).cast("int")).alias("n_sa"),
         F.sum(((F.col("m_postal_code") == 1) & (F.col("m_address") == 0)).cast("int")).alias("zip_with_addr"),
         F.sum((F.col("m_address") == 0).cast("int")).alias("n_addr"),
         F.sum("m_categories").alias("miss_categories"), F.sum("m_attributes").alias("miss_attributes"), F.sum("m_schedule").alias("miss_schedule"),
         F.sum(valid_rec.cast("int")).alias("valid"), F.sum(ready.cast("int")).alias("ready")]
C = cf.agg(*aggs).first()
N7 = int(C["n"])
assert N7 == N_BUS, "join to df_miss changed the row count"

rows = []
for c, why, _ in TIER1:
    rows.append({"Tier": "1 ALWAYS", "Field": c, "Why critical": why, "Applies To": f"all ({N7:,})",
                 "Missing": C[f"miss_{c}"], "Present but invalid": C[f"bad_{c}"] - C[f"miss_{c}"],
                 "Usable Coverage %": round((N7 - C[f"bad_{c}"]) / N7 * 100, 3)})
rows += [
    {"Tier": "2 CONDITIONAL", "Field": "address", "Why critical": "storefront location", "Applies To": f"storefront ({C['n_store']:,})",
     "Missing": C["addr_store"], "Present but invalid": 0, "Usable Coverage %": round((1 - C["addr_store"] / C["n_store"]) * 100, 3)},
    {"Tier": "2 CONDITIONAL", "Field": "postal_code", "Why critical": "complete postal address", "Applies To": f"address published ({C['n_addr']:,})",
     "Missing": C["zip_with_addr"], "Present but invalid": 0, "Usable Coverage %": round((1 - C["zip_with_addr"] / C["n_addr"]) * 100, 3)}]
for b, why in [("categories", "segmentation / category analytics"), ("attributes", "feature enrichment"), ("schedule", "opening-hours features")]:
    rows.append({"Tier": "3 EXPECTED", "Field": b, "Why critical": why, "Applies To": f"all ({N7:,})", "Missing": C[f"miss_{b}"],
                 "Present but invalid": None, "Usable Coverage %": round((1 - C[f"miss_{b}"] / N7) * 100, 3)})
crit_647_df = pd.DataFrame(rows)
print("\n📋 Critical-field coverage (present = not NULL and not masked; usable = present AND domain-valid):")
display(crit_647_df)
print(f"\n   Address missing for service-area businesses (legitimate, not counted): {C['addr_sa']:,} of {C['n_sa']:,}")
print(f"\n   VALID business records (all Tier 1 usable):           {C['valid']:,} / {N7:,} ({C['valid'] / N7 * 100:.3f}%)")
print(f"   ANALYTICS-READY (valid + Tier 2 + categories present): {C['ready']:,} / {N7:,} ({C['ready'] / N7 * 100:.3f}%)")

print("\n   Examples — storefront businesses without an address:")
display(cf.where((F.col("m_address") == 1) & storefront).select("business_id", "name", "address", "city", "state", "postal_code", "review_count").limit(8))
print("\n   Examples — address published but postal_code missing:")
display(cf.where((F.col("m_postal_code") == 1) & (F.col("m_address") == 0)).select("business_id", "name", "address", "city", "state", "postal_code").limit(8))

for c, why, rule in TIER1:
    bad = C[f"bad_{c}"]
    dq_add(findings_647, "df_business", c, f"Tier 1 — always present & valid ({why})", N7, bad,
           status="PASS" if bad == 0 else "FINDING",
           examples=dq_examples(cf, c, ~usable[c]) if bad else "",
           note=f"missing={C[f'miss_{c}']}, invalid={bad - C[f'miss_{c}']}" if bad else "")
dq_add(findings_647, "df_business", "address (storefront)", "Tier 2 — storefront businesses publish an address", C["n_store"], C["addr_store"],
       examples=dq_examples(cf.where(storefront), "name", F.col("m_address") == 1),
       note="upper bound: many are mobile businesses missed by the category-based service-area rule (pet sitting, towing, tours, detailing) — the rest are true storefront gaps (DQ-BUSINESS-032)")
dq_add(findings_647, "df_business", "postal_code (address published)", "Tier 2 — a published address carries a ZIP", C["n_addr"], C["zip_with_addr"],
       examples=dq_examples(cf.where(F.col("m_address") == 0), "name", F.col("m_postal_code") == 1), note="DQ-BUSINESS-015")
dq_add(findings_647, "df_bronze", "categories", "Tier 3 — expected for segmentation", N7, C["miss_categories"],
       note="102 of these are skeleton profiles (no attributes, no hours) — 6.4.2")
for b in ("attributes", "schedule"):
    dq_add(findings_647, "df_bronze", b, "Tier 3 — expected, not critical", N7, C[f"miss_{b}"], status="INFO",
           note="does not invalidate the record; MAR on popularity (6.4.4) and co-missing (6.4.5)")
dq_add(findings_647, "df_business", "VALID record (all Tier 1 usable)", "every business row is a valid record", N7, N7 - C["valid"],
       status="PASS" if C["valid"] == N7 else "FINDING")
dq_add(findings_647, "df_business", "ANALYTICS-READY record", "valid + conditional fields + categories", N7, N7 - C["ready"], status="INFO")
record_647 = {"valid": C["valid"], "ready": C["ready"], "addr_store": C["addr_store"], "zip_with_addr": C["zip_with_addr"]}
findings_647_df = dq_summary(findings_647, "6.4.7 Critical Field Coverage")
print("CRITICAL FIELD COVERAGE CHECK COMPLETE")
print("=" * 80)

In [0]:
# 6.4.8) Completeness Summary  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Consolidated view of 6.4.1 – 6.4.7 for the business DataFrames: one status table per sub-check, the headline
# completeness metrics, and every FINDING in one place (input for the DQ-BUSINESS notes and the Silver cleaning plan).
import pandas as pd

print("=" * 80)
print("6.4) COMPLETENESS CHECKS — CONSOLIDATED SUMMARY  (Business DataFrames)")
print("=" * 80)

SECTIONS_64 = [("6.4.1 Per-Column Missingness", "findings_641_df"), ("6.4.2 Per-Row Missingness", "findings_642_df"),
               ("6.4.3 Per-Cell Null Rate", "findings_643_df"), ("6.4.4 MCAR / MAR / MNAR", "findings_644_df"),
               ("6.4.5 Missing Value Correlation", "findings_645_df"), ("6.4.6 Temporal Gap Analysis", "findings_646_df"),
               ("6.4.7 Critical Field Coverage", "findings_647_df")]
parts, missing_sections = [], []
for sec, var in SECTIONS_64:
    if var in globals():
        parts.append(globals()[var].assign(Section=sec))
    else:
        missing_sections.append(sec)
if missing_sections:
    print(f"   ⚠️  Not yet run (re-run these cells first): {missing_sections}")
all_64 = pd.concat(parts, ignore_index=True)

status_64 = (all_64.pivot_table(index="Section", columns="Status", values="Rule", aggfunc="count", fill_value=0)
             .reindex(columns=["FINDING", "INFO", "PASS"], fill_value=0).reset_index())
status_64["Checks"] = status_64[["FINDING", "INFO", "PASS"]].sum(axis=1)
print("\n📋 Check status per sub-section:")
display(status_64)

# ── Headline completeness metrics ──
R6, C3 = row_metrics_642, completeness_643
T6, K7 = timeline_646, record_647
headline = pd.DataFrame([
    ("Column", "top-level SQL NULLs in bronze scalars", "0", "NULL-based profiling reports 100 % complete — misleading"),
    ("Column", "hidden (masked) missing cells", f"{int(col_null_df['Hidden (masked)'].sum()):,}", "'' / 'None' / '{}' — only visible after 6.3 normalization"),
    ("Column", "block gaps: attributes / schedule / categories", f"{int(_cnt['attributes']):,} / {int(_cnt['schedule']):,} / {int(_cnt['categories']):,}", "9.3 % / 15.4 % / 0.07 %"),
    ("Row", "profiles > 25 % / > 50 % missing (applicable fields)", f"{R6['gt25']:,} / {R6['gt50']:,}", f"skeleton listings: {R6['skeleton']:,}"),
    ("Cell", "raw / effective / applicable-only null rate (wide model)",
     f"{C3['raw_null_pct']:.2f} % / {C3['effective_null_pct']:.2f} % / {C3['applicable_null_pct']:.2f} %", "gap effective → applicable = structural 'not applicable' NULLs"),
    ("Mechanism", "fields classified MAR (V >= 0.10)", f"{int(mech_df['Classification'].str.startswith('MAR').sum() if 'mech_df' in globals() else 0)}",
     "popularity (review_count) and business type drive missingness; address = structural MNAR"),
    ("Correlation", "schedule ↔ attributes co-missing", f"{R6['no_attr_no_hours']:,} businesses", "thin listings lose several blocks at once; attributes vanish as a block"),
    ("Temporal", "isolated mid-week schedule holes / sparse (1–2 day) schedules", f"{T6['midweek_hole']:,} / {T6['sparse_schedule']:,}", "flag only — closure vs missing entry undecidable"),
    ("Temporal", "review month grid — missing months", f"{T6['missing_months']}", f"{T6['snap_start']} → {T6['snap_end']}; 2020 dip = COVID, not loss"),
    ("Temporal", "open but silent > 1 year / > 3 years", f"{T6['open_silent_1y']:,} / {T6['open_silent_3y']:,}", "is_open likely stale (no as-of date)"),
    ("Critical", "VALID records (all Tier 1 usable)", f"{K7['valid']:,} / {N_BUS:,}", "identity, location, rating, status always present"),
    ("Critical", "storefront w/o address / address w/o ZIP", f"{K7['addr_store']:,} / {K7['zip_with_addr']:,}", "conditional requirements"),
    ("Critical", "ANALYTICS-READY records", f"{K7['ready']:,} ({K7['ready'] / N_BUS * 100:.2f} %)", "valid + conditional + categories"),
], columns=["Level", "Metric", "Value", "Interpretation"])
print("\n📋 Headline completeness metrics:")
display(headline)

find_64 = all_64[all_64["Status"] == "FINDING"][["Section", "DataFrame", "Column", "Rule", "Checked", "Violations", "Pct", "Note"]]
print(f"\n📋 All FINDINGs in 6.4 ({len(find_64)}):")
display(find_64.reset_index(drop=True))

n_f, n_i, n_p = (int((all_64["Status"] == s).sum()) for s in ("FINDING", "INFO", "PASS"))
print(f"\n   {len(all_64)} checks | {n_f} FINDING | {n_i} INFO | {n_p} PASS")
print("\n   Verdict: every business row is a VALID record (Tier 1 = 100 %), but completeness is driven by MASKED and")
print("   STRUCTURAL gaps, not SQL NULLs. Missingness is MAR on popularity / business type (not MCAR), so dropping")
print("   incomplete rows would bias towards popular restaurants. Silver: normalise masked nulls, keep structural")
print("   'not applicable' explicit, add completeness / service-area / closed-day flags, never impute attributes.")
summary_64_df = all_64
print("\nCOMPLETENESS CHECKS COMPLETE")
print("=" * 80)

### 6.5) UNIQUENESS CHECKS — Business DataFrames
*Make sure no data is duplicated and every record can be identified uniquely, both by its surrogate key (`business_id`) and as a real-world place.*

Scope: `df_bronze` (raw, 1 row per business) and the 11 TE.2 frames (`df_business`, `df_attributes`, `df_hours`, `df_categories` and the 7 long nested-attribute frames). `business_id` is a **surrogate** key, so it is unique by construction. For a business table, the real uniqueness risk is the **same place being listed under several ids**.

- **Row-Level Uniqueness**: whole rows are duplicated.
  - *Example:* two bronze rows identical in all 14 columns, including the nested `attributes` / `hours` structs.
  - *Example:* the 13 non-key columns are identical but the `business_id` differs (the listing was re-ingested or re-keyed).
  - *Example:* rows identical except the volatile metrics (`stars`, `review_count`, `is_open`), meaning one place listed twice with each copy collecting its own reviews.
  - *Sub:* no exact duplicate rows in any TE.2 frame after restructuring.

- **Key-Level Uniqueness**: primary-key columns contain no duplicates.
  - *Example:* `business_id` is NOT NULL, unique, follows the 22-char URL-safe base64 format, and stays unique after case-folding and trimming (case-insensitive collations would otherwise merge two ids).
  - *Example:* every TE.2 frame keeps its PK, and the Silver DDL PK `yelp_business_categories (business_id, category_name)` holds (re-confirms DQ-BUSINESS-005).
  - *Sub:* key cardinality against the parent: 1:1 frames have exactly one row per business, dict frames have at most one row per contract key, and there are no orphan child keys.

- **Composite Key Uniqueness**: multi-column natural keys are unique together.
  - *Example:* `(name, address, city, state, postal_code)`: the same storefront under two `business_id`s.
  - *Example:* `(name, latitude, longitude)`: the same name at the identical geocode.
  - *Example:* normalised `(name_key, addr_key, zip_key)` catches the variants documented in 6.2/6.3 (DQ-004, -017, -021, -022, -029, -031, -014).
  - *Sub:* spatial near-duplicates: the same normalised name within 100 m. Each group is classified by `is_open` as *live duplicate* (all open), *re-listing / relocation* (open + closed) or *historical* (all closed).
  - *Sub:* `(latitude, longitude)` collisions: shared geocodes are normal in malls, but a large cluster of address-less businesses on one point is a placeholder centroid.
  - *Sub:* child natural keys after case/whitespace folding: `(business_id, category_name)` and `(business_id, dict key)`.

- **Distribution Uniqueness**: distinct-value counts in each column match domain expectations.
  - *Example:* `stars` has exactly 9 values (1.0–5.0 in half-stars), `is_open` 2, `state` at most the 14 footprint states/provinces.
  - *Example:* raw distinct counts vs. folded-key counts (`city` vs `city_key`, `name` vs `name_key`) show how much cardinality comes from spelling variants.
  - *Example:* boolean / enum attributes are neither constant nor above their domain size; nested dicts have no more keys than the contract; the category vocabulary does not collapse under case-folding.
  - *Sub:* concentration: dominant values (chains, metros, schedule templates) that drive cardinality expectations.


In [0]:
# 6.5.1) Row-Level Uniqueness  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# A business record (bronze, 1 row / business) can be duplicated at three levels of strictness:
#   A) EXACT           all 14 columns identical (incl. the nested attributes / hours structs)
#   B) ID-AGNOSTIC     all 13 non-key columns identical under a DIFFERENT business_id
#                      → the same listing re-ingested / re-keyed; invisible to any PK check
#   C) STATIC PROFILE  identical except the volatile metrics (stars, review_count, is_open)
#                      → one place listed twice, each copy collecting its own reviews
#   D) every TE.2 frame (parent + 2 wide 1:1 + 8 long 0:N) holds no exact duplicate rows after restructuring
# This cell also defines the NORMALISED MATCH KEYS reused by 6.5.2 – 6.5.4. They neutralise the representation
# noise already documented so that variants of the same value compare equal:
#   whitespace (DQ-004, DQ-029), invisible U+200B / NBSP (DQ-017), curly vs straight apostrophes & casing (DQ-022,
#   DQ-030), Saint/St. · Mount/Mt. city variants (DQ-021), USPS suffix / directional abbreviations (DQ-031),
#   truncated / spaced Canadian postal codes (DQ-014). Keys are DETECTION aids only — no value is changed.
import pandas as pd
from pyspark.sql import functions as F

print("=" * 80)
print("6.5.1) ROW-LEVEL UNIQUENESS  —  Business DataFrames")
print("    Exact, ID-agnostic and static-profile duplicate rows")
print("=" * 80)

N_BUS = df_business.count()
findings_651 = []
FRAME_NAME = {"yelp_business": "df_business", "yelp_business_attributes": "df_attributes", "yelp_business_hours": "df_hours",
              "yelp_business_categories": "df_categories", "yelp_business_ambience": "df_ambience",
              "yelp_business_best_nights": "df_best_nights", "yelp_business_parking": "df_business_parking",
              "yelp_business_dietary_restrictions": "df_dietary_restrictions", "yelp_business_good_for_meal": "df_good_for_meal",
              "yelp_business_hair_specializes_in": "df_hair_specializes_in", "yelp_business_music": "df_music"}

# ══ Shared normalised match keys (reused by 6.5.2 – 6.5.4) ══
INVISIBLE_RE = "[\u00a0\u2007\u202f\u200b\u200c\u200d\u2060\ufeff]"
ADDR_TOKENS = {"street": "st", "avenue": "ave", "av": "ave", "road": "rd", "boulevard": "blvd", "drive": "dr",
               "lane": "ln", "court": "ct", "place": "pl", "parkway": "pkwy", "highway": "hwy", "circle": "cir",
               "terrace": "ter", "square": "sq", "suite": "ste", "north": "n", "south": "s", "east": "e", "west": "w",
               "northeast": "ne", "northwest": "nw", "southeast": "se", "southwest": "sw"}

def name_key(c):
    """Brand-level match key: invisible chars → space, case-fold, '&' → 'and', keep letters/digits only."""
    c = F.regexp_replace(c, INVISIBLE_RE, " ")
    c = F.regexp_replace(F.lower(c), "&", " and ")
    return F.regexp_replace(c, "[^\\p{L}\\p{N}]", "")

def city_key(c):
    """City match key: case-fold, Saint→st / Mount→mt / Fort→ft, letters only (DQ-021 folding)."""
    c = F.lower(F.regexp_replace(c, INVISIBLE_RE, " "))
    for full, abbr in (("saint", "st"), ("mount", "mt"), ("fort", "ft")):
        c = F.regexp_replace(c, f"\\b{full}\\b", abbr)
    return F.regexp_replace(c, "[^\\p{L}]", "")

def addr_key(c):
    """Street match key: case-fold, punctuation → space, ordinals 170th → 170, USPS abbreviations, no spaces."""
    c = F.lower(F.regexp_replace(c, INVISIBLE_RE, " "))
    c = F.regexp_replace(c, "[^\\p{L}\\p{N}]+", " ")
    c = F.regexp_replace(c, "\\b(\\d+)(st|nd|rd|th)\\b", "$1")
    for full, abbr in ADDR_TOKENS.items():
        c = F.regexp_replace(c, f"\\b{full}\\b", abbr)
    return F.regexp_replace(c, " ", "")

def zip_key(c):
    """Postal match key: US → ZIP5, Canada → FSA (first 3 chars, robust to DQ-014 truncation), '' → NULL."""
    z = F.upper(F.regexp_replace(c, "\\s", ""))
    return (F.when(z.rlike("^\\d{5}"), F.substring(z, 1, 5))
             .when(z.rlike("^[A-Z]\\d[A-Z]"), F.substring(z, 1, 3))
             .when(z != "", z))

def haversine_m(lat1, lon1, lat2, lon2):
    dlat, dlon = F.radians(lat2 - lat1), F.radians(lon2 - lon1)
    a = F.pow(F.sin(dlat / 2), 2) + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.pow(F.sin(dlon / 2), 2)
    return F.lit(2 * 6371008.8) * F.asin(F.sqrt(a))

df_bkeys = df_business.select("*", name_key(F.col("name")).alias("name_key"), addr_key(F.col("address")).alias("addr_key"),
                              city_key(F.col("city")).alias("city_key"), zip_key(F.col("postal_code")).alias("zip_key"))

def dup_groups(frame, cols):
    """Groups of >1 row sharing `cols`, with the business_ids involved and how the volatile metrics differ."""
    return (frame.groupBy(*cols)
                 .agg(F.count("*").alias("rows"), F.sort_array(F.collect_list("business_id")).alias("business_ids"),
                      F.sum("review_count").alias("review_count_total"), F.countDistinct("stars").alias("stars_variants"),
                      F.countDistinct("is_open").alias("is_open_variants"))
                 .where("rows > 1"))

def group_stats(groups):
    r = groups.agg(F.count("*").alias("g"), F.sum("rows").alias("r")).first()
    g, rows = int(r["g"] or 0), int(r["r"] or 0)
    return g, rows, rows - g

# ══════════════════════════════════════════════════════════════════════
# A–C) df_bronze — three strictness levels
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─' * 60}\n📌 A–C) df_bronze — duplicate rows at three strictness levels")
n_bronze = df_bronze.count()
n_exact_distinct = df_bronze.distinct().count()
dq_add(findings_651, "df_bronze", "A) all 14 columns", "no two rows fully identical", n_bronze, n_bronze - n_exact_distinct,
       note="nested attributes / hours structs compared field by field")

NON_KEY = [c for c in df_bronze.columns if c != "business_id"]
VOLATILE = ["stars", "review_count", "is_open"]
STATIC = [c for c in NON_KEY if c not in VOLATILE]

g_noid = dup_groups(df_bronze, NON_KEY)
gB, rB, sB = group_stats(g_noid)
dq_add(findings_651, "df_bronze", "B) 13 non-key columns", "same content ⇒ same business_id", n_bronze, sB,
       examples="; ".join(str(x["business_ids"]) for x in g_noid.limit(3).collect()) if sB else "",
       note=f"{gB:,} groups / {rB:,} rows (re-keyed copies)")

g_static = dup_groups(df_bronze, STATIC)
gC, rC, sC = group_stats(g_static)
dq_add(findings_651, "df_bronze", "C) 10 static columns (excl. stars/review_count/is_open)",
       "same static profile ⇒ same business_id", n_bronze, sC,
       examples="; ".join(f"{x['name']} @ {x['address']} {x['business_ids']}" for x in g_static.limit(3).collect()) if sC else "",
       note=f"{gC:,} groups / {rC:,} rows; differ only in volatile metrics")
if sC:
    print("\n   📋 Static-profile duplicate groups (same place, metrics drifted apart):")
    display(g_static.select("name", "address", "city", "state", "rows", "business_ids", "review_count_total",
                            "stars_variants", "is_open_variants").orderBy(F.desc("rows")).limit(20))

# ══════════════════════════════════════════════════════════════════════
# D) TE.2 frames — exact duplicate rows after restructuring
#    Long frames: dict rows are (business_id, key, value) — MAP keys are unique by construction, so an exact
#    duplicate would mean the explode logic itself duplicated data. df_categories carries category_position,
#    so a repeated LABEL is not an exact duplicate row — it is checked against the DDL PK in 6.5.2.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─' * 60}\n📌 D) TE.2 frames — exact duplicate rows")
frame_rows_651 = []
for ds, fr in business_dfs.items():
    n, d = fr.count(), fr.distinct().count()
    frame_rows_651.append({"DataFrame": FRAME_NAME[ds], "Rows": n, "Distinct rows": d, "Columns": len(fr.columns)})
    dq_add(findings_651, FRAME_NAME[ds], f"all {len(fr.columns)} columns", "no exact duplicate rows", n, n - d)

findings_651_df = pd.DataFrame(findings_651)
uq_metrics = {"n_bus": N_BUS, "bronze_exact_dups": n_bronze - n_exact_distinct, "noid_dups": sB,
              "static_dups": sC, "static_groups": gC,
              "frame_exact_dups": sum(r["Rows"] - r["Distinct rows"] for r in frame_rows_651)}

print("\n   Interpretation:")
print(f"   • Exact / ID-agnostic duplicates: {uq_metrics['bronze_exact_dups']:,} / {sB:,} — no row was ingested twice, with or without a new id.")
print(f"   • Static-profile duplicates: {sC:,} surplus rows in {gC:,} groups — identical place, attributes, hours and categories,")
print("     but separate ids whose ratings / review counts / status drifted apart.")
print(f"   • TE.2 restructuring introduced {uq_metrics['frame_exact_dups']:,} exact duplicate rows across 11 frames.")
print("   • Row-level checks only catch byte-identical copies. Real-world duplicate listings (different spelling,")
print("     slightly different geocode, different metrics) are tested on natural keys in 6.5.3.")
dq_summary(findings_651, "6.5.1 Row-Level Uniqueness")
print("\nROW-LEVEL UNIQUENESS COMPLETE")
print("=" * 80)


In [0]:
# 6.5.2) Key-Level Uniqueness  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
#   A) business_id (parent PK): NOT NULL, exact uniqueness, and uniqueness under transformations a downstream
#      system may silently apply: CASE-FOLDING (case-insensitive collations in SQL Server / Power BI / Excel lookups
#      would merge two Yelp ids that differ only in case — base64 ids are case-sensitive) and TRIMMING.
#      Format: 22-char URL-safe base64 [A-Za-z0-9_-]. Ids starting with '-' are valid but are read as options by
#      CLIs and as formulas by spreadsheets (INFO).
#   B) PK of every TE.2 frame (TE.1 DATASETS) + the Silver DDL PK where it differs:
#      yelp_business_categories DDL PK = (business_id, category_name) → re-confirms DQ-BUSINESS-005.
#   C) Key cardinality vs the parent: 1:1 frames exactly one row per business; long dict frames at most one row
#      per contract key (a business cannot have 10 Ambience flags when the contract defines 9); categories 0:N profiled.
#   D) Every child business_id resolves to a parent business_id (no orphans created by restructuring).
import operator
from functools import reduce
import pandas as pd
from pyspark.sql import functions as F

print("=" * 80)
print("6.5.2) KEY-LEVEL UNIQUENESS  —  Business DataFrames")
print("    Primary keys, key robustness and key cardinality")
print("=" * 80)

findings_652 = []

# ══ A) business_id — parent PK ══
print(f"\n{'─' * 60}\n📌 A) business_id — parent primary key (df_bronze)")
ID_RE = "^[A-Za-z0-9_-]{22}$"
bid = F.col("business_id")
k = df_bronze.agg(
    F.count("*").alias("n"),
    F.sum(bid.isNull().cast("int")).alias("nulls"),
    F.countDistinct(bid).alias("d_exact"),
    F.countDistinct(F.lower(bid)).alias("d_lower"),
    F.countDistinct(F.trim(bid)).alias("d_trim"),
    F.sum((bid != F.trim(bid)).cast("int")).alias("padded"),
    F.sum((~bid.rlike(ID_RE)).cast("int")).alias("bad_fmt"),
    F.sum(bid.startswith("-").cast("int")).alias("lead_dash"),
).first()
n_nn = k["n"] - k["nulls"]
dq_add(findings_652, "df_bronze", "business_id", "NOT NULL", k["n"], k["nulls"])
dq_add(findings_652, "df_bronze", "business_id", "UNIQUE (exact, case-sensitive)", n_nn, n_nn - k["d_exact"])
dq_add(findings_652, "df_bronze", "business_id", "UNIQUE under case-folding (CI collations)", k["d_exact"], k["d_exact"] - k["d_lower"],
       note="lower(id) collisions would merge distinct businesses in case-insensitive systems")
dq_add(findings_652, "df_bronze", "business_id", "no leading / trailing whitespace", k["n"], k["padded"],
       note=f"distinct after trim: {k['d_trim']:,}")
dq_add(findings_652, "df_bronze", "business_id", "format ^[A-Za-z0-9_-]{22}$", k["n"], k["bad_fmt"],
       examples=dq_examples(df_bronze, "business_id", ~bid.rlike(ID_RE)) if k["bad_fmt"] else "")
dq_add(findings_652, "df_bronze", "business_id", "starts with '-' (CLI / spreadsheet hazard)", k["n"], k["lead_dash"],
       status="INFO" if k["lead_dash"] else "PASS", examples=dq_examples(df_bronze, "business_id", bid.startswith("-"), 4),
       note="valid base64; quote ids in CSV exports and CLI calls")

# ══ B) PK of every TE.2 frame + Silver DDL PK ══
print(f"\n{'─' * 60}\n📌 B) Primary key of every TE.2 frame (+ Silver DDL PK where it differs)")
PK_CHECKS = [(ds, spec["PK"], "TE.2 PK") for ds, spec in DATASETS.items()]
PK_CHECKS.append(("yelp_business_categories", ["business_id", "category_name"], "Silver DDL PK"))
pk_rows_652 = []
for ds, pk, basis in PK_CHECKS:
    fr = business_dfs[ds]
    n = fr.count()
    nulls = fr.where(reduce(operator.or_, [F.col(c).isNull() for c in pk])).count()
    dup_g = fr.groupBy(*pk).count().where("count > 1")
    d = fr.select(*pk).distinct().count()
    ex = "; ".join("(" + ", ".join(str(r[c]) for c in pk) + f") ×{r['count']}" for r in dup_g.limit(4).collect()) if n != d else ""
    dq_add(findings_652, FRAME_NAME[ds], "(" + ", ".join(pk) + ")", f"{basis} unique & NOT NULL", n, (n - d) + nulls, examples=ex,
           note="repeated label in the source list (DQ-BUSINESS-005)" if basis == "Silver DDL PK" else "")
    pk_rows_652.append({"DataFrame": FRAME_NAME[ds], "Key basis": basis, "PK": ", ".join(pk), "Rows": n,
                        "Distinct PK": d, "Duplicate PK rows": n - d, "NULL PK rows": nulls})
pk_652_df = pd.DataFrame(pk_rows_652)
display(pk_652_df)

# ══ C) Key cardinality vs the parent ══
print(f"\n{'─' * 60}\n📌 C) Key cardinality — rows per business_id vs the contract")
DICT_OF_TABLE = {tbl: a for a, (tbl, _) in DICT_TABLE_MAP.items()}
card_rows_652 = []
for ds, spec in DATASETS.items():
    fr = business_dfs[ds]
    per = fr.groupBy("business_id").count()
    s = per.agg(F.count("*").alias("bus"), F.sum("count").alias("rows"), F.min("count").alias("mn"),
                F.expr("percentile_approx(count, 0.5)").alias("med"), F.max("count").alias("mx")).first()
    card = spec["Cardinality"]
    if card in ("parent", "1:1"):
        missing, extra = N_BUS - s["bus"], s["rows"] - s["bus"]
        dq_add(findings_652, FRAME_NAME[ds], "business_id", "exactly 1 row per parent business", N_BUS, missing + extra,
               note=f"missing parents {missing:,}, extra rows {extra:,}")
        limit = 1
    elif ds in DICT_OF_TABLE:
        limit = len(DICT_TABLE_MAP[DICT_OF_TABLE[ds]][1])
        over = per.where(F.col("count") > limit).count()
        dq_add(findings_652, FRAME_NAME[ds], "business_id", f"≤ {limit} rows per business (contract keys)", s["bus"], over)
    else:
        limit = None
        dq_add(findings_652, FRAME_NAME[ds], "business_id", "0:N — categories per business (profile)", s["bus"], 0, status="INFO",
               note=f"min {s['mn']} / median {s['med']} / max {s['mx']}; {N_BUS - s['bus']:,} businesses without categories")
    card_rows_652.append({"DataFrame": FRAME_NAME[ds], "Cardinality": card, "Businesses covered": s["bus"],
                          "Coverage %": round(s["bus"] / N_BUS * 100, 2), "Rows": s["rows"], "Min rows / business": s["mn"],
                          "Median": s["med"], "Max": s["mx"], "Contract max": limit if limit else "—"})
card_652_df = pd.DataFrame(card_rows_652)
display(card_652_df)

# ══ D) Child keys resolve to the parent (no orphans) ══
print(f"\n{'─' * 60}\n📌 D) Child business_id → parent df_business (orphans)")
parent_ids = df_business.select("business_id")
for ds, fr in business_dfs.items():
    if ds == "yelp_business":
        continue
    child_ids = fr.select("business_id").distinct()
    orphans = child_ids.join(parent_ids, "business_id", "left_anti").count()
    dq_add(findings_652, FRAME_NAME[ds], "business_id → df_business", "no orphan child keys", child_ids.count(), orphans)

findings_652_df = pd.DataFrame(findings_652)
uq_metrics.update({"id_ci_collisions": k["d_exact"] - k["d_lower"], "id_lead_dash": k["lead_dash"],
                   "cat_pk_dups": int(pk_652_df.loc[pk_652_df["Key basis"] == "Silver DDL PK", "Duplicate PK rows"].sum())})

print("\n   Interpretation:")
print(f"   • business_id is a clean surrogate key: unique, non-null, well-formed, and robust to case-folding ({uq_metrics['id_ci_collisions']} collisions).")
print(f"   • {k['lead_dash']:,} ids start with '-' — valid, but must be quoted in CLI calls and CSV/Excel exports.")
print(f"   • Every TE.2 frame keeps its PK. The Silver DDL PK of yelp_business_categories is violated by {uq_metrics['cat_pk_dups']} rows")
print("     (DQ-BUSINESS-005) — the load must deduplicate before inserting.")
print("   • A unique surrogate key does not prove that each id is a distinct real-world business → 6.5.3.")
dq_summary(findings_652, "6.5.2 Key-Level Uniqueness")
print("\nKEY-LEVEL UNIQUENESS COMPLETE")
print("=" * 80)


---
#### 🚨 DQ-BUSINESS-037 — `business_id` Values Starting With `-` (Export / CLI Interoperability Hazard)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-037` |
| **Severity** | LOW (the key is valid and unique, but it can be silently corrupted outside Databricks) |
| **Detected In** | Phase 6.5 — Uniqueness Checks → **6.5.2 Key-Level Uniqueness** (Cell 93), check A "starts with '-' (CLI / spreadsheet hazard)" |
| **Also Confirmed In** | 6.5.2 (Cell 93): the same key passes NOT NULL, exact uniqueness, uniqueness after case-folding (0 collisions), no padding, and the `^[A-Za-z0-9_-]{22}$` format. So the issue is only the leading character |
| **Affected Columns** | `business_id` (PK of `yelp_business`; FK in all 10 business child tables, and in `yelp_review`, `yelp_tip`, `yelp_checkin`) |
| **Total Affected Rows** | **2,386** of 150,346 businesses (1.59%) |
| **Invalid Value** | `-4dYswJy7SPcbcERvitmIg`, `-ZXYcCyDjj1QimnJyGtYtg`, `-aeZuatjCDMV1X4gCTz9Ug`, `-7GDqSUaXrpC8Ql7nDBxWA` |
| **Domain Rule** | An identifier must survive every export and tool round-trip byte-for-byte. Yelp ids are 22-char URL-safe base64, so `-` and `_` are legal anywhere, including position 1 |

**Description:** These ids are correct inside Delta and Spark SQL. The risk comes from consumers outside the lakehouse:
* Excel / Google Sheets read a cell starting with `-` as a formula or negative number, giving `#NAME?` or a rewritten value. The same rule is behind CSV-injection warnings.
* Shell tools and the Databricks / Azure CLIs read `-4dYs...` as a command-line option.
* Some BI tools trim or re-type the value.

Because `business_id` is also the join key to reviews, tips and check-ins, a single corrupted export breaks joins for about 1.6% of businesses and their reviews without any error. This is an interoperability issue, not a data error, and the value must not be changed.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep `business_id` unchanged; document the hazard in the DDL column comment and enforce quoting in every export path** (`quoteAll=true` for CSV, text-typed columns for Excel, `--` before positional ids in CLI calls) | No value change, so lineage and FK joins to review / tip / checkin stay intact. The fix sits where the problem happens (the consumer) |
| 2 | **Add an integer surrogate `business_sk` in Gold dimensions** for BI tools, keeping `business_id` as the natural key | Integer keys are immune to formula parsing and join faster in BI. Costs one extra mapping, and the problem still exists for anyone exporting `business_id` |
| 3 | **Prefix or rewrite the ids (e.g. `b_-4dYs...`)** | Removes the hazard, but changes the key: every review / tip / checkin FK has to be rewritten the same way, and the ids no longer match the Yelp source |
| 4 | **Do nothing** | Correct inside Databricks, but the first spreadsheet or CLI hand-off can silently break 2,386 businesses |

**Best Logical Solution:** **Option 1 — keep the id as-is, document it, and enforce quoting on export**, with Option 2 as an optional Gold improvement. The source key stays authoritative and joinable, and the risk is handled at the boundary where it occurs.

---

In [0]:
# 6.5.3) Composite Key Uniqueness  —  Business natural keys & entity resolution
# ─────────────────────────────────────────────────────────────────────
# business_id is a SURROGATE key: unique by construction (6.5.2) but silent on whether two ids describe the same
# real-world place. For a business listing the real uniqueness rule is the NATURAL key:
#   A) (name, address, city, state, postal_code)   exact
#   B) (name, latitude, longitude)                 exact — same name at the identical geocode
#   C) (name_key, addr_key, zip_key)               normalised (6.5.1 keys) — catches 6.2/6.3 spelling variants
#   D) spatial near-duplicates: same name_key within 100 m (haversine), any address spelling
#   Every group / pair is classified by is_open:
#      all open        → LIVE duplicate listings (reviews, ratings and check-ins split between ids — most harmful)
#      open + closed   → RE-LISTING / RELOCATION (old id closed, new id opened → history fragmented)
#      all closed      → historical duplicates (low impact)
#   Legitimate look-alikes exist (hotel + its restaurant, two chain units in one mall), so D is a CANDIDATE list.
#   E) (latitude, longitude) collisions — shared geocodes are normal in malls / multi-tenant buildings, but large
#      clusters of ADDRESS-LESS businesses on ONE point are placeholder city / ZIP centroids (DQ-027, DQ-032)
#   F) (addr_key, zip_key) shared by different names — multi-tenant addresses (INFO, legitimate)
#   G) child natural keys after case / whitespace folding: (business_id, category_name) and (business_id, dict key)
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

print("=" * 80)
print("6.5.3) COMPOSITE KEY UNIQUENESS  —  natural keys & duplicate listings")
print("=" * 80)

findings_653 = []
B = df_bkeys
OPEN_NOTE = {"all open": "LIVE duplicates — reviews / ratings split", "open + closed": "re-listing / relocation",
             "all closed": "historical duplicates"}

def nat_groups(frame, keys, where=None):
    fr = frame.where(where) if where is not None else frame
    return (fr.groupBy(*keys)
              .agg(F.count("*").alias("rows"), F.sum(F.col("is_open").cast("int")).alias("open_rows"),
                   F.sum("review_count").alias("reviews"), F.countDistinct("name").alias("name_spellings"),
                   F.countDistinct("address").alias("address_spellings"),
                   F.sort_array(F.collect_list("business_id")).alias("business_ids"),
                   F.first("name").alias("sample_name"), F.first("address").alias("sample_address"),
                   F.first("city").alias("sample_city"))
              .where("rows > 1")
              .withColumn("open_status", F.when(F.col("open_rows") == F.col("rows"), "all open")
                                          .when(F.col("open_rows") == 0, "all closed").otherwise("open + closed"))
              .toPandas())

def report_groups(column, pdf, rule, note=""):
    groups = len(pdf)
    rows = int(pdf["rows"].sum()) if groups else 0
    dq_add(findings_653, "df_business", column, rule, N_BUS, rows - groups,
           examples="; ".join(f"{r.sample_name} @ {r.sample_address or '∅'} ({r.sample_city}) ×{r.rows}"
                              for r in pdf.sort_values("rows", ascending=False).head(4).itertuples()),
           note=f"{groups:,} groups / {rows:,} rows. {note}")
    if groups:
        br = pdf.groupby("open_status").agg(groups=("rows", "size"), rows=("rows", "sum")).reset_index()
        for b in br.itertuples():
            print(f"      • {b.open_status:<14} {b.groups:>5,} groups / {b.rows:>5,} rows  → {OPEN_NOTE[b.open_status]}")

# ══ A) exact natural key ══
print(f"\n{'─' * 60}\n📌 A) (name, address, city, state, postal_code) — exact")
NAT = ["name", "address", "city", "state", "postal_code"]
ga = nat_groups(B, NAT)
report_groups("(name, address, city, state, postal_code)", ga, "exact natural key unique",
              note=f"{int((ga['sample_address'] == '').sum()) if len(ga) else 0} groups have an empty address")

# ══ B) same name at the identical geocode ══
print(f"\n{'─' * 60}\n📌 B) (name, latitude, longitude) — exact")
gb = nat_groups(B, ["name", "latitude", "longitude"])
report_groups("(name, latitude, longitude)", gb, "same name never shares an exact geocode",
              note=f"{int((gb['address_spellings'] > 1).sum()) if len(gb) else 0} groups with different address spellings")

# ══ C) normalised natural key ══
print(f"\n{'─' * 60}\n📌 C) (name_key, addr_key, zip_key) — normalised (storefronts with an address only)")
gc = nat_groups(B, ["name_key", "addr_key", "zip_key"], (F.col("addr_key") != "") & (F.col("name_key") != ""))
variant_groups = int(((gc["name_spellings"] > 1) | (gc["address_spellings"] > 1)).sum()) if len(gc) else 0
report_groups("(name_key, addr_key, zip_key)", gc, "normalised natural key unique",
              note=f"{variant_groups} groups are only visible after normalisation (spelling variants)")
if variant_groups:
    print("\n   📋 Duplicates only visible after normalisation (spelling variants of the same storefront):")
    display(gc[(gc["name_spellings"] > 1) | (gc["address_spellings"] > 1)]
            [["sample_name", "sample_address", "sample_city", "rows", "name_spellings", "address_spellings", "open_status", "business_ids"]]
            .head(15))

# ══ D) spatial near-duplicates ══
print(f"\n{'─' * 60}\n📌 D) Same name_key within 100 m (haversine) — candidate duplicate listings")
P = B.where(F.col("name_key") != "").select("business_id", "name", "address", "addr_key", "city", "latitude", "longitude",
                                            "is_open", "review_count", "name_key")
a, b = P.alias("a"), P.alias("b")
pairs_653_df = (a.join(b, (F.col("a.name_key") == F.col("b.name_key")) & (F.col("a.business_id") < F.col("b.business_id")))
    .withColumn("distance_m", haversine_m(F.col("a.latitude"), F.col("a.longitude"), F.col("b.latitude"), F.col("b.longitude")))
    .where(F.col("distance_m") <= 100)
    .select(F.col("a.business_id").alias("business_id_1"), F.col("b.business_id").alias("business_id_2"),
            F.col("a.name").alias("name_1"), F.col("b.name").alias("name_2"),
            F.col("a.address").alias("address_1"), F.col("b.address").alias("address_2"), F.col("a.city").alias("city"),
            F.round("distance_m", 1).alias("distance_m"),
            F.col("a.is_open").alias("is_open_1"), F.col("b.is_open").alias("is_open_2"),
            F.col("a.review_count").alias("reviews_1"), F.col("b.review_count").alias("reviews_2"),
            ((F.col("a.addr_key") == F.col("b.addr_key")) & (F.col("a.addr_key") != "")).alias("same_address"),
            ((F.col("a.address") == "") | (F.col("b.address") == "")).alias("address_missing"))
    .toPandas())

if len(pairs_653_df):
    p = pairs_653_df
    p["match_basis"] = np.select([p["address_missing"], p["same_address"], p["distance_m"] <= 25],
                                 ["no address (service-area)", "same address (normalised)", "≤ 25 m, address differs"],
                                 "25–100 m, address differs")
    opens = p["is_open_1"].astype(int) + p["is_open_2"].astype(int)
    p["open_status"] = np.select([opens == 2, opens == 1], ["all open", "open + closed"], "all closed")
    p["confidence"] = np.where(p["same_address"] | (p["distance_m"] <= 25), "high", "medium")
    p["name_variant"] = p["name_1"] != p["name_2"]
    involved = len(set(p["business_id_1"]) | set(p["business_id_2"]))
    print("\n   📊 Candidate pairs — match basis × open status:")
    display(pd.crosstab(p["match_basis"], p["open_status"], margins=True, margins_name="Total").reset_index())
else:
    p, involved = pairs_653_df, 0

dq_add(findings_653, "df_business", "(name_key) within 100 m", "one listing per place", N_BUS, involved,
       note=f"{len(p):,} candidate pairs; {int(p['name_variant'].sum()) if len(p) else 0} with differently spelled names")
for status in ["all open", "open + closed", "all closed"]:
    n_pairs = int((p["open_status"] == status).sum()) if len(p) else 0
    n_high = int(((p["open_status"] == status) & (p["confidence"] == "high")).sum()) if len(p) else 0
    dq_add(findings_653, "df_business", f"  ↳ pairs: {status}", OPEN_NOTE[status], len(p), n_pairs,
           status=("INFO" if status == "all closed" else None) if n_pairs else "PASS", note=f"high confidence: {n_high:,}")
if len(p):
    print("\n   📋 Highest-confidence candidate pairs (same address or ≤ 25 m, closest first):")
    _ord = {"all open": 0, "open + closed": 1, "all closed": 2}
    display(p[p["confidence"] == "high"].assign(_o=p["open_status"].map(_ord)).sort_values(["_o", "distance_m"])
            [["name_1", "name_2", "address_1", "address_2", "city", "distance_m", "is_open_1", "is_open_2",
              "reviews_1", "reviews_2", "match_basis", "open_status", "business_id_1", "business_id_2"]].head(20))

# ══ E) coordinate collisions & placeholder centroids ══
print(f"\n{'─' * 60}\n📌 E) (latitude, longitude) collisions — shared geocodes vs placeholder centroids")
coord_653_df = (B.groupBy("latitude", "longitude")
                 .agg(F.count("*").alias("businesses"), F.sum((F.col("address") == "").cast("int")).alias("no_address"),
                      F.countDistinct("addr_key").alias("distinct_addresses"), F.countDistinct("name_key").alias("distinct_names"),
                      F.first("city").alias("city"), F.first("state").alias("state"))
                 .where("businesses > 1").toPandas())
coord_653_df["no_address_share"] = (coord_653_df["no_address"] / coord_653_df["businesses"]).round(3)
HOT_MIN = 10
hot = coord_653_df[(coord_653_df["businesses"] >= HOT_MIN) & (coord_653_df["no_address_share"] >= 0.5)]
shared_rows = int(coord_653_df["businesses"].sum())
dq_add(findings_653, "df_business", "(latitude, longitude) shared", "profile — multi-tenant geocodes", N_BUS, shared_rows,
       status="INFO", note=f"{len(coord_653_df):,} points shared by ≥2 businesses; max {int(coord_653_df['businesses'].max())} on one point")
dq_add(findings_653, "df_business", "(latitude, longitude) placeholder centroid",
       f"no point with ≥{HOT_MIN} businesses, ≥50% address-less", N_BUS, int(hot["businesses"].sum()),
       examples="; ".join(f"({r.latitude}, {r.longitude}) {r.city}: {r.businesses} biz, {r.no_address} w/o address"
                          for r in hot.sort_values("businesses", ascending=False).head(4).itertuples()),
       note=f"{len(hot)} hotspot points — service-area businesses geocoded to a city / ZIP centroid (DQ-032 / DQ-027)")
print("\n   📋 Largest coordinate clusters:")
display(coord_653_df.sort_values("businesses", ascending=False).head(15))

# ══ F) multi-tenant addresses (legitimate) ══
print(f"\n{'─' * 60}\n📌 F) (addr_key, zip_key) shared by different businesses — multi-tenant addresses")
mt = (B.where(F.col("addr_key") != "").groupBy("addr_key", "zip_key")
       .agg(F.countDistinct("name_key").alias("names"), F.count("*").alias("n"))
       .where("names > 1").agg(F.count("*").alias("g"), F.sum("n").alias("r"), F.max("n").alias("mx")).first())
dq_add(findings_653, "df_business", "(addr_key, zip_key)", "profile — several businesses at one address", N_BUS, int(mt["r"] or 0),
       status="INFO", note=f"{int(mt['g'] or 0):,} addresses host ≥2 different names (max {mt['mx']}) — malls, office towers; NOT a key")

# ══ G) child natural keys after folding ══
print(f"\n{'─' * 60}\n📌 G) Child natural keys after case / whitespace folding")
fold = lambda c: F.lower(F.trim(F.regexp_replace(F.regexp_replace(c, INVISIBLE_RE, " "), "\\s+", " ")))
n_cat = df_categories.count()
exact_sur = n_cat - df_categories.select("business_id", "category_name").distinct().count()
fold_sur = n_cat - df_categories.select("business_id", fold(F.col("category_name")).alias("c")).distinct().count()
dq_add(findings_653, "df_categories", "(business_id, folded category_name)", "one category per business after folding",
       n_cat, fold_sur, note=f"exact repeats {exact_sur} (DQ-BUSINESS-005) + {fold_sur - exact_sur} case / spacing variants")
for a_, (tbl, _) in DICT_TABLE_MAP.items():
    fr = business_dfs[tbl]
    n = fr.count()
    sur = n - fr.select("business_id", fold(F.col("key")).alias("k")).distinct().count()
    dq_add(findings_653, FRAME_NAME[tbl], "(business_id, folded key)", "one flag per key after folding", n, sur)

# ══ Duplicate-candidate inventory (input for Silver entity resolution) ══
cand = set()
for g in (ga, gb, gc):
    for ids in g["business_ids"]:
        cand.update(ids)
if len(p):
    cand.update(p["business_id_1"]); cand.update(p["business_id_2"])
dup_candidates_653 = sorted(cand)

findings_653_df = pd.DataFrame(findings_653)
uq_metrics.update({"nat_exact_groups": len(ga), "nat_exact_surplus": int(ga["rows"].sum() - len(ga)) if len(ga) else 0,
                   "name_geo_surplus": int(gb["rows"].sum() - len(gb)) if len(gb) else 0,
                   "nat_norm_surplus": int(gc["rows"].sum() - len(gc)) if len(gc) else 0, "nat_norm_variant_groups": variant_groups,
                   "spatial_pairs": len(p), "spatial_live": int((p["open_status"] == "all open").sum()) if len(p) else 0,
                   "spatial_relist": int((p["open_status"] == "open + closed").sum()) if len(p) else 0,
                   "dup_candidates": len(dup_candidates_653), "hot_points": len(hot), "hot_rows": int(hot["businesses"].sum()),
                   "shared_coord_rows": shared_rows, "cat_fold_extra": fold_sur - exact_sur})

print("\n   Interpretation:")
print(f"   • {uq_metrics['nat_exact_surplus']} exact and {uq_metrics['nat_norm_surplus']} normalised natural-key duplicates: the same storefront listed")
print("     under 2+ business_ids. Most pairs are open + closed (Yelp re-listed the place and closed the old id)")
print("     or both open (live duplicates). In both cases reviews and ratings are split between ids.")
print(f"   • {uq_metrics['spatial_pairs']} same-name pairs within 100 m; {len(dup_candidates_653):,} businesses in total are duplicate candidates.")
print(f"   • {len(hot)} geocodes act as placeholder centroids for {uq_metrics['hot_rows']:,} mostly address-less businesses — lat/lon is NOT")
print("     a location key for service-area businesses, and distance-based features are meaningless for them.")
print("   • Caveat: groups with 3+ rows are mostly several units of one chain inside an AIRPORT terminal (STL 10701 Natural")
print("     Bridge Rd, TPA 4100 George J Bean Pkwy, PHL 8000 Essington Ave). These are legitimate, so the natural key needs a")
print("     unit / terminal qualifier, and these groups need manual review before any merge.")
print("   • Shared addresses / geocodes for DIFFERENT names are legitimate (malls, office towers) and must not be deduplicated.")
dq_summary(findings_653, "6.5.3 Composite Key Uniqueness")
print("\nCOMPOSITE KEY UNIQUENESS COMPLETE")
print("=" * 80)


---
#### 🚨 DQ-BUSINESS-038 — Same Storefront Listed Under Several `business_id`s (Natural-Key Duplicate Listings)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-038` |
| **Severity** | HIGH (entity-level duplication: reviews, ratings and check-ins of one place are split across ids) |
| **Detected In** | Phase 6.5 — Uniqueness Checks → **6.5.3 Composite Key Uniqueness** (Cell 95), check A `(name, address, city, state, postal_code)` exact and check C `(name_key, addr_key, zip_key)` normalised |
| **Also Confirmed In** | 6.5.3 B (Cell 95): 36 surplus rows share name + identical lat/lon. 6.5.1 C (Cell 92): 0 static-profile duplicates, so the copies are **not** byte copies (categories, attributes, stars and review counts differ). The 51 extra duplicates found only after normalisation come from the spelling issues DQ-BUSINESS-021 / 022 / 029 / 031. 6.5.2 (Cell 93) shows `business_id` itself is unique, so a PK check cannot catch this |
| **Affected Columns** | `name`, `address`, `city`, `state`, `postal_code` (key); impact on `stars`, `review_count`, `is_open`, and every FK in review / tip / checkin |
| **Total Affected Rows** | **Exact:** 121 groups / 248 rows / **127 surplus** (0.084%). **Normalised:** 172 groups / 350 rows / **178 surplus** (0.118%), 41 groups only visible after normalisation |
| **Invalid Value** | `AMC Indianapolis 17 @ 4325 S Meridian St`: 2 ids, one closed (69 reviews, 4.0 stars), one open (72 reviews, 3.5 stars). `Bourbon Orleans Hotel @ 717 Orleans St`: 45 vs 306 reviews. `Fat Hen Grocery`: `7457 St Charles Ave` vs `7457 St. Charles Ave`. `Telus World of Science`, `Whitts Barbecue`, `Joe’s Cafe` / `Joe's Cafe` (name spelling only) |
| **Domain Rule** | One physical business location = one entity. Yelp keeps one business page per location, so the natural key (name + street address + postal code) must identify at most one active listing |

**Description:** The same storefront exists under two or more `business_id`s. By `is_open` status, the exact groups split as:

| Pattern | Groups | Rows | Meaning |
| --- | --- | --- | --- |
| all open | 52 | 109 | **Live duplicates**: two active pages for one place |
| open + closed | 56 | 112 | **Re-listing**: Yelp opened a new page and closed the old one, so the history is split |
| all closed | 13 | 27 | Historical duplicates, low impact |

The normalised groups split 82 / 71 / 19 the same way.

The impact is measurable:
* **Reviews:** the 121 exact groups hold 13,616 reviews, and **3,346 (24.6%) sit on the smaller copy**, invisible to anyone reading the main page.
* **Ratings:** stars differ inside 90 of 121 groups, by **1 star or more in 48 groups** (max 2.0).
* **Analytics:** per-business ratings, review counts, "top-rated" rankings and business counts per city / category are biased, and check-ins and tips are split the same way.

**Caveat:** only 3 groups have 3+ rows, and they are legitimate multi-unit venues: several Starbucks units inside airport terminals (STL `10701 Natural Bridge Rd`, TPA `4100 George J Bean Pkwy`) and XpresSpa at PHL `8000 Essington Ave`. They need a unit / terminal qualifier, not a merge.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep every `business_id`; add a deterministic `canonical_business_id` crosswalk in Silver** built from the normalised natural key (`name_key`, `addr_key`, `zip_key`). Survivorship order: open > most reviews > highest id. Send groups with 3+ rows or airport / mall addresses to manual review | Reviews, tips and check-ins keep their original FK (no cross-table rewrite). Gold can aggregate by `canonical_business_id` to re-join split reviews and recompute stars from them. Deterministic and auditable, and reversible if a merge is wrong |
| 2 | **Physically merge: keep one row, move the others to `quarantine`, and repoint review / tip / checkin FKs** | Gives the cleanest Silver table, but it is destructive and touches 4 tables. Bronze-to-Silver lineage of the dropped ids is lost, and a false merge (airport chains) is hard to undo |
| 3 | **Probabilistic entity resolution (e.g. Splink / Jaro-Winkler + geo + category)** | Higher recall (catches "Little Caesars" vs "Little Caesars Pizza"), but non-deterministic, needs tuning and labelled pairs. Better suited to a later Gold enrichment |
| 4 | **Do nothing** | Ratings and review counts stay split for 248–350 listings; every per-business KPI is wrong for them |

**Best Logical Solution:** **Option 1 — keep all ids and add a deterministic `canonical_business_id` crosswalk (normalised natural key + survivorship rules), with multi-unit venues routed to manual review.** It fixes the analytics without rewriting any FK, stays reproducible from the 6.5.1 match keys, and leaves room for Option 3 later.

---

---
#### 🚨 DQ-BUSINESS-039 — Same-Name Listings Within 100 m (Spatial Near-Duplicates, Re-Listings and Copied Geocodes)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-039` |
| **Severity** | MEDIUM (candidate duplicates; needs confidence tiers before any action) |
| **Detected In** | Phase 6.5 — Uniqueness Checks → **6.5.3 Composite Key Uniqueness** (Cell 95), check D "same `name_key` within 100 m (haversine)" |
| **Also Confirmed In** | 6.5.3 A/C (Cell 95): the 144 same-address pairs are the DQ-BUSINESS-038 groups seen spatially. 6.5.3 B (Cell 95): 13 same-name groups share an identical geocode but a **different** address text. Coordinate quality issues DQ-BUSINESS-011 / 027 explain part of the jitter |
| **Affected Columns** | `name`, `latitude`, `longitude`, `address`, `is_open` |
| **Total Affected Rows** | **397** candidate pairs involving **782** businesses (0.52%). 50 pairs have differently spelled names |
| **Invalid Value** | `IHOP` (Boone): `702 Blowing Rock Rd` vs `825 Nashville Pike, Ste D` at **0 m**. `Union Taco`: `4161 Main St` vs `4229 Main St` at 0.2 m. `Fox and Hound Pub & Grille`: `14490 Lowe Way` vs `14490 Lowes Way`. `East End Alehouse` vs `East End Ale House` (`, 712 Main St` vs `712 Main St`) |
| **Domain Rule** | Two listings of the same brand at (almost) the same point are one location, unless the venue is multi-unit (airport, mall, hotel + its restaurant). Two different street addresses cannot share the exact same rooftop geocode |

**Description:** Pairs by match basis and open status:

| Match basis | all open | open + closed | all closed | Total |
| --- | --- | --- | --- | --- |
| same address (normalised) | 67 | 58 | 19 | 144 |
| ≤ 25 m, address differs | 60 | 49 | 19 | 128 |
| 25–100 m, address differs | 47 | 53 | 21 | 121 |
| no address (service-area) | 3 | 1 | 0 | 4 |
| **Total** | **177** | **161** | **59** | **397** |

High-confidence pairs (same address or ≤ 25 m): 128 live, 108 re-listing, 38 historical. The 249 "address differs" pairs are not caught by the address-based check (DQ-BUSINESS-038), and they mix three different situations:
* **(a) Address variants of one place:** typos or suffix changes (`Lowe Way` / `Lowes Way`).
* **(b) Relocations:** one id closed, a new one opened nearby. These are two real places in time, so they should be linked rather than merged.
* **(c) Copied geocodes:** different addresses carrying an identical lat/lon (IHOP 0 m apart with two different streets). This is a coordinate error, not a duplicate.

A plain distance rule cannot tell these apart, so the list is a candidate set, not a delete list.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Tiered candidate list:** auto-link **high-confidence** pairs (same normalised address, or ≤ 25 m with the same `zip_key`) into the DQ-BUSINESS-038 `canonical_business_id` crosswalk, and flag **medium** pairs (25–100 m) with `is_possible_duplicate` for review | Uses the evidence already computed. High tier has very low false-merge risk, medium tier is surfaced without being acted on |
| 2 | **Link relocations instead of merging:** for open + closed pairs with different addresses, store `successor_business_id` on the closed listing | Keeps two real locations distinct, while letting Gold follow a business's history across the move |
| 3 | **Flag copied geocodes:** same point but different address → `geo_quality = 'shared_point'`, and re-geocode from the address where possible | Treats (c) as a coordinate problem; stops wrong coordinates feeding distance features and later dedupe runs |
| 4 | **Auto-merge every pair within 100 m** | Simple, but merges chain units in airports / malls and relocations, and the damage is hard to undo |
| 5 | **Tighten to ≤ 10 m only** | Fewer false positives, but misses address-variant duplicates whose geocodes jitter (DQ-BUSINESS-027) |

**Best Logical Solution:** **Option 1 combined with Options 2 and 3.** Auto-link the high-confidence tier, flag the medium tier, link relocations with `successor_business_id`, and mark copied geocodes. Each situation gets the treatment that matches what it really is, and nothing is merged on distance alone.

---

---
#### 🚨 DQ-BUSINESS-040 — Placeholder Centroid Coordinates Shared by Many Address-Less Businesses

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-040` |
| **Severity** | MEDIUM (lat/lon is not a real location for these rows; geo analytics are distorted) |
| **Detected In** | Phase 6.5 — Uniqueness Checks → **6.5.3 Composite Key Uniqueness** (Cell 95), check E "(latitude, longitude) placeholder centroid" |
| **Also Confirmed In** | 6.5.4 A (Cell 99): only 137,248 distinct (lat, lon) pairs for 150,346 businesses, so 13,098 rows reuse another business's geocode. 6.3.6 / DQ-BUSINESS-032: 5,127 empty-string addresses (service-area businesses). DQ-BUSINESS-027 (coarse coordinates) and DQ-BUSINESS-011 (coordinates outside every metro) are other coordinate-quality issues |
| **Affected Columns** | `latitude`, `longitude` (with `address = ''`) |
| **Total Affected Rows** | **2,238** businesses (1.49%) on **123** hotspot points (≥ 10 businesses, ≥ 50% address-less). **2,143** of them (95.8%) have no address, i.e. **41.8% of all address-less businesses** sit on these 123 points. 1,976 of the 2,238 are open |
| **Invalid Value** | `(34.420334, -119.7107494)` Santa Barbara: **146** businesses, 141 without address. `(34.4208305, -119.6981901)` Santa Barbara: 65 / 60. `(39.4835647, -119.7310213)` Reno: 51 / 48. `(27.9552692, -82.4563199)` "Tampa Bay": 35 / 33. `(39.5401545, -119.7483949)` Sparks: 34 / 34 (100%) |
| **Domain Rule** | `latitude` / `longitude` should locate the business premises. A business with no street address has no premises, so a coordinate shared by dozens of unrelated businesses is a city / ZIP centroid, not a location |

**Description:** Service-area businesses (movers, plumbers, mobile services, photographers) hide their address on Yelp. Yelp then geocodes them to a city or ZIP centroid, so up to 146 unrelated businesses sit on exactly the same point. Multi-tenant buildings also share geocodes, but they have addresses. The hotspots here are 95.8% address-less overall (the 15 largest are 89–100%), which separates the two cases. The effects:
* **Heatmaps and density maps:** artificial spikes downtown.
* **Distance and "nearby" features:** for example, distance to the closest competitor comes out as 0 m.
* **Geo-clustering:** clusters form around the fake points.
* **Other uniqueness checks:** these points create false same-place matches (`(latitude, longitude)` cannot be used as part of a location key for these rows).

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep the coordinates; add `geo_precision` in Silver** (`'rooftop'` / `'shared_point'` / `'placeholder_centroid'`), where placeholder = point shared by ≥ 10 businesses with ≥ 50% address-less, or `address` empty. Gold geo features filter or down-weight non-rooftop rows | No value is lost (the city-level position is roughly correct), the rule is deterministic and reproducible from this check, and each consumer decides how to use it |
| 2 | **Set lat/lon to NULL for placeholder rows** | Stops misuse, but throws away still-useful city-level location and turns 2,238 rows into missing coordinates for any map |
| 3 | **Re-geocode from the street address** | Gives correct points for the 95 hotspot rows that *do* have an address, but cannot work for the 2,143 address-less ones, and needs an external geocoder |
| 4 | **Add random jitter around the centroid** | Removes the visual spike, but fabricates locations. Not acceptable for a cleaned layer |

**Best Logical Solution:** **Option 1 — keep the coordinates and add a `geo_precision` flag**, plus Option 3 for the 95 hotspot rows that have an address. Coordinates are preserved, the fake precision is made explicit, and every downstream geo feature gets a reliable filter.

---

In [0]:
# 6.5.4) Distribution Uniqueness  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Is the number of distinct values in each column what the business domain predicts?
#   A) df_business — closed domains (stars 9, is_open 2, state ≤ 14 footprint codes), keys (business_id = N),
#      and RAW vs FOLDED cardinality (city vs city_key, name vs name_key, address vs addr_key, postal_code vs
#      zip_key): the gap is the cardinality inflated by spelling variants. lat/lon pairs should be ≈ N.
#   B) df_attributes — 32 attributes vs their domain size (BOOLEAN 2, price 4, enums |ENUM_DOMAINS|):
#      CONSTANT (1 value) or EMPTY columns carry no information; OVER-domain is corruption;
#      DOMINANT (top value ≥ 95 %) is low-information; LOW SUPPORT (< 1 % of businesses answered) means the
#      cardinality itself is not reliable (INFO, links DQ-BUSINESS-034).
#   C) 7 nested dicts — distinct keys ≤ contract keys; values ⊆ {TRUE, FALSE}; constant keys flagged.
#   D) df_categories — vocabulary size, stability under case / punctuation folding, single-use labels.
#   E) Concentration — dominant values (chains, metros, schedule templates) that drive cardinality expectations.
import operator
from functools import reduce
import pandas as pd
from pyspark.sql import functions as F

print("=" * 80)
print("6.5.4) DISTRIBUTION UNIQUENESS  —  Business DataFrames")
print("    Per-column cardinality vs domain expectations")
print("=" * 80)

findings_654 = []
B = df_bkeys

# ══ A) df_business cardinality ══
print(f"\n{'─' * 60}\n📌 A) df_business — cardinality per column")
CORE = ["business_id", "name", "address", "city", "state", "postal_code", "latitude", "longitude", "stars", "review_count", "is_open"]
KEYS = ["name_key", "addr_key", "city_key", "zip_key"]
d = B.agg(*[F.countDistinct(c).alias(c) for c in CORE + KEYS], F.countDistinct("latitude", "longitude").alias("lat_lon"),
          F.min("review_count").alias("rc_min"), F.max("review_count").alias("rc_max")).first()

def card(column, distinct, expected, status, assessment, target):
    """target = the distinct count the domain predicts; Gap = |actual - target|."""
    return {"Column": column, "Distinct": int(distinct), "Expected": expected, "Gap": abs(int(distinct) - int(target)),
            "Cardinality %": round(distinct / N_BUS * 100, 3), "Status": status, "Assessment": assessment}

card_rows = [
    card("business_id", d["business_id"], f"= {N_BUS:,} (PK)", "PASS" if d["business_id"] == N_BUS else "FINDING", "fully unique surrogate key", N_BUS),
    card("stars", d["stars"], "9 (1.0 … 5.0, half-stars)", "PASS" if d["stars"] == 9 else "FINDING", "closed ordinal domain", 9),
    card("is_open", d["is_open"], "2 (TRUE / FALSE)", "PASS" if d["is_open"] == 2 else "FINDING", "binary flag", 2),
    card("state", d["state"], "≤ 14 (11 Yelp metros)", "PASS" if d["state"] <= 14 else "FINDING",
         f"{d['state'] - 14} extra codes = mislabeled states (DQ-BUSINESS-009 / 010)", 14),
    card("city", d["city"], f"≈ {d['city_key']:,} (folded city_key)", "PASS" if d["city"] == d["city_key"] else "FINDING",
         f"{d['city'] - d['city_key']:,} extra values from spelling variants (DQ-BUSINESS-021)", d["city_key"]),
    card("name", d["name"], f"≈ {d['name_key']:,} (folded name_key)", "PASS" if d["name"] == d["name_key"] else "INFO",
         f"{d['name'] - d['name_key']:,} extra values from casing / punctuation variants (DQ-BUSINESS-022); repeats = chains", d["name_key"]),
    card("address", d["address"], f"≈ {d['addr_key']:,} (folded addr_key)", "PASS" if d["address"] == d["addr_key"] else "INFO",
         f"{d['address'] - d['addr_key']:,} extra values from suffix / casing variants (DQ-BUSINESS-031)", d["addr_key"]),
    card("postal_code", d["postal_code"], f"≈ {d['zip_key']:,} (ZIP5 / FSA)", "INFO",
         "gap = granularity, not noise: full 6-char CA codes collapse to their FSA (kept coarse because of DQ-BUSINESS-014 truncation)", d["zip_key"]),
    card("(latitude, longitude)", d["lat_lon"], f"≈ {N_BUS:,} (one point per place)", "INFO",
         f"{N_BUS - d['lat_lon']:,} rows reuse another business's geocode (6.5.3 E)", N_BUS),
    card("review_count", d["review_count"], "open (≥ 0)", "INFO", f"range {d['rc_min']} – {d['rc_max']:,}", d["review_count"]),
]
card_654_df = pd.DataFrame(card_rows)
display(card_654_df)
for r in card_rows:
    dq_add(findings_654, "df_business", r["Column"], f"expected distinct: {r['Expected']}", r["Distinct"],
           0 if r["Status"] == "PASS" else r["Gap"], status=r["Status"], note=r["Assessment"])

print("\n   📊 Top-value concentration:")
conc_rows = []
for c in ["name", "city", "state", "postal_code"]:
    top = B.groupBy(c).count().orderBy(F.desc("count")).limit(3).collect()
    conc_rows.append({"Column": c, "Top values": ", ".join(f"{t[c]!r} ({t['count']:,})" for t in top),
                      "Top-1 share %": round(top[0]["count"] / N_BUS * 100, 2)})
chain = B.groupBy("name_key").count()
ch = chain.agg(F.sum(F.when(F.col("count") > 1, F.col("count")).otherwise(0)).alias("rows_in_repeated"),
               F.sum(F.when(F.col("count") >= 10, 1).otherwise(0)).alias("chains10")).first()
conc_rows.append({"Column": "name_key (chains)", "Top values": f"{ch['chains10']:,} brands with ≥ 10 locations",
                  "Top-1 share %": round(ch["rows_in_repeated"] / N_BUS * 100, 2)})
display(pd.DataFrame(conc_rows))
print("      (for name_key the share column = % of businesses whose name is shared with another listing)")

# ══ B) df_attributes — 32 attributes vs domain size ══
print(f"\n{'─' * 60}\n📌 B) df_attributes — distinct values vs domain size")
attr_cols = [c for c in df_attributes.columns if c != "business_id"]
stack = "stack({}, {}) as (attr, value)".format(len(attr_cols), ", ".join(f"'{c}', cast({c} as string)" for c in attr_cols))
vc = df_attributes.selectExpr(stack).where("value is not null").groupBy("attr", "value").count().toPandas()
DOMAIN = {**{s: 2 for s in ATTR_BOOLEAN_MAP.values()}, **{s: 4 for s in ATTR_INT_MAP.values()},
          **{s: len(ENUM_DOMAINS[s]) for s in ATTR_ENUM_MAP.values()}}
attr_rows = []
for c in attr_cols:
    sub = vc[vc["attr"] == c]
    nn, nd = int(sub["count"].sum()), len(sub)
    top_share = (sub["count"].max() / nn) if nn else 0.0
    dom = DOMAIN[c]
    if nd == 0:
        flag, status = "EMPTY (never answered)", "FINDING"
    elif nd == 1:
        flag, status = "CONSTANT", "FINDING"
    elif nd > dom:
        flag, status = f"OVER domain ({nd} > {dom})", "FINDING"
    elif nn < 0.01 * N_BUS:
        flag, status = f"LOW SUPPORT ({nn} answers < 1 %)", "INFO"
    elif top_share >= 0.95:
        flag, status = "DOMINANT (top ≥ 95 %)", "INFO"
    elif nd < dom:
        flag, status = f"UNDER domain ({nd} < {dom})", "INFO"
    else:
        flag, status = "OK", "PASS"
    top_val = sub.sort_values("count", ascending=False)["value"].iloc[0] if nd else "—"
    attr_rows.append({"Attribute": c, "Answered": nn, "Answered %": round(nn / N_BUS * 100, 2), "Distinct": nd,
                      "Domain size": dom, "Top value": top_val, "Top share %": round(top_share * 100, 2), "Flag": flag, "Status": status})
    if status != "PASS":
        dq_add(findings_654, "df_attributes", c, f"distinct ≤ {dom}, not constant", nn, nn if status == "FINDING" else 0,
               status=status, note=f"{flag}; top '{top_val}' {top_share * 100:.2f} %")
attr_654_df = pd.DataFrame(attr_rows)
n_attr_ok = int((attr_654_df["Status"] == "PASS").sum())
dq_add(findings_654, "df_attributes", f"{n_attr_ok} attributes", "distinct values within domain, informative", len(attr_cols), 0, status="PASS")
display(attr_654_df.sort_values(["Status", "Top share %"], ascending=[True, False]))

# ══ C) nested dicts — keys vs contract, values ⊆ {TRUE, FALSE} ══
print(f"\n{'─' * 60}\n📌 C) Nested dicts — distinct keys / values vs contract")
dict_rows = []
for a_, (tbl, keymap) in DICT_TABLE_MAP.items():
    fr = business_dfs[tbl]
    kv = fr.where(F.col("value").isNotNull()).groupBy("key").agg(F.countDistinct("value").alias("vals"), F.count("*").alias("n")).toPandas()
    keys_all = fr.select("key").distinct().count()
    vals = fr.where(F.col("value").isNotNull()).select("value").distinct().count()
    const_keys = sorted(kv.loc[kv["vals"] == 1, "key"].tolist())
    dq_add(findings_654, FRAME_NAME[tbl], "key", f"distinct keys ≤ {len(keymap)} (contract)", len(keymap), max(keys_all - len(keymap), 0))
    if const_keys:
        dq_add(findings_654, FRAME_NAME[tbl], "value per key", "each key takes both TRUE and FALSE", len(kv), len(const_keys),
               status="INFO", examples=", ".join(const_keys), note="constant flag → no discriminative value")
    dict_rows.append({"DataFrame": FRAME_NAME[tbl], "Contract keys": len(keymap), "Distinct keys": keys_all,
                      "Distinct non-null values": vals, "Constant keys": ", ".join(const_keys) or "—"})
display(pd.DataFrame(dict_rows))

# ══ D) df_categories — vocabulary ══
print(f"\n{'─' * 60}\n📌 D) df_categories — vocabulary cardinality")
cn = F.col("category_name")
cv = df_categories.agg(F.countDistinct(cn).alias("exact"),
                       F.countDistinct(F.lower(F.trim(cn))).alias("folded"),
                       F.countDistinct(F.regexp_replace(F.lower(cn), "[^\\p{L}\\p{N}]", "")).alias("alnum")).first()
per_cat = df_categories.groupBy("category_name").agg(F.countDistinct("business_id").alias("businesses"))
single = per_cat.where("businesses = 1").count()
top_cat = per_cat.orderBy(F.desc("businesses")).limit(5).collect()
dq_add(findings_654, "df_categories", "category_name", "vocabulary stable under case / space folding", cv["exact"], cv["exact"] - cv["folded"])
dq_add(findings_654, "df_categories", "category_name", "vocabulary stable under punctuation folding", cv["exact"], cv["exact"] - cv["alnum"],
       note="e.g. 'Hair Salons' vs 'Hair-Salons' would collapse")
dq_add(findings_654, "df_categories", "category_name", "profile — single-use labels", cv["exact"], single, status="INFO",
       examples=", ".join(repr(r["category_name"]) for r in per_cat.where("businesses = 1").limit(6).collect()),
       note=f"vocabulary {cv['exact']:,} labels; incl. delimiter fragments (DQ-BUSINESS-013)")
print("   Top categories: " + ", ".join(f"{r['category_name']} ({r['businesses']:,}, {r['businesses'] / N_BUS * 100:.1f}%)" for r in top_cat))

# ══ E) df_hours — weekly schedule templates ══
print(f"\n{'─' * 60}\n📌 E) df_hours — distinct weekly schedule patterns")
hcols = [HOURS_MAP[dd] for dd in DAYS]
pat = (df_hours.where(reduce(operator.or_, [F.col(c).isNotNull() for c in hcols]))
               .select(F.concat_ws(" | ", *[F.coalesce(F.col(c), F.lit("—")) for c in hcols]).alias("weekly_pattern")))
n_sched = pat.count()
pat_vc = pat.groupBy("weekly_pattern").count()
n_pat = pat_vc.count()
top_pat = pat_vc.orderBy(F.desc("count")).limit(5).toPandas()
top_pat["share %"] = (top_pat["count"] / n_sched * 100).round(2)
dq_add(findings_654, "df_hours", "weekly pattern (Mon–Sun)", "profile — schedule templates", n_sched, n_pat, status="INFO",
       note=f"{n_pat:,} distinct weekly patterns for {n_sched:,} businesses with a schedule; top = '{top_pat['weekly_pattern'].iloc[0]}' "
            f"({top_pat['share %'].iloc[0]} %) — 0:0-0:0 template (DQ-BUSINESS-007 / 025)")
display(top_pat)

findings_654_df = pd.DataFrame(findings_654)
uq_metrics.update({"state_distinct": d["state"], "city_inflation": d["city"] - d["city_key"], "name_inflation": d["name"] - d["name_key"],
                   "addr_inflation": d["address"] - d["addr_key"], "latlon_reuse": N_BUS - d["lat_lon"],
                   "attr_flagged": int((attr_654_df["Status"] != "PASS").sum()), "cat_vocab": cv["exact"], "hours_patterns": n_pat})

print("\n   Interpretation:")
print(f"   • Closed domains hold: stars {d['stars']} values, is_open {d['is_open']}. state has {d['state']} codes vs 14 expected (DQ-009 / 010).")
print(f"   • Spelling variants inflate cardinality: city +{uq_metrics['city_inflation']:,}, name +{uq_metrics['name_inflation']:,}, "
      f"address +{uq_metrics['addr_inflation']:,} distinct values.")
print(f"   • All 32 attributes stay within their domain and none is constant; {uq_metrics['attr_flagged']} are dominant (≥ 95 % one value)")
print("     or low-support (< 1 % answered) — weak features whose distinct counts are not reliable, see table B.")
print(f"   • Category vocabulary ({cv['exact']:,} labels) is stable under folding, so it is a clean controlled list.")
dq_summary(findings_654, "6.5.4 Distribution Uniqueness")
print("\nDISTRIBUTION UNIQUENESS COMPLETE")
print("=" * 80)


---
#### 🚨 DQ-BUSINESS-041 — `Music.no_music` Is Never TRUE (Dead Flag That Contradicts All-False Music Dicts)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-041` |
| **Severity** | LOW–MEDIUM (semantic: the flag carries no information and makes all-False dicts ambiguous) |
| **Detected In** | Phase 6.5 — Uniqueness Checks → **6.5.4 Distribution Uniqueness** (Cell 99), check C "value per key: each key takes both TRUE and FALSE" on `df_music` |
| **Also Confirmed In** | Same check flags `DietaryRestrictions.kosher` as constant (never TRUE), but that dict exists for only 30 businesses (low support, DQ-BUSINESS-034), so it is documented, not treated as an error. TE.2 key profile (Cell 35) shows `no_music` present in the Music dicts like every other key |
| **Affected Columns** | `attributes.Music['no_music']` → `df_music` (`key = 'no_music'`) / `yelp_business_music.no_music` |
| **Total Affected Rows** | `no_music = TRUE` in **0** of the 6,858 bronze Music dicts that contain the key. **4,622** Music dicts contain no TRUE value at all (every music type FALSE / None *and* `no_music` FALSE) |
| **Invalid Value** | `{'dj': False, 'background_music': False, 'no_music': False, 'jukebox': False, 'live': False, 'video': False, 'karaoke': False}` says neither "some music" nor "no music" |
| **Domain Rule** | `no_music` should be the complement of the music types: `no_music = TRUE` ⇔ every other music flag is FALSE. A flag that is always FALSE is either never collected or defaulted |

**Description:** The distinct-value check shows `no_music` has a cardinality of 1 (always FALSE). This most likely means the key is never collected as TRUE upstream and only appears with a default value when the Music block is saved. As a result, 4,622 businesses (62% of the 7,417 with a Music dict) have a dict that is logically contradictory: no DJ, no background music, no jukebox, no live music, no karaoke, and also *not* "no music".

Any model or report that reads `no_music` literally learns a constant. Any rule that derives "has music" from it is wrong. The all-False pattern also suggests that, for this dict, FALSE often means "not answered" rather than a confirmed no.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep `no_music` as-is but document it as non-informative in the DDL / data dictionary; in Silver Enriched derive `has_music = dj OR background_music OR jukebox OR live OR video OR karaoke`** and use that instead | Source values are untouched (lineage kept), the derived flag is built from keys that do vary, and consumers are warned off the dead column |
| 2 | **Overwrite `no_music = NOT has_music`** | Makes the dict internally consistent, but asserts facts the source never gave: an all-False dict may mean "not answered", not "no music" |
| 3 | **Drop `no_music` from `yelp_business_music`** | Removes the noise, but changes the agreed DDL contract and the bronze-to-silver 1:1 key mapping (6.1.1 coverage check would then report an unmapped leaf) |
| 4 | **Do nothing** | Constant feature in every model; the contradiction stays hidden |

**Best Logical Solution:** **Option 1 — keep the raw flag, document it as non-informative, and derive `has_music` from the other six keys.** It respects the contract and the source values, and gives analysts a meaningful music indicator. `DietaryRestrictions.kosher` gets the same "never TRUE, low support" note in the data dictionary.

---

In [0]:
# 6.5.5) Uniqueness Summary  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Consolidated view of 6.5.1 – 6.5.4: status per sub-check, headline uniqueness metrics, and every FINDING in one
# place (input for the DQ-BUSINESS notes and the Silver deduplication / entity-resolution plan).
import pandas as pd

print("=" * 80)
print("6.5) UNIQUENESS CHECKS — CONSOLIDATED SUMMARY  (Business DataFrames)")
print("=" * 80)

SECTIONS_65 = [("6.5.1 Row-Level Uniqueness", "findings_651_df"), ("6.5.2 Key-Level Uniqueness", "findings_652_df"),
               ("6.5.3 Composite Key Uniqueness", "findings_653_df"), ("6.5.4 Distribution Uniqueness", "findings_654_df")]
parts, missing_sections = [], []
for sec, var in SECTIONS_65:
    if var in globals():
        parts.append(globals()[var].assign(Section=sec))
    else:
        missing_sections.append(sec)
if missing_sections:
    print(f"   ⚠️  Not yet run (re-run these cells first): {missing_sections}")
all_65 = pd.concat(parts, ignore_index=True)

status_65 = (all_65.pivot_table(index="Section", columns="Status", values="Rule", aggfunc="count", fill_value=0)
             .reindex(columns=["FINDING", "INFO", "PASS"], fill_value=0).reset_index())
status_65["Checks"] = status_65[["FINDING", "INFO", "PASS"]].sum(axis=1)
print("\n📋 Check status per sub-section:")
display(status_65)

M = uq_metrics
headline_65 = pd.DataFrame([
    ("Row", "exact / ID-agnostic duplicate bronze rows", f"{M['bronze_exact_dups']:,} / {M['noid_dups']:,}", "no row ingested twice"),
    ("Row", "static-profile duplicates (same place, metrics differ)", f"{M['static_dups']:,} in {M['static_groups']:,} groups", "identical place + attributes + hours + categories under 2 ids"),
    ("Row", "exact duplicate rows created by TE.2", f"{M['frame_exact_dups']:,}", "restructuring is lossless and duplicate-free"),
    ("Key", "business_id case-folding collisions / ids starting with '-'", f"{M['id_ci_collisions']:,} / {M['id_lead_dash']:,}", "surrogate key is clean; quote ids in CLI / CSV"),
    ("Key", "Silver DDL PK violations in yelp_business_categories", f"{M['cat_pk_dups']:,}", "DQ-BUSINESS-005, dedupe before load"),
    ("Composite", "exact natural-key duplicates (name, address, city, state, ZIP)", f"{M['nat_exact_surplus']:,} surplus in {M['nat_exact_groups']:,} groups", "same storefront, several ids"),
    ("Composite", "normalised natural-key duplicates / groups visible only after normalisation", f"{M['nat_norm_surplus']:,} / {M['nat_norm_variant_groups']:,}", "spelling variants hide duplicates"),
    ("Composite", "same-name pairs ≤ 100 m: total / live / re-listing", f"{M['spatial_pairs']:,} / {M['spatial_live']:,} / {M['spatial_relist']:,}", "candidate duplicate listings"),
    ("Composite", "businesses flagged as duplicate candidates", f"{M['dup_candidates']:,} ({M['dup_candidates'] / M['n_bus'] * 100:.2f} %)", "input for entity resolution"),
    ("Composite", "placeholder-centroid geocodes / businesses on them", f"{M['hot_points']:,} / {M['hot_rows']:,}", "lat/lon is not a location key for service-area businesses"),
    ("Distribution", "state codes (expected ≤ 14)", f"{M['state_distinct']}", "DQ-009 / 010"),
    ("Distribution", "cardinality inflation: city / name / address", f"+{M['city_inflation']:,} / +{M['name_inflation']:,} / +{M['addr_inflation']:,}", "DQ-021 / 022 / 031"),
    ("Distribution", "attributes flagged (dominant ≥ 95 % / low support < 1 %)", f"{M['attr_flagged']}", "weak features"),
], columns=["Level", "Metric", "Value", "Interpretation"])
print("\n📋 Headline uniqueness metrics:")
display(headline_65)

find_65 = all_65[all_65["Status"] == "FINDING"][["Section", "DataFrame", "Column", "Rule", "Checked", "Violations", "Pct", "Note"]]
print(f"\n📋 All FINDINGs in 6.5 ({len(find_65)}):")
display(find_65.reset_index(drop=True))

n_f, n_i, n_p = (int((all_65["Status"] == s).sum()) for s in ("FINDING", "INFO", "PASS"))
print(f"\n   {len(all_65)} checks | {n_f} FINDING | {n_i} INFO | {n_p} PASS")
print("\n   Verdict: the table is clean at the ROW and KEY level (no copied rows, a robust surrogate key, lossless TE.2), but")
print("   NOT at the ENTITY level: the same storefront appears under several business_ids (live duplicates and")
print("   re-listings), so reviews / ratings are split. Silver: keep every id (reviews / tips / check-ins reference them),")
print("   add a deterministic `canonical_business_id` from the normalised natural key + 100 m rule, dedupe the categories")
print("   PK, and flag placeholder-centroid coordinates.")
summary_65_df = all_65
print("\nUNIQUENESS CHECKS COMPLETE")
print("=" * 80)



### 6.6) CROSS-COLUMN CONSISTENCY CHECKS
*Validate logical relationships and contradictions across columns within a row.*

- **Logical Relationship Validation** — Derived columns match their deterministic source function.
  - *Example:* `total_votes == useful + funny + cool`.
  - *Example:* `text_length == length(text)`.
  - *Example:* `review_date == review_timestamp.cast('date')`.

- **Derived-Column Logic Checks** — Business-logic transformations are correct.
  - *Example:* ID columns follow Yelp's 22-char base62 hash format.
  - *Example:* `stars` is a whole number in the valid Yelp rating set {1, 2, 3, 4, 5}.
  - *Example:* `review_half_year` correctly maps months 1–6 → H1, 7–12 → H2.

- **Logical Contradiction Detection** — Mutually exclusive states or impossible value combinations.
  - *Example:* `stars = 0` (impossible: Yelp's scale is 1–5).
  - *Example:* `useful < 0` (negative vote count is impossible).
  - *Example:* `review_id == user_id` (different ID namespaces should never collide).

- **Cross-Column Range Logic** — Range relationships between related columns.
  - *Example:* `review_month_start_date ≤ review_date ≤ review_month_end_date`.
  - *Example:* Review date between Yelp founding (2004) and today.
  - *Example:* Vote counts within plausible bounds.

- **Dependency Rule Validation** — If condition A holds, then condition B must also hold.
  - *Example:* `is_year_start == true` → `month == 1 AND day == 1`.
  - *Example:* Vote count > 0 → text should be non-empty (votes require content).
  - *Example:* `review_time_of_day == 'Morning'` → `review_am_pm == 'AM'`.

- **Category Mapping Consistency** — Codes and descriptions align across related column pairs.
  - *Example:* `review_month == 1` → `review_month_name == 'January'`.
  - *Example:* `review_quarter == 1` → `review_quarter_name == 'Q1'`.


In [0]:
# 6.6) Cross-Column Consistency — Setup & 6.6.1 Logical Relationship Validation  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# 6.6 checks relationships BETWEEN columns of the same business (incl. across the 1:1 / 0:N TE.2 frames, which
# are one logical record split by shape). This cell defines the shared helpers + `cc_wide` used by 6.6.2 – 6.6.6.
#
# 6.6.1 = "derived column == deterministic function(source)". In this dataset every TE.2 column is DERIVED from a
# bronze leaf by a TE.1 decoder, so each one is re-derived here with an INDEPENDENT method (split / lpad / lower /
# regex token counts instead of the TE.1 decoders) and compared row by row, null-safe:
#   A) df_business    pass-through scalars unchanged; stars DOUBLE → DECIMAL(2,1) exact; is_open 1 ⇔ TRUE
#   B) df_attributes  24 BOOLEAN / 1 INT / 7 ENUM == decode(raw literal)
#   C) df_hours       raw kept as-is; start/end 'HH:mm' == zero-padded parts of 'H:M-H:M'
#   D) df_categories  rows per business == tokens in raw list; positions 0..n-1 contiguous; label order identical
#   E) 7 nested dicts per business: #keys / #TRUE / #FALSE / #NULL == token counts in the raw dict string
# TE.2 only reconciled TOTALS; this is the per-row proof that restructuring changed no value.
# Requires: df_bronze (6.1.1), TE.1 / TE.2 frames, dq_add (6.2.1).
import pandas as pd
from functools import reduce
from pyspark.sql import functions as F

print("=" * 80)
print("6.6) CROSS-COLUMN CONSISTENCY CHECKS  —  Business DataFrames (TE.2)")
print("=" * 80)

N_BUS = df_business.count()
DAYS = list(HOURS_MAP)                                    # Monday … Sunday
CC_FRAME = {"yelp_business_ambience": "df_ambience", "yelp_business_best_nights": "df_best_nights",
            "yelp_business_parking": "df_business_parking", "yelp_business_dietary_restrictions": "df_dietary_restrictions",
            "yelp_business_good_for_meal": "df_good_for_meal", "yelp_business_hair_specializes_in": "df_hair_specializes_in",
            "yelp_business_music": "df_music"}
cc_metrics = {"n_bus": N_BUS}

# ══ Shared helpers (reused by 6.6.2 – 6.6.6) ══
def chk(frame, column, rule, bad, scope=None, status="FINDING", ex=None, note=""):
    """One check: `bad` evaluated only on rows in `scope`. status = what a violation is reported as."""
    return {"frame": frame, "column": column, "rule": rule, "bad": bad,
            "scope": scope if scope is not None else F.lit(True), "status": status, "ex": ex, "note": note}

def run_checks(findings, base, checks, ex_default=None):
    """All checks in ONE aggregation pass; examples fetched only for violated checks. Returns {column: violations}."""
    conds = [(F.coalesce(c["scope"], F.lit(False)), F.coalesce(c["bad"], F.lit(False))) for c in checks]
    aggs = []
    for i, (sc, bd) in enumerate(conds):
        aggs += [F.sum(sc.cast("int")).alias(f"s{i}"), F.sum((sc & bd).cast("int")).alias(f"v{i}")]
    r = base.agg(*aggs).first()
    out = {}
    for i, c in enumerate(checks):
        s, v = int(r[f"s{i}"] or 0), int(r[f"v{i}"] or 0)
        ex = ""
        if v:
            e = c["ex"] if c["ex"] is not None else (ex_default if ex_default is not None else F.col("business_id"))
            sc, bd = conds[i]
            ex = " ; ".join(str(x[0]) for x in base.where(sc & bd).select(e.cast("string")).limit(3).collect())
        dq_add(findings, c["frame"], c["column"], c["rule"], s, v, status="PASS" if v == 0 else c["status"], examples=ex, note=c["note"])
        out[c["column"]] = v
    return out

# ══ cc_wide — ONE logical row per business, rebuilt from the TE.2 frames (1:1 joins + 0:N frames folded back) ══
_dict_maps = [business_dfs[tbl].groupBy("business_id")
              .agg(F.map_from_entries(F.collect_list(F.struct("key", "value"))).alias(f"d_{a}"))
              for a, (tbl, _) in DICT_TABLE_MAP.items()]
_cat_list = (df_categories.groupBy("business_id")
             .agg(F.array_sort(F.collect_list(F.struct("category_position", "category_name"))).alias("_c"))
             .select("business_id", F.transform("_c", lambda x: x["category_name"]).alias("cat_list")))
cc_wide = reduce(lambda l, r: l.join(r, "business_id", "left"), [df_business, df_attributes, df_hours, _cat_list, *_dict_maps])

def dk(attr, key):
    """Value of a nested-dict key (NULL if the dict or the key is absent)."""
    return F.col(f"d_{attr}").getItem(key)

def has_cat(rx):
    return F.coalesce(F.exists("cat_list", lambda x: x.rlike(rx)), F.lit(False))

def _mins(c):
    return F.when(c.isNotNull(), F.substring(c, 1, 2).cast("int") * 60 + F.substring(c, 4, 2).cast("int"))

START_M = {d: _mins(F.col(f"{HOURS_MAP[d]}_start_time")) for d in DAYS}   # minutes after midnight
END_M   = {d: _mins(F.col(f"{HOURS_MAP[d]}_end_time")) for d in DAYS}
HAS_HOURS = reduce(lambda a, b: a | b, [F.col(HOURS_MAP[d]).isNotNull() for d in DAYS])
EX_BIZ = F.concat_ws(" | ", "business_id", "name", "city", "state")
print(f"   cc_wide: {N_BUS:,} businesses × {len(cc_wide.columns)} columns (df_business ⋈ df_attributes ⋈ df_hours ⋈ categories ⋈ 7 dict maps)")

# ══════════════════════════════════════════════════════════════════════
# 6.6.1) Logical Relationship Validation
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─' * 60}\n📌 6.6.1) Logical Relationship Validation — every TE.2 value == independent re-derivation from bronze")
findings_661 = []

# ── A) df_business ──
print("\n   ── A) df_business vs bronze scalars ──")
jb = df_business.alias("t").join(df_bronze.alias("r"), "business_id")
A = [chk("df_business", c, "typed == bronze (pass-through, null-safe)", ~F.col(f"t.{c}").eqNullSafe(F.col(f"r.{c}")))
     for c in ["name", "address", "city", "state", "postal_code", "latitude", "longitude"]]
A += [chk("df_business", "stars", "DECIMAL(2,1) == bronze DOUBLE (exact)", ~F.col("t.stars").cast("double").eqNullSafe(F.col("r.stars"))),
      chk("df_business", "review_count", "BIGINT == bronze value", ~F.col("t.review_count").eqNullSafe(F.col("r.review_count").cast("bigint"))),
      chk("df_business", "is_open", "TRUE ⇔ raw 1, FALSE ⇔ raw 0", ~F.col("t.is_open").eqNullSafe(F.col("r.is_open").cast("int") == 1))]
run_checks(findings_661, jb, A)

# ── B) df_attributes ── independent decoders: lower()→cast boolean, cast int, strip quotes by regex
print("\n   ── B) df_attributes vs raw Python literals (32 attributes) ──")
ja = df_attributes.alias("t").join(df_bronze.select("business_id", "attributes").alias("r"), "business_id")
B_ = []
for b, s in ATTR_BOOLEAN_MAP.items():
    raw = F.col(f"r.attributes.{b}")
    B_.append(chk("df_attributes", s, "== lower(raw).cast(boolean); 'None' → NULL",
                  ~F.col(f"t.{s}").eqNullSafe(F.when(raw.isin("True", "False"), F.lower(raw).cast("boolean"))),
                  ex=F.concat_ws(" → ", raw, F.col(f"t.{s}").cast("string"))))
for b, s in ATTR_INT_MAP.items():
    raw = F.col(f"r.attributes.{b}")
    B_.append(chk("df_attributes", s, "== raw.cast(int); 'None' → NULL", ~F.col(f"t.{s}").eqNullSafe(F.when(raw != "None", raw.cast("int"))),
                  ex=F.concat_ws(" → ", raw, F.col(f"t.{s}").cast("string"))))
for b, s in ATTR_ENUM_MAP.items():
    raw = F.col(f"r.attributes.{b}")
    B_.append(chk("df_attributes", s, "== raw with u'…' / '…' quotes stripped; 'None' → NULL",
                  ~F.col(f"t.{s}").eqNullSafe(F.when(raw != "None", F.regexp_replace(raw, "^u?'|'$", ""))),
                  ex=F.concat_ws(" → ", raw, F.col(f"t.{s}"))))
run_checks(findings_661, ja, B_)

# ── C) df_hours ── split on '-' / ':' and lpad, instead of the TE.1 regex + format_string
print("\n   ── C) df_hours vs raw 'H:M-H:M' (7 days × raw / start / end) ──")
jh = df_hours.alias("t").join(df_bronze.select("business_id", "hours").alias("r"), "business_id")
C_ = []
for d in DAYS:
    hc, raw = HOURS_MAP[d], F.col(f"r.hours.{d}")
    p = F.split(raw, "[-:]")
    ok = raw.rlike(r"^\d{1,2}:\d{1,2}-\d{1,2}:\d{1,2}$")
    exp_s = F.when(ok, F.concat(F.lpad(p[0], 2, "0"), F.lit(":"), F.lpad(p[1], 2, "0")))
    exp_e = F.when(ok, F.concat(F.lpad(p[2], 2, "0"), F.lit(":"), F.lpad(p[3], 2, "0")))
    C_ += [chk("df_hours", hc, "raw range kept as-is", ~F.col(f"t.{hc}").eqNullSafe(raw)),
           chk("df_hours", f"{hc}_start_time", "== lpad(H):lpad(M) of left side", ~F.col(f"t.{hc}_start_time").eqNullSafe(exp_s),
               ex=F.concat_ws(" → ", raw, F.col(f"t.{hc}_start_time"))),
           chk("df_hours", f"{hc}_end_time", "== lpad(H):lpad(M) of right side", ~F.col(f"t.{hc}_end_time").eqNullSafe(exp_e),
               ex=F.concat_ws(" → ", raw, F.col(f"t.{hc}_end_time")))]
run_checks(findings_661, jh, C_)

# ── D) df_categories ──
print("\n   ── D) df_categories vs raw ', '-separated list ──")
_cat_agg = (df_categories.groupBy("business_id")
            .agg(F.count("*").alias("n_rows"), F.min("category_position").alias("pmin"), F.max("category_position").alias("pmax"),
                 F.countDistinct("category_position").alias("pdist"),
                 F.array_sort(F.collect_list(F.struct("category_position", "category_name"))).alias("_c"))
            .withColumn("labels", F.transform("_c", lambda x: x["category_name"])))
jc = (df_bronze.select("business_id", F.col("categories").alias("raw"), F.split("categories", ", ").alias("raw_list"))
      .join(_cat_agg, "business_id", "left"))
D_ = [chk("df_categories", "rows per business", "== number of tokens in raw list", F.coalesce(F.col("n_rows"), F.lit(0)) != F.size("raw_list"),
          scope=F.col("raw").isNotNull(), ex=F.concat_ws(" → ", "raw", F.col("n_rows").cast("string"))),
      chk("df_categories", "category_position", "0 … n-1, contiguous, unique",
          (F.col("pmin") != 0) | (F.col("pmax") != F.col("n_rows") - 1) | (F.col("pdist") != F.col("n_rows")), scope=F.col("n_rows").isNotNull()),
      chk("df_categories", "category_name (ordered)", "labels by position == raw tokens (same order, as-is)", ~(F.col("labels") == F.col("raw_list")),
          scope=F.col("raw").isNotNull() & F.col("n_rows").isNotNull(), ex="raw"),
      chk("df_categories", "raw NULL ⇒ no rows", "business without categories has no category rows", F.col("n_rows").isNotNull(),
          scope=F.col("raw").isNull())]
run_checks(findings_661, jc, D_)

# ── E) 7 nested dicts ── per-business token counts in the raw string vs rows / values in the long frame
print("\n   ── E) 7 nested-dict frames vs raw dict-string token counts (per business) ──")
_tok = lambda s, pat: F.when(s.isNotNull(), F.size(F.regexp_extract_all(s, F.lit(pat), F.lit(0)))).otherwise(F.lit(0))
raw_cnt = [F.col("business_id")]
for a in DICT_TABLE_MAP:
    s = F.when(F.col(f"attributes.{a}").rlike(DICT_RE), F.col(f"attributes.{a}"))
    raw_cnt += [s.isNotNull().alias(f"{a}_isdict"), _tok(s, DICT_KEY_RE).alias(f"{a}_rk"), _tok(s, r":\s*True").alias(f"{a}_rT"),
                _tok(s, r":\s*False").alias(f"{a}_rF"), _tok(s, r":\s*None").alias(f"{a}_rN")]
je = df_bronze.select(*raw_cnt)
for a, (tbl, _) in DICT_TABLE_MAP.items():
    je = je.join(business_dfs[tbl].groupBy("business_id").agg(
        F.count("*").alias(f"{a}_k"), F.sum((F.col("value") == True).cast("int")).alias(f"{a}_T"),
        F.sum((F.col("value") == False).cast("int")).alias(f"{a}_F"), F.sum(F.col("value").isNull().cast("int")).alias(f"{a}_N")),
        "business_id", "left")
E_ = []
for a, (tbl, _) in DICT_TABLE_MAP.items():
    z = lambda c: F.coalesce(F.col(c), F.lit(0))
    bad = (z(f"{a}_k") != F.col(f"{a}_rk")) | (z(f"{a}_T") != F.col(f"{a}_rT")) | (z(f"{a}_F") != F.col(f"{a}_rF")) | (z(f"{a}_N") != F.col(f"{a}_rN"))
    E_ += [chk(CC_FRAME[tbl], "key / TRUE / FALSE / NULL counts", f"per business == tokens in raw attributes.{a}", bad, scope=F.col(f"{a}_isdict")),
           chk(CC_FRAME[tbl], "non-dict raw ⇒ no rows", f"'None' / NULL attributes.{a} produces no long rows", F.col(f"{a}_k").isNotNull(),
               scope=~F.col(f"{a}_isdict"))]
run_checks(findings_661, je, E_)

findings_661_df = pd.DataFrame(findings_661)
n_f = int((findings_661_df["Status"] == "FINDING").sum())
cc_metrics["rel_checks"], cc_metrics["rel_findings"] = len(findings_661_df), n_f
print(f"\n   {len(findings_661_df)} relationship checks | {n_f} FINDING  — "
      f"{'every TE.2 value is an exact, lossless function of its bronze source' if n_f == 0 else 'see FINDINGs above'}")
print("=" * 80)


In [0]:
# 6.6.2) Derived-Column Logic Checks  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Business-logic derivations (not plain decoding): is the meaning encoded in a value / column name / derived
# indicator consistent with the rule that produces it?
#   A) Identifier logic   business_id = 22-char base62 Yelp hash ([A-Za-z0-9_-]); every child frame row has one
#   B) Rating logic       stars is a mean rounded to the half-star grid 1.0 … 5.0 (step 0.5)
#   C) Schedule logic     each (start, end) pair derives ONE daily window class + duration:
#                         normal (end > start) | overnight (end < start, wraps past midnight) | '00:00-00:00' (all-day /
#                         unknown convention, DQ-007) | same non-midnight time (zero-length or 24 h — undecidable)
#   D) Naming logic       contract column names are valid, unique snake_case; dict key → column = '-' → '_' only;
#                         hours column = 'hours_' + lower(day)
#   E) Derived indicator  Music.no_music must equal NOT(any music type TRUE) — the rule a Silver `has_music` relies on
import re
from functools import reduce
from pyspark.sql import functions as F
import pandas as pd

print(f"\n{'─' * 60}\n📌 6.6.2) Derived-Column Logic Checks")
findings_662 = []

# ── A + B) df_business ──
print("\n   ── A/B) Identifier & rating logic ──")
AB = [chk("df_business", "business_id", "22-char base62 Yelp hash [A-Za-z0-9_-]{22}", ~F.col("business_id").rlike("^[A-Za-z0-9_-]{22}$")),
      chk("df_business", "stars", "on half-star grid: stars × 2 integer, 1.0 ≤ stars ≤ 5.0",
          (F.col("stars") * 2 != F.floor(F.col("stars") * 2)) | ~F.col("stars").between(1, 5), scope=F.col("stars").isNotNull(), ex="stars")]
run_checks(findings_662, df_business, AB)
for tbl, fr in business_dfs.items():
    if tbl == "yelp_business":
        continue
    n_bad = fr.where(~F.col("business_id").rlike("^[A-Za-z0-9_-]{22}$") | F.col("business_id").isNull()).count()
    dq_add(findings_662, {"yelp_business_attributes": "df_attributes", "yelp_business_hours": "df_hours",
                          "yelp_business_categories": "df_categories"}.get(tbl, CC_FRAME.get(tbl, tbl)),
           "business_id (FK side)", "22-char base62 hash, NOT NULL", fr.count(), n_bad)

# ── C) Schedule window class per business-day ──
print("\n   ── C) Daily window derivation (start, end) → class + duration ──")
day_rows = reduce(lambda l, r: l.unionByName(r), [
    df_hours.select("business_id", F.lit(d).alias("day"), F.col(HOURS_MAP[d]).alias("raw"),
                    _mins(F.col(f"{HOURS_MAP[d]}_start_time")).alias("s"), _mins(F.col(f"{HOURS_MAP[d]}_end_time")).alias("e"))
    .where(F.col("raw").isNotNull()) for d in DAYS])
win_class = (F.when((F.col("s") == 0) & (F.col("e") == 0), "00:00-00:00 (all-day / unknown)")
              .when(F.col("s") == F.col("e"), "same non-midnight time")
              .when(F.col("e") > F.col("s"), "normal (end > start)")
              .when(F.col("e") < F.col("s"), "overnight (wraps midnight)")
              .otherwise("unparseable"))
duration = (F.when(F.col("s") == F.col("e"), F.lit(1440))
             .when(F.col("e") > F.col("s"), F.col("e") - F.col("s"))
             .otherwise(F.col("e") + 1440 - F.col("s")))
day_rows = day_rows.withColumn("win_class", win_class).withColumn("dur_min", duration)
cls_pd = (day_rows.groupBy("win_class").agg(F.count("*").alias("business_days"), F.countDistinct("business_id").alias("businesses"),
                                            F.round(F.avg("dur_min") / 60, 2).alias("avg_hours"))
          .orderBy(F.desc("business_days")).toPandas())
cls_pd["share %"] = (cls_pd["business_days"] / cls_pd["business_days"].sum() * 100).round(2)
display(cls_pd)
C2 = [chk("df_hours", "(start, end) parseable", "every non-NULL day derives a window class", F.col("win_class") == "unparseable", ex="raw"),
      chk("df_hours", "same non-midnight start/end", "start == end only allowed as the 00:00-00:00 convention",
          F.col("win_class") == "same non-midnight time", ex=F.concat_ws(" ", "business_id", "day", "raw"),
          note="e.g. '9:0-9:0': zero-length or 24 h cannot be decided → derived duration is undefined"),
      chk("df_hours", "00:00-00:00 days", "derived class is unambiguous", F.col("win_class").startswith("00:00"), status="INFO",
          note="all-day vs 'hours unknown' convention (DQ-BUSINESS-007 / 025); duration derived as 24 h here"),
      chk("df_hours", "daily duration", "derived duration ∈ (0, 24 h]", ~F.col("dur_min").between(1, 1440), ex=F.concat_ws(" ", "day", "raw"))]
v2c = run_checks(findings_662, day_rows, C2)
cc_metrics["same_time_days"] = v2c["same non-midnight start/end"]
cc_metrics["midnight_days"] = v2c["00:00-00:00 days"]

# ── D) Contract naming logic (metadata) ──
print("\n   ── D) Contract naming derivations ──")
_ident = re.compile(r"^[a-z][a-z0-9_]*$")
all_attr_cols = list(ATTR_BOOLEAN_MAP.values()) + list(ATTR_INT_MAP.values()) + list(ATTR_ENUM_MAP.values())
bad_ident = [c for c in all_attr_cols if not (_ident.match(c) and c.startswith("attr_"))]
dq_add(findings_662, "df_attributes", "column names", "'attr_' + snake_case, unique", len(all_attr_cols),
       len(bad_ident) + (len(all_attr_cols) - len(set(all_attr_cols))), examples=", ".join(bad_ident))
key_pairs = [(k, c) for _, (_, km) in DICT_TABLE_MAP.items() for k, c in km.items()]
bad_keys = [f"{k}→{c}" for k, c in key_pairs if c != k.replace("-", "_") or not _ident.match(c)]
dq_add(findings_662, "7 dict frames", "key → silver column", "column == key with '-' → '_' (valid identifier)", len(key_pairs), len(bad_keys),
       examples=", ".join(bad_keys))
bad_hours = [f"{d}→{c}" for d, c in HOURS_MAP.items() if c != f"hours_{d.lower()}"]
dq_add(findings_662, "df_hours", "day → column", "column == 'hours_' + lower(day)", len(HOURS_MAP), len(bad_hours), examples=", ".join(bad_hours))

# ── E) Derived indicator: no_music ⇔ NOT any music type ──
print("\n   ── E) Derived indicator logic: Music.no_music ──")
music_types = [k for k in DICT_TABLE_MAP["Music"][1] if k != "no_music"]
any_music = reduce(lambda a, b: a | b, [F.coalesce(dk("Music", k), F.lit(False)) for k in music_types])
all_answered = reduce(lambda a, b: a & b, [dk("Music", k).isNotNull() for k in music_types])
E2 = [chk("df_music", "no_music vs music types", "no_music == NOT(any of dj / background_music / jukebox / live / video / karaoke)",
          dk("Music", "no_music") != ~any_music, scope=dk("Music", "no_music").isNotNull() & all_answered,
          ex=F.col("d_Music").cast("string"),
          note="evaluated only where all six types are answered; DQ-BUSINESS-041 (no_music never TRUE)")]
v2e = run_checks(findings_662, cc_wide, E2)
cc_metrics["no_music_bad"] = v2e["no_music vs music types"]

findings_662_df = pd.DataFrame(findings_662)
print(f"\n   {len(findings_662_df)} derived-logic checks | {int((findings_662_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((findings_662_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


---
#### 📝 6.6.2 Derived-Column Logic — Re-Detected Errors (no new ID)

The two FINDINGs in **6.6.2 Derived-Column Logic Checks (Cell 104)** are errors that are already documented. The 6.6 view adds detail, so the existing notes are updated here instead of being given new numbers.

| Check (Cell 104) | Violations | Existing Error | What 6.6 adds |
| --- | --- | --- | --- |
| C) `same non-midnight start/end` (start == end, not `0:0-0:0`) | **1,492** business-days, 238 businesses | **DQ-BUSINESS-007** (zero-length ranges) | Most of these values are **not random**. `6:0-6:0` (600 days / 93 businesses), `5:0-5:0` (396 / 58) and `9:0-9:0` (334 / 49) make up 89%. This is the pattern a 24-hour business produces when its operating day is entered as starting at 05:00 or 06:00 (e.g. White Castle `5:0-5:0` every day, flagged `latenight`). The derived duration is undefined (0 h or 24 h) |
| E) `no_music == NOT(any music type)` | **2,876** of 5,226 Music dicts where all six music types are answered | **DQ-BUSINESS-041** | The derivation rule a Silver `has_music` / `no_music` indicator would rely on fails for 55% of fully answered dicts. This confirms that `no_music` cannot be used as an indicator |

**Updated recommendation for DQ-BUSINESS-007 (same-time ranges):**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Add `is_same_time_range` in Silver; set the derived duration to NULL; additionally set `likely_24h = TRUE` when the same start = end value is repeated on ≥ 5 days of the week** | The repeated-boundary pattern is a strong but not certain 24 h signal. Flagging keeps the raw value and makes the inference explicit and reversible |
| 2 | Treat every start = end as 24 h | Correct for most of the 5:00 / 6:00 cases, but wrong for isolated values such as `0:30-0:30` or `12:0-12:0`, which look like data-entry errors |
| 3 | Treat every start = end as closed | Removes 24 h businesses from "open now" and late-night analyses |

**Best Logical Solution:** **Option 1.** It keeps the source value, adds a documented heuristic only where the weekly pattern supports it, and leaves single-day cases unknown.

> **Reviewed and not raised in 6.6.1 / 6.6.2:** all 81 logical-relationship checks in Cell 103 pass. Every TE.2 value (scalars, 32 attributes, 21 hour columns, category list and order, and key/TRUE/FALSE/NULL counts of the 7 nested dicts) equals an independent re-derivation from bronze, so the restructuring is lossless row by row. `business_id` is a 22-char hash in all 11 frames, `stars` is always on the half-star grid, and all contract naming rules hold. The 73,908 `0:0-0:0` days (INFO) are the existing DQ-BUSINESS-007 / 025 ambiguity.

---

In [0]:
# 6.6.3) Logical Contradiction Detection  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Mutually exclusive states inside ONE business profile (owner / crowd answers that cannot both be true).
# Severity reflects how hard the exclusion is:
#   FINDING = logically impossible  (21+ venue that is good for kids, a rating without reviews, …)
#   INFO    = commercially unlikely but possible (upscale ambience at the $ tier, …) — profiled, not an error
#   A) Rating        stars present but review_count = 0
#   B) Audience      AgesAllowed 21plus / 18plus vs GoodForKids
#   C) Positioning   RestaurantsAttire / Ambience / price tier / table service
#   D) Music         no_music = TRUE together with a music type (DQ-BUSINESS-041) ; all-FALSE dicts
#   E) Operations    ByAppointmentOnly vs Open24Hours ; DriveThru vs TakeOut ; Delivery vs TakeOut ;
#                    Smoking = 'outdoor' vs OutdoorSeating = FALSE
#   F) Diet          vegan TRUE but vegetarian FALSE (DQ-BUSINESS-020, re-checked across the frames)
#   G) Identity      name identical to address / city (field shifted into the wrong column)
from functools import reduce
from pyspark.sql import functions as F
import pandas as pd

print(f"\n{'─' * 60}\n📌 6.6.3) Logical Contradiction Detection — mutually exclusive states within a business")
findings_663 = []

T_ = lambda c: F.coalesce(c, F.lit(False))                      # answered TRUE
F_ = lambda c: F.coalesce(~c, F.lit(False))                     # answered FALSE
att, ages, kids = F.col("attr_restaurants_attire"), F.col("attr_ages_allowed"), F.col("attr_good_for_kids")
price = F.col("attr_restaurants_price_range_2")
music_types = [k for k in DICT_TABLE_MAP["Music"][1] if k != "no_music"]
any_music = reduce(lambda a, b: a | b, [T_(dk("Music", k)) for k in music_types])
music_all_false = reduce(lambda a, b: a & b, [F_(dk("Music", k)) for k in DICT_TABLE_MAP["Music"][1]])
FOOD_SERVICE = has_cat("^(Restaurants|Fast Food|Coffee & Tea|Cafes|Donuts|Burgers|Ice Cream & Frozen Yogurt|Juice Bars & Smoothies|Bagels)$")
EX_A = lambda *cs: F.concat_ws(" | ", "business_id", "name", *[F.concat_ws("=", F.lit(n), c.cast("string")) for n, c in cs])

checks_663 = [
    # A) rating
    chk("df_business", "stars ∧ review_count = 0", "a rating requires ≥ 1 review", F.col("review_count") == 0,
        scope=F.col("stars").isNotNull(), ex=EX_A(("stars", F.col("stars")), ("rc", F.col("review_count")))),
    # B) audience
    chk("df_attributes", "ages_allowed 21plus/18plus ∧ good_for_kids", "adults-only venue cannot be good for kids",
        T_(kids), scope=ages.isin("21plus", "18plus"), ex=EX_A(("ages", ages), ("kids", kids))),
    chk("df_attributes", "ages_allowed allages ∧ NOT good_for_kids", "all-ages venue answered 'not good for kids'",
        F_(kids), scope=ages == "allages", status="INFO", note="admission vs suitability — possible (e.g. bars admitting minors)"),
    # C) positioning
    chk("df_attributes + df_ambience", "attire formal ∧ ambience casual/divey", "formal dress code excludes casual / divey ambience",
        T_(dk("Ambience", "casual")) | T_(dk("Ambience", "divey")), scope=att == "formal",
        ex=EX_A(("attire", att), ("ambience", F.col("d_Ambience")))),
    chk("df_attributes + df_categories", "attire formal ∧ category Fast Food", "a fast-food outlet has no formal dress code",
        has_cat("^Fast Food$"), scope=att == "formal", ex=EX_A(("attire", att)), note="chain units answered 'formal' (Subway, Taco Bell, Culver's …)"),
    chk("df_ambience", "divey ∧ (classy ∨ upscale)", "a dive is neither classy nor upscale",
        T_(dk("Ambience", "classy")) | T_(dk("Ambience", "upscale")), scope=T_(dk("Ambience", "divey")),
        ex=EX_A(("ambience", F.col("d_Ambience")))),
    chk("df_ambience + df_attributes", "upscale ∧ price tier $", "upscale ambience at the lowest price tier",
        price == 1, scope=T_(dk("Ambience", "upscale")), status="INFO", ex=EX_A(("price", price))),
    chk("df_ambience + df_attributes", "divey ∧ price tier $$$$", "divey ambience at the highest price tier",
        price == 4, scope=T_(dk("Ambience", "divey")), status="INFO", ex=EX_A(("price", price))),
    chk("df_attributes", "attire formal ∧ NOT table_service", "formal dining without table service",
        F_(F.col("attr_restaurants_table_service")), scope=att == "formal", status="INFO"),
    # D) music
    chk("df_music", "no_music ∧ any music type", "no_music = TRUE excludes dj / background / jukebox / live / video / karaoke",
        any_music, scope=T_(dk("Music", "no_music")), ex=F.col("d_Music").cast("string")),
    chk("df_music", "all music keys FALSE (incl. no_music)", "dict states neither 'some music' nor 'no music'",
        music_all_false, scope=F.col("d_Music").isNotNull() & (F.size("d_Music") > 0), status="INFO",
        note="DQ-BUSINESS-041 — FALSE likely means 'not answered' in this dict"),
    # E) operations
    chk("df_attributes", "by_appointment_only ∧ open_24_hours", "appointment-only business cannot be open 24 h",
        T_(F.col("attr_open_24_hours")), scope=T_(F.col("attr_by_appointment_only")),
        ex=EX_A(("appt", F.col("attr_by_appointment_only")), ("24h", F.col("attr_open_24_hours")))),
    chk("df_attributes + df_categories", "drive_thru ∧ NOT take_out (food businesses)", "a food drive-thru IS take-out",
        F_(F.col("attr_restaurants_take_out")), scope=T_(F.col("attr_drive_thru")) & FOOD_SERVICE,
        ex=EX_A(("drive_thru", F.col("attr_drive_thru")), ("take_out", F.col("attr_restaurants_take_out"))),
        note="scoped to restaurants / coffee / fast food: pharmacy & bank drive-thrus legitimately have take_out = FALSE"),
    chk("df_attributes", "delivery ∧ NOT take_out", "delivers food but no pick-up", F_(F.col("attr_restaurants_take_out")),
        scope=T_(F.col("attr_restaurants_delivery")), status="INFO", note="possible for delivery-only / ghost kitchens and caterers"),
    chk("df_attributes", "smoking outdoor ∧ NOT outdoor_seating", "outdoor smoking area implies an outdoor area",
        F_(F.col("attr_outdoor_seating")), scope=F.col("attr_smoking") == "outdoor", status="INFO",
        note="a patio may exist without seating — soft rule"),
    # F) diet
    chk("df_dietary_restrictions", "vegan ∧ NOT vegetarian", "every vegan option is vegetarian", F_(dk("DietaryRestrictions", "vegetarian")),
        scope=T_(dk("DietaryRestrictions", "vegan")), ex=F.col("d_DietaryRestrictions").cast("string"), note="DQ-BUSINESS-020"),
    # G) identity fields
    chk("df_business", "name == address", "business name copied into the address field", F.lower(F.trim("name")) == F.lower(F.trim("address")),
        scope=F.col("address") != "", status="INFO", ex=EX_A(("address", F.col("address")))),
    chk("df_business", "name == city", "business name equals the city name", F.lower(F.trim("name")) == F.lower(F.trim("city")),
        status="INFO", ex=EX_A(("city", F.col("city")))),
]
v663 = run_checks(findings_663, cc_wide, checks_663, ex_default=EX_BIZ)
cc_metrics.update({"adult_kids": v663["ages_allowed 21plus/18plus ∧ good_for_kids"],
                   "formal_casual": v663["attire formal ∧ ambience casual/divey"],
                   "divey_classy": v663["divey ∧ (classy ∨ upscale)"],
                   "appt_24h": v663["by_appointment_only ∧ open_24_hours"],
                   "drive_thru_no_takeout": v663["drive_thru ∧ NOT take_out (food businesses)"]})

findings_663_df = pd.DataFrame(findings_663)
print("\n   📋 Contradiction matrix (scope = rows where the premise holds):")
display(findings_663_df[["DataFrame", "Column", "Rule", "Checked", "Violations", "Pct", "Status", "Note"]])
print(f"\n   {len(findings_663_df)} contradiction checks | {int((findings_663_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((findings_663_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


---
#### 🚨 DQ-BUSINESS-042 — `RestaurantsAttire = 'formal'` on Fast-Food and Casual Venues

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-042` |
| **Severity** | MEDIUM (logical contradiction; invalidates the top tier of an ordinal attribute) |
| **Detected In** | Phase 6.6 — Cross-Column Consistency Checks → **6.6.3 Logical Contradiction Detection** (Cell 106): checks `attire formal ∧ category Fast Food` (24) and `attire formal ∧ ambience casual/divey` (3) |
| **Also Confirmed In** | Same cell, INFO `attire formal ∧ NOT table_service` (11 of 70, incl. Domino's and Taco Bell). Independent SQL on bronze: 25 distinct businesses meet either rule |
| **Affected Columns** | `attributes.RestaurantsAttire` → `df_attributes.attr_restaurants_attire`; cross-checked with `df_categories.category_name` and `df_ambience` (`casual`, `divey`) |
| **Total Affected Rows** | **25** of the **70** businesses with `formal` attire (**36%** of the whole `formal` class) |
| **Invalid Value** | `Subway` / `McDonald's` / `Church's Chicken` / `Culver's` with `attire = formal`. `Subway` and `Culver's` also have `Ambience.casual = True` |
| **Domain Rule** | `formal` is the highest dress-code tier (jacket required). A fast-food counter or a casual / divey place cannot also require formal dress |

**Description:** `formal` is a rare value (70 businesses), and more than a third of it is contradicted by the business's own category or ambience. Most hits are units of national fast-food chains. This suggests a wrong drop-down choice by the owner, or a bulk chain-level default, rather than a real dress code. Because the class is so small, these errors noticeably distort any "formal dining" segment and the ordinal `casual < dressy < formal` feature.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep the raw value; add `attire_conflict_flag` in Silver; in Silver Enriched expose `attr_restaurants_attire_validated` = NULL when the flag is set** | The source stays auditable, analysts get a consistent field, and no unsupported value (e.g. `casual`) is invented |
| 2 | **Recode the 25 to `casual`** | Probably right for fast food, but it asserts a value the source never gave |
| 3 | **Set the attribute to NULL in Silver Cleaned** | Consistent, but loses lineage of what the owner reported |
| 4 | **Do nothing** | 36% of the `formal` tier stays wrong |

**Best Logical Solution:** **Option 1 — flag and expose a validated column.** The contradiction is certain, but the correct value is not, so NULL-on-conflict in the enriched layer is the safest outcome.

---

---
#### 🚨 DQ-BUSINESS-043 — Adults-Only Venues (`AgesAllowed` 21plus / 18plus) Marked `GoodForKids = True`

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-043` |
| **Severity** | LOW–MEDIUM (logical contradiction; small volume but affects family-suitability filters) |
| **Detected In** | Phase 6.6 — Cross-Column Consistency Checks → **6.6.3 Logical Contradiction Detection** (Cell 106): check `ages_allowed 21plus/18plus ∧ good_for_kids` |
| **Also Confirmed In** | Independent SQL on bronze: 3 rows with `21plus` + `GoodForKids = True` (+1 with `18plus` = 4). The reverse soft rule (`allages` ∧ NOT good_for_kids, 5) is INFO only |
| **Affected Columns** | `attributes.AgesAllowed` → `attr_ages_allowed`; `attributes.GoodForKids` → `attr_good_for_kids` |
| **Total Affected Rows** | **4** of **52** adults-only businesses (7.7%) |
| **Invalid Value** | `Morgan's Pier` (21plus, kids = True), `Bollywood Beats` (18plus, kids = True), `Red Door Chinese Eatery` (21plus, kids = True) |
| **Domain Rule** | A venue that admits only adults (18+ / 21+) cannot be suitable for children |

**Description:** Both attributes describe the audience. `AgesAllowed` is an admission rule, so it is the stronger statement. A Chinese restaurant tagged `21plus` suggests the *age* answer is the wrong one in some cases, so neither field can be trusted on its own.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Flag (`audience_conflict_flag`); in Silver Enriched set a derived `family_friendly` = NULL for flagged rows** | The contradiction is certain, the true side is not (restaurant with `21plus` vs pier bar with kids = True) |
| 2 | **Give `AgesAllowed` precedence and set `GoodForKids = FALSE`** | An admission rule normally beats a suitability opinion, but this is wrong for misentered ages (`Red Door Chinese Eatery`) |
| 3 | **Do nothing** | 4 adults-only venues appear in "good for kids" searches |

**Best Logical Solution:** **Option 1 — flag and NULL the derived audience field.** At 4 rows it costs nothing and avoids guessing.

---

---
#### 🚨 DQ-BUSINESS-044 — `Ambience` Simultaneously `divey` and `classy`

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-044` |
| **Severity** | LOW–MEDIUM (mutually exclusive flags inside one nested dict) |
| **Detected In** | Phase 6.6 — Cross-Column Consistency Checks → **6.6.3 Logical Contradiction Detection** (Cell 106): check `divey ∧ (classy ∨ upscale)` |
| **Also Confirmed In** | Independent SQL on the raw `attributes.Ambience` string: 108 with `divey` + `classy`, **0** with `divey` + `upscale`. INFO checks in the same cell: `divey ∧ $$$$` (3) and `upscale ∧ $` (3) |
| **Affected Columns** | `attributes.Ambience['divey']`, `['classy']` → `df_ambience` (`key` = divey / classy) / `yelp_business_ambience.divey`, `.classy` |
| **Total Affected Rows** | **108** of **1,468** businesses with `divey = True` (7.4%) |
| **Invalid Value** | `The Howlin' Wolf` and `The Blarney Stone`: `{divey: True, classy: True, casual: True/False, upscale: False}` |
| **Domain Rule** | "Divey" (rough, cheap, unpretentious) and "classy" (refined, elegant) are opposite ends of the same scale |

**Description:** Ambience flags come from crowd votes, and Yelp shows a key as TRUE once enough reviewers pick it, so opposing tags can both pass the threshold. The pattern is concentrated in bars and music venues. Because `upscale` never co-occurs with `divey`, reviewers seem to use "classy" loosely ("a classy dive"), so this is a weak signal rather than corrupted data. It still breaks a one-hot or "dominant ambience" feature.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep both flags; add `ambience_conflict_flag`; when Silver Enriched derives one `primary_ambience`, return `mixed` for flagged rows** | Crowd-sourced flags are opinions, so both are real observations. A derived single label must not pick one arbitrarily |
| 2 | **Drop `classy` when `divey` is TRUE (or vice versa)** | Consistent, but an arbitrary precedence between two crowd opinions |
| 3 | **Set both to NULL** | Removes valid information about the 108 venues |

**Best Logical Solution:** **Option 1 — keep, flag, and resolve only in the derived label.**

---

---
#### 🚨 DQ-BUSINESS-045 — Food Drive-Thrus With `RestaurantsTakeOut = False` (Chain-Level Default)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-045` |
| **Severity** | MEDIUM (logical contradiction; systematic, chain-wide) |
| **Detected In** | Phase 6.6 — Cross-Column Consistency Checks → **6.6.3 Logical Contradiction Detection** (Cell 106): check `drive_thru ∧ NOT take_out (food businesses)` |
| **Also Confirmed In** | The unscoped version of the rule gave 536 hits, mostly pharmacy drive-thrus (e.g. CVS Pharmacy). Those are legitimate and were removed by restricting the rule to restaurant / coffee / fast-food categories. Independent SQL by business name confirms the chain concentration below |
| **Affected Columns** | `attributes.DriveThru` → `attr_drive_thru`; `attributes.RestaurantsTakeOut` → `attr_restaurants_take_out`; scope from `df_categories` |
| **Total Affected Rows** | **162** of **3,674** food businesses with a drive-thru (4.4%). **Jack in the Box (93)** and **Burger King (34)** make up 78% |
| **Invalid Value** | `Jack in the Box` / `Burger King` / `Chick-fil-A` / `Starbucks`: `DriveThru = True`, `RestaurantsTakeOut = False` |
| **Domain Rule** | Ordering at a food drive-thru *is* take-out, so `DriveThru = True` ⇒ `RestaurantsTakeOut ≠ False` for food businesses |

**Description:** The concentration in two chains points to a bulk / chain-level attribute load in which take-out was defaulted to FALSE, not to individual owner mistakes. Any "offers take-out" analysis under-counts these chains, and models learn a false negative correlation between the two flags.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Silver Enriched: derive `offers_take_out = TRUE` when `DriveThru = TRUE` for food categories; keep the raw `attr_restaurants_take_out`; add `takeout_inferred_flag`** | The implication is a business-logic certainty (drive-thru ⊆ take-out), so the inference is safe. The flag keeps it transparent |
| 2 | **Overwrite `RestaurantsTakeOut = TRUE` in Silver Cleaned** | Same result, but loses the record of what the source said |
| 3 | **NULL `RestaurantsTakeOut` for the 162** | Safe, but discards a value that can be inferred with certainty |
| 4 | **Do nothing** | Major chains appear as "no take-out" |

**Best Logical Solution:** **Option 1 — rule-based inference in the enriched layer with a flag.** This is one of the few contradictions where the correct value follows logically.

---

> **Cross-references from 6.6.3 (already documented, no new ID):** `vegan ∧ NOT vegetarian` (3 of 14) → **DQ-BUSINESS-020** (count confirmed across frames). `all music keys FALSE incl. no_music` (2,876 of 7,417) → **DQ-BUSINESS-041**.
>
> **Reviewed and not raised (plausible business reality, INFO):** `delivery ∧ NOT take_out` 1,586 (Walmart / CVS delivery, ghost kitchens), `smoking outdoor ∧ NOT outdoor_seating` 293 (patio without seating), `allages ∧ NOT good_for_kids` 5, `upscale ∧ $` 3, `divey ∧ $$$$` 3 (gentlemen's clubs), `name == address` 6 / `name == city` 11 (e.g. a business literally named after its street or town). Passed: 0 ratings without reviews, 0 `no_music = TRUE` with music, 0 appointment-only + 24 h.

---

In [0]:
# 6.6.4) Cross-Column Range Logic  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# The valid range of one column DEPENDS on another column of the same business:
#   A) Geography   (latitude, longitude) must lie inside the bounding box of the labelled `state`
#                  (6.2.2 only tested a fixed metro footprint — a Tampa point labelled 'PA' passed there)
#   B) Geography   (latitude, longitude) within 50 km of the median point of businesses sharing its ZIP5 / FSA
#   C) Schedule    end_time vs start_time of the same day: an overnight window (end < start) should end in the
#                  early morning (≤ 06:00); later "overnight" ends signal AM/PM swaps or 18 h+ windows
#   D) Schedule    no overlap between days: an overnight window on day d must end before day d+1 opens
#   E) Schedule    a same-day window > 20 h that is not the 00:00-00:00 convention ('0:0-23:59', '6:0-5:0', …)
#   F) Profile     stars at the extremes (1.0 / 5.0) given review_count ≥ 100 (rounded mean of many reviews)
from functools import reduce
from pyspark.sql import functions as F, Window
import pandas as pd

print(f"\n{'─' * 60}\n📌 6.6.4) Cross-Column Range Logic")
findings_664 = []

# ── A) state bounding boxes (lat_min, lat_max, lon_min, lon_max), the 14 footprint states/provinces ──
STATE_BBOX = {
    "PA": (39.72, 42.27, -80.52, -74.69), "NJ": (38.93, 41.36, -75.56, -73.89), "DE": (38.45, 39.84, -75.79, -75.05),
    "FL": (24.40, 31.00, -87.63, -80.03), "IN": (37.77, 41.76, -88.10, -84.78), "TN": (34.98, 36.68, -90.31, -81.65),
    "AZ": (31.33, 37.00, -114.82, -109.05), "LA": (28.93, 33.02, -94.04, -88.82), "NV": (35.00, 42.00, -120.01, -114.04),
    "CA": (32.53, 42.01, -124.41, -114.13), "ID": (41.99, 49.00, -117.24, -111.04), "MO": (35.99, 40.61, -95.77, -89.10),
    "IL": (36.97, 42.51, -91.51, -87.49), "AB": (49.00, 60.00, -120.00, -110.00)}
TOL = 0.05                                                       # ≈ 5 km tolerance on every edge
in_box = F.lit(False)
for s_, (la0, la1, lo0, lo1) in STATE_BBOX.items():
    in_box = in_box | ((F.col("state") == s_) & F.col("latitude").between(la0 - TOL, la1 + TOL) & F.col("longitude").between(lo0 - TOL, lo1 + TOL))
EX_GEO = F.concat_ws(" | ", "name", "city", "state", "postal_code", F.format_string("%.4f,%.4f", "latitude", "longitude"))

print("\n   ── A) Coordinates vs labelled state ──")
vA = run_checks(findings_664, df_business, [
    chk("df_business", "(lat, lon) ∈ bbox(state)", "point lies inside the labelled state's bounding box (±0.05°)", ~in_box,
        scope=F.col("state").isin(list(STATE_BBOX)) & F.col("latitude").isNotNull(), ex=EX_GEO,
        note="state or coordinates wrong; states outside the 14-code footprint (XMS etc.) are DQ-BUSINESS-009/010")])
cc_metrics["geo_state_bbox"] = vA["(lat, lon) ∈ bbox(state)"]

# ── B) distance to ZIP centroid ──
def hav_km(la1, lo1, la2, lo2):
    a = (F.pow(F.sin(F.radians(la2 - la1) / 2), 2)
         + F.cos(F.radians(la1)) * F.cos(F.radians(la2)) * F.pow(F.sin(F.radians(lo2 - lo1) / 2), 2))
    return F.lit(2 * 6371.0088) * F.asin(F.sqrt(a))

zip5 = F.upper(F.regexp_replace("postal_code", r"\s", ""))
zip5 = F.when(zip5.rlike(r"^\d{5}"), F.substring(zip5, 1, 5)).when(zip5.rlike(r"^[A-Z]\d[A-Z]"), F.substring(zip5, 1, 3))
zb = df_business.select("business_id", "name", "city", "state", "postal_code", "latitude", "longitude", zip5.alias("zip5")).where(F.col("zip5").isNotNull())
zc = zb.groupBy("zip5").agg(F.count("*").alias("zip_n"), F.percentile_approx("latitude", 0.5).alias("c_lat"),
                            F.percentile_approx("longitude", 0.5).alias("c_lon"))
zd = zb.join(zc, "zip5").withColumn("km", hav_km(F.col("latitude"), F.col("longitude"), F.col("c_lat"), F.col("c_lon")))
print("\n   ── B) Coordinates vs postal code ──")
vB = run_checks(findings_664, zd, [
    chk("df_business", "(lat, lon) ≤ 50 km from ZIP median", "point within 50 km of the median point of its ZIP5 / FSA (≥ 3 businesses)",
        F.col("km") > 50, scope=F.col("zip_n") >= 3, ex=F.concat_ws(" | ", "name", "zip5", "state", F.format_string("%.0f km", "km")),
        note="either the ZIP or the geocode is wrong (ZIP typo, or default/centroid geocode)")])
cc_metrics["geo_zip_far"] = vB["(lat, lon) ≤ 50 km from ZIP median"]
print("   Distance-to-ZIP-median distribution (km):")
display(zd.where("zip_n >= 3").select(*[F.round(F.percentile_approx("km", q), 2).alias(f"p{q * 100:g}") for q in (0.5, 0.9, 0.99, 0.999)],
                                      F.round(F.max("km"), 1).alias("max")))

# ── C–E) schedule ranges ──
print("\n   ── C–E) Opening-window ranges ──")
days_long = reduce(lambda l, r: l.unionByName(r), [
    df_hours.select("business_id", F.lit(i).alias("dow"), F.lit(d).alias("day"), F.col(HOURS_MAP[d]).alias("raw"),
                    START_M[d].alias("s"), END_M[d].alias("e"),
                    START_M[DAYS[(i + 1) % 7]].alias("next_s"), END_M[DAYS[(i + 1) % 7]].alias("next_e"),
                    F.col(HOURS_MAP[DAYS[(i + 1) % 7]]).alias("next_raw"))
    .where(F.col("raw").isNotNull()) for i, d in enumerate(DAYS)])
overnight = F.col("e") < F.col("s")
next_real = F.col("next_raw").isNotNull() & ~((F.col("next_s") == 0) & (F.col("next_e") == 0))
EX_DAY = F.concat_ws(" ", "business_id", "day", "raw", F.lit("→ next:"), F.coalesce("next_raw", F.lit("closed")))
vC = run_checks(findings_664, days_long, [
    chk("df_hours", "overnight end ≤ 06:00", "end < start (wraps midnight) ⇒ end in the early morning", F.col("e") > 360,
        scope=overnight, ex=EX_DAY, note="e.g. '17:0-11:0' / '11:0-10:0' → 18–23 h windows or AM/PM swaps"),
    chk("df_hours", "overnight end ≤ next day's start", "window spilling past midnight must close before the next day opens",
        F.col("e") > F.col("next_s"), scope=overnight & next_real, ex=EX_DAY,
        note="overlapping consecutive days → double-counted open time"),
    chk("df_hours", "same-day window ≤ 20 h", "end > start, duration ≤ 20 h unless the 00:00-00:00 convention",
        (F.col("e") - F.col("s")) > 1200, scope=F.col("e") > F.col("s"), status="INFO", ex=EX_DAY,
        note="mostly '0:0-23:59' / '0:0-23:0' = alternative all-day spellings (DQ-BUSINESS-025)")])
cc_metrics.update({"overnight_late_end": vC["overnight end ≤ 06:00"], "day_overlap": vC["overnight end ≤ next day's start"]})

# ── F) rating extremes vs review volume ──
print("\n   ── F) Rating extremes vs review volume ──")
run_checks(findings_664, df_business, [
    chk("df_business", "stars ∈ {1.0, 5.0} ∧ review_count ≥ 100", "a mean of ≥ 100 reviews rarely rounds to an extreme",
        F.col("stars").cast("double").isin(1.0, 5.0), scope=F.col("review_count") >= 100, status="INFO",
        ex=F.concat_ws(" | ", "name", "city", F.col("stars").cast("string"), F.col("review_count").cast("string")),
        note="possible (≥ 4.75 average) — profile for rating-manipulation review")])
print("   stars × review_count bucket (share of 1.0 / 5.0 ratings):")
rc_bucket = (F.when(F.col("review_count") < 10, "a) <10").when(F.col("review_count") < 50, "b) 10-49")
              .when(F.col("review_count") < 100, "c) 50-99").when(F.col("review_count") < 500, "d) 100-499").otherwise("e) 500+"))
display(df_business.groupBy(rc_bucket.alias("review_count_bucket")).agg(
    F.count("*").alias("businesses"),
    F.round(F.avg((F.col("stars") == 5).cast("int")) * 100, 2).alias("pct_5_0"),
    F.round(F.avg((F.col("stars") == 1).cast("int")) * 100, 2).alias("pct_1_0"),
    F.round(F.avg("stars"), 2).alias("avg_stars")).orderBy("review_count_bucket"))

findings_664_df = pd.DataFrame(findings_664)
print(f"\n   {len(findings_664_df)} range checks | {int((findings_664_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((findings_664_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


---
#### 🚨 DQ-BUSINESS-046 — Address Block (city / state / ZIP) Contradicts the Coordinates

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-046` |
| **Severity** | HIGH (accuracy; the business is placed in a different state or metro than its address) |
| **Detected In** | Phase 6.6 — Cross-Column Consistency Checks → **6.6.4 Cross-Column Range Logic** (Cell 111): A) `(lat, lon) ∈ bbox(state)` (18) and B) `(lat, lon) ≤ 50 km from ZIP median` (4) |
| **Also Confirmed In** | 6.6.6 D) `metro(lat, lon) → state` (Cell 121): 15 in-footprint conflicts such as `Los Angeles CA 90041` and `Tampa FL 33609` with Philadelphia coordinates, or `Chattanooga TN` / `Salinas CA` with Tampa coordinates. Independent SQL with the same boxes and ZIP medians: **20** distinct businesses (18 ∪ 4, 2 in both). Most of them passed the metro-footprint check in 6.2.2 because the point does lie inside a Yelp metro. Only the far-away cases (e.g. `Broadway Veterinary Hospital`, 626 km from any metro) overlap DQ-BUSINESS-011 |
| **Affected Columns** | `latitude`, `longitude` vs `state`, `postal_code`, `city` (`df_business`) |
| **Total Affected Rows** | **20** businesses (18 of 150,325 outside the bounding box of their own state; 4 of 149,839 more than 50 km from the median point of their ZIP, up to 1,899 km) |
| **Invalid Value** | `Hellz Bellz · Los Angeles, CA 90041 · 39.661, -75.570` (Delaware); `Loft 17 · Tampa, FL 33609 · 40.042, -75.481` (Philadelphia suburbs, 1,492 km from its ZIP); `Intensity Fit Club · Tampa, CA 33613 · 28.076, -82.416` (correct Tampa point, wrong state) |
| **Domain Rule** | Coordinates, state and ZIP describe one location. The point must lie in the labelled state and near the other businesses with the same ZIP |

**Description:** Two patterns:

| Pattern | Example | Which field is wrong |
| --- | --- | --- |
| Coordinates in a Yelp metro, the whole address block from somewhere else | `Los Angeles CA 90041` / `Tampa FL 33609` at Philadelphia coordinates | Usually the **address** (Yelp's footprint is defined by the metros, so the geocode is the more reliable witness). Possibly a chain's HQ / default address copied into the listing |
| Address and coordinates agree, only `state` differs | `Tampa, CA 33613` at a Tampa point | **`state`** (overlaps DQ-BUSINESS-023) |

These rows put businesses into the wrong state for every state-level aggregate, and into the wrong metro for distance or "nearby" features.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Flag `geo_address_conflict`; adjudicate by majority of three witnesses (coordinates, ZIP prefix, city): correct `state` when ZIP + city + coordinates agree against it; otherwise keep values and set the flag** | Uses independent evidence, fixes only the field that is outvoted, and leaves unresolved cases visible |
| 2 | **Reverse-geocode the coordinates and overwrite city / state / ZIP** | Most complete fix, but needs an external service and assumes the geocode is always right (not true for DQ-BUSINESS-040 placeholder centroids) |
| 3 | **Quarantine the 20 businesses** | Clean, but removes reviewed businesses for an error that is often fixable |
| 4 | **Do nothing** | 20 businesses in the wrong state / metro |

**Best Logical Solution:** **Option 1 — flag, then correct only by a 2-of-3 witness rule.** It is deterministic, auditable and needs no external dependency. Reverse geocoding (Option 2) can be added later for the rows still unresolved.

---

---
#### 🚨 DQ-BUSINESS-047 — False "Overnight" Windows: Inverted Start/End and 12-Hour-Clock Entries

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-047` |
| **Severity** | MEDIUM (wrong opening hours; 18–23 h windows computed for normal businesses) |
| **Detected In** | Phase 6.6 — Cross-Column Consistency Checks → **6.6.4 Cross-Column Range Logic** (Cell 111): C) `overnight end ≤ 06:00` |
| **Also Confirmed In** | Independent SQL split on `[-:]` over the raw bronze strings: the same 339 business-days. DQ-BUSINESS-007 treated every end < start range as a genuine overnight window that "can be computed". This check shows a subset that is not overnight at all |
| **Affected Columns** | `hours.<Day>` → `df_hours.hours_<day>`, `_start_time`, `_end_time` |
| **Total Affected Rows** | **339** of 58,501 overnight business-days (0.58%), at most **102 businesses** (70 + 31 + 1 by pattern) |
| **Invalid Value** | See breakdown below |
| **Domain Rule** | A range that wraps midnight belongs to a late-night business and ends in the early morning (≤ 06:00). An "overnight" range ending at 07:00–22:00 implies an 18–23 h opening |

| Pattern | Business-days | Businesses | Examples | Likely truth |
| --- | --- | --- | --- | --- |
| Start/end **swapped** (start ≥ 15:00) | 240 | 70 | `22:0-17:0`, `20:0-18:0`, `22:0-21:0`, `23:0-22:0` | `17:00-22:00`, `18:00-20:00` … |
| **12-hour clock** entry (both ≤ 12:00) | 98 | 31 | `9:0-7:0`, `11:0-10:0`, `10:0-9:0`, `11:30-8:0` | `09:00-19:00`, `11:00-22:00` … (PM end entered without +12) |
| Other | 1 | 1 | — | manual review |

**Description:** Naive modulo-24 h arithmetic (DQ-BUSINESS-007 Option 1) turns `22:0-17:0` into a 19-hour day and `9:0-7:0` into 22 hours. That inflates opening-hours features and "open now" / late-night coverage for up to 102 ordinary businesses. Because the pattern usually repeats on every day of the week for a business, it is a systematic entry error, not a real schedule.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Flag `hours_suspect_inverted` / `hours_suspect_12h`; in Silver Enriched compute duration and "open at" only from a corrected range (swap for pattern 1, end + 12 h for pattern 2) and keep the raw range** | Both corrections are deterministic and reversible. The flags let consumers exclude the rows instead |
| 2 | **Set duration to NULL for these rows** | Safe, but loses a correction that is almost certainly right |
| 3 | **Overwrite the raw string in Silver Cleaned** | Changes source values with no record of the original |
| 4 | **Treat as genuine overnight** (current DQ-007 rule) | Produces 18–23 h days for ordinary businesses |

**Best Logical Solution:** **Option 1 — flag and correct in the derived layer only.** Raw hours stay untouched, which keeps lineage and the EDA rule "never change values".

---

---
#### 🚨 DQ-BUSINESS-048 — Overnight Window Overlaps the Next Day's Opening

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-048` |
| **Severity** | LOW (small volume; double-counted open time) |
| **Detected In** | Phase 6.6 — Cross-Column Consistency Checks → **6.6.4 Cross-Column Range Logic** (Cell 111): D) `overnight end ≤ next day's start` |
| **Also Confirmed In** | Partly overlaps DQ-BUSINESS-047 (e.g. `Monday 20:0-17:0 → Tuesday 8:0-17:0` is an inverted window that also overlaps) |
| **Affected Columns** | `hours.<Day>` and `hours.<Day+1>` (Sunday wraps to Monday) → `df_hours` start / end columns |
| **Total Affected Rows** | **20** of 56,338 overnight business-days followed by a real opening the next day (0.04%) |
| **Invalid Value** | `Thursday 17:0-1:0 → Friday 0:0-3:0`; `Friday 15:0-2:0 → Saturday 1:30-2:0`; `Monday 20:0-17:0 → Tuesday 8:0-17:0` |
| **Domain Rule** | If a window on day *d* runs past midnight, it must end before (or exactly when) day *d+1* opens. Two windows cannot cover the same clock time |

**Description:** The overlaps come either from late-night windows also recorded as the next day's early-morning window (`0:0-3:0` after `17:0-1:0`), or from inverted ranges (DQ-BUSINESS-047). Weekly opening hours and "open now" counts are double-counted for those hours.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **In Silver Enriched build a weekly interval table (business_id, open_ts, close_ts on a 7-day clock) and merge overlapping intervals; flag `hours_overlap_flag`** | Interval merge is the standard, lossless way to represent schedules; raw values stay unchanged |
| 2 | **Truncate the earlier window at the next day's opening** | Simple, but assumes which of the two windows is right |
| 3 | **Do nothing** | 20 business-days double-count opening time |

**Best Logical Solution:** **Option 1 — merged weekly intervals plus a flag.** It also gives a clean base for all opening-hours analytics (DQ-007, 012, 025, 047).

---

> **Reviewed and not raised in 6.6.4 (INFO):** 778 same-day windows longer than 20 h are almost all `0:0-23:59` / `0:0-23:0`, which are alternative all-day spellings already in **DQ-BUSINESS-025**. 307 businesses with ≥ 100 reviews and an extreme rating (1.0 / 5.0) are statistically possible (a 5.0 needs a ≥ 4.75 average). The bucket table shows the 5.0 share falls from 18.4% (< 10 reviews) to 0.7% (500+), which is the expected small-sample effect, not a data error.

---

In [0]:
# 6.6.5) Dependency Rule Validation  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# "If A holds then B must hold" — across columns AND across the TE.2 frames of one business.
#   A) Attribute scope ⇒ business type (df_attributes / dicts ⇒ df_categories)
#        restaurant-only attributes ⇒ food / drink / hospitality category; HairSpecializesIn ⇒ hair / beauty;
#        AcceptsInsurance ⇒ health / auto / professional; BestNights / GoodForDancing / CoatCheck ⇒ nightlife / venue
#      INFO: Yelp shows attribute questions by category, so a hit means the category list or the attribute is wrong
#   B) Attribute ⇒ schedule (dicts / df_attributes ⇒ df_hours, only for businesses that publish hours)
#        BestNights.<day> TRUE ⇒ open that day ; GoodForMeal breakfast / lunch / dinner / latenight ⇒ open at that time
#        Open24Hours TRUE ⇒ every listed day is all-day ; all 7 days all-day ⇒ Open24Hours ≠ FALSE (DQ-BUSINESS-026)
#   C) Attribute ⇒ attribute
#        BYOBCorkage yes_* ⇒ BYOB ≠ FALSE ; Corkage TRUE ⇒ BYOB ≠ FALSE ; yes_free ⇒ Corkage ≠ TRUE (DQ-BUSINESS-024)
#        HappyHour TRUE ⇒ Alcohol ≠ 'none'
#   D) Profile
#        review_count > 0 ⇒ stars NOT NULL ; is_open TRUE ⇒ a published schedule (coverage, INFO)
from functools import reduce
from pyspark.sql import functions as F
import pandas as pd

print(f"\n{'─' * 60}\n📌 6.6.5) Dependency Rule Validation — if A then B")
findings_665 = []
T_ = lambda c: F.coalesce(c, F.lit(False))
F_ = lambda c: F.coalesce(~c, F.lit(False))

FOOD_RX   = ("Restaurants|Food|Bars|Nightlife|Cafe|Coffee|Bakeries|Caterers|Brewer|Winer|Distiller|Pubs|Lounges|Venues|Hotels|"
             "Casinos|Event Planning|Delis|Diners|Desserts|Juice|Tea|Ice Cream|Donuts|Bagels|Butcher|Grocery|Golf|Bowling|Cinema")
HAIR_RX   = "Hair|Barber|Beauty|Salon|Spa|Blow Dry|Wig|Cosmetic"
HEALTH_RX = ("Health|Medical|Doctor|Dentist|Dental|Pet|Veterinar|Automotive|Auto |Insurance|Optometr|Eyewear|Chiropract|Therap|"
             "Counsel|Pharmac|Hospital|Clinic|Massage|Acupuncture|Fitness|Professional Services|Home Services|Beauty|Medi")
NIGHT_RX  = "Nightlife|Bars|Pubs|Lounges|Clubs|Venues|Restaurants|Comedy|Karaoke|Casinos|Arts & Entertainment|Event"
EX_CAT = F.concat_ws(" | ", "business_id", "name", F.concat_ws(", ", F.slice("cat_list", 1, 4)))

# ── A) attribute scope ⇒ category ──
print("\n   ── A) Attribute scope ⇒ business type (categories) ──")
A5 = [chk("df_attributes ⇒ df_categories", f"{c} answered ⇒ food/drink category", "restaurant-only question implies a food / drink / hospitality category",
          ~has_cat(FOOD_RX), scope=F.col(c).isNotNull(), status="INFO", ex=EX_CAT)
      for c in ["attr_restaurants_table_service", "attr_restaurants_attire", "attr_restaurants_reservations", "attr_alcohol"]]
A5 += [chk("df_good_for_meal ⇒ df_categories", "GoodForMeal present ⇒ food/drink category", "meal suitability implies a food business",
           ~has_cat(FOOD_RX), scope=F.col("d_GoodForMeal").isNotNull(), status="INFO", ex=EX_CAT),
       chk("df_hair_specializes_in ⇒ df_categories", "HairSpecializesIn present ⇒ hair/beauty category", "hair specialisation implies a salon / barber",
           ~has_cat(HAIR_RX), scope=F.col("d_HairSpecializesIn").isNotNull(), status="INFO", ex=EX_CAT),
       chk("df_attributes ⇒ df_categories", "attr_accepts_insurance answered ⇒ health/auto/pro category", "insurance question implies a medical / auto / professional service",
           ~has_cat(HEALTH_RX), scope=F.col("attr_accepts_insurance").isNotNull(), status="INFO", ex=EX_CAT),
       chk("df_best_nights ⇒ df_categories", "BestNights present ⇒ nightlife/venue category", "best-nights question implies nightlife / venue",
           ~has_cat(NIGHT_RX), scope=F.col("d_BestNights").isNotNull(), status="INFO", ex=EX_CAT),
       chk("df_attributes ⇒ df_categories", "good_for_dancing / coat_check answered ⇒ nightlife/venue", "dancing / coat-check question implies nightlife / venue",
           ~has_cat(NIGHT_RX), scope=F.col("attr_good_for_dancing").isNotNull() | F.col("attr_coat_check").isNotNull(), status="INFO", ex=EX_CAT)]
vA5 = run_checks(findings_665, cc_wide, A5)

# ── B) attribute ⇒ schedule ──
print("\n   ── B) Attribute ⇒ schedule (businesses with published hours) ──")
PRESENT = {d: F.col(HOURS_MAP[d]).isNotNull() for d in DAYS}
ALLDAY  = {d: PRESENT[d] & (START_M[d] == 0) & ((END_M[d] == 0) | (END_M[d] >= 1439)) for d in DAYS}
OVERN   = {d: PRESENT[d] & (END_M[d] < START_M[d]) for d in DAYS}
any_day = lambda fn: reduce(lambda a, b: a | b, [F.coalesce(PRESENT[d] & fn(d), F.lit(False)) for d in DAYS])
all_listed = lambda fn: reduce(lambda a, b: a & b, [F.coalesce(~PRESENT[d] | fn(d), F.lit(True)) for d in DAYS])
open_at = lambda m: any_day(lambda d: ALLDAY[d] | (START_M[d] <= m) & ((END_M[d] >= m) | OVERN[d]) | OVERN[d] & (END_M[d] >= m))
bn_key = {d: d.lower() for d in DAYS}
best_night_closed = reduce(lambda a, b: a | b, [T_(dk("BestNights", bn_key[d])) & ~PRESENT[d] for d in DAYS])
EX_SCH = F.concat_ws(" | ", "business_id", "name", *[F.concat_ws("=", F.lit(d[:3]), F.coalesce(F.col(HOURS_MAP[d]), F.lit("—"))) for d in DAYS])
B5 = [chk("df_best_nights ⇒ df_hours", "BestNights.<day> TRUE ⇒ open that day", "a best night must be an opening day",
          best_night_closed, scope=F.col("d_BestNights").isNotNull() & HAS_HOURS, ex=F.concat_ws(" || ", EX_SCH, F.col("d_BestNights").cast("string"))),
      chk("df_good_for_meal ⇒ df_hours", "breakfast TRUE ⇒ open by 10:30", "some day opens ≤ 10:30 (or all-day / overnight)",
          ~any_day(lambda d: ALLDAY[d] | OVERN[d] | (START_M[d] <= 630)), scope=T_(dk("GoodForMeal", "breakfast")) & HAS_HOURS, ex=EX_SCH),
      chk("df_good_for_meal ⇒ df_hours", "lunch TRUE ⇒ open at 12:30", "some day covers 12:30",
          ~open_at(750), scope=T_(dk("GoodForMeal", "lunch")) & HAS_HOURS, ex=EX_SCH),
      chk("df_good_for_meal ⇒ df_hours", "dinner TRUE ⇒ open at 19:00", "some day covers 19:00",
          ~open_at(1140), scope=T_(dk("GoodForMeal", "dinner")) & HAS_HOURS, ex=EX_SCH),
      chk("df_good_for_meal ⇒ df_hours", "latenight TRUE ⇒ open past 22:00", "some day closes ≥ 22:00, overnight or all-day",
          ~any_day(lambda d: ALLDAY[d] | OVERN[d] | (END_M[d] >= 1320)), scope=T_(dk("GoodForMeal", "latenight")) & HAS_HOURS, ex=EX_SCH),
      chk("df_attributes ⇒ df_hours", "open_24_hours TRUE ⇒ every listed day all-day", "24 h flag implies 00:00-00:00 / 00:00-23:59 on each listed day",
          ~all_listed(lambda d: ALLDAY[d]), scope=T_(F.col("attr_open_24_hours")) & HAS_HOURS, ex=EX_SCH, note="DQ-BUSINESS-026"),
      chk("df_hours ⇒ df_attributes", "7 days all-day ⇒ open_24_hours ≠ FALSE", "a 24/7 schedule contradicts Open24Hours = FALSE",
          F_(F.col("attr_open_24_hours")), scope=reduce(lambda a, b: a & b, [F.coalesce(ALLDAY[d], F.lit(False)) for d in DAYS]),
          ex=EX_SCH, note="DQ-BUSINESS-026 (reverse direction)")]
vB5 = run_checks(findings_665, cc_wide, B5)

# ── C) attribute ⇒ attribute ──
print("\n   ── C) Attribute ⇒ attribute ──")
byob, bc, cork = F.col("attr_byob"), F.col("attr_byob_corkage"), F.col("attr_corkage")
EX_BY = F.concat_ws(" | ", "business_id", "name", F.concat_ws("=", F.lit("BYOB"), byob.cast("string")),
                    F.concat_ws("=", F.lit("BYOBCorkage"), bc), F.concat_ws("=", F.lit("Corkage"), cork.cast("string")),
                    F.concat_ws("=", F.lit("Alcohol"), F.col("attr_alcohol")), F.concat_ws("=", F.lit("HappyHour"), F.col("attr_happy_hour").cast("string")))
C5 = [chk("df_attributes", "BYOBCorkage yes_* ⇒ BYOB ≠ FALSE", "corkage policy for own bottles implies BYOB allowed", F_(byob),
          scope=bc.isin("yes_free", "yes_corkage"), ex=EX_BY, note="DQ-BUSINESS-024"),
      chk("df_attributes", "Corkage TRUE ⇒ BYOB ≠ FALSE", "a corkage fee implies own bottles are allowed", F_(byob), scope=T_(cork), ex=EX_BY, note="DQ-BUSINESS-024"),
      chk("df_attributes", "BYOBCorkage yes_free ⇒ Corkage ≠ TRUE", "free corkage contradicts a corkage fee", T_(cork), scope=bc == "yes_free",
          ex=EX_BY, note="DQ-BUSINESS-024"),
      chk("df_attributes", "BYOBCorkage no ⇒ BYOB ≠ TRUE", "'no' (bottles not allowed) contradicts BYOB = TRUE", T_(byob), scope=bc == "no",
          ex=EX_BY, note="DQ-BUSINESS-024"),
      chk("df_attributes", "HappyHour TRUE ⇒ Alcohol ≠ 'none'", "happy hour (drink specials) implies alcohol is served",
          F.col("attr_alcohol") == "none", scope=T_(F.col("attr_happy_hour")), ex=EX_BY,
          note="possible for BYOB-only venues with food specials — check BYOB in examples")]
vC5 = run_checks(findings_665, cc_wide, C5)

# ── D) profile ──
print("\n   ── D) Profile dependencies ──")
D5 = [chk("df_business", "review_count > 0 ⇒ stars NOT NULL", "reviewed business must carry a rating", F.col("stars").isNull(),
          scope=F.col("review_count") > 0),
      chk("df_business ⇒ df_hours", "is_open TRUE ⇒ published schedule", "open business publishes at least one day", ~HAS_HOURS,
          scope=T_(F.col("is_open")), status="INFO", note="coverage gap, not an error — service-area / by-appointment businesses (DQ-BUSINESS-032 / 033)")]
vD5 = run_checks(findings_665, cc_wide, D5)
cc_metrics.update({"bestnight_closed": vB5["BestNights.<day> TRUE ⇒ open that day"],
                   "meal_hours_conflicts": sum(vB5[k] for k in list(vB5)[1:5]),
                   "open24_conflicts": vB5["open_24_hours TRUE ⇒ every listed day all-day"] + vB5["7 days all-day ⇒ open_24_hours ≠ FALSE"],
                   "byob_conflicts": sum(vC5[k] for k in list(vC5)[:4]),
                   "happyhour_no_alcohol": vC5["HappyHour TRUE ⇒ Alcohol ≠ 'none'"],
                   "scope_leaks": sum(vA5.values())})

findings_665_df = pd.DataFrame(findings_665)
print(f"\n   {len(findings_665_df)} dependency checks | {int((findings_665_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((findings_665_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


---
#### 🚨 DQ-BUSINESS-049 — `BestNights` Recommends Days on Which the Business Is Closed

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-049` |
| **Severity** | MEDIUM (cross-frame dependency violation between `df_best_nights` and `df_hours`) |
| **Detected In** | Phase 6.6 — Cross-Column Consistency Checks → **6.6.5 Dependency Rule Validation** (Cell 115): B) `BestNights.<day> TRUE ⇒ open that day` (only businesses that publish hours) |
| **Also Confirmed In** | Independent SQL on the raw strings: 235 (business, day) pairs. Monday 58, Tuesday 54, Sunday 40, Saturday 30, Wednesday 23, Thursday 18, Friday 12 |
| **Affected Columns** | `attributes.BestNights` → `df_best_nights` (`key` = weekday, `value` = TRUE) vs `hours.<Day>` → `df_hours.hours_<day>` (NULL) |
| **Total Affected Rows** | **208** of 5,387 businesses with both BestNights and hours (3.9%), 235 business-days |
| **Invalid Value** | `Visual Arts Collective`: only Saturday hours, BestNights `friday = True`; `21st Amendment at La Louisiane`: open Thu–Sat, BestNights `tuesday = True` |
| **Domain Rule** | A "best night" must be a night the venue is open: `BestNights[d] = TRUE ⇒ hours[d] IS NOT NULL` |

**Description:** Two snapshots taken at different times disagree. BestNights is a crowd vote that accumulates over years, while hours is the current schedule. The Mon/Tue/Sun skew matches typical closing days, so the most likely cause is that the venue stopped opening on those days after the votes were cast. A second possible cause is that a NULL day in `hours` means "not provided" rather than "closed" (DQ-BUSINESS-033).

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep both; in Silver Enriched derive `best_night_open_<day> = best_night AND hours_<day> IS NOT NULL` and add `bestnight_hours_conflict_flag`** | The current schedule is the operational truth for "go tonight" use cases, while the raw vote is preserved for history |
| 2 | **Set the conflicting BestNights keys to FALSE** | Asserts the crowd was wrong; loses the historical signal |
| 3 | **Treat the NULL hours day as open** | Contradicts DQ-BUSINESS-033 (NULL day = closed for 55,837 business-days) |

**Best Logical Solution:** **Option 1 — derived, schedule-aware flag plus conflict flag.**

---

---
#### 🚨 DQ-BUSINESS-050 — `GoodForMeal` Flags That the Opening Hours Make Impossible

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-050` |
| **Severity** | MEDIUM (cross-frame dependency violation between `df_good_for_meal` and `df_hours`) |
| **Detected In** | Phase 6.6 — Cross-Column Consistency Checks → **6.6.5 Dependency Rule Validation** (Cell 115): B) four meal ⇒ schedule rules |
| **Also Confirmed In** | The breakfast rule was first tested with a 10:00 cut-off and loosened to 10:30 so that 10:30 openers (e.g. QDOBA) are not counted. All counts below use the tolerant rule |
| **Affected Columns** | `attributes.GoodForMeal` → `df_good_for_meal` (`breakfast`, `lunch`, `dinner`, `latenight`) vs `df_hours` start / end times |
| **Total Affected Rows** | **762** rule violations among businesses that publish hours (see table) |
| **Invalid Value** | See table |
| **Domain Rule** | A meal flag must fall inside at least one opening window of the week |

| Rule | Violations | Of | Example |
| --- | --- | --- | --- |
| `lunch = TRUE` ⇒ open at 12:30 on some day | **553** | 13,627 (4.1%) | `Sal DeForte's Ristorante` opens 16:00 every day |
| `dinner = TRUE` ⇒ open at 19:00 on some day | **116** | 12,678 (0.9%) | `Corner Bakery` 6:30–15:00; `Sweet Soulfood` 11:00–17:00 |
| `latenight = TRUE` ⇒ closes ≥ 22:00 / overnight / all-day | **58** | 1,289 (4.5%) | `Khyber Pass Pub` 12:00–21:00; `White Castle` `5:0-5:0` (see DQ-007) |
| `breakfast = TRUE` ⇒ opens ≤ 10:30 on some day | **35** | 2,904 (1.2%) | `Roundabout Grill` opens 16:00–17:00 |

**Description:** Like DQ-BUSINESS-049, GoodForMeal is a long-run crowd opinion, while hours is the current schedule. A restaurant that stops serving lunch keeps its old `lunch = TRUE` tag. Some hits are caused by other hours errors (`5:0-5:0` same-time ranges, DQ-BUSINESS-007; inverted ranges, DQ-BUSINESS-047), so fixing those first reduces this count.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep the raw meal flags; in Silver Enriched derive `serves_<meal>_now = good_for_<meal> AND open during the meal window`, and add `meal_hours_conflict_flag`; evaluate after the DQ-047 / DQ-007 hour fixes** | Separates crowd opinion from current service, so meal-time recommendations use the current schedule; raw values stay intact |
| 2 | **Set the conflicting meal flags to FALSE** | Probably right for most lunch cases, but assumes the hours are always correct (not true for DQ-047 rows) |
| 3 | **Do nothing** | 553 restaurants are recommended for lunch while closed at lunch time |

**Best Logical Solution:** **Option 1 — schedule-aware derived flags, evaluated after the hours corrections.**

---

---
#### 🚨 DQ-BUSINESS-051 — `HappyHour = True` at Businesses With `Alcohol = 'none'`

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-051` |
| **Severity** | LOW–MEDIUM (dependency violation; partly explained by BYOB) |
| **Detected In** | Phase 6.6 — Cross-Column Consistency Checks → **6.6.5 Dependency Rule Validation** (Cell 115): C) `HappyHour TRUE ⇒ Alcohol ≠ 'none'` |
| **Also Confirmed In** | Independent SQL on bronze (both quoting styles of `'none'`, DQ-BUSINESS-003): **438**. Split by BYOB: NULL 314, True 61, False 62, `'None'` literal 1 |
| **Affected Columns** | `attributes.HappyHour` → `attr_happy_hour`; `attributes.Alcohol` → `attr_alcohol` (`'none'` is a valid category, DQ-BUSINESS-002); `attr_byob` |
| **Total Affected Rows** | **438** of 9,721 businesses with a happy hour (4.5%) |
| **Invalid Value** | `HappyHour = True`, `Alcohol = 'none'` |
| **Domain Rule** | A happy hour is a discounted-drinks period, so it implies alcohol is served: `HappyHour = TRUE ⇒ Alcohol ∈ {beer_and_wine, full_bar}` |

**Description:** Only the 61 BYOB = True rows have a plausible explanation (food specials at an unlicensed BYOB place). The other 377 contradict each other: either the venue obtained a licence and `Alcohol` is stale, or `HappyHour` is used for food / coffee discounts. Any "bars with happy hour" or alcohol-licence analysis is affected.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep both raw values; add `happyhour_alcohol_conflict_flag` for `Alcohol = 'none'` AND `BYOB IS NOT TRUE` (377); treat the 61 BYOB rows as valid** | Separates the explainable case from the real conflicts, and no value is guessed |
| 2 | **Set `HappyHour = FALSE` when `Alcohol = 'none'`** | Wrong for venues whose `Alcohol` answer is out of date |
| 3 | **Upgrade `Alcohol` to `beer_and_wine`** | Invents a licence level |

**Best Logical Solution:** **Option 1 — BYOB-aware conflict flag.**

---

---
#### 🚨 DQ-BUSINESS-052 — Ambiguous Meaning of `BYOB`: Corkage Fee With `BYOB = False`

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-052` |
| **Severity** | MEDIUM (semantic ambiguity in the attribute definition; 56% of corkage businesses affected) |
| **Detected In** | Phase 6.6 — Cross-Column Consistency Checks → **6.6.5 Dependency Rule Validation** (Cell 115): C) `Corkage TRUE ⇒ BYOB ≠ FALSE` |
| **Also Confirmed In** | Independent SQL on bronze: 612, split by Alcohol: `full_bar` 457, `beer_and_wine` 138, `none` 16, NULL 1. For comparison, `Corkage = True` + `BYOB = True`: 272. This rule was **not** part of DQ-BUSINESS-024 (which only tested rules anchored on `BYOBCorkage`) |
| **Affected Columns** | `attributes.BYOB` → `attr_byob`; `attributes.Corkage` → `attr_corkage`; `attr_alcohol` |
| **Total Affected Rows** | **612** of 1,096 businesses with `Corkage = True` (55.8%) |
| **Invalid Value** | `Top Shelf Sports Lounge`: full_bar, `BYOB = False`, `Corkage = True` |
| **Domain Rule** | A corkage fee is charged on bottles brought in by guests, so outside bottles must be allowed |

**Description:** This is mainly a **definition** problem, not 612 typos. In 595 of the 612 cases (97%), the venue is licensed (`full_bar` / `beer_and_wine`). Licensed restaurants often accept a guest's wine for a fee while not being a "BYOB restaurant". So in practice `BYOB = False` means "not a BYOB-concept (unlicensed) venue", **not** "outside bottles forbidden". Only the **16** unlicensed rows (`Alcohol = 'none'`, BYOB = False, Corkage = True) are true contradictions. This also changes how DQ-BUSINESS-024 should be read: its Option 1 precedence rule assumes BYOB means "outside bottles allowed".

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Document in the DDL / data dictionary that `attr_byob` = "BYOB-concept venue"; in the DQ-024 `byob_policy` derivation treat `Corkage = TRUE` as "outside bottles allowed with fee" regardless of BYOB; flag only the 16 unlicensed contradictions** | Matches observed behaviour in 97% of cases, prevents a false 612-row error count, and keeps a single consistent policy field |
| 2 | **Treat all 612 as conflicts and NULL BYOB** | Destroys valid information for licensed restaurants |
| 3 | **Overwrite `BYOB = TRUE` when Corkage = TRUE** | Mislabels full-bar restaurants as BYOB venues |

**Best Logical Solution:** **Option 1 — fix the semantics in the data dictionary and in the `byob_policy` derivation, and flag only the 16 real contradictions.**

---

---
#### 🚨 DQ-BUSINESS-053 — Attributes Answered Outside Their Business-Type Scope

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-053` |
| **Severity** | LOW (INFO-level: misapplied attributes or incomplete category lists) |
| **Detected In** | Phase 6.6 — Cross-Column Consistency Checks → **6.6.5 Dependency Rule Validation** (Cell 115): A) attribute scope ⇒ business type (9 INFO checks) |
| **Also Confirmed In** | Related to DQ-BUSINESS-034 (structural "not applicable" NULLs): there the attributes are missing where they don't apply; here they are *present* where they don't apply |
| **Affected Columns** | `attr_restaurants_table_service`, `attr_restaurants_attire`, `attr_restaurants_reservations`, `attr_alcohol`, `GoodForMeal`, `attr_accepts_insurance`, `BestNights`, `attr_good_for_dancing` / `attr_coat_check` vs `df_categories` |
| **Total Affected Rows** | **743** rule hits: dancing / coat check 333, reservations 237, table service 119, attire 25, alcohol 24, GoodForMeal 2, AcceptsInsurance 2, BestNights 1; HairSpecializesIn 0 |
| **Invalid Value** | `Target` / `Petco` with `GoodForDancing` / `CoatCheck` and `RestaurantsReservations`; `U.S. Bank Branch` with `Alcohol`; `The Children's Museum of Indianapolis` with `RestaurantsAttire` |
| **Domain Rule** | Yelp shows each attribute question only to the business types it applies to, so an answered attribute implies a matching category |

**Description:** Two causes. (a) Chain-level bulk attribute templates applied to every unit (Target, Petco, European Wax Center), which fill restaurant / nightlife questions with (usually FALSE) answers. (b) Real but uncategorised services, e.g. a stadium or museum with an on-site restaurant (`Citizens Bank Park` with attire). Case (a) adds a misleading FALSE where the honest value is "not applicable", and it biases attribute prevalence statistics (e.g. share of venues without coat check).

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep raw values; add a category-scope map in Silver Enriched and expose scoped attributes as NULL ("not applicable") outside their scope, with `attr_out_of_scope_flag`** | Not-applicable is semantically different from FALSE; the source stays auditable; prevalence metrics use the right denominator |
| 2 | **Delete out-of-scope answers in Silver Cleaned** | Same analytical effect, but loses lineage and the cases where the business genuinely offers the service |
| 3 | **Do nothing** | Template FALSEs distort attribute rates for restaurants and nightlife |

**Best Logical Solution:** **Option 1 — scope-aware enriched view with a flag.**

---

> **Cross-references from 6.6.5 (already documented, no new ID):** `Open24Hours TRUE ⇒ every listed day all-day` 15 of 19 → **DQ-BUSINESS-026** (count identical). BYOB trio: `BYOBCorkage yes_* ⇒ BYOB ≠ FALSE` 49, `yes_free ⇒ Corkage ≠ TRUE` 11, `no ⇒ BYOB ≠ TRUE` 10 → **DQ-BUSINESS-024** (counts identical to Cell 60). `is_open TRUE ⇒ published schedule` (INFO) → **DQ-BUSINESS-032 / 033**.
>
> **Passed:** 0 reviewed businesses without a rating; 0 seven-day all-day schedules with `Open24Hours = FALSE`; 0 HairSpecializesIn outside hair / beauty categories.

---

In [0]:
# 6.6.6) Category Mapping Consistency  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Code ↔ label pairs must map consistently (functional dependencies across rows):
#   A) ZIP5 → state          a US ZIP belongs to exactly one state         (FINDING for minority-state rows)
#   B) ZIP5 → city           one ZIP → one (folded) city name               (INFO: USPS ZIPs can span towns)
#   C) FSA → province        Canadian FSA first letter T ⇒ AB (Canada Post)
#   D) metro → state         the Yelp metro a point sits in fixes the admissible state codes
#   E) raw enum ↔ decoded    raw literal → exactly one decoded value; decoded value ← spellings differing only by u'…'
#   F) dict key ↔ column     long-frame keys ⊆ contract key map; BestNights keys ↔ the 7 hours days
# The state ↔ postal-country and state ↔ ZIP3 rules were tested in 6.3.1 (DQ-BUSINESS-023); this cell adds the
# reverse direction (code → label) plus the coordinate-based mapping.
from pyspark.sql import functions as F, Window
import pandas as pd

print(f"\n{'─' * 60}\n📌 6.6.6) Category Mapping Consistency")
findings_666 = []

def fd_violations(frame, code, label, label_display=None):
    """Functional dependency code → label: rows whose label is NOT the code's modal label."""
    label_display = label_display or label
    g = frame.groupBy(code, label).agg(F.count("*").alias("n"), F.first(label_display).alias("disp"))
    w = Window.partitionBy(code).orderBy(F.desc("n"), F.col(label))
    g = g.withColumn("rk", F.row_number().over(w)).withColumn("labels", F.count("*").over(Window.partitionBy(code)))
    multi = g.where("labels > 1")
    x = multi.agg(F.countDistinct(code).alias("codes"), F.sum(F.when(F.col("rk") > 1, F.col("n")).otherwise(0)).alias("rows")).first()
    ex = (multi.groupBy(code).agg(F.concat_ws(" / ", F.sort_array(F.collect_list(F.format_string("%s (%d)", "disp", "n")), False)).alias("m"))
               .limit(5).collect())
    return int(x["codes"] or 0), int(x["rows"] or 0), " ; ".join(f"{r[code]} → {r['m']}" for r in ex), multi

zip_raw = F.upper(F.regexp_replace("postal_code", r"\s", ""))
city_fold = F.regexp_replace(F.regexp_replace(F.regexp_replace(F.lower("city"), r"\bsaint\b", "st"), r"\bmount\b", "mt"), "[^a-z]", "")
geo = df_business.select("business_id", "name", "city", "state", "postal_code", "latitude", "longitude",
                         F.when(zip_raw.rlike(r"^\d{5}$"), zip_raw).alias("zip5"),
                         F.when(zip_raw.rlike(r"^[A-Z]\d[A-Z]"), F.substring(zip_raw, 1, 3)).alias("fsa"),
                         city_fold.alias("city_fold"))
us = geo.where(F.col("zip5").isNotNull())
n_us = us.count()

# ── A / B) ZIP5 → state, ZIP5 → city ──
print("\n   ── A/B) ZIP5 → state / city ──")
zc_codes, zc_rows, zc_ex, _ = fd_violations(us, "zip5", "state")
dq_add(findings_666, "df_business", "postal_code (ZIP5) → state", "each ZIP5 maps to exactly one state", n_us, zc_rows,
       examples=zc_ex, note=f"{zc_codes} ZIP5 codes carry >1 state; violating rows = minority-state rows")
zcity_codes, zcity_rows, zcity_ex, _ = fd_violations(us, "zip5", "city_fold", "city")
dq_add(findings_666, "df_business", "postal_code (ZIP5) → city (folded)", "each ZIP5 maps to one city name", n_us, zcity_rows,
       status="INFO" if zcity_rows else "PASS", examples=zcity_ex,
       note=f"{zcity_codes} ZIPs with >1 folded city: neighbourhood / suburb names used as city (DQ-BUSINESS-018 / 021) or ZIPs spanning towns")
cc_metrics.update({"zip_state_rows": zc_rows, "zip_state_codes": zc_codes, "zip_city_rows": zcity_rows})

# ── C) FSA → province ──
ca = geo.where(F.col("fsa").isNotNull())
dq_add(findings_666, "df_business", "FSA first letter → province", "'T' FSA ⇔ AB (Canada Post allocation)", ca.count(),
       ca.where((F.col("fsa").startswith("T")) != (F.col("state") == "AB")).count(),
       examples=dq_examples(ca, F.concat_ws(" | ", "name", "state", "postal_code"), (F.col("fsa").startswith("T")) != (F.col("state") == "AB")))

# ── D) metro → admissible states ──
print("\n   ── D) Yelp metro (by coordinates) → admissible state codes ──")
METROS = {  # centre (lat, lon), admissible state codes (incl. cross-border suburbs)
    "Philadelphia": ((39.9526, -75.1652), {"PA", "NJ", "DE"}), "Tampa": ((27.9506, -82.4572), {"FL"}),
    "Indianapolis": ((39.7684, -86.1581), {"IN"}), "Nashville": ((36.1627, -86.7816), {"TN"}),
    "Tucson": ((32.2226, -110.9747), {"AZ"}), "New Orleans": ((29.9511, -90.0715), {"LA"}),
    "Reno": ((39.5296, -119.8138), {"NV", "CA"}), "Edmonton": ((53.5461, -113.4938), {"AB"}),
    "Santa Barbara": ((34.4208, -119.6982), {"CA"}), "Boise": ((43.6150, -116.2023), {"ID"}),
    "St. Louis": ((38.6270, -90.1994), {"MO", "IL"})}
dist = F.array(*[F.struct(hav_km(F.col("latitude"), F.col("longitude"), F.lit(la), F.lit(lo)).alias("km"), F.lit(m).alias("metro"))
                 for m, ((la, lo), _) in METROS.items()])
adm = F.create_map(*[x for m, (_, st) in METROS.items() for x in (F.lit(m), F.array(*[F.lit(s) for s in sorted(st)]))])
gm = (geo.where(F.col("latitude").isNotNull()).withColumn("near", F.array_min(dist))
         .withColumn("metro", F.col("near.metro")).withColumn("metro_km", F.col("near.km")))
vD = run_checks(findings_666, gm, [
    chk("df_business", "metro(lat, lon) → state", "state ∈ admissible codes of the nearest Yelp metro (≤ 150 km)",
        ~F.array_contains(adm[F.col("metro")], F.col("state")), scope=F.col("metro_km") <= 150,
        ex=F.concat_ws(" | ", "name", "city", "state", "postal_code", "metro", F.format_string("%.0f km", "metro_km"))),
    chk("df_business", "point → some metro", "every point maps to a Yelp metro (≤ 150 km)", F.col("metro_km") > 150, status="INFO",
        ex=F.concat_ws(" | ", "name", "state", "metro", F.format_string("%.0f km", "metro_km")), note="DQ-BUSINESS-011")])
cc_metrics["metro_state"] = vD["metro(lat, lon) → state"]
print("   state × metro cross-tab (off-diagonal = mapping conflicts):")
display(gm.where("metro_km <= 150").groupBy("metro").pivot("state").count().na.fill(0).orderBy("metro"))

# ── E) raw enum literal ↔ decoded value ──
print("\n   ── E) Enum attributes: raw literal ↔ decoded value ──")
je6 = df_attributes.join(df_bronze.select("business_id", "attributes"), "business_id")
enum_rows, bad_fwd, bad_rev, n_raw = [], 0, 0, 0
for b, s in ATTR_ENUM_MAP.items():
    pairs = (je6.where(F.col(f"attributes.{b}").isNotNull() & (F.col(f"attributes.{b}") != "None"))
                .groupBy(F.col(f"attributes.{b}").alias("raw"), F.col(s).alias("decoded")).count().toPandas())
    fwd = pairs.groupby("raw")["decoded"].nunique()
    rev_bad = pairs[pairs.apply(lambda r: r["decoded"] is None or r["raw"] not in (f"'{r['decoded']}'", f"u'{r['decoded']}'"), axis=1)]
    bad_fwd += int(pairs[pairs["raw"].isin(fwd[fwd > 1].index)]["count"].sum())
    bad_rev += int(rev_bad["count"].sum())
    n_raw += int(pairs["count"].sum())
    enum_rows.append({"Attribute": s, "Raw spellings": len(pairs), "Decoded values": pairs["decoded"].nunique(),
                      "Raw→decoded 1:1": bool((fwd <= 1).all()), "Rows w/ spelling ≠ 'x' / u'x'": int(rev_bad["count"].sum()),
                      "Mapping": " ; ".join(f"{r.raw}→{r.decoded}" for r in pairs.sort_values("decoded").itertuples())})
display(pd.DataFrame(enum_rows))
dq_add(findings_666, "df_attributes (7 enums)", "raw literal → decoded", "each raw literal decodes to exactly one value", n_raw, bad_fwd)
dq_add(findings_666, "df_attributes (7 enums)", "decoded ← raw spellings", "raw spellings of one value differ only by the u'…' prefix", n_raw, bad_rev,
       note="quoting-style variants themselves = DQ-BUSINESS-003")

# ── F) dict key ↔ contract column; BestNights keys ↔ hours days ──
print("\n   ── F) Nested-dict keys ↔ contract columns ──")
for a, (tbl, keymap) in DICT_TABLE_MAP.items():
    fr = business_dfs[tbl]
    dq_add(findings_666, CC_FRAME[tbl], "key → contract column", f"key ∈ {{{', '.join(keymap)}}}", fr.count(),
           fr.where(~F.col("key").isin(list(keymap))).count(),
           examples=dq_examples(fr, "key", ~F.col("key").isin(list(keymap))))
bn_keys = set(DICT_TABLE_MAP["BestNights"][1])
day_keys = {d.lower() for d in DAYS}
dq_add(findings_666, "df_best_nights ↔ df_hours", "BestNights key ↔ hours day", "the 7 BestNights keys == the 7 hours days", 7,
       len(bn_keys ^ day_keys), examples=", ".join(sorted(bn_keys ^ day_keys)))

findings_666_df = pd.DataFrame(findings_666)
print(f"\n   {len(findings_666_df)} mapping checks | {int((findings_666_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((findings_666_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


---
#### 🚨 DQ-BUSINESS-054 — One ZIP Code Mapped to Several City Names (Township / Neighbourhood / Mall Names Used as `city`)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-054` |
| **Severity** | LOW (standardization; breaks city-level grouping, not location accuracy) |
| **Detected In** | Phase 6.6 — Cross-Column Consistency Checks → **6.6.6 Category Mapping Consistency** (Cell 121): B) `postal_code (ZIP5) → city (folded)` |
| **Also Confirmed In** | The city key is already folded for case, punctuation and Saint/St, Mount/Mt (DQ-BUSINESS-021), so these are different *names*, not spelling variants. Related to DQ-BUSINESS-018 (polluted city values) |
| **Affected Columns** | `city` vs `postal_code` (`df_business`) |
| **Total Affected Rows** | **8,385** of 144,716 US businesses (5.8%) are filed under a non-majority city name for their ZIP5 |
| **Invalid Value** | `08002 → Cherry Hill (445) / Cherry Hill Mall (1) / W Cherry Hill (1)`; `08012 → Blackwood (137) / Turnersville (125) / Gloucester Township (18) / Washington Township (5) / Hilltop (1) / Gloucester (1)`; `08009 → Berlin (92) / Berlin Boro / West Berlin / Winslow` |
| **Domain Rule** | A city dimension should hold one canonical place name per locality. A ZIP has one USPS default city, even if other names are "acceptable" |

**Description:** Many of these are legitimate USPS "acceptable" names (Blackwood vs Turnersville, both valid for 08012), so not every row is wrong. But the dimension mixes three levels: postal city, municipality / township (`Gloucester Township`) and points of interest (`Cherry Hill Mall`). City-level counts and rankings therefore split one locality into several groups.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep raw `city`; add `city_canonical` = the USPS default city for the ZIP (or the modal city per ZIP within the dataset when no reference is available), plus `city_is_alias`** | Gives one consistent grouping key and preserves the owner-entered locality |
| 2 | **Overwrite `city` with the ZIP's modal city** | Consistent, but removes valid locality detail (Turnersville is a real place) |
| 3 | **Do nothing** | City-level aggregates stay fragmented |

**Best Logical Solution:** **Option 1 — add a canonical city from a USPS ZIP reference table (fall back to the modal city).**

---

> **Cross-references from 6.6.6 (already documented, no new ID):** `postal_code (ZIP5) → state`: 31 minority-state rows in 29 ZIPs (e.g. `08054 → NJ 411 / NV 1`, `19014 → PA 138 / DE 1`). Independent SQL shows **all 31** are among the 71 state ↔ ZIP-prefix conflicts of **DQ-BUSINESS-023**. `metro(lat, lon) → state`: 35 conflicts = 20 off-footprint state codes (CO, MT, WA, MA, NC, …) → **DQ-BUSINESS-010**, plus 15 in-footprint ones (`Eagle CA 83616` in Boise, `Mount Laurel NV 08054` in Philadelphia, `New Orleans FL 70122`) → **DQ-BUSINESS-023** / **DQ-BUSINESS-046**. `point → some metro` (2 > 150 km) → **DQ-BUSINESS-011**.
>
> **Passed:** 0 Canadian FSA ↔ province mismatches (5,555); every enum raw literal decodes to exactly one value and spellings differ only by the `u'…'` prefix (183,307 values); all nested-dict keys belong to the contract key map; BestNights keys ↔ hours days match 1:1.

---

In [0]:
# 6.6.7) Cross-Column Consistency — Consolidated Summary  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Status per sub-check, headline metrics and every FINDING of 6.6.1 – 6.6.6 in one place
# (input for the DQ-BUSINESS notes and the Silver cleaning rules).
import pandas as pd

print("=" * 80)
print("6.6) CROSS-COLUMN CONSISTENCY — CONSOLIDATED SUMMARY  (Business DataFrames)")
print("=" * 80)

SECTIONS_66 = [("6.6.1 Logical Relationship", "findings_661_df"), ("6.6.2 Derived-Column Logic", "findings_662_df"),
               ("6.6.3 Logical Contradiction", "findings_663_df"), ("6.6.4 Cross-Column Range", "findings_664_df"),
               ("6.6.5 Dependency Rules", "findings_665_df"), ("6.6.6 Category Mapping", "findings_666_df")]
parts, missing_sections = [], []
for sec, var in SECTIONS_66:
    (parts.append(globals()[var].assign(Section=sec)) if var in globals() else missing_sections.append(sec))
if missing_sections:
    print(f"   ⚠️  Not yet run (re-run these cells first): {missing_sections}")
all_66 = pd.concat(parts, ignore_index=True)

status_66 = (all_66.pivot_table(index="Section", columns="Status", values="Rule", aggfunc="count", fill_value=0)
             .reindex(columns=["FINDING", "INFO", "PASS"], fill_value=0).reset_index())
status_66["Checks"] = status_66[["FINDING", "INFO", "PASS"]].sum(axis=1)
print("\n📋 Check status per sub-section:")
display(status_66)

M = cc_metrics
headline_66 = pd.DataFrame([
    ("Relationship", "TE.2 values ≠ independent re-derivation from bronze", f"{M['rel_findings']} of {M['rel_checks']} checks", "restructuring is lossless row by row"),
    ("Derived", "same non-midnight start/end days / 00:00-00:00 days", f"{M['same_time_days']:,} / {M['midnight_days']:,}", "window duration undefined / ambiguous (DQ-007, 025)"),
    ("Derived", "Music dicts where no_music ≠ NOT(any music)", f"{M['no_music_bad']:,}", "DQ-041: no_music is not a usable indicator"),
    ("Contradiction", "21+/18+ venues good for kids", f"{M['adult_kids']:,}", "audience attributes contradict"),
    ("Contradiction", "formal attire with casual/divey ambience · divey + classy/upscale", f"{M['formal_casual']:,} · {M['divey_classy']:,}", "crowd-sourced ambience noise"),
    ("Contradiction", "appointment-only + 24 h · drive-thru without take-out", f"{M['appt_24h']:,} · {M['drive_thru_no_takeout']:,}", "operational flags contradict"),
    ("Range", "coordinates outside the labelled state's bounding box", f"{M['geo_state_bbox']:,}", "wrong state or wrong geocode"),
    ("Range", "coordinates > 50 km from their ZIP median", f"{M['geo_zip_far']:,}", "wrong ZIP or wrong geocode"),
    ("Range", "overnight windows ending after 06:00 · overlapping consecutive days", f"{M['overnight_late_end']:,} · {M['day_overlap']:,}", "AM/PM swaps / double-counted time"),
    ("Dependency", "BestNights on a closed day · meal flags outside opening hours", f"{M['bestnight_closed']:,} · {M['meal_hours_conflicts']:,}", "attributes vs schedule"),
    ("Dependency", "Open24Hours vs schedule · BYOB trio · HappyHour without alcohol", f"{M['open24_conflicts']:,} · {M['byob_conflicts']:,} · {M['happyhour_no_alcohol']:,}", "DQ-026 / DQ-024 + new"),
    ("Dependency", "attribute scope leaks (attribute not matching category)", f"{M['scope_leaks']:,}", "INFO — category list incomplete or attribute misapplied"),
    ("Mapping", "ZIP5 → state minority rows (ZIP codes)", f"{M['zip_state_rows']:,} ({M['zip_state_codes']:,})", "one ZIP, two states"),
    ("Mapping", "ZIP5 → city minority rows · metro → state conflicts", f"{M['zip_city_rows']:,} · {M['metro_state']:,}", "neighbourhood-as-city / mislabeled state"),
], columns=["Level", "Metric", "Value", "Interpretation"])
print("\n📋 Headline cross-column metrics:")
display(headline_66)

find_66 = all_66[all_66["Status"] == "FINDING"][["Section", "DataFrame", "Column", "Rule", "Checked", "Violations", "Pct", "Examples", "Note"]]
print(f"\n📋 All FINDINGs in 6.6 ({len(find_66)}):")
display(find_66.sort_values("Violations", ascending=False).reset_index(drop=True))
info_66 = all_66[all_66["Status"] == "INFO"][["Section", "DataFrame", "Column", "Checked", "Violations", "Pct", "Note"]]
print(f"\n📋 INFO profiles in 6.6 ({len(info_66)}):")
display(info_66.reset_index(drop=True))

n_f, n_i, n_p = (int((all_66["Status"] == s).sum()) for s in ("FINDING", "INFO", "PASS"))
print(f"\n   {len(all_66)} checks | {n_f} FINDING | {n_i} INFO | {n_p} PASS")
summary_66_df = all_66
print("\nCROSS-COLUMN CONSISTENCY CHECKS COMPLETE")
print("=" * 80)


---
#### 📋 6.6 Cross-Column Consistency — Error Register

New errors raised in Phase 6.6 (next free ID: **DQ-BUSINESS-055**). Cell numbers refer to the detecting code cell.

| Error ID | Title | Severity | Detected In (Cell) | Affected | Best Solution (short) |
| --- | --- | --- | --- | --- | --- |
| DQ-BUSINESS-042 | `RestaurantsAttire = 'formal'` on fast-food / casual venues | MEDIUM | 6.6.3 (106) | 25 of 70 formal | Flag; validated attire = NULL on conflict |
| DQ-BUSINESS-043 | Adults-only venues marked GoodForKids | LOW–MEDIUM | 6.6.3 (106) | 4 of 52 | Flag; derived family flag = NULL |
| DQ-BUSINESS-044 | Ambience both `divey` and `classy` | LOW–MEDIUM | 6.6.3 (106) | 108 of 1,468 | Keep both, flag, `primary_ambience = mixed` |
| DQ-BUSINESS-045 | Food drive-thru with TakeOut = False (chain default) | MEDIUM | 6.6.3 (106) | 162 of 3,674 | Infer `offers_take_out = TRUE` + flag |
| DQ-BUSINESS-046 | Address block contradicts coordinates | HIGH | 6.6.4 (111) | 20 businesses | Flag; 2-of-3 witness correction |
| DQ-BUSINESS-047 | False overnight windows (inverted / 12 h clock) | MEDIUM | 6.6.4 (111) | 339 business-days | Flag; corrected range in enriched layer |
| DQ-BUSINESS-048 | Overnight window overlaps next day | LOW | 6.6.4 (111) | 20 business-days | Merged weekly intervals + flag |
| DQ-BUSINESS-049 | BestNights on closed days | MEDIUM | 6.6.5 (115) | 208 of 5,387 | Schedule-aware derived flag |
| DQ-BUSINESS-050 | GoodForMeal flags outside opening hours | MEDIUM | 6.6.5 (115) | 762 rule hits | Schedule-aware `serves_<meal>_now` |
| DQ-BUSINESS-051 | HappyHour with Alcohol = 'none' | LOW–MEDIUM | 6.6.5 (115) | 438 (377 excl. BYOB) | BYOB-aware conflict flag |
| DQ-BUSINESS-052 | Ambiguous BYOB meaning (corkage without BYOB) | MEDIUM | 6.6.5 (115) | 612 (16 true conflicts) | Fix data dictionary + `byob_policy` rule |
| DQ-BUSINESS-053 | Attributes outside their business-type scope | LOW | 6.6.5 (115) | 743 rule hits | Scope map; out-of-scope → NULL in enriched |
| DQ-BUSINESS-054 | One ZIP, several city names | LOW | 6.6.6 (121) | 8,385 of 144,716 | `city_canonical` from USPS ZIP reference |

**Re-detected existing errors (counts updated, no new ID):** DQ-BUSINESS-007 (1,492 same-time ranges, 89% at 5:00 / 6:00 / 9:00 → likely 24 h, Cell 104), DQ-BUSINESS-020 (vegan without vegetarian, 3), DQ-BUSINESS-023 (ZIP → state, 31 ⊆ 71), DQ-BUSINESS-024 (49 / 11 / 10, identical), DQ-BUSINESS-026 (Open24Hours vs schedule, 15), DQ-BUSINESS-041 (no_music derivation fails on 2,876 dicts), DQ-BUSINESS-010 / 011 (off-footprint states and remote points in the metro mapping).

**Clean result:** all 81 logical-relationship checks of 6.6.1 (Cell 103) pass, so the TE.1 / TE.2 restructuring is proven lossless row by row.

---


### 6.7) REFERENTIAL INTEGRITY CHECKS
*Make sure every relationship that has `yelp_business` as its parent is valid, both inside the Silver business model and across the sibling Yelp tables.*

`yelp_business.business_id` is the **parent key** of two groups of relationships:
* **Internal (Silver business model, TE.2):** 10 child frames → `df_business`. Two are 1:1 (`df_attributes`, `df_hours`). Eight are 0:N (`df_categories` plus the 7 nested-dict long frames).
* **External (bronze_final siblings):** `yelp_review`, `yelp_tip` and `yelp_checkin` all reference `business_id`. `yelp_business` also stores **pre-computed aggregates** of the review fact table (`review_count`, `stars`) and an activity flag (`is_open`).

- **Foreign Key Validity** — every FK value exists in the parent PK. The parent PK itself must be unique and NOT NULL, and each child PK and declared cardinality must hold.
  - *View 1 — Orphan records:* a child `business_id` (attributes, hours, categories, dict frames, review, tip, check-in) with no matching `df_business.business_id`.
  - *View 2 — Missing children:* a business with no child row. This is a FINDING only when the relationship is mandatory. Every Yelp business has at least 5 reviews, so a business with no reviews means the extract is broken. A business without tips or check-ins is only INFO.
  - *View 3 — Invalid FK format:* the FK is NULL, padded with whitespace, or not the 22-char URL-safe base64 id. Case-insensitive collisions are also checked.

- **Lookup Table Conformance** — values that will become Silver lookup / dimension keys must exist in their reference domain, and `yelp_business` must be a conformed dimension across the fact tables.
  - *Reference lookups:* `state` → jurisdiction list and Yelp footprint, the 7 enum attributes → `ENUM_DOMAINS`, `category_name` → a clean category vocabulary (no delimiter fragments, no case / space variants).
  - *Sibling conformance:* the share of businesses that appear in review, tip and check-in, the overlap between them, coverage per state and per `is_open`, and business reviews written by users missing from `yelp_user`.

- **Cross-Table Aggregate Reconciliation** — stored values must match the values computed from the fact tables, and the *definition* of each stored value has to be identified.
  - *Example:* `yelp_business.review_count` vs `COUNT(*)` and vs `COUNT(DISTINCT user_id)` of the business's reviews.
  - *Example:* `yelp_business.stars` vs `AVG(review.stars)` rounded to the half-star grid, using all reviews vs only the latest review per user.
  - *Example:* `is_open` vs the last review / tip / check-in date (open but silent, or closed but recently active).

In [0]:
# 6.7.1) Foreign Key Validity  —  yelp_business.business_id as the parent key
# ─────────────────────────────────────────────────────────────────────
#   0) Parent PK prerequisites  : df_business.business_id unique, NOT NULL, 22-char URL-safe base64, case-insensitive unique
#   A) Internal (TE.2 model)    : 10 child frames → df_business  (1:1 attributes / hours, 0:N categories + 7 dict frames)
#   B) External (bronze_final)  : yelp_review / yelp_tip / yelp_checkin .business_id → yelp_business
# Views per relationship:  View 1 orphans · View 2 parents without children · View 3 invalid FK format
# plus child-PK uniqueness and declared cardinality (1:1, 0:1, 0:N).
# Business ids that start with '-' are valid base64 (DQ-BUSINESS-037) and are matched by ID_RE.
from pyspark.sql import functions as F
import pandas as pd

print("=" * 80)
print("6.7.1) FOREIGN KEY VALIDITY  —  yelp_business.business_id as parent key")
print("=" * 80)

findings_671, ri_metrics = [], {}
ID_RE = r"^[A-Za-z0-9_-]{22}$"
info_or_pass = lambda v: "INFO" if v else "PASS"
SIBLINGS = {"yelp_review":  "yelp_poc_uc.bronze_final.yelp_review",
            "yelp_tip":     "yelp_poc_uc.bronze_final.yelp_tip",
            "yelp_checkin": "yelp_poc_uc.bronze_final.yelp_checkin"}
sib_raw = {t: spark.table(p) for t, p in SIBLINGS.items()}

# ══ 0) Parent key prerequisites ══
print(f"\n{'─' * 60}\n📌 0) Parent key — df_business.business_id")
pk = df_business.agg(F.count("*").alias("n"), F.countDistinct("business_id").alias("d"),
                     F.sum(F.col("business_id").isNull().cast("int")).alias("nul"),
                     F.countDistinct(F.lower("business_id")).alias("d_ci"),
                     F.sum((~F.col("business_id").rlike(ID_RE)).cast("int")).alias("bad"),
                     F.sum((F.col("business_id") != F.trim("business_id")).cast("int")).alias("ws")).first()
N_PARENT = int(pk["n"])
dq_add(findings_671, "df_business", "business_id (PK)", "unique and NOT NULL", N_PARENT, (N_PARENT - int(pk["d"])) + int(pk["nul"] or 0))
dq_add(findings_671, "df_business", "business_id (PK) format", "22-char URL-safe base64, no padding", N_PARENT, int(pk["bad"] or 0) + int(pk["ws"] or 0))
ci = int(pk["d"]) - int(pk["d_ci"])
dq_add(findings_671, "df_business", "business_id (PK) case-insensitive", "ids stay unique under case-insensitive comparison", int(pk["d"]), ci,
       status=info_or_pass(ci), note="only matters for case-insensitive engines / collations downstream")
parent_keys = df_business.select("business_id").withColumn("_parent", F.lit(1))

# ══ A) Internal model — TE.2 child frames → df_business ══
print(f"\n{'─' * 60}\n📌 A) Internal model — TE.2 child frames → df_business ({N_PARENT:,} parents)")
CHILD_FRAMES = {"yelp_business_attributes": "df_attributes", "yelp_business_hours": "df_hours",
                "yelp_business_categories": "df_categories", **CC_FRAME}
NO_CHILD_NOTE = {"df_attributes": "all 32 attributes NULL — structural 'not applicable' block (DQ-BUSINESS-034 / 035)",
                 "df_hours": "no published schedule (all 7 days NULL; DQ-BUSINESS-033, 6.4)",
                 "df_categories": "business without any category — cannot be placed in the category dimension"}
internal_rows = []
for tbl, fname in CHILD_FRAMES.items():
    fr, spec = business_dfs[tbl], DATASETS[tbl]
    pk_cols, card = spec["PK"], spec["Cardinality"]
    payload = [c for c in fr.columns if c not in pk_cols]
    has_data = F.lit(False)
    for c in payload:
        has_data = has_data | F.col(c).isNotNull()
    r = (fr.join(parent_keys, "business_id", "left")
           .agg(F.count("*").alias("rows"), F.countDistinct("business_id").alias("biz"),
                F.sum(F.col("business_id").isNull().cast("int")).alias("nul"),
                F.sum((F.col("_parent").isNull() & F.col("business_id").isNotNull()).cast("int")).alias("orph"),
                F.sum((~F.col("business_id").rlike(ID_RE)).cast("int")).alias("bad"),
                F.countDistinct(*pk_cols).alias("pk_d"),
                F.countDistinct(F.when(has_data, F.col("business_id"))).alias("biz_data"))).first()
    rows, biz, orph, nul = int(r["rows"]), int(r["biz"]), int(r["orph"] or 0), int(r["nul"] or 0)
    pk_dup, bad, biz_data = rows - int(r["pk_d"]), int(r["bad"] or 0), int(r["biz_data"])
    no_data = N_PARENT - biz_data
    print(f"\n   ── {fname}  ({card}, PK = {', '.join(pk_cols)}) ──")
    dq_add(findings_671, fname, "business_id → df_business (View 1)", "child FK NOT NULL and present in the parent", rows, orph + nul)
    dq_add(findings_671, fname, "business_id format (View 3)", "22-char URL-safe base64", rows, bad)
    dq_add(findings_671, fname, f"PK ({', '.join(pk_cols)})", "child PK unique", rows, pk_dup)
    if card == "1:1":
        dq_add(findings_671, fname, "cardinality 1:1", "exactly one row per parent business", N_PARENT, abs(N_PARENT - biz) + (rows - biz))
    dq_add(findings_671, fname, "parents without child data (View 2)", "business has ≥ 1 non-NULL value in this child", N_PARENT, no_data,
           status=info_or_pass(no_data),
           note=NO_CHILD_NOTE.get(fname, "dict absent for the business — expected for out-of-scope business types (DQ-BUSINESS-034 / 053)"))
    internal_rows.append({"Child frame": fname, "Cardinality": card, "PK": ", ".join(pk_cols), "Rows": rows,
                          "Parents referenced": biz, "Parents with data": biz_data, "Coverage %": round(biz_data / N_PARENT * 100, 2),
                          "Orphan / NULL FK": orph + nul, "Malformed FK": bad, "PK duplicates": pk_dup})

# Silver DDL natural key of the category table: (business_id, category_name)
cat_nk = (df_categories.groupBy("business_id", "category_name").count().where("count > 1")
          .agg(F.sum(F.col("count") - 1).alias("x"), F.count("*").alias("g")).first())
n_cat = df_categories.count()
dq_add(findings_671, "df_categories", "natural key (business_id, category_name)", "a business lists each category once", n_cat,
       int(cat_nk["x"] or 0), examples=dq_examples(df_categories.groupBy("business_id", "category_name").count().where("count > 1"),
                                                    F.concat_ws(" | ", "business_id", "category_name"), F.lit(True), 3),
       note=f"{int(cat_nk['g'] or 0)} (business, category) groups; breaks the Silver DDL PK — DQ-BUSINESS-005")
print("\n📋 Internal child frames — FK / PK / cardinality profile:")
display(pd.DataFrame(internal_rows))

# ══ B) External — sibling bronze tables → yelp_business ══
print(f"\n{'─' * 60}\n📌 B) External — bronze_final siblings .business_id → yelp_business")
SIB_RULE = {  # declared cardinality, mandatory?, rationale
    "yelp_review":  ("1:N", True,  "every business in the Yelp Open Dataset has ≥ 5 reviews → a business without reviews means a broken extract"),
    "yelp_tip":     ("0:N", False, "tips are optional user content"),
    "yelp_checkin": ("0:1", False, "check-ins are optional; bronze stores 1 row per business with a comma-separated timestamp list")}
external_rows, sib_ids = [], {}
for t, raw in sib_raw.items():
    card, mandatory, why = SIB_RULE[t]
    per = raw.groupBy("business_id").agg(F.count("*").alias("child_rows"))
    sib_ids[t] = per
    j = per.join(parent_keys, "business_id", "left")
    is_orph = F.col("_parent").isNull() & F.col("business_id").isNotNull()
    r = j.agg(F.count("business_id").alias("ids"), F.sum("child_rows").alias("rows"),
              F.sum(F.when(is_orph, 1).otherwise(0)).alias("orph_ids"),
              F.sum(F.when(is_orph, F.col("child_rows")).otherwise(0)).alias("orph_rows"),
              F.sum(F.when(F.col("business_id").isNull(), F.col("child_rows")).otherwise(0)).alias("null_rows"),
              F.sum(F.when(~F.col("business_id").rlike(ID_RE), F.col("child_rows")).otherwise(0)).alias("bad_rows"),
              F.sum(F.when(F.col("business_id") != F.trim("business_id"), F.col("child_rows")).otherwise(0)).alias("ws_rows"),
              F.max("child_rows").alias("max_per_biz")).first()
    rows, ids = int(r["rows"]), int(r["ids"])
    orph_rows, null_rows = int(r["orph_rows"] or 0), int(r["null_rows"] or 0)
    missing = parent_keys.join(per, "business_id", "left_anti").count()
    print(f"\n   ── {t}.business_id → yelp_business  ({card}; {rows:,} rows, {ids:,} distinct businesses) ──")
    dq_add(findings_671, t, "business_id → yelp_business (View 1)", "child FK NOT NULL and present in yelp_business", rows, orph_rows + null_rows,
           examples=dq_examples(j, "business_id", is_orph, 5), note=f"{int(r['orph_ids'] or 0):,} orphan ids")
    dq_add(findings_671, t, "business_id format (View 3)", "22-char URL-safe base64, no padding", rows,
           int(r["bad_rows"] or 0) + int(r["ws_rows"] or 0))
    if card == "0:1":
        dq_add(findings_671, t, "cardinality 0:1", "at most one bronze row per business", rows, rows - ids)
    dq_add(findings_671, t, "businesses without children (View 2)", f"business has ≥ 1 {t} row", N_PARENT, missing,
           status=("FINDING" if mandatory else "INFO") if missing else "PASS", note=why)
    external_rows.append({"Sibling": t, "Cardinality": card, "Mandatory": mandatory, "Rows": rows, "Distinct businesses": ids,
                          "Orphan rows": orph_rows, "NULL FK rows": null_rows, "Malformed FK rows": int(r["bad_rows"] or 0) + int(r["ws_rows"] or 0),
                          "Max rows / business": int(r["max_per_biz"]), "Businesses w/o rows": missing,
                          "Coverage %": round((N_PARENT - missing) / N_PARENT * 100, 2)})
    ri_metrics[f"{t}_orphan_rows"], ri_metrics[f"{t}_missing"] = orph_rows + null_rows, missing
print("\n📋 External siblings — FK profile:")
display(pd.DataFrame(external_rows))

# ── Who are the businesses without tips / check-ins? (MNAR profile for View 2) ──
sibling_cov = df_business.select("business_id", "state", "is_open", "review_count", "stars")
for t, per in sib_ids.items():
    sibling_cov = sibling_cov.join(per.select("business_id", F.lit(True).alias(f"in_{t[5:]}")), "business_id", "left")
sibling_cov = sibling_cov.na.fill(False, subset=[f"in_{t[5:]}" for t in sib_ids])
print("\n📋 Profile of businesses by sibling presence (low-activity businesses are the ones without tips / check-ins):")
display(sibling_cov.groupBy("in_review", "in_tip", "in_checkin")
        .agg(F.count("*").alias("businesses"), F.round(F.count("*") / F.lit(N_PARENT) * 100, 2).alias("pct"),
             F.round(F.avg(F.col("is_open").cast("int")) * 100, 1).alias("open_pct"),
             F.percentile_approx("review_count", 0.5).alias("median_review_count"),
             F.round(F.avg(F.col("stars").cast("double")), 2).alias("avg_stars"))
        .orderBy(F.desc("businesses")))

findings_671_df = pd.DataFrame(findings_671)
print(f"\n   {len(findings_671_df)} FK checks | {int((findings_671_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((findings_671_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


In [0]:
# 6.7.2) Lookup Table Conformance  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
#   A) Reference lookups  — values that become Silver lookup / dimension keys must exist in their reference domain
#        A1 state          → jurisdiction list (US + CA) and the Yelp 11-metro footprint      (6.2.1 constants)
#        A2 7 enum attrs   → ENUM_DOMAINS (DDL comments)
#        A3 category_name  → a clean category vocabulary (no delimiter fragments, no case / space variants, no blanks)
#        (dict keys → contract key map was proven in 6.6.6 F and is not repeated here)
#   B) Sibling conformance — yelp_business must be one conformed dimension for review / tip / check-in
#        B1 presence overlap of the three fact tables     B2 coverage per state (systematic regional gaps?)
#        B3 business reviews written by users missing from yelp_user (reviewer join loses rows)
from pyspark.sql import functions as F
import pandas as pd

print("=" * 80)
print("6.7.2) LOOKUP TABLE CONFORMANCE  —  Business DataFrames")
print("=" * 80)
findings_672 = []

# ══ A1) state → jurisdiction / footprint lookup ══
print(f"\n{'─' * 60}\n📌 A1) state → jurisdiction lookup ({len(VALID_JURISDICTIONS)} codes) and Yelp footprint ({len(YELP_FOOTPRINT_STATES)} codes)")
run_checks(findings_672, df_business, [
    chk("df_business", "state → jurisdiction lookup", "state ∈ USPS states / territories ∪ Canadian provinces",
        ~F.col("state").isin(sorted(VALID_JURISDICTIONS)), scope=F.col("state").isNotNull(),
        ex=F.concat_ws(" | ", "name", "city", "state"), note="DQ-BUSINESS-009"),
    chk("df_business", "state → Yelp footprint lookup", "state ∈ the 14 codes of the 11 Yelp metros",
        ~F.col("state").isin(sorted(YELP_FOOTPRINT_STATES)), scope=F.col("state").isNotNull(),
        ex=F.concat_ws(" | ", "name", "city", "state"), note="DQ-BUSINESS-010")])

# ══ A2) enum attributes → ENUM_DOMAINS ══
print(f"\n{'─' * 60}\n📌 A2) Enum attributes → ENUM_DOMAINS lookup")
run_checks(findings_672, df_attributes, [
    chk("df_attributes", f"{c} → enum lookup", f"value ∈ {{{', '.join(sorted(dom))}}}", ~F.col(c).isin(sorted(dom)),
        scope=F.col(c).isNotNull(), ex=F.col(c)) for c, dom in ENUM_DOMAINS.items()])
print("   Enum value distribution (lookup rows that a Silver dimension would need):")
display(pd.DataFrame([{"Attribute": c, "Lookup values": ", ".join(sorted(dom)),
                       "Observed": " ; ".join(f"{r[0]}={r[1]:,}" for r in df_attributes.where(F.col(c).isNotNull()).groupBy(c).count().orderBy(F.desc("count")).collect())}
                      for c, dom in ENUM_DOMAINS.items()]))

# ══ A3) category_name → category vocabulary ══
print(f"\n{'─' * 60}\n📌 A3) category_name → category dimension vocabulary")
# fragments produced when a multi-word label containing ', ' is split by the list delimiter (DQ-BUSINESS-013)
DELIM_FRAGMENTS = {"Beer, Wine & Spirits": ["Beer", "Wine & Spirits"], "Books, Mags, Music & Video": ["Books", "Mags", "Music & Video"],
                   "Used, Vintage & Consignment": ["Used", "Vintage & Consignment"], "Wills, Trusts, & Probates": ["Wills", "Trusts", "& Probates"]}
FRAG = sorted({f for v in DELIM_FRAGMENTS.values() for f in v})
fold = F.regexp_replace(F.lower(F.trim("category_name")), r"\s+", " ")
vocab = df_categories.groupBy("category_name").agg(F.countDistinct("business_id").alias("biz"), F.count("*").alias("rows"))
n_labels = vocab.count()
collide = (vocab.withColumn("k", fold).groupBy("k").agg(F.count("*").alias("variants"), F.sum("rows").alias("rows"),
                                                        F.concat_ws(" / ", F.collect_list("category_name")).alias("labels"))
           .where("variants > 1"))
c_rows = collide.agg(F.sum(F.col("variants") - 1).alias("v"), F.sum("rows").alias("r")).first()
print(f"   {n_labels:,} distinct category labels across {n_cat:,} category rows")
run_checks(findings_672, df_categories, [
    chk("df_categories", "category_name blank / padded", "label non-empty and trimmed",
        (F.trim("category_name") == "") | (F.col("category_name") != F.trim("category_name")), ex=F.col("category_name")),
    chk("df_categories", "category_name delimiter fragment", "label is not a fragment of a ', '-containing Yelp category",
        F.col("category_name").isin(FRAG), ex=F.col("category_name"), note="DQ-BUSINESS-013 — fragments would become fake dimension members"),
    chk("df_categories", "category_name starts with a letter/digit", "label begins with [A-Za-z0-9]",
        ~F.col("category_name").rlike(r"^[A-Za-z0-9]"), ex=F.col("category_name"), note="'& Probates' = DQ-BUSINESS-013 fragment")])
dq_add(findings_672, "df_categories", "category_name case / space variants", "one label per folded (lower, single-space) key",
       n_labels, int(c_rows["v"] or 0), examples=dq_examples(collide, "labels", F.lit(True), 5))
n_single = vocab.where("biz = 1").count()
dq_add(findings_672, "df_categories", "category_name used by one business", "label shared by ≥ 2 businesses", n_labels, n_single,
       status=info_or_pass(n_single), examples=dq_examples(vocab, "category_name", F.col("biz") == 1, 8),
       note="rare but genuine Yelp categories; no external taxonomy available to validate them")
frag_biz = df_categories.where(F.col("category_name").isin(FRAG)).select("business_id").distinct().count()
ri_metrics.update({"cat_labels": n_labels, "cat_fragment_biz": frag_biz, "cat_singletons": n_single})
print("   Fragment labels and the Yelp category they come from:")
display(vocab.where(F.col("category_name").isin(FRAG))
        .withColumn("source_category", F.create_map(*[x for full, fr in DELIM_FRAGMENTS.items() for f in fr for x in (F.lit(f), F.lit(full))])[F.col("category_name")])
        .orderBy("source_category", "category_name"))

# ══ B) Sibling conformance — one business dimension for all fact tables ══
print(f"\n{'─' * 60}\n📌 B1) Presence of each business in review / tip / check-in")
venn = (sibling_cov.groupBy("in_review", "in_tip", "in_checkin").count()
        .withColumn("pct", F.round(F.col("count") / F.lit(N_PARENT) * 100, 2)).orderBy(F.desc("count")))
display(venn)
all3 = sibling_cov.where("in_review AND in_tip AND in_checkin").count()
dq_add(findings_672, "yelp_business ↔ review/tip/checkin", "present in all three fact tables", "conformed dimension coverage",
       N_PARENT, N_PARENT - all3, status=info_or_pass(N_PARENT - all3),
       note=f"{all3 / N_PARENT * 100:.1f}% in all three; gaps are tip / check-in only (review coverage is 100%, 6.7.1)")

print(f"\n{'─' * 60}\n📌 B2) Coverage per state (≥ 100 businesses) — systematic regional gaps?")
by_state = (sibling_cov.groupBy("state")
            .agg(F.count("*").alias("businesses"), *[F.round(F.avg(F.col(f"in_{k}").cast("int")) * 100, 1).alias(f"{k}_pct") for k in ("review", "tip", "checkin")],
                 F.round(F.avg(F.col("is_open").cast("int")) * 100, 1).alias("open_pct"))
            .orderBy(F.desc("businesses")))
glob = sibling_cov.agg(*[F.avg(F.col(f"in_{k}").cast("int")).alias(k) for k in ("tip", "checkin")]).first()
big = by_state.where("businesses >= 100")
display(big)
gap_states = big.where((F.col("tip_pct") < glob["tip"] * 100 - 10) | (F.col("checkin_pct") < glob["checkin"] * 100 - 10))
n_gap = gap_states.count()
dq_add(findings_672, "yelp_business ↔ tip/checkin", "coverage per state", "no state (≥ 100 businesses) > 10 pp below global tip / check-in coverage",
       big.count(), n_gap, status=info_or_pass(n_gap), examples=dq_examples(gap_states, F.concat_ws(" ", "state", "tip_pct", "checkin_pct"), F.lit(True)),
       note=f"global: tip {glob['tip'] * 100:.1f}%, check-in {glob['checkin'] * 100:.1f}%")

print(f"\n{'─' * 60}\n📌 B3) Business reviews written by users missing from yelp_user")
users = spark.table("yelp_poc_uc.bronze_final.yelp_user").select("user_id")
rv_orph = sib_raw["yelp_review"].select("business_id", "user_id").join(users, "user_id", "left_anti")
ro = rv_orph.agg(F.count("*").alias("rows"), F.countDistinct("business_id").alias("biz"), F.countDistinct("user_id").alias("users")).first()
dq_add(findings_672, "yelp_review → yelp_user (business view)", "businesses with reviews from unknown users",
       "every reviewer of a business exists in yelp_user", N_PARENT, int(ro["biz"]), status=info_or_pass(int(ro["biz"])),
       examples=dq_examples(rv_orph, "business_id", F.lit(True), 5),
       note=f"{int(ro['rows'])} reviews by {int(ro['users'])} users; owned by the review / user EDA — a business → reviewer join drops these rows")
ri_metrics.update({"all3": all3, "orphan_user_reviews": int(ro["rows"]), "orphan_user_biz": int(ro["biz"])})

findings_672_df = pd.DataFrame(findings_672)
print(f"\n   {len(findings_672_df)} lookup checks | {int((findings_672_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((findings_672_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


In [0]:
# 6.7.3) Cross-Table Aggregate Reconciliation  —  yelp_business stored values vs the fact tables
# ─────────────────────────────────────────────────────────────────────
# yelp_business carries values that are DERIVED from yelp_review. Each one is tested against more than one candidate
# definition, so that a genuine data error can be told apart from a definition mismatch:
#   review_count  vs  COUNT(*)                        (every review row)
#                 vs  COUNT(DISTINCT user_id)          (Yelp keeps only the CURRENT review per user; older ones are superseded)
#   stars         vs  half-up round of AVG(stars) over all reviews
#                 vs  half-up round of AVG(stars) over the latest review per user   (+ the ± 0.25 rounding bound)
#   review_count  ≥ 5                                  (dataset inclusion rule)
#   is_open       vs  the last review / tip / check-in (open but silent ≥ 1 y; closed but active in the final 30 days)
# The residual mismatches are then related to how close they are to the snapshot date (counters taken at a different time).
from pyspark.sql import functions as F, Window
import pandas as pd

print("=" * 80)
print("6.7.3) CROSS-TABLE AGGREGATE RECONCILIATION  —  stored business values vs fact tables")
print("=" * 80)
findings_673 = []
half_up = lambda c: F.floor(c * 2 + F.lit(0.5 + 1e-9)) / 2

# ══ Computed aggregates per business ══
rv = sib_raw["yelp_review"].select("review_id", "business_id", "user_id", "stars", F.to_timestamp("date").alias("ts"))
rv = rv.withColumn("rk", F.row_number().over(Window.partitionBy("business_id", "user_id").orderBy(F.desc("ts"), F.desc("review_id"))))
agg_r = rv.groupBy("business_id").agg(
    F.count("*").alias("n_reviews"), F.countDistinct("user_id").alias("n_reviewers"),
    F.sum((F.col("rk") > 1).cast("int")).alias("n_superseded"),
    F.avg("stars").alias("avg_all"), F.avg(F.when(F.col("rk") == 1, F.col("stars"))).alias("avg_current"),
    F.min("ts").alias("first_review"), F.max("ts").alias("last_review"))
agg_t = sib_raw["yelp_tip"].groupBy("business_id").agg(F.max(F.to_timestamp("date")).alias("last_tip"))
agg_c = sib_raw["yelp_checkin"].groupBy("business_id").agg(
    F.max(F.array_max(F.transform(F.split("date", ","), lambda x: F.to_timestamp(F.trim(x))))).alias("last_checkin"))

rec = (df_business.select("business_id", "name", "city", "state", "is_open", "review_count", F.col("stars").cast("double").alias("stars"))
       .join(agg_r, "business_id", "left").join(agg_t, "business_id", "left").join(agg_c, "business_id", "left")
       .withColumn("stars_all", half_up(F.col("avg_all"))).withColumn("stars_current", half_up(F.col("avg_current")))
       .withColumn("last_activity", F.greatest("last_review", "last_tip", "last_checkin")))
SNAP = rec.agg(F.max("last_activity")).first()[0]
rec = rec.withColumn("days_before_snap", F.datediff(F.lit(SNAP), F.col("last_review")))
print(f"   Snapshot (latest review / tip / check-in): {SNAP}")

# ══ A) review_count ══
print(f"\n{'─' * 60}\n📌 A) review_count — which definition does Yelp store?")
has_r = F.col("n_reviews").isNotNull()
ex_rc = F.concat_ws(" | ", "business_id", "name", F.format_string("stored=%d rows=%d reviewers=%d", "review_count", "n_reviews", "n_reviewers"))
vA = run_checks(findings_673, rec, [
    chk("df_business ↔ yelp_review", "review_count = COUNT(*) reviews", "stored counter equals the number of review rows",
        F.col("review_count") != F.col("n_reviews"), scope=has_r, status="INFO", ex=ex_rc,
        note="definition mismatch, not an error: superseded (older) reviews of the same user are in the file but not in the counter"),
    chk("df_business ↔ yelp_review", "review_count = COUNT(DISTINCT user_id)", "stored counter equals the number of distinct reviewers",
        F.col("review_count") != F.col("n_reviewers"), scope=has_r, ex=ex_rc, note="residual = counter snapshot drift (see recency table)"),
    chk("df_business", "review_count ≥ 5", "dataset inclusion rule (stored)", F.col("review_count") < 5, ex=ex_rc),
    chk("yelp_review", "COUNT(DISTINCT user_id) ≥ 5", "dataset inclusion rule (computed from the fact table)", F.col("n_reviewers") < 5,
        scope=has_r, ex=ex_rc, note="reviews missing from the extract for low-volume businesses")])
ri_metrics.update({"rc_vs_rows": vA["review_count = COUNT(*) reviews"], "rc_vs_reviewers": vA["review_count = COUNT(DISTINCT user_id)"]})
sup = rec.agg(F.sum("n_superseded").alias("rows"), F.sum((F.col("n_superseded") > 0).cast("int")).alias("biz")).first()
ri_metrics.update({"superseded_rows": int(sup["rows"]), "superseded_biz": int(sup["biz"])})
print(f"   Superseded reviews (same user, same business, not the latest): {int(sup['rows']):,} rows in {int(sup['biz']):,} businesses")

print("\n   Direction and size of the residual (stored − distinct reviewers):")
display(rec.where(has_r & (F.col("review_count") != F.col("n_reviewers")))
        .withColumn("diff", F.col("review_count") - F.col("n_reviewers")).groupBy("diff")
        .agg(F.count("*").alias("businesses"), F.round(F.avg(F.col("is_open").cast("int")) * 100, 1).alias("open_pct"),
             F.percentile_approx("days_before_snap", 0.5).alias("median_days_last_review_before_snap"))
        .orderBy("diff"))
print("   Is the residual a snapshot-timing effect? Share of businesses whose last review is ≤ 30 days before the snapshot:")
display(rec.where(has_r).withColumn("counter_matches", F.col("review_count") == F.col("n_reviewers")).groupBy("counter_matches")
        .agg(F.count("*").alias("businesses"),
             F.round(F.avg((F.col("days_before_snap") <= 30).cast("int")) * 100, 1).alias("pct_last_review_le_30d"),
             F.round(F.avg((F.col("days_before_snap") <= 7).cast("int")) * 100, 1).alias("pct_last_review_le_7d"),
             F.percentile_approx("days_before_snap", 0.5).alias("median_days_before_snap")))

# ══ B) stars ══
print(f"\n{'─' * 60}\n📌 B) stars — half-star rounding of which average?")
ex_st = F.concat_ws(" | ", "business_id", "name", F.format_string("stored=%.1f avg_current=%.3f avg_all=%.3f n=%d",
                                                              "stars", "avg_current", "avg_all", "n_reviewers"))
vB = run_checks(findings_673, rec, [
    chk("df_business ↔ yelp_review", "stars = round½(AVG all reviews)", "stored rating = half-up rounding of the mean of every review row",
        F.col("stars") != F.col("stars_all"), scope=has_r, status="INFO", ex=ex_st,
        note="definition mismatch: superseded reviews are excluded from the Yelp rating"),
    chk("df_business ↔ yelp_review", "stars = round½(AVG current review per user)", "stored rating = half-up rounding of the mean of each user's latest review",
        F.col("stars") != F.col("stars_current"), scope=has_r, ex=ex_st),
    chk("df_business ↔ yelp_review", "|stars − AVG current| ≤ 0.25", "stored rating is within the half-star rounding bound of the computed mean",
        F.abs(F.col("stars") - F.col("avg_current")) > 0.25 + 1e-9, scope=has_r, ex=ex_st)])
ri_metrics.update({"stars_vs_all": vB["stars = round½(AVG all reviews)"], "stars_vs_current": vB["stars = round½(AVG current review per user)"],
                   "stars_bound": vB["|stars − AVG current| ≤ 0.25"]})
print("   Stars residual vs counter residual (do the two drifts hit the same businesses?):")
display(rec.where(has_r).groupBy((F.col("review_count") == F.col("n_reviewers")).alias("count_matches"),
                                (F.col("stars") == F.col("stars_current")).alias("stars_matches")).count().orderBy("count_matches", "stars_matches"))

# ══ C) is_open vs activity ══
print(f"\n{'─' * 60}\n📌 C) is_open vs last review / tip / check-in")
days_act = F.datediff(F.lit(SNAP), F.col("last_activity"))
ex_op = F.concat_ws(" | ", "name", "city", "state", F.date_format("last_activity", "yyyy-MM-dd"))
vC = run_checks(findings_673, rec.withColumn("days_act", days_act), [
    chk("df_business ↔ review/tip/checkin", "is_open = TRUE ⇒ activity in the last 365 days", "an open business shows some activity in the final year",
        F.col("days_act") > 365, scope=F.col("is_open") == True, status="INFO", ex=ex_op,
        note="stale is_open — DQ-BUSINESS-036 (re-measured on review + tip + check-in)"),
    chk("df_business ↔ review/tip/checkin", "is_open = FALSE ⇒ no activity in the last 30 days", "a closed business is not reviewed / visited right before the snapshot",
        F.col("days_act") <= 30, scope=F.col("is_open") == False, status="INFO", ex=ex_op,
        note="recently closed (activity still trailing) or a wrong closed flag")])
ri_metrics.update({"open_silent": vC["is_open = TRUE ⇒ activity in the last 365 days"], "closed_active": vC["is_open = FALSE ⇒ no activity in the last 30 days"]})
print("   Days from last activity to snapshot, by is_open:")
display(rec.withColumn("days_act", days_act).groupBy("is_open")
        .agg(F.count("*").alias("businesses"), *[F.percentile_approx("days_act", q).alias(f"p{int(q * 100)}") for q in (0.1, 0.5, 0.9)])
        .orderBy("is_open"))

recon_673 = rec  # per-business reconciliation frame, reused by the DQ notes
findings_673_df = pd.DataFrame(findings_673)
print(f"\n   {len(findings_673_df)} reconciliation checks | {int((findings_673_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((findings_673_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


---
#### 🚨 DQ-BUSINESS-055 — `review_count` / `stars` Count Only Each User's Current Review (Undocumented Aggregate Definition)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-055` |
| **Severity** | MEDIUM (semantic: a naive re-derivation in Silver / Gold silently changes 45% of counters and 5.6% of ratings) |
| **Detected In** | Phase 6.7 — Referential Integrity → **6.7.3 Cross-Table Aggregate Reconciliation**: A) `review_count = COUNT(*)`, B) `stars = round½(AVG all reviews)` |
| **Also Confirmed In** | Independent SQL over `bronze_final.yelp_review`. `review_count = COUNT(DISTINCT user_id)` for 150,037 / 150,346 businesses (99.79%). `stars = half-up round½(AVG of each user's latest review)` for 150,327 (99.99%) |
| **Affected Columns** | `yelp_business.review_count`, `yelp_business.stars` ↔ `yelp_review (business_id, user_id, stars, date)` |
| **Total Affected Rows** | **244,520** superseded review rows (same user, same business, not the latest) in **67,301** businesses. `review_count ≠ COUNT(*)` for **67,362** businesses. `stars ≠ round½(AVG all)` for **8,456** businesses |
| **Invalid Value** | `T-Mobile (mGFsPRHdE6SMV1gYQ6GzgQ)`: stored 2.0, mean of current reviews 2.167 → 2.0 ✔, mean of all reviews 2.429 → 2.5 ✘. Max per business: 289 more review rows than the stored counter |
| **Domain Rule** | On Yelp, a user's newer review of a business **replaces** the older one in the business's count and rating. Older reviews stay visible as "previous reviews" but do not count. The review extract includes these superseded rows |

**Description:** The stored counters are correct. The definition behind them is not written down anywhere. A Silver / Gold job that recomputes `review_count` with `COUNT(*)` or `stars` with `AVG(stars)` over `yelp_review` will disagree with Yelp for tens of thousands of businesses. Each reconciliation run will then raise false alarms. Review-level analytics (sentiment, rating trends) also double-count users who reviewed the same business more than once.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Add `is_current_review` to Silver `yelp_review` (latest review per `(business_id, user_id)`, tie-break on `review_id`). Derive every business aggregate from current reviews only, and document the definition in the DDL column comments of `review_count` / `stars`** | Reproduces Yelp's numbers exactly and keeps the history |
| 2 | Delete superseded reviews | Matches the counters, but destroys rating-history data (244k rows) |
| 3 | Keep the stored values and never recompute | Correct today, but drifts the moment new reviews land (see DQ-BUSINESS-056) |

**Best Logical Solution:** **Option 1 — flag current reviews and derive aggregates from them, using the documented formula `stars = floor(2·AVG + 0.5) / 2`.**

---

---
#### 🚨 DQ-BUSINESS-056 — `review_count` / `stars` Taken at a Different Moment Than the Review Extract (Snapshot Drift)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-056` |
| **Severity** | LOW (309 businesses, ± 1–2 reviews, max half a star) |
| **Detected In** | Phase 6.7 — Referential Integrity → **6.7.3 Cross-Table Aggregate Reconciliation**: `review_count = COUNT(DISTINCT user_id)`, `stars = round½(AVG current)`, `\|stars − AVG current\| ≤ 0.25` |
| **Also Confirmed In** | Independent SQL: 279 businesses have stored < computed and 30 have stored > computed, with a maximum gap of 2 |
| **Affected Columns** | `yelp_business.review_count`, `yelp_business.stars` |
| **Total Affected Rows** | **309** businesses with a counter mismatch (276 at −1, 3 at −2, 30 at +1). **19** of them also have a rating mismatch, 16 of which fall outside the ± 0.25 rounding bound |
| **Invalid Value** | `Roy's Appliance Service`: stored 5, distinct reviewers 6. `Mr.Smoke`: stored 2.5, mean of current reviews 3.0 |
| **Domain Rule** | A derived counter has to reflect the same snapshot as the fact table it summarises |

**Description:** The residual is a timing effect, not random corruption. In 59.5% of the mismatched businesses the last review is ≤ 30 days before the snapshot (2022-01-19), against 17.2% of businesses that match. The median gap is 13 days vs 236 days. Every one of the 19 rating mismatches is inside the 309 counter mismatches. So the business file was frozen slightly before (stored < computed) or after (stored > computed: reviews later removed) the review file.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Recompute `review_count` / `stars` in Silver Enriched from current reviews (DQ-BUSINESS-055). Keep the Yelp values as `review_count_source` / `stars_source` with a `aggregates_in_sync` flag** | The aggregates become consistent with the fact table, and the source value stays auditable |
| 2 | Overwrite the stored values with the recomputed ones | Consistent, but loses the source value |
| 3 | Do nothing | 309 businesses disagree between the dimension and the fact table |

**Best Logical Solution:** **Option 1 — recompute from current reviews and keep the source values with an in-sync flag.**

---

In [0]:
# 6.7.4) Referential Integrity — Consolidated Summary  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Status per sub-check, headline metrics and every FINDING / INFO of 6.7.1 – 6.7.3 in one place.
import pandas as pd

print("=" * 80)
print("6.7) REFERENTIAL INTEGRITY — CONSOLIDATED SUMMARY  (Business DataFrames)")
print("=" * 80)

SECTIONS_67 = [("6.7.1 FK Validity", "findings_671_df"), ("6.7.2 Lookup Conformance", "findings_672_df"),
               ("6.7.3 Aggregate Reconciliation", "findings_673_df")]
parts, missing_sections = [], []
for sec, var in SECTIONS_67:
    (parts.append(globals()[var].assign(Section=sec)) if var in globals() else missing_sections.append(sec))
if missing_sections:
    print(f"   ⚠️  Not yet run (re-run these cells first): {missing_sections}")
all_67 = pd.concat(parts, ignore_index=True)

status_67 = (all_67.pivot_table(index="Section", columns="Status", values="Rule", aggfunc="count", fill_value=0)
             .reindex(columns=["FINDING", "INFO", "PASS"], fill_value=0).reset_index())
status_67["Checks"] = status_67[["FINDING", "INFO", "PASS"]].sum(axis=1)
print("\n📋 Check status per sub-section:")
display(status_67)

M = ri_metrics
headline_67 = pd.DataFrame([
    ("FK", "orphan review / tip / check-in rows", f"{M['yelp_review_orphan_rows']:,} / {M['yelp_tip_orphan_rows']:,} / {M['yelp_checkin_orphan_rows']:,}", "every fact row joins to a business"),
    ("FK", "businesses without reviews / tips / check-ins", f"{M['yelp_review_missing']:,} / {M['yelp_tip_missing']:,} / {M['yelp_checkin_missing']:,}", "reviews mandatory; tips / check-ins optional (low-activity businesses)"),
    ("Lookup", "distinct category labels · businesses carrying delimiter fragments", f"{M['cat_labels']:,} · {M['cat_fragment_biz']:,}", "fragments = fake dimension members (DQ-013)"),
    ("Lookup", "businesses in all three fact tables", f"{M['all3']:,} ({M['all3'] / N_PARENT * 100:.1f}%)", "conformed dimension coverage"),
    ("Lookup", "reviews by users missing from yelp_user (businesses)", f"{M['orphan_user_reviews']:,} ({M['orphan_user_biz']:,})", "owned by review / user EDA"),
    ("Aggregate", "review_count ≠ COUNT(*) · ≠ COUNT(DISTINCT user_id)", f"{M['rc_vs_rows']:,} · {M['rc_vs_reviewers']:,}", "counter = distinct current reviewers"),
    ("Aggregate", "superseded review rows (businesses)", f"{M['superseded_rows']:,} ({M['superseded_biz']:,})", "must be excluded when re-deriving counters"),
    ("Aggregate", "stars ≠ round½(AVG all) · ≠ round½(AVG current) · outside ± 0.25", f"{M['stars_vs_all']:,} · {M['stars_vs_current']:,} · {M['stars_bound']:,}", "rating = half-up mean of current reviews"),
    ("Aggregate", "open but silent ≥ 1 y · closed but active ≤ 30 d", f"{M['open_silent']:,} · {M['closed_active']:,}", "stale is_open (DQ-036)"),
], columns=["Level", "Metric", "Value", "Interpretation"])
print("\n📋 Headline referential-integrity metrics:")
display(headline_67)

find_67 = all_67[all_67["Status"] == "FINDING"][["Section", "DataFrame", "Column", "Rule", "Checked", "Violations", "Pct", "Examples", "Note"]]
print(f"\n📋 All FINDINGs in 6.7 ({len(find_67)}):")
display(find_67.sort_values("Violations", ascending=False).reset_index(drop=True))
info_67 = all_67[all_67["Status"] == "INFO"][["Section", "DataFrame", "Column", "Checked", "Violations", "Pct", "Note"]]
print(f"\n📋 INFO profiles in 6.7 ({len(info_67)}):")
display(info_67.reset_index(drop=True))

n_f, n_i, n_p = (int((all_67["Status"] == s).sum()) for s in ("FINDING", "INFO", "PASS"))
print(f"\n   {len(all_67)} checks | {n_f} FINDING | {n_i} INFO | {n_p} PASS")
summary_67_df = all_67
print("\nREFERENTIAL INTEGRITY CHECKS COMPLETE")
print("=" * 80)


---
#### 📋 6.7 Referential Integrity — Error Register

New errors raised in Phase 6.7 (next free ID: **DQ-BUSINESS-057**).

| Error ID | Title | Severity | Detected In | Affected | Best Solution (short) |
| --- | --- | --- | --- | --- | --- |
| DQ-BUSINESS-055 | Stored aggregates count only each user's current review | MEDIUM | 6.7.3 A / B | 244,520 superseded reviews in 67,301 businesses | `is_current_review` flag; aggregates from current reviews |
| DQ-BUSINESS-056 | Counter snapshot drift | LOW | 6.7.3 A / B | 309 counters, 19 ratings | Recompute + keep `*_source` and an in-sync flag |

**Clean results (PASS):** `business_id` is unique, NOT NULL, a well-formed 22-char id and case-insensitive unique. All 10 TE.2 child frames have 0 orphan / malformed FKs and 0 PK duplicates, and the 1:1 frames hold exactly one row per business. Review (6,990,280), tip (908,915) and check-in (131,930) rows have 0 orphan, NULL or malformed `business_id`. Every business has reviews, and check-in is 0:1. All 7 enum attributes conform to `ENUM_DOMAINS`. The category vocabulary (1,311 labels) has no blank / padded labels and no case / space variants. Tip / check-in coverage shows no state-level gap.

**Re-detected existing errors (no new ID):** DQ-BUSINESS-005 (43 duplicate `(business_id, category_name)` pairs break the Silver category PK), DQ-BUSINESS-009 (`XMS`) / -010 (21 off-footprint states) in the state lookup, DQ-BUSINESS-013 (10 delimiter-fragment labels, 9,371 rows in 4,150 businesses, would become fake category-dimension members, including `& Probates`), DQ-BUSINESS-036 (28,038 open businesses with no review / tip / check-in in the final year, plus 130 closed businesses active in the final 30 days).

**INFO (expected, documented):** businesses without tips (44,153) or check-ins (18,416) are low-activity listings (median 7–9 reviews vs 23). Dict-frame coverage ranges from 0.02% (DietaryRestrictions) to 59% (BusinessParking), which is structural not-applicable (DQ-BUSINESS-034 / 053). 33 reviews in 32 businesses come from users missing from `yelp_user` and are owned by the review / user EDA.

---


### 6.8) BUSINESS RULE & ACCURACY CHECKS
*Checks whether business listings are plausible in a Yelp business-domain context. These go beyond schema, null, range and cross-column checks (6.1–6.7 are cross-referenced, not repeated). Most checks run on `b68`, a one-row-per-business frame built in 6.8.1 from the TE.2 frames plus review / tip / check-in metrics.*

- **Business Rule Violations** — listings that contradict how Yelp businesses work.
  - *Entity & name accuracy:* Yelp event pages (`Yelp Events` category) listed as businesses and marked open; the name carries an operational status ("- CLOSED", "Coming Soon"), copies the address or city, is a placeholder token, or contains a phone number or web domain.
  - *Category ↔ attribute plausibility:* `$$$$` price on fast-food / food-truck formats, an `upscale` ambience at price `$`, a restaurant sub-category without the `Restaurants` parent.
  - *Category ↔ hours plausibility:* bars that close every day before 17:00, Breakfast & Brunch that never opens before 15:00, restaurants open < 10 h / week, weekly hours > 168.
  - *Rating ↔ volume plausibility:* a perfect 5.0 or ≤ 1.5 rating with ≥ 100 reviews.

- **Statistical Outlier Detection** — extreme values of the business-level metrics: `review_count`, categories per business, attributes answered, weekly open hours, tips, check-ins and name length. Heavy-tailed counts use Tukey 3×IQR fences on log1p. Contextual outliers: review volume vs the city (robust z) and a chain unit's rating vs its chain median.

- **Distribution Anomaly Detection** — the shape of the business population: rating heaping by review-count band (small-sample effect), left-truncation of `review_count` at 5, `is_open` rate by state, cohort (first-activity year) sizes and survival, category frequency (long tail), and opening-time minute heaping.

- **Concentration & Pareto Analysis** — whether a small share of businesses, chains, cities or categories dominates reviews, check-ins, tips or listings (Gini coefficient, top-k% share, Lorenz table), overall and by state.

- **Business Activity Span Analysis** — each business's lifecycle across review / tip / check-in: activity before Yelp existed, compressed lifecycles and single-day review bursts, long dormancy followed by reactivation (possible re-used listing), new businesses near the snapshot, and dormancy by `is_open`.


In [0]:
# 6.8.1) Business Rule Violations  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Builds b68 = ONE row per business (df_business + categories + attributes + hours summary + review/tip/check-in
# metrics from 6.7.3 recon_673). b68 is reused by 6.8.2 – 6.8.5.
# Rules (not covered in 6.2 – 6.7; earlier findings are cross-referenced):
#   A) Entity & name accuracy  — Yelp event pages listed as businesses, status text in the name, placeholder / address-as-name,
#                                phone / domain / no-letter names
#   B) Category ↔ attribute    — $$$$ on budget formats, upscale at $, restaurant sub-category without 'Restaurants'
#   C) Category ↔ hours        — bars closing before 17:00, brunch opening ≥ 15:00, restaurants < 10 h/wk, > 168 h/wk
#   D) Rating ↔ volume         — perfect 5.0 / ≤ 1.5 stars with ≥ 100 reviews
from pyspark.sql import functions as F
import pandas as pd

print("=" * 80)
print("6.8.1) BUSINESS RULE VIOLATIONS  —  Business DataFrames")
print("=" * 80)
assert "recon_673" in globals(), "Run 6.7.3 first — b68 reuses its per-business review / tip / check-in metrics (recon_673)"
findings_681, m68 = [], {}

print("\n   Previously verified (cross-reference, not repeated):")
for chk_name, ref in [("stars on the half-star grid / review_count ≥ 5", "6.2.1 / 6.7.3"),
                      ("attire / kids / ambience / drive-thru contradictions", "6.6.3 (DQ-042 … 045)"),
                      ("BestNights / GoodForMeal vs hours, HappyHour w/o alcohol", "6.6.5 (DQ-049 … 051)"),
                      ("attributes answered outside business-type scope", "6.6.5 (DQ-053)"),
                      ("implausibly short / inverted / overlapping windows", "6.2.2 / 6.6.4 (DQ-012, 047, 048)"),
                      ("stale is_open", "6.4.6 / 6.7.3 (DQ-036)")]:
    print(f"   🔗 {chk_name:<58} → {ref}")

# ══ Build b68 ══
_m = lambda c: F.split(c, ":")[0].cast("int") * 60 + F.split(c, ":")[1].cast("int")
day_cols, h_exprs = [d.lower() for d in DAYS], []
for d in day_cols:
    s, e = _m(F.col(f"hours_{d}_start_time")), _m(F.col(f"hours_{d}_end_time"))
    h_exprs += [s.alias(f"s_{d}"), e.alias(f"e_{d}"),
                F.when(s.isNull() | e.isNull(), None).when(e > s, e - s).otherwise(e + 1440 - s).alias(f"dur_{d}")]  # e ≤ s → cross-midnight / 24 h
hrs = df_hours.select("business_id", *h_exprs)
hrs = hrs.select("business_id",
    sum(F.when(F.col(f"dur_{d}").isNotNull(), 1).otherwise(0) for d in day_cols).alias("open_days"),
    F.when(F.greatest(*[F.col(f"dur_{d}").isNotNull() for d in day_cols]),
           sum(F.coalesce(F.col(f"dur_{d}"), F.lit(0)) for d in day_cols) / 60.0).alias("weekly_hours"),
    sum(F.when((F.col(f"e_{d}") > F.col(f"s_{d}")) & (F.col(f"e_{d}") <= 17 * 60), 1).otherwise(0) for d in day_cols).alias("days_close_by_17"),
    sum(F.when(F.col(f"s_{d}") >= 15 * 60, 1).otherwise(0) for d in day_cols).alias("days_open_from_15"),
    F.array(*[F.col(f"s_{d}") for d in day_cols]).alias("start_minutes"))

attr_cols = [c for c in df_attributes.columns if c != "business_id"]
attr_sum = df_attributes.select("business_id", "attr_restaurants_price_range_2",
                                sum(F.col(c).isNotNull().cast("int") for c in attr_cols).alias("n_attr"))
upscale = df_ambience.where((F.col("key") == "upscale") & (F.col("value") == True)).select("business_id", F.lit(True).alias("amb_upscale"))
cats = df_categories.groupBy("business_id").agg(F.collect_set("category_name").alias("cats"), F.count("*").alias("n_categories"))
tips_n = sib_raw["yelp_tip"].groupBy("business_id").agg(F.count("*").alias("n_tips"), F.min(F.to_timestamp("date")).alias("first_tip"))
chk_ts = F.transform(F.split("date", ","), lambda x: F.to_timestamp(F.trim(x)))
chk_n = sib_raw["yelp_checkin"].select("business_id", F.size(F.split("date", ",")).alias("n_checkins"),
                                       F.array_min(chk_ts).alias("first_checkin"))
b68 = (df_business
       .join(cats, "business_id", "left").join(attr_sum, "business_id", "left").join(upscale, "business_id", "left")
       .join(hrs, "business_id", "left")
       .join(recon_673.select("business_id", "n_reviews", "n_reviewers", "first_review", "last_review", "last_tip", "last_checkin", "last_activity"),
             "business_id", "left")
       .join(tips_n, "business_id", "left").join(chk_n, "business_id", "left")
       .withColumn("n_tips", F.coalesce("n_tips", F.lit(0))).withColumn("n_checkins", F.coalesce("n_checkins", F.lit(0)))
       .withColumn("n_categories", F.coalesce("n_categories", F.lit(0)))
       .withColumn("name_len", F.length(F.trim("name")))
       .withColumn("first_activity", F.least("first_review", "first_tip", "first_checkin")))
has = lambda c: F.coalesce(F.array_contains("cats", c), F.lit(False))
any_of = lambda cs: F.greatest(*[has(c) for c in cs])
nm = F.lower(F.trim("name"))
EX = F.concat_ws(" | ", "name", "city", "state")

# ══ A) Entity & name accuracy ══
print(f"\n{'─' * 60}\n📌 A) Entity & name accuracy")
is_event = has("Yelp Events")
STATUS_RE = r"(?i)(permanently closed|temporarily closed|\bclosed\b|\bmoved\b|out of business|coming soon|opening soon)"
CLOSED_RE = r"(?i)(permanently closed|temporarily closed|\bclosed\b|\bmoved\b|out of business)"
PLACEHOLDER = ["test", "none", "null", "n/a", "na", "unknown", "tbd", "business", "restaurant", "store", "-", "."]
vA = run_checks(findings_681, b68, [
    chk("df_categories", "listing is a Yelp event, not a business", "no row carries the 'Yelp Events' category (community events, not places)",
        is_event, ex=F.concat_ws(" | ", "name", "city", F.col("review_count").cast("string")),
        note="Elite / community event pages; 'reviews' are event feedback — distort business-level stats"),
    chk("df_business", "Yelp event flagged is_open = TRUE", "a finite event is not an open business", F.col("is_open") == True,
        scope=is_event, ex=F.concat_ws(" | ", "name", F.date_format("last_activity", "yyyy-MM-dd"))),
    chk("df_business", "name carries operational status", "name does not embed 'closed' / 'moved' / 'coming soon' — status belongs in is_open",
        F.col("name").rlike(STATUS_RE), ex=F.concat_ws(" | ", "name", F.col("is_open").cast("string"))),
    chk("df_business", "name says closed but is_open = TRUE", "a listing whose name says closed / moved is not flagged open",
        F.col("is_open") == True, scope=F.col("name").rlike(CLOSED_RE), ex=F.col("name")),
    chk("df_business", "name = address or city", "business name is not a copy of its street address or city",
        (nm == F.lower(F.trim("address"))) | (nm == F.lower(F.trim("city"))), ex=F.concat_ws(" | ", "name", "address", "city"),
        note="placeholder name; a few are genuine landmarks / venues named after the place"),
    chk("df_business", "name is a placeholder token", f"name ∉ {PLACEHOLDER}", nm.isin(PLACEHOLDER), ex=F.col("name")),
    chk("df_business", "name contains a phone number", "no NANP phone pattern in the name",
        F.col("name").rlike(r"\(?\b\d{3}\)?[-. ]\d{3}[-. ]\d{4}\b"), ex=F.col("name")),
    chk("df_business", "name contains a web domain", "name is not a URL / domain", F.col("name").rlike(r"(?i)(www\.|https?://|\.(com|net|org|io|co)\b)"),
        status="INFO", ex=F.col("name"), note="mostly genuine online-first brands (e.g. webuyanycar.com)"),
    chk("df_business", "name has no letter", "name contains ≥ 1 letter", ~F.col("name").rlike("[A-Za-z]"), status="INFO", ex=F.col("name"),
        note="numeric brand names (e.g. '1984', '360') — legit but weak identity for fuzzy matching"),
    chk("df_business", "name ≤ 2 characters", "name length ≥ 3", F.col("name_len") <= 2, status="INFO", ex=F.col("name")),
    chk("df_business", "name fully upper-case", "name is not ALL CAPS (≥ 4 letters)",
        (F.col("name") == F.upper("name")) & (F.length(F.regexp_replace("name", "[^A-Za-z]", "")) >= 4), status="INFO", ex=F.col("name"),
        note="style noise for display / matching; many are acronyms or brand styling")])
m68["name_status"] = vA["name carries operational status"]
m68["yelp_events"] = vA["listing is a Yelp event, not a business"]

# ══ B) Category ↔ attribute plausibility ══
print(f"\n{'─' * 60}\n📌 B) Category ↔ attribute plausibility")
BUDGET = ["Fast Food", "Food Trucks", "Hot Dogs", "Food Stands", "Convenience Stores", "Street Vendors"]
# data-driven restaurant sub-categories: ≥ 200 businesses and ≥ 99 % of them also carry 'Restaurants'
REST_CHILD = [r["category_name"] for r in
              df_categories.join(cats.select("business_id", has("Restaurants").alias("r")), "business_id")
              .groupBy("category_name").agg(F.countDistinct("business_id").alias("n"), F.avg(F.col("r").cast("int")).alias("p"))
              .where("n >= 200 AND p >= 0.99 AND category_name NOT IN ('Restaurants', 'Food')").collect()]
print(f"   {len(REST_CHILD)} restaurant sub-categories derived from the data (e.g. {', '.join(sorted(REST_CHILD)[:6])}, …)")
price = F.col("attr_restaurants_price_range_2")
vB = run_checks(findings_681, b68, [
    chk("df_attributes × df_categories", "$$$$ on a budget format", f"RestaurantsPriceRange2 ≠ 4 for {', '.join(BUDGET)}",
        price == 4, scope=any_of(BUDGET) & price.isNotNull(), ex=F.concat_ws(" | ", "name", "city", F.array_join("cats", ", "))),
    chk("df_attributes × df_ambience", "upscale ambience at price $", "Ambience.upscale = TRUE ⇒ RestaurantsPriceRange2 > 1",
        price == 1, scope=F.coalesce(F.col("amb_upscale"), F.lit(False)) & price.isNotNull(), ex=EX),
    chk("df_categories", "restaurant sub-category without 'Restaurants'", "category hierarchy complete (child ⇒ parent)",
        ~has("Restaurants"), scope=any_of(REST_CHILD), ex=F.concat_ws(" | ", "name", F.array_join("cats", ", ")),
        note="Yelp expands parent categories; 0 here means the hierarchy is consistent")])

# ══ C) Category ↔ hours plausibility ══
print(f"\n{'─' * 60}\n📌 C) Category ↔ hours plausibility")
open_any = F.col("open_days") > 0
vC = run_checks(findings_681, b68, [
    chk("df_hours × df_categories", "Bars close before 17:00 every open day", "a bar is open in the evening on ≥ 1 day",
        F.col("days_close_by_17") == F.col("open_days"), scope=has("Bars") & open_any, status="INFO", ex=EX,
        note="daytime-only 'bars' (juice / oxygen / brunch bars) or stale hours"),
    chk("df_hours × df_categories", "Breakfast & Brunch never opens before 15:00", "a breakfast place opens before 15:00 on ≥ 1 day",
        F.col("days_open_from_15") == F.col("open_days"), scope=has("Breakfast & Brunch") & open_any, status="INFO", ex=EX,
        note="late-night brunch concepts, or 12-hour-clock entry (cf. DQ-047)"),
    chk("df_hours × df_categories", "Restaurants open < 10 h per week", "a restaurant with hours is open ≥ 10 h / week",
        F.col("weekly_hours") < 10, scope=has("Restaurants") & open_any, status="INFO",
        ex=F.concat_ws(" | ", "name", F.format_string("%.1f h/wk", "weekly_hours")), note="pop-ups / partial schedules; cf. DQ-012"),
    chk("df_hours", "weekly open hours ≤ 168", "a week has 168 hours", F.col("weekly_hours") > 168, scope=open_any,
        ex=F.concat_ws(" | ", "name", F.format_string("%.1f h/wk", "weekly_hours")), note="only possible with overlapping windows (DQ-048)")])

# ══ D) Rating ↔ volume plausibility ══
print(f"\n{'─' * 60}\n📌 D) Rating ↔ volume plausibility")
stars_d = F.col("stars").cast("double")
vD = run_checks(findings_681, b68, [
    chk("df_business", "perfect 5.0 with ≥ 100 reviews", "high-volume businesses rarely keep a perfect mean", stars_d == 5.0,
        scope=F.col("review_count") >= 100, status="INFO", ex=F.concat_ws(" | ", "name", F.col("review_count").cast("string")),
        note="service businesses (optometry, tattoo, tours) — genuine, but candidates for solicited-review review"),
    chk("df_business", "≤ 1.5 stars with ≥ 100 reviews", "high-volume businesses rarely sit at the bottom of the scale", stars_d <= 1.5,
        scope=F.col("review_count") >= 100, status="INFO", ex=F.concat_ws(" | ", "name", F.col("review_count").cast("string")),
        note="utilities / telecom / insurers — complaint-driven, genuine")])
print("   Top categories of the ≤ 1.5-star, ≥ 100-review businesses (domain explanation):")
display(b68.where((stars_d <= 1.5) & (F.col("review_count") >= 100)).select(F.explode("cats").alias("category"))
        .groupBy("category").count().orderBy(F.desc("count")).limit(10))

findings_681_df = pd.DataFrame(findings_681)
print(f"\n   {len(findings_681_df)} rule checks | {int((findings_681_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((findings_681_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


In [0]:
# 6.8.2) Statistical Outlier Detection  —  business-level metrics (b68 from 6.8.1)
# ─────────────────────────────────────────────────────────────────────
#   A) Univariate  — Tukey 3×IQR fences. Heavy-tailed counts are fenced on log1p(x) (raw fences flag a third of the data),
#                    near-symmetric metrics on the raw scale (both tails).
#   B) Contextual  — review_count vs its city (robust z on log1p, cities ≥ 50 businesses)
#                    a chain unit's stars vs its chain median (chains ≥ 20 units, |Δ| ≥ 2.0 stars)
# Outliers are profiled (INFO) — they are extreme, not invalid; impossible values were handled in 6.2.
from pyspark.sql import functions as F, Window
import pandas as pd, numpy as np

print("=" * 80)
print("6.8.2) STATISTICAL OUTLIER DETECTION  —  business-level metrics")
print("=" * 80)
findings_682 = []

METRICS = [  # metric, scale, tails, scope, meaning
    ("review_count", "log", "upper", None,                                  "listing popularity"),
    ("n_tips",       "log", "upper", None,                                  "tips per business"),
    ("n_checkins",   "log", "upper", None,                                  "check-ins per business"),
    ("n_categories", "raw", "both",  F.col("n_categories") > 0,             "categories per business (incl. parents)"),
    ("n_attr",       "raw", "both",  F.col("n_attr") > 0,                   "single-value attributes answered"),
    ("weekly_hours", "raw", "both",  F.col("weekly_hours").isNotNull(),     "weekly open hours"),
    ("name_len",     "raw", "both",  F.col("name").isNotNull(),             "name length (characters)")]
X = lambda m, sc: F.log1p(F.col(m).cast("double")) if sc == "log" else F.col(m).cast("double")

# ══ A) Univariate — one pass for the quantiles, one pass for the outlier counts ══
print(f"\n{'─' * 60}\n📌 A) Univariate outliers")
q = b68.agg(*[F.percentile_approx(F.when(sc_ if sc_ is not None else F.lit(True), X(m, s)), [0.25, 0.75]).alias(f"{m}_q")
              for m, s, _, sc_, _ in METRICS],
            *[F.percentile_approx(F.when(sc_ if sc_ is not None else F.lit(True), F.col(m)), [0.0, 0.01, 0.5, 0.99, 0.999, 1.0]).alias(f"{m}_p")
              for m, _, _, sc_, _ in METRICS],
            *[F.skewness(F.when(sc_ if sc_ is not None else F.lit(True), F.col(m))).alias(f"{m}_sk") for m, _, _, sc_, _ in METRICS]).first()
checks, fences, prof = [], {}, []
for m, s, tails, sc_, meaning in METRICS:
    q1, q3 = q[f"{m}_q"]; iqr = q3 - q1
    lo, hi = q1 - 3 * iqr, q3 + 3 * iqr
    back = (lambda v: float(np.expm1(v))) if s == "log" else float
    fences[m] = (back(lo) if tails == "both" else None, back(hi))
    bad = (X(m, s) > hi) if tails == "upper" else ((X(m, s) > hi) | (X(m, s) < lo))
    checks.append(chk("b68", f"{m} extreme outlier", f"Tukey 3×IQR on {'log1p' if s == 'log' else 'raw'} ({tails} tail)", bad, scope=sc_,
                      status="INFO", ex=F.concat_ws(" | ", "name", "city", F.col(m).cast("string")), note=meaning))
    p = q[f"{m}_p"]
    prof.append({"Metric": m, "Scale": s, "Min": p[0], "P01": p[1], "Median": p[2], "P99": p[3], "P99.9": p[4], "Max": p[5],
                 "Skew": round(q[f"{m}_sk"] or 0, 2),
                 "Lower fence": None if fences[m][0] is None else round(fences[m][0], 1), "Upper fence": round(fences[m][1], 1)})
vA = run_checks(findings_682, b68, checks)
print("\n   Metric profile and fences (fences shown on the original scale):")
display(pd.DataFrame(prof))
print("   Most extreme businesses per metric (top 3):")
top_rows = []
for m, *_ in METRICS:
    for r in b68.where(F.col(m).isNotNull()).orderBy(F.desc(m)).select("name", "city", "state", F.col(m).cast("double").alias("v")).limit(3).collect():
        top_rows.append({"Metric": m, "Name": r["name"], "City": r["city"], "State": r["state"], "Value": r["v"]})
display(pd.DataFrame(top_rows))

# ══ B1) review_count vs city peers ══
print(f"\n{'─' * 60}\n📌 B1) Contextual — review_count vs city peers (robust z on log1p, cities ≥ 50 businesses)")
lx = F.log1p(F.col("review_count").cast("double"))
wc = Window.partitionBy("city_k")
ctx = (b68.withColumn("city_k", F.lower(F.trim("city"))).withColumn("lx", lx)
          .withColumn("city_n", F.count("*").over(wc)).withColumn("city_med", F.percentile_approx("lx", 0.5).over(wc)))
ctx = ctx.withColumn("absdev", F.abs(F.col("lx") - F.col("city_med"))).withColumn("city_mad", F.percentile_approx("absdev", 0.5).over(wc))
ctx = ctx.withColumn("rz", F.try_divide(F.col("lx") - F.col("city_med"), 1.4826 * F.col("city_mad")))  # MAD = 0 in tiny cities → NULL
vB = run_checks(findings_682, ctx, [
    chk("b68", "review_count vs city (robust z > 3.5)", "popularity within 3.5 robust SD of the city median", F.col("rz") > 3.5,
        scope=(F.col("city_n") >= 50) & (F.col("city_mad") > 0), status="INFO",
        ex=F.concat_ws(" | ", "name", "city", F.col("review_count").cast("string")),
        note="local landmarks / tourist magnets; dominate city-level aggregates")])
display(ctx.where((F.col("city_n") >= 50) & (F.col("rz") > 3.5)).groupBy("city", "state")
        .agg(F.count("*").alias("outliers"), F.max("review_count").alias("max_review_count"), F.first("city_n").alias("city_businesses"))
        .orderBy(F.desc("outliers")).limit(10))

# ══ B2) chain unit rating vs chain median ══
print(f"\n{'─' * 60}\n📌 B2) Contextual — chain unit stars vs chain median (chains ≥ 20 units)")
wk = Window.partitionBy("chain_k")
ch = (b68.withColumn("chain_k", F.regexp_replace(F.lower(F.trim("name")), r"[^a-z0-9]", ""))
         .withColumn("units", F.count("*").over(wk))
         .withColumn("chain_med", F.percentile_approx(F.col("stars").cast("double"), 0.5).over(wk))
         .withColumn("d", F.col("stars").cast("double") - F.col("chain_med")))
vB2 = run_checks(findings_682, ch, [
    chk("b68", "chain unit |stars − chain median| ≥ 2.0", "a chain unit's rating within 2 stars of its chain median", F.abs(F.col("d")) >= 2.0,
        scope=F.col("units") >= 20, status="INFO",
        ex=F.concat_ws(" | ", "name", "city", F.format_string("%.1f vs %.1f", F.col("stars").cast("double"), "chain_med"), F.col("review_count").cast("string")),
        note="mostly low-volume units (small-sample ratings); a few could be mis-attributed brands (same name, different business)")])
print("   Chain outliers by review volume (small samples explain most of them):")
display(ch.where((F.col("units") >= 20) & (F.abs(F.col("d")) >= 2.0))
        .groupBy(F.when(F.col("review_count") < 10, "5–9").when(F.col("review_count") < 30, "10–29").otherwise("30+").alias("review_band"))
        .count().orderBy("review_band"))

findings_682_df = pd.DataFrame(findings_682)
print(f"\n   {len(findings_682_df)} outlier checks | {int((findings_682_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((findings_682_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


In [0]:
# 6.8.3) Distribution Anomaly Detection  —  shape of the business population (b68 from 6.8.1)
# ─────────────────────────────────────────────────────────────────────
#   A) Rating mix by review-count band  — extreme ratings (≤ 1.5 / 5.0) heap where samples are small
#   B) review_count lower truncation    — the dataset only keeps businesses with ≥ 5 reviews (left-censored)
#   C) is_open rate by state            — regional closure-rate deviations (> 10 pp from the global rate)
#   D) Cohorts by first-activity year   — entry-cohort size shocks and survivorship (older cohorts close more)
#   E) Category frequency               — long tail / head dominance of the category vocabulary
#   F) Opening-time heaping             — start minutes off the :00 / :15 / :30 / :45 grid
from pyspark.sql import functions as F
import pandas as pd, numpy as np

print("=" * 80)
print("6.8.3) DISTRIBUTION ANOMALY DETECTION  —  business population")
print("=" * 80)
findings_683 = []
N68 = b68.count()
stars_d = F.col("stars").cast("double")

# ══ A) Rating mix by review-count band ══
print(f"\n{'─' * 60}\n📌 A) Rating distribution by review-count band")
band = (F.when(F.col("review_count") < 10, "a) 5–9").when(F.col("review_count") < 50, "b) 10–49")
         .when(F.col("review_count") < 200, "c) 50–199").otherwise("d) 200+"))
mix = (b68.withColumn("band", band).withColumn("s", F.col("stars").cast("string"))
          .groupBy("band").pivot("s", [f"{x / 2:.1f}" for x in range(2, 11)]).count().na.fill(0).orderBy("band"))
mix_pd = mix.toPandas().set_index("band")
mix_pct = (mix_pd.div(mix_pd.sum(axis=1), axis=0) * 100).round(1)
mix_pct["businesses"] = mix_pd.sum(axis=1)
display(mix_pct.reset_index())
ext = lambda b: float(mix_pct.loc[b, [c for c in mix_pd.columns if float(c) <= 1.5 or float(c) == 5.0]].sum())
r_small, r_big = ext("a) 5–9"), ext("d) 200+")
ratio = r_small / r_big if r_big else float("inf")
print(f"   Extreme-rating share (≤ 1.5 or 5.0): 5–9 reviews {r_small:.1f}%  vs  200+ reviews {r_big:.1f}%  (×{ratio:.1f})")
dq_add(findings_683, "df_business", "stars heaping at the extremes (small samples)", "extreme-rating share in the 5–9 band ≤ 3× the 200+ band",
       int(mix_pd.loc["a) 5–9"].sum()), int(round(r_small / 100 * mix_pd.loc["a) 5–9"].sum())), status="INFO" if ratio > 3 else "PASS",
       note=f"×{ratio:.1f}: a mean of 5–9 reviews is noisy — use a Bayesian / shrunk rating for ranking")

# ══ B) review_count left-truncation ══
print(f"\n{'─' * 60}\n📌 B) review_count lower tail (dataset inclusion rule ≥ 5)")
low = b68.where("review_count <= 12").groupBy("review_count").count().orderBy("review_count").toPandas()
low["pct_of_all"] = (low["count"] / N68 * 100).round(2)
display(low)
min_rc = int(b68.agg(F.min("review_count")).first()[0])
at5 = int(low.loc[low["review_count"] == 5, "count"].sum())
dq_add(findings_683, "df_business", "review_count left-truncated at 5", "population is not censored below 5 reviews", N68, N68 if min_rc == 5 else 0,
       status="INFO" if min_rc == 5 else "PASS",
       note=f"min = {min_rc}; mode at 5 ({at5:,} businesses) — businesses with < 5 reviews are absent, so averages of review_count / stars are biased upwards vs Yelp overall")

# ══ C) is_open rate by state ══
print(f"\n{'─' * 60}\n📌 C) is_open rate by state (≥ 100 businesses)")
glob_open = b68.agg(F.avg(F.col("is_open").cast("int"))).first()[0] * 100
st = (b68.groupBy("state").agg(F.count("*").alias("businesses"), F.round(F.avg(F.col("is_open").cast("int")) * 100, 1).alias("open_pct"),
                               F.round(F.avg(stars_d), 2).alias("avg_stars"), F.percentile_approx("review_count", 0.5).alias("median_reviews"))
         .where("businesses >= 100").withColumn("delta_pp", F.round(F.col("open_pct") - F.lit(glob_open), 1)).orderBy("open_pct"))
display(st)
dev = st.where(F.abs("delta_pp") > 10)
n_dev = dev.count()
dq_add(findings_683, "df_business", "is_open rate by state", "no state (≥ 100 businesses) deviates > 10 pp from the global open rate",
       st.count(), n_dev, status="INFO" if n_dev else "PASS", examples=dq_examples(dev, F.concat_ws(" ", "state", "open_pct"), F.lit(True)),
       note=f"global open rate {glob_open:.1f}%")

# ══ D) Cohorts by first-activity year ══
print(f"\n{'─' * 60}\n📌 D) Entry cohorts (year of first review / tip / check-in) — size and survival")
coh = (b68.withColumn("cohort", F.year("first_activity")).groupBy("cohort")
          .agg(F.count("*").alias("businesses"), F.round(F.avg(F.col("is_open").cast("int")) * 100, 1).alias("open_pct"),
               F.percentile_approx("review_count", 0.5).alias("median_reviews"))
          .orderBy("cohort").toPandas())
coh["yoy_pct"] = (coh["businesses"].pct_change() * 100).round(1)
display(coh)
inner = coh.iloc[1:-1]  # first and last cohorts are partial years
shock = inner[(inner["yoy_pct"].abs() > 40) & (inner["cohort"] >= 2010)]
dq_add(findings_683, "df_business", "entry-cohort size shock", "inner cohorts (2010+) change < 40 % year-over-year", len(inner[inner["cohort"] >= 2010]), len(shock),
       status="INFO" if len(shock) else "PASS", examples=", ".join(f"{int(r.cohort)}: {r.yoy_pct:+.0f}%" for r in shock.itertuples()),
       note="first activity ≠ opening date; post-2018 cohorts are small because a business needs ≥ 5 reviews to be included (right-censoring)")
surv = coh[coh["businesses"] >= 500]
mono = float(np.corrcoef(surv["cohort"], surv["open_pct"])[0, 1]) if len(surv) > 2 else 0
print(f"   Correlation cohort-year ↔ open %: {mono:+.2f} (survivorship: older listings have had more time to close)")

# ══ E) Category frequency ══
print(f"\n{'─' * 60}\n📌 E) Category frequency (vocabulary head / tail)")
freq = df_categories.groupBy("category_name").agg(F.countDistinct("business_id").alias("businesses")).orderBy(F.desc("businesses")).toPandas()
freq["share_of_businesses_pct"] = (freq["businesses"] / N68 * 100).round(2)
display(freq.head(20))
tail = int((freq["businesses"] < 10).sum())
print(f"   {len(freq):,} labels | top-10 labels appear on {freq.head(10)['businesses'].sum() / freq['businesses'].sum() * 100:.1f}% of category rows | "
      f"{tail:,} labels used by < 10 businesses")
dq_add(findings_683, "df_categories", "long-tail categories (< 10 businesses)", "category usable as a model / dashboard dimension", len(freq), tail,
       status="INFO" if tail else "PASS", examples=", ".join(freq[freq["businesses"] < 10]["category_name"].head(6)),
       note="group into the parent category (or 'Other') for analytics; 'Restaurants' / 'Food' / 'Shopping' are parent labels")

# ══ F) Opening-time heaping ══
print(f"\n{'─' * 60}\n📌 F) Opening-time minute heaping")
st_min = b68.select("business_id", F.explode("start_minutes").alias("s")).where(F.col("s").isNotNull()).withColumn("minute", F.col("s") % 60)
min_mix = st_min.groupBy("minute").count().orderBy(F.desc("count")).toPandas()
min_mix["pct"] = (min_mix["count"] / min_mix["count"].sum() * 100).round(3)
display(min_mix.head(10))
off = st_min.where(~F.col("minute").isin(0, 15, 30, 45))
n_off_biz = off.select("business_id").distinct().count()
dq_add(findings_683, "df_hours", "start time off the quarter-hour grid", "opening minute ∈ {00, 15, 30, 45}", int(min_mix["count"].sum()), off.count(),
       status="INFO" if n_off_biz else "PASS", examples=dq_examples(b68.join(off.select("business_id").distinct(), "business_id"), "name", F.lit(True), 5),
       note=f"{n_off_biz:,} businesses; genuine (e.g. 10:10) or typos — heaping on :00 / :30 is the normal human pattern")

findings_683_df = pd.DataFrame(findings_683)
print(f"\n   {len(findings_683_df)} distribution checks | {int((findings_683_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((findings_683_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


In [0]:
# 6.8.4) Concentration & Pareto Analysis  —  who dominates the business data? (b68 from 6.8.1)
# ─────────────────────────────────────────────────────────────────────
#   A) Engagement across businesses  — Gini + top-k% share of reviews (stored review_count), check-ins, tips; Lorenz table
#   B) Review concentration by state — is the head heavier in some markets?
#   C) Listings across chains        — share of listings held by multi-unit brands (normalised name)
#   D) Listings across cities        — metro dominance (top cities) and city Gini
#   E) Category concentration        — share of businesses carrying the top labels
# Concentration is not an error; it is profiled (INFO) because it drives sampling / weighting decisions downstream.
from pyspark.sql import functions as F
import pandas as pd, numpy as np

print("=" * 80)
print("6.8.4) CONCENTRATION & PARETO ANALYSIS  —  Business DataFrames")
print("=" * 80)
findings_684 = []

def gini(x):
    x = np.sort(np.asarray(x, dtype=float)); n = len(x)
    return float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum())) if n and x.sum() else 0.0

def top_share(x, p):
    x = np.sort(np.asarray(x, dtype=float))[::-1]; k = max(1, int(round(len(x) * p)))
    return float(x[:k].sum() / x.sum() * 100) if x.sum() else 0.0

lvl = lambda g: "very high" if g >= 0.7 else "high" if g >= 0.5 else "moderate" if g >= 0.3 else "low"
eng = b68.select("business_id", "state", "review_count", "n_checkins", "n_tips").toPandas()

# ══ A) Engagement across businesses ══
print(f"\n{'─' * 60}\n📌 A) Engagement concentration across {len(eng):,} businesses")
rows = []
for m, label in [("review_count", "reviews"), ("n_checkins", "check-ins"), ("n_tips", "tips")]:
    g = gini(eng[m])
    rows.append({"Metric": label, "Total": int(eng[m].sum()), "Gini": round(g, 3), "Level": lvl(g),
                 **{f"Top {int(p * 100)}% share": round(top_share(eng[m], p), 1) for p in (0.01, 0.05, 0.10, 0.20)},
                 "Zero share %": round(float((eng[m] == 0).mean() * 100), 1)})
    s20 = top_share(eng[m], 0.2)
    dq_add(findings_684, "b68", f"{label} concentration", f"top 20 % of businesses hold < 50 % of {label}", len(eng),
           int(round(len(eng) * 0.2)) if s20 >= 50 else 0, status="INFO" if s20 >= 50 else "PASS",
           note=f"Gini {g:.3f} ({lvl(g)}); top 20 % of businesses hold {top_share(eng[m], 0.2):.1f}% of {label}")
conc = pd.DataFrame(rows)
display(conc)
print("   Lorenz table — cumulative share of reviews held by the bottom x% of businesses:")
xs = np.sort(eng["review_count"].values.astype(float)); cum = np.cumsum(xs) / xs.sum()
display(pd.DataFrame({"bottom_pct_of_businesses": [10, 25, 50, 75, 90, 95, 99],
                      "cum_pct_of_reviews": [round(float(cum[int(len(xs) * p / 100) - 1] * 100), 1) for p in [10, 25, 50, 75, 90, 95, 99]]}))

# ══ B) Review concentration by state ══
print(f"\n{'─' * 60}\n📌 B) Review concentration by state (≥ 100 businesses)")
by_st = (eng.groupby("state").filter(lambda d: len(d) >= 100).groupby("state")[["review_count"]]
            .apply(lambda d: pd.Series({"businesses": len(d), "reviews": int(d["review_count"].sum()), "gini": round(gini(d["review_count"]), 3),
                                        "top_1pct_share": round(top_share(d["review_count"], 0.01), 1)}))
            .reset_index().sort_values("gini", ascending=False))
display(by_st)
g_all = gini(eng["review_count"])
heavy = by_st[by_st["gini"] > g_all + 0.05]
dq_add(findings_684, "b68", "state-level review concentration", "state Gini within +0.05 of the global Gini", len(by_st), len(heavy),
       status="INFO" if len(heavy) else "PASS", examples=", ".join(f"{r.state} {r.gini}" for r in heavy.itertuples()),
       note="tourist markets (e.g. New Orleans / Nashville) concentrate reviews on a few landmarks")

# ══ C) Listings across chains ══
print(f"\n{'─' * 60}\n📌 C) Listings held by multi-unit brands (normalised name)")
chains = (b68.withColumn("chain_k", F.regexp_replace(F.lower(F.trim("name")), r"[^a-z0-9]", ""))
             .groupBy("chain_k").agg(F.first("name").alias("example_name"), F.count("*").alias("units"),
                                     F.sum("review_count").alias("reviews"), F.round(F.avg(F.col("stars").cast("double")), 2).alias("avg_stars"))
             .toPandas().sort_values("units", ascending=False))
multi = chains[chains["units"] >= 10]
share_multi = multi["units"].sum() / chains["units"].sum() * 100
print(f"   {len(chains):,} distinct normalised names | {len(multi):,} brands with ≥ 10 units hold {share_multi:.1f}% of listings "
      f"but {multi['reviews'].sum() / chains['reviews'].sum() * 100:.1f}% of reviews")
display(chains.head(15))
dq_add(findings_684, "b68", "listings held by brands with ≥ 10 units", "chain listings < 10 % of the population", int(chains["units"].sum()),
       int(multi["units"].sum()), status="INFO" if share_multi >= 10 else "PASS",
       note=f"{share_multi:.1f}% of listings; chain units are review-light — stratify by brand to avoid chain-dominated models")

# ══ D) Listings across cities ══
print(f"\n{'─' * 60}\n📌 D) Listings across cities")
cities = b68.groupBy("city", "state").agg(F.count("*").alias("businesses"), F.sum("review_count").alias("reviews")).toPandas().sort_values("businesses", ascending=False)
cities["share_pct"] = (cities["businesses"] / cities["businesses"].sum() * 100).round(2)
display(cities.head(12))
top10_city = cities.head(10)["businesses"].sum() / cities["businesses"].sum() * 100
g_city = gini(cities["businesses"])
print(f"   {len(cities):,} (city, state) values | top-10 cities hold {top10_city:.1f}% of businesses | city Gini {g_city:.3f}")
n_tiny = int((cities["businesses"] < 5).sum())
dq_add(findings_684, "b68", "business listings across cities", "(city, state) value holds ≥ 5 businesses", len(cities), n_tiny,
       status="INFO" if n_tiny else "PASS",
       note=f"Gini {g_city:.3f}; {int((cities['businesses'] < 5).sum()):,} 'cities' with < 5 businesses (suburbs + spelling variants DQ-021)")

# ══ E) Category concentration ══
print(f"\n{'─' * 60}\n📌 E) Category concentration")
cat_biz = df_categories.groupBy("category_name").agg(F.countDistinct("business_id").alias("b")).toPandas()
g_cat = gini(cat_biz["b"])
rest_food = b68.where(has("Restaurants") | has("Food")).count() / b68.count() * 100
print(f"   category Gini {g_cat:.3f} | {rest_food:.1f}% of businesses are Restaurants and/or Food")
k1 = max(1, int(round(len(cat_biz) * 0.01))); s1 = top_share(cat_biz["b"], 0.01)
dq_add(findings_684, "df_categories", "category usage concentration", "top 1 % of labels cover < 25 % of category rows", len(cat_biz),
       k1 if s1 >= 25 else 0, status="INFO" if s1 >= 25 else "PASS", note=f"Gini {g_cat:.3f}; top {k1} labels = {s1:.1f}% of rows; {rest_food:.1f}% of businesses are Restaurants / Food — food-heavy population")

findings_684_df = pd.DataFrame(findings_684)
print(f"\n   {len(findings_684_df)} concentration checks | {int((findings_684_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


In [0]:
# 6.8.5) Business Activity Span Analysis  —  lifecycle of each business across review / tip / check-in
# ─────────────────────────────────────────────────────────────────────
#   A) Source windows      — first / last timestamp per source; nothing before Yelp existed (2004-10-01) or after the snapshot
#   B) Lifecycle profile   — active span (first → last activity), dormancy before the snapshot, reviews per active year
#   C) Suspicious shapes   — compressed lifecycle (≥ 10 reviews within ≤ 7 days), single-day bursts (≥ 50 % of reviews on one day),
#                            long silence (≥ 3 y without reviews) followed by reactivation (re-used listing / re-opening)
#   D) Newcomers           — first activity in the final 90 days but already ≥ 5 reviews
# Builds on b68 (6.8.1) and SNAP (6.7.3). Stale is_open was measured in 6.4.6 / 6.7.3 (DQ-036) and is only profiled here.
from pyspark.sql import functions as F, Window
import pandas as pd

print("=" * 80)
print("6.8.5) BUSINESS ACTIVITY SPAN ANALYSIS  —  lifecycle across review / tip / check-in")
print("=" * 80)
findings_685 = []
YELP_LAUNCH = "2004-10-01"
print(f"   Snapshot: {SNAP}   |   Yelp launch reference: {YELP_LAUNCH}")

# ══ A) Source windows ══
print(f"\n{'─' * 60}\n📌 A) Source time windows")
rv_ts = sib_raw["yelp_review"].select("business_id", F.to_timestamp("date").alias("ts"))
tp_ts = sib_raw["yelp_tip"].select("business_id", F.to_timestamp("date").alias("ts"))
ck_ts = sib_raw["yelp_checkin"].select("business_id", F.explode(F.split("date", ",")).alias("d")).select("business_id", F.to_timestamp(F.trim("d")).alias("ts"))
win_rows = []
for src, fr in [("review", rv_ts), ("tip", tp_ts), ("checkin", ck_ts)]:
    r = fr.agg(F.count("*").alias("n"), F.min("ts").alias("first"), F.max("ts").alias("last"), F.sum(F.col("ts").isNull().cast("int")).alias("unparsed"),
               F.sum((F.col("ts") < F.lit(YELP_LAUNCH)).cast("int")).alias("pre"), F.sum((F.col("ts") > F.lit(SNAP)).cast("int")).alias("post")).first()
    win_rows.append({"Source": src, "Events": int(r["n"]), "First": r["first"], "Last": r["last"], "Unparsed": int(r["unparsed"])})
    dq_add(findings_685, f"yelp_{src}", f"{src} timestamps within [Yelp launch, snapshot]", f"{YELP_LAUNCH} ≤ ts ≤ snapshot, parseable",
           int(r["n"]), int(r["pre"]) + int(r["post"]) + int(r["unparsed"]))
display(pd.DataFrame(win_rows))

# ══ B) Lifecycle profile ══
print(f"\n{'─' * 60}\n📌 B) Lifecycle profile (days)")
daily = rv_ts.groupBy("business_id", F.to_date("ts").alias("d")).agg(F.count("*").alias("n"))
w = Window.partitionBy("business_id").orderBy("d")
gaps = daily.withColumn("gap", F.datediff("d", F.lag("d").over(w)))
per = gaps.groupBy("business_id").agg(F.sum("n").alias("rv_rows"), F.max("n").alias("max_day"), F.count("*").alias("review_days"),
                                      F.datediff(F.max("d"), F.min("d")).alias("review_span_days"), F.max("gap").alias("max_gap_days"))
life = (b68.select("business_id", "name", "city", "state", "is_open", "review_count", "first_activity", "last_activity")
           .join(per, "business_id", "left")
           .withColumn("span_days", F.datediff("last_activity", "first_activity"))
           .withColumn("dormant_days", F.datediff(F.lit(SNAP), "last_activity"))
           .withColumn("reviews_per_year", F.col("rv_rows") / F.greatest(F.col("span_days") / 365.25, F.lit(1 / 12))))  # floor at 1 month
qs = [0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
prof = life.agg(*[F.percentile_approx(c, qs).alias(c) for c in ("span_days", "dormant_days", "reviews_per_year", "max_gap_days")]).first()
display(pd.DataFrame({c: [round(float(v), 1) for v in prof[c]] for c in prof.asDict()}, index=[f"p{int(q * 100)}" for q in qs]).reset_index(names="quantile"))
print("   Dormancy before the snapshot by is_open (share of businesses):")
dorm = F.when(F.col("dormant_days") <= 30, "a) ≤ 30 d").when(F.col("dormant_days") <= 365, "b) 31 d – 1 y") \
        .when(F.col("dormant_days") <= 3 * 365, "c) 1 – 3 y").otherwise("d) > 3 y")
display(life.groupBy(dorm.alias("dormancy")).pivot("is_open", [True, False]).count().na.fill(0)
        .withColumnRenamed("true", "open").withColumnRenamed("false", "closed").orderBy("dormancy"))

# ══ C) Suspicious lifecycle shapes + D) newcomers ══
print(f"\n{'─' * 60}\n📌 C / D) Suspicious lifecycle shapes and newcomers")
EXL = F.concat_ws(" | ", "name", "city", F.format_string("%d reviews", "rv_rows"))
vC = run_checks(findings_685, life, [
    chk("b68 × yelp_review", "compressed lifecycle (≥ 10 reviews in ≤ 7 days)", "review history of ≥ 10 reviews spans > 7 days",
        F.col("review_span_days") <= 7, scope=F.col("rv_rows") >= 10, status="INFO",
        ex=F.concat_ws(" | ", EXL, F.format_string("%d d", "review_span_days")), note="one-off events / pop-ups, or a review campaign"),
    chk("b68 × yelp_review", "single-day review burst (≥ 50 % on one day)", "no single day carries ≥ 50 % of a business's ≥ 10 reviews",
        F.col("max_day") >= 0.5 * F.col("rv_rows"), scope=F.col("rv_rows") >= 10, status="INFO",
        ex=F.concat_ws(" | ", EXL, F.format_string("max/day %d", "max_day")), note="event venues / festivals, or solicited-review bursts"),
    chk("b68 × yelp_review", "silence ≥ 3 y then reactivated", "no ≥ 3-year gap between consecutive reviews",
        F.col("max_gap_days") >= 3 * 365, scope=F.col("review_days") >= 2, status="INFO",
        ex=F.concat_ws(" | ", EXL, F.format_string("gap %d d", "max_gap_days")),
        note="re-opening / new owner under the same listing — reviews before the gap may describe a different business"),
    chk("b68", "newcomer: first activity ≤ 90 days before snapshot", "profile of very young listings",
        F.col("dormant_days").isNotNull() & (F.datediff(F.lit(SNAP), "first_activity") <= 90), status="INFO",
        ex=F.concat_ws(" | ", "name", "city", F.date_format("first_activity", "yyyy-MM-dd"), F.col("review_count").cast("string")),
        note="needed ≥ 5 reviews within 90 days to enter the dataset — fast-growing or burst-reviewed"),
    chk("b68", "is_open = FALSE but dormant ≤ 30 days", "profile of recently closed listings", F.col("dormant_days") <= 30,
        scope=F.col("is_open") == False, status="INFO", ex=F.concat_ws(" | ", "name", "city", F.date_format("last_activity", "yyyy-MM-dd")),
        note="same 130 as 6.7.3 C — cross-reference DQ-BUSINESS-036")])
print("   Largest single-day bursts:")
display(life.where("rv_rows >= 10 AND max_day >= 0.5 * rv_rows")
        .select("name", "city", "state", "is_open", "rv_rows", "max_day", "review_days", "review_span_days").orderBy(F.desc("max_day")).limit(10))
print("   Longest silences followed by reactivation:")
display(life.where("max_gap_days >= 1095").select("name", "city", "state", "is_open", "rv_rows", "max_gap_days", "first_activity", "last_activity")
        .orderBy(F.desc("max_gap_days")).limit(10))
life_685 = life
findings_685_df = pd.DataFrame(findings_685)
print(f"\n   {len(findings_685_df)} activity checks | {int((findings_685_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((findings_685_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


In [0]:
# 6.8.6) Business Rule & Accuracy — Consolidated Summary  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# Status per sub-section and every FINDING / INFO of 6.8.1 – 6.8.5 in one place.
import pandas as pd

print("=" * 80)
print("6.8) BUSINESS RULE & ACCURACY — CONSOLIDATED SUMMARY  (Business DataFrames)")
print("=" * 80)

SECTIONS_68 = [("6.8.1 Business Rules", "findings_681_df"), ("6.8.2 Statistical Outliers", "findings_682_df"),
               ("6.8.3 Distribution Anomalies", "findings_683_df"), ("6.8.4 Concentration & Pareto", "findings_684_df"),
               ("6.8.5 Activity Span", "findings_685_df")]
parts, missing_sections = [], []
for sec, var in SECTIONS_68:
    (parts.append(globals()[var].assign(Section=sec)) if var in globals() else missing_sections.append(sec))
if missing_sections:
    print(f"   ⚠️  Not yet run (re-run these cells first): {missing_sections}")
all_68 = pd.concat(parts, ignore_index=True)

status_68 = (all_68.pivot_table(index="Section", columns="Status", values="Rule", aggfunc="count", fill_value=0)
             .reindex(columns=["FINDING", "INFO", "PASS"], fill_value=0).reset_index())
status_68["Checks"] = status_68[["FINDING", "INFO", "PASS"]].sum(axis=1)
print("\n📋 Check status per sub-section:")
display(status_68)

find_68 = all_68[all_68["Status"] == "FINDING"][["Section", "DataFrame", "Column", "Rule", "Checked", "Violations", "Pct", "Examples", "Note"]]
print(f"\n📋 All FINDINGs in 6.8 ({len(find_68)}):")
display(find_68.sort_values("Violations", ascending=False).reset_index(drop=True))
info_68 = all_68[all_68["Status"] == "INFO"][["Section", "DataFrame", "Column", "Checked", "Violations", "Pct", "Note"]]
print(f"\n📋 INFO profiles in 6.8 ({len(info_68)}):")
display(info_68.reset_index(drop=True))

n_f, n_i, n_p = (int((all_68["Status"] == s).sum()) for s in ("FINDING", "INFO", "PASS"))
print(f"\n   {len(all_68)} checks | {n_f} FINDING | {n_i} INFO | {n_p} PASS")
summary_68_df = all_68
print("\nBUSINESS RULE & ACCURACY CHECKS COMPLETE")
print("=" * 80)


---
#### 🚨 DQ-BUSINESS-057 — Yelp Event Pages Listed as Businesses

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-057` |
| **Severity** | MEDIUM (few rows, but they are not businesses, and 79% are marked open) |
| **Detected In** | Phase 6.8 — Business Rule & Accuracy → **6.8.1 A** — `listing is a Yelp event, not a business`, `Yelp event flagged is_open = TRUE` |
| **Also Confirmed In** | 6.8.5 C: compressed lifecycles and single-day review bursts (`Elites Go Free: Jurassic Quest`: 12 reviews in 4 days; `Elite Experience: Raising Cane's Holiday Soiree`: 15 of 25 reviews on one day) |
| **Affected Columns** | `yelp_business.categories` (`Yelp Events`, always with `Local Flavor`), `name`, `is_open` |
| **Total Affected Rows** | **48** event listings (1,668 reviews). **38** of them have `is_open = TRUE`, 23 have no hours and 6 have no address |
| **Invalid Value** | `Elite Event: Elite Week at Pig Shack Eatery` (Reno, open, 57 reviews), `Yelp's Totally Bazaar: Freakshakes at TeeJay's Sweet Tooth` (Indianapolis, open, 93 reviews) |
| **Domain Rule** | The business dimension holds places that operate over time. A one-off community / Elite event is a different entity type |

**Description:** Yelp hosts Elite and community events as pages that look like business listings. Their "reviews" are attendee feedback for one evening, so their rating, review counts and lifecycle (span of days, single-day bursts) are not comparable to a real business. Because events are never formally "closed", 38 of 48 still say `is_open = TRUE`. They inflate open-business counts and add noise to popularity and rating models.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Add `is_yelp_event = array_contains(categories, 'Yelp Events')` in Silver. Keep the rows, but exclude them from business-level aggregates, open-business KPIs and model training by default** | Data stays complete and auditable; consumers get a clean business population |
| 2 | Route the rows to a separate `yelp_event` table | Cleaner model, but needs a new entity and FK handling for their reviews / tips |
| 3 | Delete the rows | Loses 1,668 reviews and breaks review → business FKs |

**Best Logical Solution:** **Option 1 — flag with `is_yelp_event` and exclude by default.**

---

---
#### 🚨 DQ-BUSINESS-058 — `name` Carries Operational Status or a Copy of the Address / City

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-058` |
| **Severity** | LOW (25 businesses) |
| **Detected In** | Phase 6.8 — Business Rule & Accuracy → **6.8.1 A** — `name carries operational status`, `name = address or city` |
| **Affected Columns** | `yelp_business.name` |
| **Total Affected Rows** | **8** names with status text (`- Closed`, `- CLOSED`, `moved`, `coming soon`; all 7 "closed" names are correctly `is_open = FALSE`). **17** names equal to their address or city |
| **Invalid Value** | `Reginelli's Pizzeria - Closed`, `MedExpress Urgent Care - CLOSED`, name `Philadelphia` with empty address in city `Philadelphia`, name `King of Prussia` |
| **Domain Rule** | `name` holds the trading name only. Status belongs in `is_open`, location in `address` / `city` |

**Description:** Two kinds of contamination of the identity field. (a) Status suffixes added by editors when a place closed: the flag already holds the information, but the suffix breaks name matching, chain grouping (DQ-022) and dedup (DQ-038 / 039). (b) Place names used as the business name: most have empty or generic addresses (overlap with DQ-019 / 032 / 040) and look like placeholder listings. A few are real venues named after their location (e.g. `Blackett's Ridge`).

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **In Silver add `name_clean` with the status suffix removed (regex on `- closed` / `(closed)` / `moved` / `coming soon`), and a flag `name_is_location`. Keep the original `name`** | Matching and chain logic use `name_clean`; the source value stays auditable |
| 2 | Overwrite `name` | Simpler, but loses the editor's signal |
| 3 | Do nothing | Duplicate / chain detection misses these rows |

**Best Logical Solution:** **Option 1 — `name_clean` + `name_is_location` flag.**

---

---
#### 🚨 DQ-BUSINESS-059 — `RestaurantsPriceRange2` Contradicts the Business Type

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-059` |
| **Severity** | LOW (32 businesses) |
| **Detected In** | Phase 6.8 — Business Rule & Accuracy → **6.8.1 B** — `$$$$ on a budget format`, `upscale ambience at price $` |
| **Affected Columns** | `attr_restaurants_price_range_2`, `categories`, `Ambience.upscale` |
| **Total Affected Rows** | **29** of 8,892 budget-format businesses (Fast Food, Food Trucks, Hot Dogs, Food Stands, Convenience Stores, Street Vendors) are `$$$$`. **3** of 428 upscale venues are `$` |
| **Invalid Value** | `CVS Pharmacy` (Spring Hill) and `Wawa` (Palmyra) at price level 4. `Savaya Coffee Market` upscale at price level 1 |
| **Domain Rule** | Price level is consistent with the format: budget formats are `$` / `$$`, upscale venues are at least `$$` |

**Description:** `RestaurantsPriceRange2` is crowd-sourced and the 1–4 scale is easy to invert (1 = cheap). The `$$$$` budget formats are mostly chain units (CVS, Wawa, convenience stores) where luxury pricing is not credible, which points to inverted or misclicked answers. Price level is a key feature for recommendation and segmentation, so a few inverted values on high-volume chains have a large impact.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep the value and add a `price_range_suspect` flag in Silver (budget format × 4, or upscale × 1). Optionally impute the chain's modal price level into `price_range_imputed`** | Contradictions are visible; models can use the flag or the imputed value |
| 2 | Set suspect values to NULL | Removes noise, but loses data that may be correct (e.g. a premium food truck) |
| 3 | Do nothing | Inverted values bias price-level features |

**Best Logical Solution:** **Option 1 — flag, with optional chain-mode imputation.**

---

---
#### 📋 6.8 Business Rule & Accuracy — Error Register

New errors raised in Phase 6.8 (next free ID: **DQ-BUSINESS-060**).

| Error ID | Title | Severity | Detected In | Affected | Best Solution (short) |
| --- | --- | --- | --- | --- | --- |
| DQ-BUSINESS-057 | Yelp event pages listed as businesses | MEDIUM | 6.8.1 A | 48 listings (38 open, 1,668 reviews) | `is_yelp_event` flag, excluded by default |
| DQ-BUSINESS-058 | Name carries status or location text | LOW | 6.8.1 A | 8 status + 17 name = address / city | `name_clean` + `name_is_location` |
| DQ-BUSINESS-059 | Price range contradicts business type | LOW | 6.8.1 B | 29 budget × $$$$, 3 upscale × $ | `price_range_suspect` flag (+ chain-mode imputation) |

**Clean results (PASS):** no placeholder or phone-number names, no restaurant sub-category without `Restaurants` (hierarchy complete), no week longer than 168 h, no closed-name listing marked open. All review (6.99 M), tip (909 K) and check-in (13.36 M) timestamps parse and fall inside [2004-10-01, snapshot]. No state deviates by more than 10 pp from the global open rate (79.6%).

**Re-detected existing errors (no new ID):** DQ-BUSINESS-036 (stale `is_open`: 5,542 open businesses silent > 3 years, 130 closed businesses active in the last 30 days), DQ-BUSINESS-021 / 018 (868 of 1,467 `(city, state)` values hold < 5 businesses, mostly suburbs and spelling variants), DQ-BUSINESS-022 (chain name variants affect brand grouping), DQ-BUSINESS-007 / 026 (168 h/week schedules on non-24h types, e.g. a law office, are consistent with the ambiguous `0:0-0:0` encoding), DQ-BUSINESS-055 (stored `review_count` used for concentration).

**INFO (expected, documented — analytics caveats for Silver / Gold):**
* **Small-sample ratings:** 24.8% of businesses with 5–9 reviews have an extreme rating (≤ 1.5 or 5.0), vs 1.6% at 200+ reviews (× 15.5). 279 perfect-5.0 and 116 ≤ 1.5-star businesses still have ≥ 100 reviews (the low ones are banks, fast food, insurers). Use a Bayesian / shrunk rating for ranking.
* **Inclusion bias:** `review_count` is left-truncated at 5 (mode 5 = 9.9% of businesses), so averages are not representative of all Yelp listings. Recent entry cohorts are right-censored (2021: 2,220, 2022: 19), and their open rate rises to 98% (survivorship, cohort ↔ open % correlation + 0.96). The 2010 cohort jump (+ 119%) coincides with check-ins starting on 2009-12-30, so `first_activity` mixes sources.
* **Concentration:** reviews Gini 0.678 (top 1% of businesses = 19.6% of reviews), check-ins 0.814, tips 0.764. LA is the most concentrated state (Gini 0.753, French Quarter landmarks). Brands with ≥ 10 units hold 16.0% of listings but only 10.0% of reviews. Top 10 cities hold 48.7% of businesses. Stratify / weight samples by city and brand.
* **Outliers (extreme, not invalid):** 32 review-count outliers on the log scale (Acme Oyster House 7,568), 909 businesses far above their city median, 232 chain units ≥ 2 stars off their chain median (163 of them with only 5–9 reviews), 38 opening minutes off the quarter-hour grid, 236 categories used by < 10 businesses.
* **Lifecycle shapes:** 8 businesses with ≥ 10 reviews inside ≤ 7 days and single-day bursts (mostly events / pop-ups, see DQ-057), businesses reactivated after a ≥ 3-year silence (possible new owner under the same listing), newcomers with ≥ 5 reviews within 90 days.

---

### 6.9) TIMELINESS CHECKS
*Business-table assessment of snapshot currency, ingestion observability, staleness of current-state fields, timestamp integrity and boundary effects.*

`yelp_poc_uc.bronze_final.yelp_business` is a **current-state snapshot without a single timestamp column**. `is_open`, `stars`, `review_count`, `hours` and `attributes` describe the business *as of an unknown moment*. Timeliness is therefore a **historical-snapshot quality** question (how old is each business's state, and can we date it?), not a real-time SLA question. The only time evidence is in the facts linked by `business_id`: `yelp_review.date`, `yelp_tip.date`, the `yelp_checkin.date` list, plus `yelp_user.yelping_since` for author chronology. The sibling bronze tables are only read, never modified.

| Sub-section | Definition (generic) | Adaptation to the business frames |
| --- | --- | --- |
| **6.9.1 Snapshot Freshness & Coverage** | Earliest / latest timestamps, calendar span, recent-year concentration, staleness relative to today | Infer the **as-of date** (latest linked review / tip / check-in), check that all sources share one cut-off, measure snapshot age, left-censoring of tips (2009) and check-ins (2009-12), yearly volume and recent-year share per source, and the **recency band of every business** (last activity) by `is_open`. Builds `t69` (1 row / business). |
| **6.9.2 Ingestion Observability** | Row-level load metadata; Delta history only as a batch-level proxy | Row-level ingestion / as-of / valid-from columns in all 5 tables, lineage `TBLPROPERTIES`, Delta commit history (one batch load, no later data changes, row counts unchanged), and **publication lag** (load date − as-of date). |
| **6.9.3 Snapshot Lag & Boundary Period Risk** | Latest event far older than the latest commit; incomplete first / last periods bias trends | How old is the **state of each business**: market-level cut-off and activity freshness per metro, `is_open` currency (cross-ref DQ-BUSINESS-036), hours / attributes of long-silent listings, **rating currency** (`stars` built on old reviews), `review_count` counter drift (cross-ref DQ-BUSINESS-056), partial **boundary months / years**, and **source regime shifts** right before the cut-off (one feed thinning out earlier than the others). |
| **6.9.4 Temporal Consistency of Raw Timestamps** | Clean parsing, parsed components agree with the text, no future-dated or malformed values | Format / parse / round-trip / future / pre-launch checks for `review.date`, `tip.date`, `user.yelping_since` and every token of the check-in list; check-in list ordering and duplicate tokens; time-of-day heaping (date-only precision); **author chronology** (content before registration); **time-zone interpretation** (UTC vs local business time — no time-zone column), including a daylight-saving *spring-forward* test. |
| **6.9.5 Temporal Gaps & Summary** | Gaps in a time series | Gaps were measured in **6.4.6** (completeness: weekly schedule holes, month grid, silent periods, stale `is_open`) and **6.8.5** (lifecycle shapes, reactivation). 6.9.5 cross-references them and consolidates all 6.9 checks. |

**Status convention:** `FINDING` = a defect in the data that the Silver layer must handle; `INFO` = a property of a historical snapshot that analysts must know (documented, not "fixed"); `PASS` = rule holds.

In [0]:
# 6.9.1) Snapshot Freshness & Coverage  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
# yelp_business has no timestamp → its as-of date is INFERRED from the time-stamped facts linked by business_id.
#   A) Source windows & cut-off  — first / last event per source (review, tip, check-in, user registration), one shared cut-off?
#   B) Snapshot age              — as-of date vs today
#   C) Coverage boundaries       — source start dates (feature launch → left-censoring), partial first / last month
#   D) Business recency mix      — last activity per business (review ∪ tip ∪ check-in) in bands, by is_open
#   E) Recent-year concentration — yearly volume per covered day, share of the final 1 / 3 years, last full year vs peak,
#                                  listing vintage (first activity) and businesses whose whole observed life is pre-COVID
# Builds t69 = 1 row / business with first / last event per source; reused by 6.9.3 – 6.9.5.
# Read-only: the sibling bronze tables are only read, never modified.
from pyspark.sql import functions as F
import pandas as pd
from datetime import date

print("=" * 80)
print("6.9.1) SNAPSHOT FRESHNESS & COVERAGE  —  inferred as-of date, source cut-offs, business recency")
print("=" * 80)
assert "df_business" in globals(), "Run TE.1 / TE.2 first (df_business, df_hours, df_attributes)"

# ══ Shared helpers — identical to 6.2.1 / 6.6.1; re-declared only if this section is run stand-alone ══
if "dq_add" not in globals():
    _ICON = {"PASS": "✅", "FINDING": "⚠️", "INFO": "ℹ️"}
    def dq_add(findings, frame, column, rule, checked, violations, status=None, examples="", note=""):
        checked, violations = int(checked or 0), int(violations or 0)
        status = status or ("PASS" if violations == 0 else "FINDING")
        findings.append({"DataFrame": frame, "Column": column, "Rule": rule, "Checked": checked, "Violations": violations,
                         "Pct": round(violations / checked * 100, 3) if checked else 0.0, "Status": status,
                         "Examples": examples, "Note": note})
        print(f"   {_ICON[status]} {column:<40} {violations:>9,} / {checked:>9,}  — {status}  [{rule}]")
        if examples and status != "PASS":
            print(f"      e.g. {examples}")
if "run_checks" not in globals():
    def chk(frame, column, rule, bad, scope=None, status="FINDING", ex=None, note=""):
        return {"frame": frame, "column": column, "rule": rule, "bad": bad,
                "scope": scope if scope is not None else F.lit(True), "status": status, "ex": ex, "note": note}
    def run_checks(findings, base, checks, ex_default=None):
        conds = [(F.coalesce(c["scope"], F.lit(False)), F.coalesce(c["bad"], F.lit(False))) for c in checks]
        aggs = []
        for i, (sc, bd) in enumerate(conds):
            aggs += [F.sum(sc.cast("int")).alias(f"s{i}"), F.sum((sc & bd).cast("int")).alias(f"v{i}")]
        r = base.agg(*aggs).first()
        out = {}
        for i, c in enumerate(checks):
            s, v = int(r[f"s{i}"] or 0), int(r[f"v{i}"] or 0)
            ex = ""
            if v:
                e = c["ex"] if c["ex"] is not None else (ex_default if ex_default is not None else F.col("business_id"))
                ex = " ; ".join(str(x[0]) for x in base.where(conds[i][0] & conds[i][1]).select(e.cast("string")).limit(3).collect())
            dq_add(findings, c["frame"], c["column"], c["rule"], s, v, status="PASS" if v == 0 else c["status"], examples=ex, note=c["note"])
            out[c["column"]] = v
        return out

findings_691 = []
TS_FMT_69 = "yyyy-MM-dd HH:mm:ss"
YELP_LAUNCH_69 = "2004-10-01"
TODAY_69 = date.today()
SOURCES_69 = {"yelp_business": "yelp_poc_uc.bronze_final.yelp_business",
              "yelp_review":   "yelp_poc_uc.bronze_final.yelp_review",
              "yelp_tip":      "yelp_poc_uc.bronze_final.yelp_tip",
              "yelp_checkin":  "yelp_poc_uc.bronze_final.yelp_checkin",
              "yelp_user":     "yelp_poc_uc.bronze_final.yelp_user"}
raw_69 = {t: spark.table(p) for t, p in SOURCES_69.items()}
ts_ = lambda c: F.try_to_timestamp(c, F.lit(TS_FMT_69))          # NULL instead of an error on a malformed string

# Event frames: one row per time-stamped fact (raw string kept for 6.9.4 format checks)
ev_69 = {
    "review":  raw_69["yelp_review"].select("business_id", "user_id", F.col("date").alias("raw")),
    "tip":     raw_69["yelp_tip"].select("business_id", "user_id", F.col("date").alias("raw")),
    "checkin": raw_69["yelp_checkin"].select("business_id", F.lit(None).cast("string").alias("user_id"),
                                            F.explode(F.split("date", ",")).alias("raw"))
                                    .withColumn("raw", F.trim("raw")),   # ', '-separated list
}
ev_69 = {k: v.withColumn("ts", ts_(F.col("raw"))) for k, v in ev_69.items()}
usr_69 = raw_69["yelp_user"].select("user_id", F.col("yelping_since").alias("raw")).withColumn("ts", ts_(F.col("raw")))

# ══ A) Source windows & cut-off ══
print(f"\n{'─' * 60}\n📌 A) Source time windows (all business-linked facts + user registrations)")
win = []
for src, fr in [*ev_69.items(), ("user (yelping_since)", usr_69)]:
    r = fr.agg(F.count("*").alias("n"), F.min("ts").alias("first"), F.max("ts").alias("last"),
               F.sum(F.col("ts").isNull().cast("int")).alias("unparsed")).first()
    win.append({"Source": src, "Events": int(r["n"]), "First": r["first"], "Last": r["last"], "Unparsed": int(r["unparsed"])})
win_691 = pd.DataFrame(win)
SNAP_69 = max(w for w in win_691.loc[win_691["Source"] != "user (yelping_since)", "Last"])    # as-of = latest business-linked fact
SNAP_D69 = SNAP_69.date()
win_691["Days last → snapshot"] = [(SNAP_69 - l).total_seconds() / 86400 for l in win_691["Last"]]
win_691["Days first → Yelp launch"] = [(f.date() - date.fromisoformat(YELP_LAUNCH_69)).days for f in win_691["First"]]
display(win_691.round(2))
print(f"   Inferred as-of date of yelp_business (latest review / tip / check-in): {SNAP_69}")
if "SNAP" in globals():
    print(f"   Cross-check with 6.7.3 SNAP: {SNAP}  →  {'identical' if SNAP == SNAP_69 else 'DIFFERENT'}")

off = win_691[win_691["Days last → snapshot"] > 1]
dq_add(findings_691, "review / tip / check-in / user", "single extract cut-off", "every source's latest event is ≤ 1 day before the as-of date",
       len(win_691), len(off), examples=", ".join(f"{s}: {d:.0f} d" for s, d in zip(off["Source"], off["Days last → snapshot"])),
       note=f"all sources end on {SNAP_D69} → one consistent extract; business state is interpreted as of this date")

# ══ B) Snapshot age ══
print(f"\n{'─' * 60}\n📌 B) Snapshot age")
age_days = (TODAY_69 - SNAP_D69).days
print(f"   As-of {SNAP_D69}  →  today {TODAY_69}: {age_days:,} days (~{age_days / 365.25:.1f} years)")
dq_add(findings_691, "yelp_business (inferred)", "snapshot age", "as-of date within 365 days of today (fit for current-state use)",
       1, int(age_days > 365), status="INFO", examples=f"{age_days:,} days old",
       note="historical snapshot: is_open / hours / attributes / stars must not be presented as today's state of the business")

# ══ C) Coverage boundaries ══
print(f"\n{'─' * 60}\n📌 C) Coverage boundaries")
first_rv = win_691.loc[win_691["Source"] == "review", "First"].iloc[0]
late = win_691[(win_691["Source"].isin(["tip", "checkin"])) & ((win_691["First"] - first_rv).dt.days > 365)]
for _, w in win_691.iterrows():
    print(f"   {w['Source']:<22} starts {w['First']:%Y-%m-%d}  ({(w['First'] - first_rv).days / 365.25:+.1f} y vs first review)")
dq_add(findings_691, "tip / check-in", "source starts with the review history", "tip and check-in windows start within 1 year of the first review",
       2, len(late), status="INFO", examples=", ".join(f"{s} from {f:%Y-%m-%d}" for s, f in zip(late["Source"], late["First"])),
       note="features launched later (tips 2009, check-ins end-2009) → tip / check-in based first_activity and "
            "pre-2010 engagement are left-censored, not missing")
import calendar
last_m_days = calendar.monthrange(SNAP_D69.year, SNAP_D69.month)[1]
first_d = first_rv.date()
first_m_days = calendar.monthrange(first_d.year, first_d.month)[1]
cov_last = SNAP_D69.day / last_m_days
cov_first = (first_m_days - first_d.day + 1) / first_m_days
print(f"   First month {first_d:%Y-%m}: {first_m_days - first_d.day + 1}/{first_m_days} days covered | "
      f"last month {SNAP_D69:%Y-%m}: {SNAP_D69.day}/{last_m_days} days covered ({cov_last * 100:.0f}%)")
dq_add(findings_691, "review / tip / check-in", "boundary months complete", "first and last calendar month fully covered",
       2, int(cov_first < 1) + int(cov_last < 1), status="INFO",
       examples=f"{first_d:%Y-%m} {cov_first * 100:.0f}% ; {SNAP_D69:%Y-%m} {cov_last * 100:.0f}%",
       note="monthly business KPIs for the boundary months are understated — exclude or pro-rate them")

# ══ D) Business recency mix ══
print(f"\n{'─' * 60}\n📌 D) Business recency — last activity (review ∪ tip ∪ check-in) before the as-of date")
_per = None
for src, fr in ev_69.items():
    a = fr.groupBy("business_id").agg(F.count("*").alias(f"n_{src}"), F.min("ts").alias(f"first_{src}"), F.max("ts").alias(f"last_{src}"))
    _per = a if _per is None else _per.join(a, "business_id", "full")
t69 = (df_business.select("business_id", "name", "city", "state", "is_open", "review_count")
       .join(_per, "business_id", "left")
       .withColumn("first_activity", F.least("first_review", "first_tip", "first_checkin"))
       .withColumn("last_activity", F.greatest("last_review", "last_tip", "last_checkin"))
       .withColumn("days_silent", F.datediff(F.lit(SNAP_D69), F.to_date("last_activity"))))
band = (F.when(F.col("days_silent").isNull(), "0) no activity").when(F.col("days_silent") <= 30, "a) ≤ 30 d")
         .when(F.col("days_silent") <= 90, "b) 31 – 90 d").when(F.col("days_silent") <= 365, "c) 91 d – 1 y")
         .when(F.col("days_silent") <= 3 * 365, "d) 1 – 3 y").otherwise("e) > 3 y"))
rec_691 = (t69.groupBy(band.alias("last_activity_band")).agg(F.count("*").alias("businesses"),
           F.sum(F.col("is_open").cast("int")).alias("open"), F.sum((~F.col("is_open")).cast("int")).alias("closed"))
           .orderBy("last_activity_band").toPandas())
N_BUS_69 = int(rec_691["businesses"].sum())
rec_691["pct"] = (rec_691["businesses"] / N_BUS_69 * 100).round(2)
rec_691["open_pct"] = (rec_691["open"] / rec_691["businesses"] * 100).round(1)
display(rec_691)
nb = dict(zip(rec_691["last_activity_band"], rec_691["businesses"]))
dq_add(findings_691, "df_business × review/tip/checkin", "business has any time-stamped activity", "every business has ≥ 1 review / tip / check-in",
       N_BUS_69, nb.get("0) no activity", 0), note="without any linked fact the business cannot be dated at all")
run_checks(findings_691, t69, [
    chk("df_business × review/tip/checkin", "last activity > 1 year before as-of", "profile of businesses not observed in the final year",
        F.col("days_silent") > 365, status="INFO", ex=F.concat_ws(" | ", "name", "city", F.date_format("last_activity", "yyyy-MM-dd")),
        note="their current-state fields are ≥ 1 year older than the snapshot (see 6.9.3)"),
    chk("df_business × review/tip/checkin", "last activity > 3 years before as-of", "profile of long-dormant listings",
        F.col("days_silent") > 3 * 365, status="INFO", ex=F.concat_ws(" | ", "name", "city", F.date_format("last_activity", "yyyy-MM-dd")))])

# ══ E) Recent-year concentration & listing vintage ══
print(f"\n{'─' * 60}\n📌 E) Recent-year concentration — yearly volume per covered day, final 1 / 3 years, listing vintage")
yr_691 = None
for src, fr in ev_69.items():
    p = fr.groupBy(F.year("ts").alias("year")).agg(F.count("*").alias(src)).toPandas()
    yr_691 = p if yr_691 is None else yr_691.merge(p, on="year", how="outer")
yr_691 = yr_691.sort_values("year").fillna(0).reset_index(drop=True)
yr_691["year"] = yr_691["year"].astype(int)
for s in ev_69:
    yr_691[s] = yr_691[s].astype(int)
WIN_LO_69 = first_rv.date()                                     # dataset window = first review → as-of date
yr_691["days_covered"] = [(min(date(y, 12, 31), SNAP_D69) - max(date(y, 1, 1), WIN_LO_69)).days + 1 for y in yr_691["year"]]
for s in ev_69:
    yr_691[f"{s}_per_day"] = (yr_691[s] / yr_691["days_covered"]).round(1)
display(yr_691)
full_y = yr_691[yr_691["days_covered"] >= 365]                   # complete calendar years only
share_rows = []
for src, fr in ev_69.items():
    d = F.to_date("ts")
    r = fr.agg(F.count("*").alias("n"), F.sum((d > F.date_sub(F.lit(SNAP_D69), 365)).cast("int")).alias("y1"),
               F.sum((d > F.date_sub(F.lit(SNAP_D69), 3 * 365)).cast("int")).alias("y3")).first()
    fy = full_y[full_y[src] > 0]
    peak, last_full = fy.loc[fy[f"{src}_per_day"].idxmax()], fy.iloc[-1]
    share_rows.append({"Source": src, "Events": int(r["n"]), "Share final 1 y %": round(r["y1"] / r["n"] * 100, 1),
                       "Share final 3 y %": round(r["y3"] / r["n"] * 100, 1), "Peak year": int(peak["year"]),
                       "Peak per day": peak[f"{src}_per_day"], "Last full year": int(last_full["year"]),
                       "Last full year per day": last_full[f"{src}_per_day"],
                       "Last full / peak": round(last_full[f"{src}_per_day"] / peak[f"{src}_per_day"], 2)})
share_691 = pd.DataFrame(share_rows)
display(share_691)
for _, s in share_691.iterrows():
    dq_add(findings_691, f"yelp_{s['Source']}", "last full year vs peak year (per day)",
           "the last complete year carries ≥ 50 % of the peak-year daily volume", 1, int(s["Last full / peak"] < 0.5), status="INFO",
           examples=f"peak {s['Peak year']} = {s['Peak per day']:,.0f}/d → {s['Last full year']} = {s['Last full year per day']:,.0f}/d "
                    f"({s['Last full / peak']:.2f}); final 1 y = {s['Share final 1 y %']}% of all events",
           note="the most recent state of a business rests on far fewer observations than its peak years (COVID + platform volume decline)")
vint_691 = (t69.groupBy(F.year("first_activity").alias("first_activity_year"))
               .agg(F.count("*").alias("businesses"), F.round(F.avg(F.col("is_open").cast("int")) * 100, 1).alias("open_pct"),
                    F.percentile_approx("days_silent", 0.5).alias("median_days_silent")).orderBy("first_activity_year").toPandas())
print("   Listing vintage — year of the first linked activity (proxy for the listing's age on Yelp):")
display(vint_691)
COVID_69 = "2020-03-01"
EX_E = F.concat_ws(" | ", "name", "city", F.date_format("last_activity", "yyyy-MM-dd"), F.col("is_open").cast("string"))
run_checks(findings_691, t69, [
    chk("df_business × review/tip/checkin", "whole observed life before 2020-03 (pre-COVID)",
        "profile of businesses whose state was last observed before the COVID shock",
        F.col("last_activity") < F.lit(COVID_69).cast("timestamp"), status="INFO", ex=EX_E,
        note="their is_open / hours / attributes describe the pre-COVID market — not comparable with post-2020 listings"),
    chk("df_business × review/tip/checkin", "open, last observed before 2020-03", "an open business has been observed after the COVID shock",
        F.col("last_activity") < F.lit(COVID_69).cast("timestamp"), scope=F.col("is_open") == True, status="INFO", ex=EX_E,
        note="COVID closures were often never flagged → strongest stale-is_open cohort (DQ-BUSINESS-036)")])

findings_691_df = pd.DataFrame(findings_691)
print(f"\n   {len(findings_691_df)} freshness checks | {int((findings_691_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((findings_691_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


In [0]:
# 6.9.2) Ingestion Observability & Table History  —  yelp_business + the 4 linked bronze tables
# ─────────────────────────────────────────────────────────────────────
# Three levels of observability:
#   A) Row level    — load / ingestion / as-of / valid-from columns in the row schema (per table)
#   B) Table level  — lineage TBLPROPERTIES that record the source snapshot (date / version / file)
#   C) Commit level — Delta history: one batch load? any data-changing commit after it? publication lag vs the as-of date
# The business table holds CURRENT-STATE fields (is_open, stars, review_count, hours, attributes); without an as-of
# column their capture date can only be inferred (6.9.1: SNAP_69). Silver rule R6 keeps lineage in TBLPROPERTIES, not columns.
from pyspark.sql import functions as F
import pandas as pd, re

print("=" * 80)
print("6.9.2) INGESTION OBSERVABILITY & TABLE HISTORY  —  row / table / commit level")
print("=" * 80)
assert "SNAP_69" in globals(), "Run 6.9.1 first"
findings_692 = []

# ══ A) Row-level metadata ══
print(f"\n{'─' * 60}\n📌 A) Row-level ingestion / as-of metadata")
META_RE = r"(?i)(ingest|load|etl|batch|_ts$|timestamp|created|updated|modified|as_of|asof|valid_(from|to)|snapshot|extract|_metadata|_rescued|source_file|file_name)"
row_meta = []
for t, fr in raw_69.items():
    hits = [c for c in fr.columns if re.search(META_RE, c)]
    row_meta.append({"Table": t, "Columns": len(fr.columns), "Metadata columns": ", ".join(hits) or "—",
                     "Event-time column": {"yelp_review": "date", "yelp_tip": "date", "yelp_checkin": "date (list)",
                                           "yelp_user": "yelping_since"}.get(t, "— (none)")})
row_meta_692 = pd.DataFrame(row_meta)
display(row_meta_692)
no_meta = row_meta_692[row_meta_692["Metadata columns"] == "—"]["Table"].tolist()
dq_add(findings_692, "5 bronze tables", "row-level ingestion metadata", "each table carries a load / ingestion timestamp or source-file column",
       len(row_meta_692), len(no_meta), status="INFO", examples=", ".join(no_meta),
       note="per-row load latency cannot be measured; all rows came from one batch (see C)")
STATE_FIELDS = ["is_open", "stars", "review_count", "hours", "attributes"]
dq_add(findings_692, "yelp_business", "as-of date for current-state fields", "is_open / stars / review_count / hours / attributes have a capture or valid-from date",
       len(STATE_FIELDS), len(STATE_FIELDS), status="INFO", examples=", ".join(STATE_FIELDS),
       note=f"capture date must be inferred = {SNAP_69:%Y-%m-%d} (6.9.1); Silver should record it once in TBLPROPERTIES (R6), not per row")

# ══ B) Lineage table properties ══
print(f"\n{'─' * 60}\n📌 B) Lineage table properties")
LINEAGE_RE = r"(?i)(source|snapshot|as_of|asof|extract|lineage|version|origin|dataset)"
props = []
for t, p in SOURCES_69.items():
    tp = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {p}").collect()}
    lin = {k: v for k, v in tp.items() if re.search(LINEAGE_RE, k) and not k.startswith(("delta.", "spark.", "clusteringColumns"))}
    cmt = spark.sql(f"DESCRIBE TABLE EXTENDED {p}").where("col_name = 'Comment'").select("data_type").collect()
    props.append({"Table": t, "Properties": len(tp), "Lineage properties": "; ".join(f"{k}={v}" for k, v in lin.items()) or "—",
                  "Table comment": (cmt[0][0] if cmt else None) or "—"})
props_692 = pd.DataFrame(props)
display(props_692)
no_lin = props_692[props_692["Lineage properties"] == "—"]["Table"].tolist()
dq_add(findings_692, "5 bronze tables", "source snapshot recorded in table metadata", "a TBLPROPERTY records the source dataset version / snapshot date",
       len(props_692), len(no_lin), status="INFO", examples=", ".join(no_lin),
       note="the Yelp Open Dataset release is not documented on the tables — add e.g. source.snapshot_date in Silver DDL")

# ══ C) Delta commit history ══
print(f"\n{'─' * 60}\n📌 C) Delta commit history")
DATA_OPS = {"WRITE", "MERGE", "UPDATE", "DELETE", "TRUNCATE", "RESTORE", "STREAMING UPDATE", "COPY INTO",
            "CREATE OR REPLACE TABLE AS SELECT", "REPLACE TABLE AS SELECT", "CREATE TABLE AS SELECT", "REPLACE TABLE"}
hist = []
for t, p in SOURCES_69.items():
    for r in spark.sql(f"DESCRIBE HISTORY {p}").select("version", "timestamp", "operation", "userName",
                                                      F.col("operationMetrics").getItem("numOutputRows").alias("rows")).collect():
        hist.append({"Table": t, "Version": r["version"], "Commit": r["timestamp"], "Operation": r["operation"],
                     "User": r["userName"], "Rows written": r["rows"], "Changes data": r["operation"] in DATA_OPS})
hist_692 = pd.DataFrame(hist).sort_values(["Table", "Version"]).reset_index(drop=True)
display(hist_692)
loads = hist_692[hist_692["Changes data"]]
initial = loads.sort_values("Version").groupby("Table").head(1)
later = loads.drop(initial.index)
print(f"   Data-loading commits: {len(loads)} | initial loads: {len(initial)} | later data-changing commits: {len(later)}")
dq_add(findings_692, "5 bronze tables", "no data change after the initial load", "only metadata commits (properties, stats, optimize) after the batch load",
       len(hist_692), len(later), examples="; ".join(f"{a} v{b} {c}" for a, b, c in zip(later["Table"], later["Version"], later["Operation"])),
       note="bronze_final is an immutable one-off snapshot → every EDA result refers to the same data version")
cnt_now = {t: raw_69[t].count() for t in initial["Table"]}
row_drift = [f"{t}: loaded {int(w):,} vs now {cnt_now[t]:,}" for t, w in zip(initial["Table"], initial["Rows written"]) if int(w) != cnt_now[t]]
dq_add(findings_692, "5 bronze tables", "current row count = rows written at load", "numOutputRows of the initial load equals today's COUNT(*)",
       len(initial), len(row_drift), examples="; ".join(row_drift), note="confirms the EDA reads exactly the loaded snapshot")
spread_min = (initial["Commit"].max() - initial["Commit"].min()).total_seconds() / 60
print(f"   Initial loads between {initial['Commit'].min()} and {initial['Commit'].max()} (spread {spread_min:.1f} min)")
dq_add(findings_692, "5 bronze tables", "tables loaded in one batch", "initial-load commits of the 5 tables lie within 60 minutes",
       len(initial), int(spread_min > 60), examples=f"spread {spread_min:.1f} min",
       note="same batch → business, review, tip, check-in and user are mutually consistent in time")
pub_lag = (initial["Commit"].min().date() - SNAP_D69).days
print(f"   Publication lag (initial load − as-of date): {pub_lag:,} days (~{pub_lag / 365.25:.1f} years)")
dq_add(findings_692, "yelp_business", "publication lag", "loaded within 30 days of the source as-of date",
       1, int(pub_lag > 30), status="INFO", examples=f"as-of {SNAP_D69} → loaded {initial['Commit'].min():%Y-%m-%d} ({pub_lag:,} d)",
       note="historical backfill of a public dataset release, not a delayed pipeline — no SLA breach, but no live freshness either")

timeliness_obs_692 = {"initial_load": initial["Commit"].min(), "pub_lag_days": pub_lag, "later_changes": len(later)}
findings_692_df = pd.DataFrame(findings_692)
print(f"\n   {len(findings_692_df)} observability checks | {int((findings_692_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((findings_692_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


In [0]:
# 6.9.3) Snapshot Lag & Currency of Current-State Fields  —  Business DataFrames (t69 from 6.9.1)
# ─────────────────────────────────────────────────────────────────────
# Every business row is one snapshot of a CURRENT state. The questions here are how old that state is per business / market:
#   A) Market cut-off      — does any state (metro) stop earlier than the global as-of date? (partial regional extract)
#   B) is_open currency    — open but silent for years / closed but active right before the snapshot (cross-ref DQ-BUSINESS-036)
#   C) Profile currency    — hours / attributes still published for listings silent > 3 years (state of the past, no valid-from)
#   D) Counter currency    — stored review_count vs the reviews in the extract (counter captured at a different moment)
#   E) Boundary months     — activity per day in the last 13 months per source (partial final month)
#   F) Boundary years      — partial first / last calendar year (yearly KPIs / YoY trends)
#   G) Source regime shift — does one feed (tip / check-in) thin out before the cut-off relative to reviews? (seasonality-controlled)
#   H) Rating currency     — is the published `stars` of an open business backed by recent reviews?
from pyspark.sql import functions as F
import pandas as pd

print("=" * 80)
print("6.9.3) SNAPSHOT LAG & CURRENCY OF CURRENT-STATE FIELDS")
print("=" * 80)
assert "t69" in globals(), "Run 6.9.1 first"
findings_693 = []
print(f"   As-of date (6.9.1): {SNAP_69}")

# ══ A) Market-level cut-off ══
print(f"\n{'─' * 60}\n📌 A) Market-level cut-off — latest event per state")
ev_all = None
for src, fr in ev_69.items():
    p = fr.select("business_id", "ts", F.lit(src).alias("src"))
    ev_all = p if ev_all is None else ev_all.unionByName(p)
st_ev = (ev_all.join(df_business.select("business_id", "state"), "business_id")
         .groupBy("state").agg(F.count("*").alias("events"), F.max("ts").alias("last_event")))
st_biz = t69.groupBy("state").agg(F.count("*").alias("businesses"), F.round(F.avg(F.col("is_open").cast("int")) * 100, 1).alias("open_pct"),
                                  F.percentile_approx("days_silent", 0.5).alias("median_days_silent"),
                                  F.round(F.avg((F.col("days_silent") <= 90).cast("int")) * 100, 1).alias("pct_active_90d"))
mkt_693 = (st_biz.join(st_ev, "state", "left")
           .withColumn("days_last_event_before_asof", F.round((F.lit(SNAP_69).cast("long") - F.col("last_event").cast("long")) / 86400, 1))
           .orderBy(F.desc("businesses")).toPandas())
display(mkt_693)
METRO_MIN = 1000
metro = mkt_693[mkt_693["businesses"] >= METRO_MIN]
metro_late = metro[metro["days_last_event_before_asof"] > 7]
dq_add(findings_693, "df_business × review/tip/checkin", f"metro cut-off (states ≥ {METRO_MIN:,} businesses)",
       "every metro market has events within 7 days of the as-of date", len(metro), len(metro_late),
       examples=", ".join(f"{s}: {d:.0f} d" for s, d in zip(metro_late["state"], metro_late["days_last_event_before_asof"])),
       note="no metro was extracted earlier than the others → cross-market comparisons share one as-of date")
# Market freshness: a metro can share the cut-off but still be far less 'alive' (coverage decay) → its business state is older
yr = (ev_69["review"].withColumn("yr", F.year("ts")).where(F.col("yr").isin(2019, 2021))
      .join(df_business.select("business_id", "state"), "business_id")
      .groupBy("state").agg(F.sum((F.col("yr") == 2019).cast("int")).alias("reviews_2019"),
                            F.sum((F.col("yr") == 2021).cast("int")).alias("reviews_2021")).toPandas())
metro = metro.merge(yr, on="state", how="left")
metro["reviews_2021_vs_2019"] = (metro["reviews_2021"] / metro["reviews_2019"]).round(2)
med_active, med_ratio = metro["pct_active_90d"].median(), metro["reviews_2021_vs_2019"].median()
metro["stale_market"] = (metro["pct_active_90d"] < 0.5 * med_active) | (metro["reviews_2021_vs_2019"] < 0.5 * med_ratio)
print(f"   Metro medians — businesses active in last 90 d: {med_active:.1f}% | 2021 / 2019 review volume: {med_ratio:.2f}")
display(metro[["state", "businesses", "open_pct", "median_days_silent", "pct_active_90d", "reviews_2019", "reviews_2021",
               "reviews_2021_vs_2019", "stale_market"]])
stale_m = metro[metro["stale_market"]]
dq_add(findings_693, "df_business × review/tip/checkin", "metro activity freshness",
       "no metro has < ½ the median share of businesses active in the last 90 d, or < ½ the median 2021/2019 review ratio",
       len(metro), len(stale_m),
       examples=", ".join(f"{s}: {a:.1f}% active 90 d, 2021/2019 = {r:.2f}, median silence {d} d, open {o:.0f}%" for s, a, r, d, o in
                         zip(stale_m["state"], stale_m["pct_active_90d"], stale_m["reviews_2021_vs_2019"], stale_m["median_days_silent"], stale_m["open_pct"])),
       note="coverage decayed in this market → its is_open / hours / attributes are systematically older than elsewhere")
metro_693 = metro
stray = mkt_693[(mkt_693["businesses"] < METRO_MIN) & (mkt_693["days_last_event_before_asof"] > 365)]
dq_add(findings_693, "df_business × review/tip/checkin", f"non-metro states silent > 1 y", "profile of out-of-footprint states (< 1,000 businesses)",
       int((mkt_693["businesses"] < METRO_MIN).sum()), len(stray), status="INFO",
       examples=", ".join(f"{s} ({b} biz, {d:.0f} d)" for s, b, d in zip(stray["state"], stray["businesses"], stray["days_last_event_before_asof"])),
       note="stray listings outside the 11 Yelp metros (cf. DQ-BUSINESS-009/021) — low volume, not a truncated extract")

# ══ B) is_open currency ══
print(f"\n{'─' * 60}\n📌 B) is_open currency (cross-reference DQ-BUSINESS-036)")
EX_B = F.concat_ws(" | ", "name", "city", "state", F.date_format("last_activity", "yyyy-MM-dd"))
run_checks(findings_693, t69, [
    chk("df_business × review/tip/checkin", "is_open = TRUE, silent > 1 y", "an open business shows activity in the final year",
        F.col("days_silent") > 365, scope=F.col("is_open") == True, status="INFO", ex=EX_B, note="stale is_open — DQ-BUSINESS-036"),
    chk("df_business × review/tip/checkin", "is_open = TRUE, silent > 3 y", "an open business shows activity in the final 3 years",
        F.col("days_silent") > 3 * 365, scope=F.col("is_open") == True, status="INFO", ex=EX_B, note="very likely closed — DQ-BUSINESS-036"),
    chk("df_business × review/tip/checkin", "is_open = FALSE, active ≤ 30 d", "a closed business is not reviewed / visited right before the as-of date",
        F.col("days_silent") <= 30, scope=F.col("is_open") == False, status="INFO", ex=EX_B,
        note="recently closed (closure flag fresher than activity) or wrong flag — DQ-BUSINESS-036")])

# ══ C) Profile currency ══
print(f"\n{'─' * 60}\n📌 C) Profile currency — hours / attributes of long-silent listings")
hours_cols = [HOURS_MAP[d] for d in DAYS]
attr_cols_69 = [c for c in df_attributes.columns if c != "business_id"]
prof = (t69.join(df_hours.select("business_id", F.greatest(*[F.col(c).isNotNull() for c in hours_cols]).alias("has_hours")), "business_id", "left")
           .join(df_attributes.select("business_id", F.greatest(*[F.col(c).isNotNull() for c in attr_cols_69]).alias("has_attrs")), "business_id", "left"))
silent_band = (F.when(F.col("days_silent") <= 365, "a) ≤ 1 y").when(F.col("days_silent") <= 3 * 365, "b) 1 – 3 y")
                .when(F.col("days_silent") <= 5 * 365, "c) 3 – 5 y").otherwise("d) > 5 y"))
prof_693 = (prof.groupBy(silent_band.alias("silent_for")).agg(F.count("*").alias("businesses"),
            F.round(F.avg(F.col("is_open").cast("int")) * 100, 1).alias("open_pct"),
            F.round(F.avg(F.col("has_hours").cast("int")) * 100, 1).alias("with_hours_pct"),
            F.round(F.avg(F.col("has_attrs").cast("int")) * 100, 1).alias("with_attributes_pct"),
            F.expr("percentile_approx(year(last_activity), 0.5)").alias("median_last_year")).orderBy("silent_for").toPandas())
display(prof_693)
EX_C = F.concat_ws(" | ", "name", "city", F.date_format("last_activity", "yyyy-MM-dd"), F.col("is_open").cast("string"))
run_checks(findings_693, prof, [
    chk("df_hours", "hours published, silent > 3 y", "a published schedule belongs to a business observed in the last 3 years",
        F.col("has_hours") == True, scope=F.col("days_silent") > 3 * 365, status="INFO", ex=EX_C,
        note="schedule has no valid-from date and may date from the business's last active period (pre-COVID for most)"),
    chk("df_hours", "hours published, open + silent > 3 y", "profile of 'open' listings whose schedule is presented as current",
        F.col("has_hours") == True, scope=(F.col("days_silent") > 3 * 365) & (F.col("is_open") == True), status="INFO", ex=EX_C,
        note="the risky subset: shown as open with opening hours, but no sign of life for > 3 years"),
    chk("df_attributes", "attributes published, silent > 3 y", "attribute values belong to a business observed in the last 3 years",
        F.col("has_attrs") == True, scope=F.col("days_silent") > 3 * 365, status="INFO", ex=EX_C,
        note="amenities / price / service options describe the past state of the listing")])

# ══ D) Counter currency ══
print(f"\n{'─' * 60}\n📌 D) Counter currency — stored review_count vs reviews in the extract")
rv_users = ev_69["review"].groupBy("business_id").agg(F.countDistinct("user_id").alias("n_reviewers"))
cnt = (t69.join(rv_users, "business_id", "left")
          .withColumn("rel", F.when(F.col("review_count") > F.col("n_reviewers"), "stored > extract")
                             .when(F.col("review_count") < F.col("n_reviewers"), "stored < extract").otherwise("equal"))
          .withColumn("last_rv_days", F.datediff(F.lit(SNAP_D69), F.to_date("last_review"))))
cnt_693 = (cnt.groupBy("rel").agg(F.count("*").alias("businesses"),
           F.round(F.avg((F.col("last_rv_days") <= 30).cast("int")) * 100, 1).alias("pct_last_review_le_30d"),
           F.percentile_approx("last_rv_days", 0.5).alias("median_days_last_review"),
           F.sum(F.abs(F.col("review_count") - F.col("n_reviewers"))).alias("abs_diff_total")).orderBy("rel").toPandas())
display(cnt_693)
EX_D = F.concat_ws(" | ", "name", "city", F.format_string("stored=%d extract=%d", "review_count", "n_reviewers"), F.date_format("last_review", "yyyy-MM-dd"))
run_checks(findings_693, cnt, [
    chk("df_business ↔ yelp_review", "review_count > distinct reviewers", "stored counter ≤ reviews present in the extract",
        F.col("review_count") > F.col("n_reviewers"), scope=F.col("n_reviewers").isNotNull(), status="INFO", ex=EX_D,
        note="counter captured after the review extract (or counts reviews filtered out) — DQ-BUSINESS-056"),
    chk("df_business ↔ yelp_review", "review_count < distinct reviewers", "stored counter ≥ reviews present in the extract",
        F.col("review_count") < F.col("n_reviewers"), scope=F.col("n_reviewers").isNotNull(), status="INFO", ex=EX_D,
        note="counter captured before the latest reviews — DQ-BUSINESS-056")])

# ══ E) Boundary months ══
print(f"\n{'─' * 60}\n📌 E) Boundary months — events per covered day, last 13 months")
m0 = F.add_months(F.trunc(F.lit(SNAP_D69), "month"), -12)
mon = (ev_all.where(F.col("ts") >= m0).groupBy(F.date_format("ts", "yyyy-MM").alias("month")).pivot("src", list(ev_69)).count()
          .orderBy("month").toPandas())
mon["days_covered"] = [SNAP_D69.day if m == f"{SNAP_D69:%Y-%m}" else pd.Period(m).days_in_month for m in mon["month"]]
for s in ev_69:
    mon[f"{s}_per_day"] = (mon[s] / mon["days_covered"]).round(0)
mon_693 = mon
display(mon_693)
last_rate = mon.iloc[-1]["review_per_day"]
prev_rate = mon.iloc[-4:-1]["review_per_day"].mean()
raw_drop = (1 - mon.iloc[-1]["review"] / mon.iloc[-4:-1]["review"].mean()) * 100
print(f"   Final month {SNAP_D69:%Y-%m}: raw review volume {raw_drop:.0f}% below the previous 3-month mean, "
      f"but per covered day {last_rate:,.0f} vs {prev_rate:,.0f} ({(last_rate / prev_rate - 1) * 100:+.0f}%)")
dq_add(findings_693, "review / tip / check-in", f"final month {SNAP_D69:%Y-%m} complete", "the as-of month is fully covered",
       1, int(SNAP_D69.day < pd.Period(f'{SNAP_D69:%Y-%m}').days_in_month), status="INFO",
       examples=f"{SNAP_D69.day}/{pd.Period(f'{SNAP_D69:%Y-%m}').days_in_month} days; raw volume −{raw_drop:.0f}%, per-day {(last_rate / prev_rate - 1) * 100:+.0f}%",
       note="an apparent month-end 'collapse' is the cut-off — use per-day rates or drop the partial month in trends")

# ══ F) Boundary years ══
print(f"\n{'─' * 60}\n📌 F) Boundary years — partial first / last calendar year")
assert "yr_691" in globals(), "Run 6.9.1 (section E) first"
part_y = yr_691[yr_691["days_covered"] < 365]
for _, y in part_y.iterrows():
    print(f"   {int(y['year'])}: {int(y['days_covered'])} days covered | reviews {int(y['review']):,} ({y['review_per_day']:,.0f}/d)")
dq_add(findings_693, "review / tip / check-in", "boundary calendar years complete", "first and last calendar year of the window fully covered",
       len(yr_691), len(part_y), status="INFO",
       examples=", ".join(f"{int(y['year'])}: {int(y['days_covered'])}/365 d" for _, y in part_y.iterrows()),
       note="yearly business KPIs / YoY growth for these years are understated — restrict year-level trends to complete years")

# ══ G) Source regime shift before the cut-off ══
print(f"\n{'─' * 60}\n📌 G) Source regime shift — tip / check-in volume relative to reviews before the cut-off")
RECENT_D, BASE_D = 56, 182                      # final 8 weeks vs the 18 weeks before them
d_ev = F.to_date("ts")
def _win(lo, hi):                               # events with SNAP - lo < date ≤ SNAP - hi, per source
    r = (ev_all.where((d_ev > F.date_sub(F.lit(SNAP_D69), lo)) & (d_ev <= F.date_sub(F.lit(SNAP_D69), hi)))
               .groupBy("src").count().collect())
    return {x["src"]: int(x["count"]) for x in r}
wins = {"base": _win(BASE_D, RECENT_D), "recent": _win(RECENT_D, 0),
        "base_prev_year": _win(BASE_D + 365, RECENT_D + 365), "recent_prev_year": _win(RECENT_D + 365, 365)}
wk = (ev_all.where(d_ev > F.date_sub(F.lit(SNAP_D69), BASE_D)).groupBy(F.date_trunc("week", "ts").alias("week"))
            .pivot("src", list(ev_69)).count().orderBy("week").toPandas().fillna(0))
regime_rows = []
for s in ["tip", "checkin"]:
    rt = {k: v.get(s, 0) / max(v.get("review", 0), 1) * 100 for k, v in wins.items()}
    wk[f"{s}_per_100_reviews"] = (wk[s] / wk["review"] * 100).round(2)
    shift, seas = rt["recent"] / rt["base"], rt["recent_prev_year"] / rt["base_prev_year"]
    below = (wk[f"{s}_per_100_reviews"] < 0.7 * rt["base"]).tolist()
    onset = next((wk["week"].iloc[i] for i in range(len(below)) if all(below[i:])), None)   # first week from which it stays low
    regime_rows.append({"Source": s, "per 100 reviews (base 18 wk)": round(rt["base"], 2), "per 100 reviews (final 8 wk)": round(rt["recent"], 2),
                        "final / base": round(shift, 2), "same windows 1 y earlier (final / base)": round(seas, 2),
                        "seasonality-adjusted ratio": round(shift / seas, 2), "stays < 70 % of base from week": onset})
regime_693 = pd.DataFrame(regime_rows)
wk_693 = wk
display(regime_693)
display(wk_693)
for _, r in regime_693.iterrows():
    bad = int(r["seasonality-adjusted ratio"] < 0.7)
    dq_add(findings_693, f"yelp_{r['Source']} ↔ yelp_review", f"{r['Source']} feed stable until the cut-off",
           "per-review volume in the final 8 weeks ≥ 70 % of the preceding 18 weeks (adjusted for the same windows one year earlier)",
           1, bad, status="FINDING" if bad else "PASS",
           examples=(f"{r['per 100 reviews (base 18 wk)']} → {r['per 100 reviews (final 8 wk)']} per 100 reviews "
                     f"(×{r['final / base']}, prior year ×{r['same windows 1 y earlier (final / base)']}); "
                     f"low from week {pd.Timestamp(r['stays < 70 % of base from week']):%Y-%m-%d}") if bad else "",
           note="a step change only in this feed → the source stops earlier / was collected differently than reviews; recency features "
                "(last_tip, tips in last 90 d) are biased low for the final weeks")

# ══ H) Rating currency ══
print(f"\n{'─' * 60}\n📌 H) Rating currency — how recent are the reviews behind the published stars?")
rv_age = (ev_69["review"].withColumn("age_d", F.datediff(F.lit(SNAP_D69), F.to_date("ts")))
          .groupBy("business_id").agg(F.count("*").alias("n_rv"), F.avg((F.col("age_d") <= 3 * 365).cast("int")).alias("share_rv_3y"),
                                       F.percentile_approx("age_d", 0.5).alias("median_review_age_d")))
rat = t69.join(rv_age, "business_id", "left").join(df_business.select("business_id", "stars"), "business_id", "left")
age_band = (F.when(F.col("median_review_age_d") <= 365, "a) ≤ 1 y").when(F.col("median_review_age_d") <= 3 * 365, "b) 1 – 3 y")
             .when(F.col("median_review_age_d") <= 5 * 365, "c) 3 – 5 y").otherwise("d) > 5 y"))
rat_693 = (rat.groupBy(age_band.alias("median_review_age")).pivot("is_open", [True, False]).count().na.fill(0)
              .withColumnRenamed("true", "open").withColumnRenamed("false", "closed").orderBy("median_review_age").toPandas())
display(rat_693)
EX_H = F.concat_ws(" | ", "name", "city", F.col("stars").cast("string"),
                   F.format_string("%d reviews, %.0f%% in last 3 y, median age %d d", "n_rv", F.col("share_rv_3y") * 100, "median_review_age_d"))
run_checks(findings_693, rat, [
    chk("df_business ↔ yelp_review", "open, ≥ 20 reviews, < 10 % from the final 3 y", "the stars of an established open business rest on recent reviews",
        F.col("share_rv_3y") < 0.1, scope=(F.col("is_open") == True) & (F.col("n_rv") >= 20), status="INFO", ex=EX_H,
        note="stars is a lifetime average without recency weighting → describes the business as it was years ago"),
    chk("df_business ↔ yelp_review", "open, median review age > 5 y", "half of an open business's reviews are younger than 5 years",
        F.col("median_review_age_d") > 5 * 365, scope=F.col("is_open") == True, status="INFO", ex=EX_H,
        note="Silver / Gold should expose a recency-weighted or trailing-3-year rating next to the published stars")])

findings_693_df = pd.DataFrame(findings_693)
print(f"\n   {len(findings_693_df)} currency checks | {int((findings_693_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((findings_693_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


In [0]:
# 6.9.4) Temporal Consistency of Business-Linked Timestamps  —  review / tip / check-in / user registration
# ─────────────────────────────────────────────────────────────────────
# We validate every raw timestamp source that informs business timeliness:
#   A) review.date, tip.date, user.yelping_since — format / parseability / future / pre-launch / round-trip / time-of-day heaping
#   B) checkin.date list                         — token format / parseability / sorting / duplicate timestamps (rows + excess tokens)
#   C) author chronology                         — review / tip written before the user registered? (size of the gap)
#   D) time-zone interpretation                  — UTC vs local-business time (the dataset has no time-zone column):
#                                                  calendar-day shift, quiet-hour alignment, DST spring-forward hour occupancy
from pyspark.sql import functions as F
import pandas as pd

print("=" * 80)
print("6.9.4) TEMPORAL CONSISTENCY OF BUSINESS-LINKED TIMESTAMPS")
print("=" * 80)
assert "ev_69" in globals() and "usr_69" in globals(), "Run 6.9.1 first"
findings_694 = []
YELP_LAUNCH_TS_69 = F.to_timestamp(F.lit(f"{YELP_LAUNCH_69} 00:00:00"))
EX_S = lambda *cs: F.concat_ws(" | ", *[F.col(c).cast("string") if isinstance(c, str) else c for c in cs])

# ══ A) Scalar timestamp strings ══
print(f"\n{'─' * 60}\n📌 A) Scalar timestamp strings — review.date, tip.date, user.yelping_since")
scalar_rows = []
for src, fr, raw_col in [("review", ev_69["review"], "raw"), ("tip", ev_69["tip"], "raw"), ("user", usr_69, "raw")]:
    r = fr.agg(F.count("*").alias("n"),
               F.sum((F.length(raw_col) != 19).cast("int")).alias("bad_len"),
               F.sum((F.col("ts").isNull() & F.col(raw_col).isNotNull()).cast("int")).alias("bad_parse"),
               F.sum((F.col("ts") > F.lit(SNAP_69)).cast("int")).alias("future"),
               F.sum((F.col("ts") < YELP_LAUNCH_TS_69).cast("int")).alias("pre_launch"),
               F.sum((F.date_format("ts", TS_FMT_69) != F.col(raw_col)).cast("int")).alias("roundtrip")).first()
    scalar_rows.append({"Source": src, "Rows": int(r["n"]), "Bad length": int(r["bad_len"]), "Parse failures": int(r["bad_parse"]),
                        "Future vs as-of": int(r["future"]), "Before 2004-10-01": int(r["pre_launch"]), "Round-trip mismatch": int(r["roundtrip"])})
scalar_694 = pd.DataFrame(scalar_rows)
display(scalar_694)
RULE_694 = {"Bad length": "raw string has 19 characters (yyyy-MM-dd HH:mm:ss)", "Parse failures": "raw string parses as a timestamp",
            "Future vs as-of": "no timestamp after the as-of date", "Before 2004-10-01": "no timestamp before Yelp launched",
            "Round-trip mismatch": "formatted parse reproduces the raw string (components agree with the text)"}
for row in scalar_rows:
    checked = row["Rows"]
    for colm, rule in RULE_694.items():
        dq_add(findings_694, f"yelp_{row['Source']}", f"{'yelping_since' if row['Source'] == 'user' else 'date'}: {colm.lower()}", rule,
               checked, row[colm], status="PASS" if row[colm] == 0 else "FINDING")
# Time-of-day heaping: a feed that only knew the DATE would pile up at 00:00:00; rounded clocks pile up at :00 seconds
heap_rows = []
for src, fr in [("review", ev_69["review"]), ("tip", ev_69["tip"]), ("checkin", ev_69["checkin"]), ("user", usr_69)]:
    r = fr.agg(F.count("*").alias("n"), F.sum((F.date_format("ts", "HH:mm:ss") == "00:00:00").cast("int")).alias("midnight"),
               F.sum((F.second("ts") == 0).cast("int")).alias("sec0")).first()
    n = int(r["n"])
    heap_rows.append({"Source": src, "Rows": n, "Exactly 00:00:00": int(r["midnight"]), "Expected (uniform)": round(n / 86400, 1),
                      "Midnight lift": round(int(r["midnight"]) / (n / 86400), 2), "Seconds = 00 %": round(int(r["sec0"]) / n * 100, 3),
                      "Expected %": round(100 / 60, 3)})
heap_694 = pd.DataFrame(heap_rows)
display(heap_694)
for _, h in heap_694.iterrows():
    bad = int(h["Midnight lift"] > 5 or h["Seconds = 00 %"] > 2 * h["Expected %"])
    dq_add(findings_694, f"yelp_{h['Source']}", "time-of-day heaping (date-only / rounded values)",
           "00:00:00 ≤ 5× and :00 seconds ≤ 2× their uniform expectation", int(h["Rows"]), bad,
           examples=f"00:00:00 lift ×{h['Midnight lift']}, :00 s = {h['Seconds = 00 %']}%",
           note="full second-level precision → time-of-day / hour-of-week features are usable")

# ══ B) Check-in timestamp list ══
print(f"\n{'─' * 60}\n📌 B) Check-in timestamp list")
ck_list = raw_69["yelp_checkin"].select("business_id", F.split("date", ",").alias("arr"))
ck_tok = ck_list.select("business_id", F.explode("arr").alias("raw")).withColumn("raw", F.trim("raw")).withColumn("ts", ts_(F.col("raw")))
ck_r = ck_list.withColumn("trimmed", F.transform("arr", lambda x: F.trim(x))).agg(
    F.count("*").alias("rows"),
    F.sum((F.col("trimmed") != F.array_sort("trimmed")).cast("int")).alias("unsorted_rows"),
    F.sum((F.size("trimmed") != F.size(F.array_distinct("trimmed"))).cast("int")).alias("dup_rows"),
    F.sum(F.size("trimmed") - F.size(F.array_distinct("trimmed"))).alias("dup_tokens")
).first()
ck_tok_r = ck_tok.agg(F.count("*").alias("n"), F.sum((F.length("raw") != 19).cast("int")).alias("bad_len"),
                         F.sum((F.col("ts").isNull() & F.col("raw").isNotNull()).cast("int")).alias("bad_parse"),
                         F.sum((F.col("ts") > F.lit(SNAP_69)).cast("int")).alias("future"),
                         F.sum((F.col("ts") < YELP_LAUNCH_TS_69).cast("int")).alias("pre_launch"),
                         F.sum((F.date_format("ts", TS_FMT_69) != F.col("raw")).cast("int")).alias("roundtrip")).first()
print(f"   Check-in business rows: {int(ck_r['rows']):,} | exploded timestamps: {int(ck_tok_r['n']):,}")
dq_add(findings_694, "yelp_checkin", "timestamp token length", "every exploded timestamp token has length 19", int(ck_tok_r["n"]), int(ck_tok_r["bad_len"]))
dq_add(findings_694, "yelp_checkin", "timestamp token parseability", "every exploded timestamp token parses", int(ck_tok_r["n"]), int(ck_tok_r["bad_parse"]))
dq_add(findings_694, "yelp_checkin", "timestamp token round-trip", "parsed check-in tokens reproduce the original string", int(ck_tok_r["n"]), int(ck_tok_r["roundtrip"]))
dq_add(findings_694, "yelp_checkin", "events after the as-of date", "no check-in exceeds the dataset as-of date", int(ck_tok_r["n"]), int(ck_tok_r["future"]))
dq_add(findings_694, "yelp_checkin", "events before Yelp launch", "no check-in predates 2004-10-01", int(ck_tok_r["n"]), int(ck_tok_r["pre_launch"]))
dq_add(findings_694, "yelp_checkin", "chronological ordering inside each list", "check-in timestamps are sorted ascending inside each business row",
       int(ck_r["rows"]), int(ck_r["unsorted_rows"]), status="PASS" if int(ck_r["unsorted_rows"]) == 0 else "FINDING")
dq_add(findings_694, "yelp_checkin", "duplicate timestamps inside each list", "a business does not repeat an identical check-in timestamp",
       int(ck_r["rows"]), int(ck_r["dup_rows"]), status="INFO", examples=f"{int(ck_r['dup_tokens']):,} surplus tokens",
       note="already handled by the check-in pipeline (DQ-CHECKIN-005, quarantined); inflates check-in counts per business if not de-duplicated")

# ══ C) Author chronology ══
print(f"\n{'─' * 60}\n📌 C) Author chronology — content before the user registered?")
usr_min = usr_69.select("user_id", F.col("ts").alias("user_registered_ts"))
rv_usr = ev_69["review"].join(usr_min, "user_id", "left")
tp_usr = ev_69["tip"].join(usr_min, "user_id", "left")
chron = []
for src, fr in [("review", rv_usr), ("tip", tp_usr)]:
    r = fr.agg(F.count("*").alias("n"), F.sum(F.col("user_registered_ts").isNull().cast("int")).alias("no_user"),
               F.sum((F.col("ts") < F.col("user_registered_ts")).cast("int")).alias("before_reg"),
               F.sum((F.to_date("ts") < F.to_date("user_registered_ts")).cast("int")).alias("before_reg_date")).first()
    chron.append({"Source": src, "Rows": int(r["n"]), "Missing user": int(r["no_user"]),
                  "ts < yelping_since": int(r["before_reg"]), "date(ts) < date(yelping_since)": int(r["before_reg_date"])})
chron_694 = pd.DataFrame(chron)
display(chron_694)
gap_r = (rv_usr.where(F.col("ts") < F.col("user_registered_ts"))
               .select(((F.col("user_registered_ts").cast("long") - F.col("ts").cast("long")) / 86400).alias("d"), "user_id")
               .agg(F.countDistinct("user_id").alias("users"), F.sum((F.col("d") < 1).cast("int")).alias("lt1d"),
                    F.percentile_approx("d", 0.5).alias("p50"), F.max("d").alias("mx")).first())
n_before = int(chron_694.loc[chron_694["Source"] == "review", "ts < yelping_since"].iloc[0])
if n_before:
    print(f"   Reviews before registration: {n_before:,} by {int(gap_r['users']):,} users | gap < 1 day: {int(gap_r['lt1d']):,} | "
          f"median gap {gap_r['p50']:.0f} d | max {gap_r['mx']:.0f} d")
dq_add(findings_694, "yelp_review ↔ yelp_user", "review before user registration", "a user is registered before writing a review",
       chron_694.loc[chron_694["Source"] == "review", "Rows"].iloc[0], n_before, status="INFO",
       examples=(f"{int(gap_r['users']):,} users; median gap {gap_r['p50']:.0f} d, max {gap_r['mx']:.0f} d; {int(gap_r['lt1d'])} within 1 day") if n_before else "",
       note="gaps of years (not hours) → not a time-zone artefact; yelping_since was reset (re-created / merged account) — "
            "user-table issue, negligible for business timeliness")
dq_add(findings_694, "yelp_tip ↔ yelp_user", "tip before user registration", "a user is registered before writing a tip",
       chron_694.loc[chron_694["Source"] == "tip", "Rows"].iloc[0], chron_694.loc[chron_694["Source"] == "tip", "ts < yelping_since"].iloc[0],
       status="INFO")

# ══ D) Time-zone interpretation ══
print(f"\n{'─' * 60}\n📌 D) Time-zone interpretation — UTC or local-business time?")
TZ_69 = {"PA": "America/New_York", "FL": "America/New_York", "LA": "America/Chicago", "TN": "America/Chicago", "MO": "America/Chicago",
         "IN": "America/Indiana/Indianapolis", "AZ": "America/Phoenix", "NV": "America/Los_Angeles", "CA": "America/Los_Angeles",
         "NJ": "America/New_York", "ID": "America/Boise", "AB": "America/Edmonton", "DE": "America/New_York", "IL": "America/Chicago"}
tz_map_69 = F.create_map(*[x for k, v in TZ_69.items() for x in (F.lit(k), F.lit(v))])
tz_probe = (ev_69["review"].join(df_business.select("business_id", "state"), "business_id")
            .where(F.col("state").isin(*TZ_69))
            .withColumn("tz", tz_map_69[F.col("state")])
            .withColumn("local_ts", F.from_utc_timestamp("ts", F.col("tz"))))
shift_694 = tz_probe.agg(F.count("*").alias("rows"),
                         F.sum((F.to_date("ts") != F.to_date("local_ts")).cast("int")).alias("local_day_shift"),
                         F.sum((F.dayofweek("ts") != F.dayofweek("local_ts")).cast("int")).alias("weekday_shift")).first()
by_hour_utc = tz_probe.groupBy("state", F.hour("ts").alias("hour_utc")).count().toPandas()
by_hour_loc = tz_probe.groupBy("state", F.hour("local_ts").alias("hour_local")).count().toPandas()
quiet_rows = []
for state in sorted(TZ_69):
    u = by_hour_utc[by_hour_utc["state"] == state].set_index("hour_utc")["count"].reindex(range(24), fill_value=0)
    l = by_hour_loc[by_hour_loc["state"] == state].set_index("hour_local")["count"].reindex(range(24), fill_value=0)
    quiet3_u = min(range(24), key=lambda h: int(u[(h - 1) % 24]) + int(u[h]) + int(u[(h + 1) % 24]))
    quiet3_l = min(range(24), key=lambda h: int(l[(h - 1) % 24]) + int(l[h]) + int(l[(h + 1) % 24]))
    quiet_rows.append({"State": state, "Quiet 3h window center (UTC)": quiet3_u, "Quiet 3h window center (local)": quiet3_l})
quiet_694 = pd.DataFrame(quiet_rows)
display(quiet_694)
shift_pct = int(shift_694["local_day_shift"]) / int(shift_694["rows"]) * 100
quiet_loc_ok = int(quiet_694["Quiet 3h window center (local)"].between(2, 6).sum())
quiet_utc_ok = int(quiet_694["Quiet 3h window center (UTC)"].between(2, 6).sum())
print(f"   Reviews that move to a different local calendar day after timezone conversion: {int(shift_694['local_day_shift']):,} / "
      f"{int(shift_694['rows']):,} ({shift_pct:.1f}%)")
print(f"   States whose quiet window lies at 02–06 h: raw clock {quiet_utc_ok}/{len(quiet_694)} | after UTC→local {quiet_loc_ok}/{len(quiet_694)}")
# DST spring-forward: local wall-clock 02:00–02:59 does not exist on that Sunday in DST zones. Local timestamps would leave it empty.
DST_STATES_69 = [s for s in TZ_69 if s != "AZ"]                  # Arizona does not observe DST
dst = (tz_probe.where(F.col("state").isin(*DST_STATES_69)).withColumn("d", F.to_date("ts")).withColumn("y", F.year("ts"))
       .withColumn("dst_start", F.when(F.col("y") >= 2007, F.next_day(F.make_date("y", F.lit(3), F.lit(7)), "Sun"))      # 2nd Sunday of March
                                .otherwise(F.next_day(F.make_date("y", F.lit(3), F.lit(31)), "Sun")))                    # 1st Sunday of April
       .where(~((F.col("state") == "IN") & (F.col("y") < 2006)))                                                   # Indiana: DST only since 2006
       .where((F.dayofweek("d") == 1) & (F.hour("ts") == 2) & (F.abs(F.datediff("d", "dst_start")) <= 21)))
dr = dst.agg(F.sum((F.col("d") == F.col("dst_start")).cast("int")).alias("in_gap"),
             F.countDistinct(F.when(F.col("d") == F.col("dst_start"), F.col("d"))).alias("gap_days"),
             F.sum((F.col("d") != F.col("dst_start")).cast("int")).alias("ctrl"),
             F.countDistinct(F.when(F.col("d") != F.col("dst_start"), F.col("d"))).alias("ctrl_days")).first()
dst_exp = int(dr["ctrl"]) / max(int(dr["ctrl_days"]), 1) * int(dr["gap_days"])
dst_694 = {"reviews_in_nonexistent_local_hour": int(dr["in_gap"]), "expected_if_utc": round(dst_exp), "dst_sundays": int(dr["gap_days"])}
print(f"   DST spring-forward Sundays ({dst_694['dst_sundays']}): {dst_694['reviews_in_nonexistent_local_hour']:,} reviews at raw 02:xx "
      f"(a local hour that does not exist) vs {dst_694['expected_if_utc']:,} expected from neighbouring Sundays")
TZ_EVIDENCE = (f"{shift_pct:.1f}% change calendar day; quiet hours at 02–06 h in {quiet_loc_ok}/{len(quiet_694)} states only after conversion "
               f"(raw {quiet_utc_ok}); {dst_694['reviews_in_nonexistent_local_hour']:,} reviews in the non-existent DST hour "
               f"(~{dst_694['expected_if_utc']:,} expected)")
dq_add(findings_694, "yelp_review / tip / check-in", "time-zone of the raw timestamps documented",
       "timestamps are local business time, or the time zone is declared", int(shift_694["rows"]), int(shift_694["local_day_shift"]),
       status="FINDING", examples=TZ_EVIDENCE,
       note="timestamps are UTC but carry no zone → day / weekday / hour features and 'reviews per day' are wrong for ~1 in 4 events "
            "unless converted with the business's state time zone")

findings_694_df = pd.DataFrame(findings_694)
print(f"\n   {len(findings_694_df)} timestamp-consistency checks | {int((findings_694_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((findings_694_df['Status'] == 'INFO').sum())} INFO")
print("=" * 80)


In [0]:
# 6.9.5) Temporal Gaps Cross-Reference & Timeliness Summary  —  Business DataFrames
# ─────────────────────────────────────────────────────────────────────
#   A) Temporal gaps  — per-source month-grid continuity (review / tip / check-in) between first and last event.
#                       Business-level gaps (silent periods, schedule holes, stale is_open, reactivation) were measured in
#                       6.4.6 (completeness) and 6.8.5 (lifecycle) and are only cross-referenced here.
#   B) Consolidated 6.9 register — all checks of 6.9.1 – 6.9.5 with status counts per sub-section.
from pyspark.sql import functions as F
import pandas as pd

print("=" * 80)
print("6.9.5) TEMPORAL GAPS CROSS-REFERENCE & TIMELINESS SUMMARY")
print("=" * 80)
for v in ["findings_691_df", "findings_692_df", "findings_693_df", "findings_694_df", "ev_all"]:
    assert v in globals(), f"Run 6.9.1 – 6.9.4 first ({v} missing)"
findings_695 = []

# ══ A) Month-grid continuity per source ══
print(f"\n{'─' * 60}\n📌 A) Month-grid continuity per source")
mon_all = ev_all.groupBy("src", F.trunc(F.to_date("ts"), "month").alias("m")).count().toPandas()
mon_all["m"] = pd.to_datetime(mon_all["m"])
grid_rows = []
for s, g in mon_all.groupby("src"):
    grid = pd.date_range(g["m"].min(), g["m"].max(), freq="MS")
    missing = sorted(set(grid) - set(g["m"]))
    g = g.sort_values("m").set_index("m")["count"]
    yoy = (g / g.shift(12) - 1) * 100
    worst = yoy.idxmin() if yoy.notna().any() else None
    grid_rows.append({"Source": s, "First month": f"{grid[0]:%Y-%m}", "Last month": f"{grid[-1]:%Y-%m}", "Expected months": len(grid),
                      "Missing months": len(missing), "Worst YoY month": f"{worst:%Y-%m}" if worst is not None else "",
                      "Worst YoY %": round(float(yoy.min()), 1) if worst is not None else None})
    dq_add(findings_695, f"yelp_{s}", "month grid continuity", "no empty calendar month between the first and last event",
           len(grid), len(missing), examples=", ".join(f"{m:%Y-%m}" for m in missing[:12]))
grid_695 = pd.DataFrame(grid_rows)
display(grid_695)
print("   Worst YoY months coincide with the COVID lockdown (2020-04) where applicable → real-world shock, not a collection gap.")

print("\n   Cross-reference — business-level temporal gaps measured earlier:")
if "timeline_646" in globals():
    t = timeline_646
    print(f"   • 6.4.6: {t.get('gap2y', 0):,} businesses with a > 2-year silent period inside their review history "
          f"(> 3 y: {t.get('gap3y', 0):,}); {t.get('open_silent_1y', 0):,} open businesses silent > 1 y (DQ-BUSINESS-036); "
          f"{t.get('midweek_hole', 0):,} isolated mid-week schedule holes; {t.get('missing_months', 0)} missing review months")
else:
    print("   • 6.4.6 not run in this session — see DQ-BUSINESS-036 (stale is_open) and the 6.4.6 schedule-gap checks")
if "findings_685_df" in globals():
    react = findings_685_df[findings_685_df["Column"].str.contains("reactiv|silence", case=False, regex=True)]
    for _, r in react.iterrows():
        print(f"   • 6.8.5: {r['Column']}: {r['Violations']:,} / {r['Checked']:,} ({r['Status']})")
else:
    print("   • 6.8.5 not run in this session — lifecycle shapes (compressed lifecycles, reactivation after ≥ 3 y silence) are reported there")
findings_695_df = pd.DataFrame(findings_695)

# ══ B) Consolidated 6.9 register ══
print(f"\n{'─' * 60}\n📌 B) Consolidated 6.9 timeliness register")
parts = {"6.9.1 Freshness & coverage": findings_691_df, "6.9.2 Ingestion observability": findings_692_df,
         "6.9.3 Snapshot lag & currency": findings_693_df, "6.9.4 Timestamp consistency": findings_694_df,
         "6.9.5 Temporal gaps": findings_695_df}
summary_69_df = pd.concat([d.assign(Section=k) for k, d in parts.items()], ignore_index=True)
summary_69_df = summary_69_df[["Section", "DataFrame", "Column", "Rule", "Checked", "Violations", "Pct", "Status", "Examples", "Note"]]
status_69 = (summary_69_df.groupby(["Section", "Status"]).size().unstack(fill_value=0)
             .reindex(columns=["PASS", "INFO", "FINDING"], fill_value=0).reset_index())
status_69["Checks"] = status_69[["PASS", "INFO", "FINDING"]].sum(axis=1)
display(status_69)
print("   FINDINGs (defects the Silver layer must handle):")
display(summary_69_df[summary_69_df["Status"] == "FINDING"])
print("   INFO (snapshot properties to document for analysts):")
display(summary_69_df[summary_69_df["Status"] == "INFO"])
print(f"\n   6.9 total: {len(summary_69_df)} checks | {int((summary_69_df['Status'] == 'FINDING').sum())} FINDING | "
      f"{int((summary_69_df['Status'] == 'INFO').sum())} INFO | {int((summary_69_df['Status'] == 'PASS').sum())} PASS")
print(f"   As-of date of yelp_business (inferred): {SNAP_69}  |  snapshot age today: {(TODAY_69 - SNAP_D69).days:,} days")
print("=" * 80)


---
#### 🚨 DQ-BUSINESS-060 — Tip Feed Halves in the Final Weeks Before the Cut-Off (Source Regime Shift)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-060` |
| **Severity** | MEDIUM (timeliness / boundary bias on tip-based business features) |
| **Detected In** | Phase 6.9 — Timeliness → **6.9.3 Snapshot Lag & Currency**, section G (source regime shift) |
| **Also Confirmed In** | Independent weekly tip / review ratio query (same step change); 6.9.3 E monthly table: Dec-2021 tips 1,777 vs Nov-2021 2,785 (−36 %) while reviews rose |
| **Affected Columns** | business-level tip features derived from `yelp_tip.date` (`last_tip`, tips per period, tip recency) |
| **Total Affected Rows** | Tips per 100 reviews fall from **5.75** (18 weeks before) to **3.60** (final 8 weeks), ×0.63; same windows one year earlier ×0.97 → seasonality-adjusted ×0.65. Ratio stays < 70 % of baseline from the week of **2021-12-06** to the cut-off (2022-01-19). Check-ins stay stable (×0.99 adjusted). |
| **Invalid Value** | week 2022-01-10: 346 tips vs 11,725 reviews (2.95 per 100), baseline ≈ 6 per 100 |
| **Domain Rule** | All sources of one snapshot should be complete up to the same cut-off; a feed-specific step change right before the cut-off indicates a collection / product change, not customer behaviour |

**Description:** Reviews and check-ins keep their usual level until 2022-01-19, but tips drop to about half of their normal rate relative to reviews from early December 2021. The drop is a step, not a slow trend, and it does not appear in the same weeks of the previous year, so it is not seasonal. Most likely the tip feed was captured with a lag (late-arriving tips not yet in the extract) or Yelp changed how tips were collected. For the business table this means tip-based recency ("tips in the last 90 days", `last_tip`) is biased low for every business in the final weeks, and a trend chart would show a false collapse of tip engagement.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep all tips; set a tip-specific effective cut-off (`tip_complete_until = 2021-11-30`) in Silver TBLPROPERTIES and exclude later weeks from tip trends / recency features (or flag them `is_partial_period`)** | Source data untouched; analyses use only the complete window |
| 2 | Re-scale late-period tip counts by the review ratio | Invents volume; hides the issue |
| 3 | Drop tips after 2021-11-30 | Loses valid rows; not needed if the cut-off is documented |

**Best Logical Solution:** **Option 1 — document a tip-specific completeness date and apply it in recency features and trends.**

---

---
#### 🚨 DQ-BUSINESS-061 — Alberta (Edmonton) Market Coverage Decay: Business State Systematically Older

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-061` |
| **Severity** | MEDIUM (timeliness differs by market → biased cross-market comparisons) |
| **Detected In** | Phase 6.9 — Timeliness → **6.9.3 Snapshot Lag & Currency**, section A (metro activity freshness) |
| **Also Confirmed In** | Independent SQL on `yelp_review ⋈ yelp_business` (2019 = 10,302, 2021 = 3,895 reviews, ratio 0.38; PA 0.57, FL 0.79) |
| **Affected Columns** | `is_open`, `stars`, `review_count`, `hours`, `attributes` of `state = 'AB'` businesses |
| **Total Affected Rows** | **5,573** AB businesses. Only **18.5 %** had activity in the last 90 days (metro median 37.9 %); median silence **592 days** (other metros 135–237 d); 2021 / 2019 review volume **0.38** (metro median 0.69). The market still has events up to the cut-off (no truncated extract). |
| **Invalid Value** | `state = 'AB'`, `pct_active_90d = 18.5`, `reviews_2021_vs_2019 = 0.38` |
| **Domain Rule** | Markets of one snapshot should be observed with comparable freshness before comparing their current-state fields |

**Description:** Alberta is the only Canadian metro and its coverage has thinned much faster than all US metros. The extract was not cut early (the last AB event is on 2022-01-19), but far fewer AB businesses are still observed. Their `is_open`, hours and attributes therefore reflect an older state than in other markets, while the open rate (78 %) looks normal — which suggests closures are flagged less in AB. Comparing open rates, ratings or amenity shares between AB and US states mixes different snapshot ages.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep the rows; add market-level freshness metrics (`pct_active_90d`, `median_days_silent`) to a Silver / Gold market dimension and use the business-level `days_since_last_activity` (DQ-BUSINESS-036) as a filter in cross-market analysis** | Transparent; no values changed |
| 2 | Exclude AB from cross-market comparisons | Simple, but loses a whole metro |
| 3 | Re-weight AB metrics | Model-dependent; hard to justify in Silver |

**Best Logical Solution:** **Option 1 — expose market freshness and filter on activity recency, rather than removing the market.**

---

---
#### 🚨 DQ-BUSINESS-062 — Business-Linked Timestamps Are UTC but Carry No Time Zone

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-BUSINESS-062` |
| **Severity** | MEDIUM (wrong calendar day / weekday / hour for about 1 in 4 events if read as local time) |
| **Detected In** | Phase 6.9 — Timeliness → **6.9.4 Temporal Consistency**, section D (time-zone interpretation) |
| **Also Confirmed In** | Three independent tests: (1) quiet 3-hour window per state lies at 08–11 h on the raw clock but at 03–04 h in **14 / 14** states after UTC→local conversion; (2) DST spring-forward test: **1,440** reviews at raw 02:xx on the 14 DST-start Sundays (a local hour that does not exist) vs ~1,267 expected — impossible for local wall-clock time; (3) separate exploratory query gave the same 1,440 |
| **Affected Columns** | `yelp_review.date`, `yelp_tip.date`, `yelp_checkin.date` (and every business-level date feature derived from them) |
| **Total Affected Rows** | **2,004,782** of 6,989,984 reviews in the 14 metro states (**28.7 %**) fall on a different local calendar day after conversion (mostly the previous day) |
| **Invalid Value** | e.g. a review stored as `2021-03-14 02:30:00` for a Philadelphia business (that local time does not exist) |
| **Domain Rule** | Event timestamps must be local business time, or the time zone must be declared |

**Description:** The raw timestamp strings have no offset, and the dataset has no time-zone column. Their behaviour is that of UTC: activity is lowest at 03–04 h local time only after conversion, and reviews exist in the hour that daylight-saving skips. Reading them as local time shifts every evening review (after 19–20 h local in the US East) to the next day. For the business table this distorts `last_activity`, days-since-last-activity, weekday / hour-of-week patterns compared with `hours`, and "active in opening hours" checks (e.g. DQ-BUSINESS-050).

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Silver: parse as `TIMESTAMP` in UTC (declared in the column comment / TBLPROPERTIES), add a business time-zone attribute derived from `state` (plus city for split states) and compute local-time columns only where needed (`event_local_ts`, `event_local_date`)** | Keeps the source instant; local features become correct and explicit |
| 2 | Convert and store only local time | Loses the original instant; DST ambiguity in the fall-back hour |
| 3 | Leave as-is, document only | Analysts will keep using the wrong day / hour |

**Best Logical Solution:** **Option 1 — store UTC explicitly and derive local-time columns from the business's time zone.**

---

---
#### 📋 6.9 Timeliness — Error Register

New errors raised in Phase 6.9 (next free ID: **DQ-BUSINESS-063**). Totals from 6.9.5: **68 checks — 3 FINDING, 30 INFO, 35 PASS**.

| Error ID | Title | Severity | Detected In | Affected | Best Solution (short) |
| --- | --- | --- | --- | --- | --- |
| DQ-BUSINESS-060 | Tip feed halves in the final weeks before the cut-off | MEDIUM | 6.9.3 G | tips / 100 reviews 5.75 → 3.60 from 2021-12-06 (seasonality-adjusted ×0.65) | tip-specific completeness date; exclude partial weeks from tip recency / trends |
| DQ-BUSINESS-061 | Alberta market coverage decay | MEDIUM | 6.9.3 A | 5,573 AB businesses; 18.5 % active in 90 d vs 37.9 % median; 2021/2019 = 0.38 | market freshness metrics + activity-recency filter |
| DQ-BUSINESS-062 | Business-linked timestamps are UTC without a time zone | MEDIUM | 6.9.4 D | 2,004,782 / 6,989,984 reviews change local day (28.7 %) | store UTC explicitly; derive local-time columns from business time zone |

**Clean results (PASS):** one shared cut-off (2022-01-19) for review / tip / check-in in all 14 metros; tables loaded in one batch with no later data change and unchanged row counts; every review, tip, check-in token and `yelping_since` value has 19 characters, parses, round-trips, and lies in [2004-10-01, as-of]; check-in lists are sorted; no time-of-day heaping (full second precision); no empty month in any source; check-in feed stable up to the cut-off.

**Snapshot properties documented (INFO, no new ID):** snapshot is **1,739 days** old (as-of 2022-01-19); no row-level ingestion metadata and no lineage TBLPROPERTIES (add `source.snapshot_date` in Silver, R6); tips start 2009-04 and check-ins 2009-12 (left-censored); 2005 and 2022 are partial years and 2022-01 a partial month; last full year volume vs peak: reviews 0.68, tips 0.32, check-ins 0.28; 41,328 businesses last observed before 2020-03; 2,846 established open businesses have < 10 % of reviews from the last 3 years and 36,314 open businesses a median review age > 5 years (stars without recency); hours / attributes still published for 18,013 / 23,747 businesses silent > 3 years; 117 reviews by 72 users predate `yelping_since` (median 648 days — account reset, user-table issue); 2,664 check-in lists contain duplicate tokens (DQ-CHECKIN-005).

**Re-detected existing errors (no new ID):** DQ-BUSINESS-036 (stale `is_open`: 22,496 + 5,542 open businesses silent 1–3 y / > 3 y; 130 closed businesses active in the last 30 days), DQ-BUSINESS-056 (counter drift: 279 businesses with `review_count` below and 30 above the distinct reviewers in the extract, mostly reviewed in the final weeks).

---